In [ ]:
# === MODULE: src/namethatllm/config.py ===
TRAINING_CONFIG = {'version': '3.1.0', 'name': 'qwen3_14b_inhouse_v2', 'description': 'Kaggle 2xT4 (~8.8-9.3h inside 10h budget): Fresh Qwen/Qwen3-14B-Base V2 on clean_master_pool_v2 (36,000 samples, 1 full cosine epoch = 1,125 opt steps) + Stage 2 prompt-grouped SupCon/SWA + Bench-1800 evals', 'backbone': 'Qwen/Qwen3-14B-Base', 'backbone_family': 'qwen', 'fresh_init': True, 'resume_from_checkpoint': None, 'stabilize_fp16': True, 'keep_layers': 27, 'layers': [2, 4, 6, 9, 13, 17, 22, 27], 'max_len': 384, 'end_id': 151643, 'lora_r': 32, 'lora_alpha': 64, 'lora_dropout': 0.05, 'lora_target_modules': ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'], 'family_margin': 0.42, 'boost_hard_sisters': True, 'supcon_weight': 0.25, 'hard_class_weight': 1.0, 'output_dir': 'export_inhouse_qwen3_14b_v2', 'stage1': {'enabled': True, 'use_v2_only': True, 'use_match_weight': True, 'match_weight_clip': [0.5, 2.5], 'lr_lora': 0.00015, 'lr_head': 0.0008, 'lora_plus_ratio': 2.5, 'batch_size': 32, 'gradient_accumulation_steps': 1, 'epochs': 1, 'max_samples': 36000, 'max_hours': 7.2, 'warmup_steps': 65, 'save_steps': 250, 'eval_steps': 375, 'eval_batch_size': 4, 'eval_subset_size': 540, 'supcon_weight': 0.0, 'label_smoothing': 0.05, 'class_quotas': {'gpt-5.5': 1900, 'gpt-5.6-luna': 1900, 'gpt-5.6-sol': 2250, 'gpt-5.6-terra': 2300, 'gpt-6-astra': 1900, 'gemini-3.1-pro-preview': 1900, 'gemini-3.6-flash': 1900, 'gemini-3.7-flash': 2350, 'gemini-3.8-flash': 2300, 'claude-opus-4-6': 1900, 'claude-opus-4-8': 1900, 'claude-opus-5': 1900, 'claude-sonnet-5': 1900, 'claude-fable-5': 2000, 'qwen3.8-max': 1000, 'glm-5.3': 2350, 'glm-5.3-flash': 2150, 'kimi-k3': 2200}, 'class_weights': {'kimi-k3': 1.05, 'glm-5.3': 1.05, 'glm-5.3-flash': 1.02, 'gpt-5.6-terra': 1.04, 'gpt-5.6-sol': 1.04, 'gemini-3.7-flash': 1.04, 'gemini-3.8-flash': 1.03, 'qwen3.8-max': 1.15}}, 'stage2': {'enabled': True, 'lr_lora': 1.2e-05, 'lr_head': 0.00015, 'lora_plus_ratio': 2.5, 'batch_size': 2, 'gradient_accumulation_steps': 2, 'epochs': 2, 'centered_weight': 0.5, 'supcon_weight': 0.12, 'supcon_warmup_frac': 0.25, 'label_smoothing': 0.05, 'warmup_steps': 15, 'n_splits': 5, 'check_overfitting': True, 'swa': True, 'swa_start_frac': 0.5, 'warmstart_centered_from_plain': True, 'centered_init_norm': 0.65, 'plain_ridge_weight': 0.0, 'centered_ridge_weight': 0.0, 'class_weights': {}}, 'weight_decay': 0.01}

"""
Canonical Configuration, Constants & Path Resolution
====================================================
Master canonical label ordering, provider families, simplex helpers,
and Kaggle filesystem asset discovery.
"""

import json
from pathlib import Path
from typing import Dict, List, Optional, Tuple, Any
import numpy as np

# Canonical 18-Class Labels (Strict order from NTL two-stage LoRA part two)
LABELS = [
    "gpt-5.5", "gpt-5.6-luna", "gpt-5.6-sol", "gpt-5.6-terra", "gpt-6-astra",
    "gemini-3.1-pro-preview", "gemini-3.6-flash", "gemini-3.7-flash", "gemini-3.8-flash",
    "claude-opus-4-6", "claude-opus-4-8", "claude-opus-5", "claude-sonnet-5", "claude-fable-5",
    "qwen3.8-max", "glm-5.3", "glm-5.3-flash", "kimi-k3"
]
K = len(LABELS)
LAB2I = {l: i for i, l in enumerate(LABELS)}
LABEL2ID = LAB2I

FAMILIES = {
    "gpt": [0, 1, 2, 3, 4],
    "gemini": [5, 6, 7, 8],
    "claude": [9, 10, 11, 12, 13],
    "glm": [15, 16],
}

HARD_CLASS_INDICES = [1, 2, 3, 6, 7, 8, 13, 14, 15, 16, 17]

SPECIALIST_GROUPS = {
    "gpt": [0, 1, 2, 3, 4],
    "gpt_56_triad": [1, 2, 3],
    "gemini": [5, 6, 7, 8],
    "gemini_flash_triad": [6, 7, 8],
    "gemini_37_38_pair": [7, 8],
    "claude": [9, 10, 11, 12, 13],
    "glm": [15, 16],
    "glm_kimi_triad": [15, 16, 17],
    "frontier_and_fable": [10, 13, 14, 15, 16, 17],
    "kimi_gpt55_qwen": [0, 14, 17],
    "claude_glm": [9, 10, 11, 12, 13, 15, 16],
    "gpt_qwen_kimi": [0, 1, 2, 3, 4, 14, 17],
}

PRIOR_SCALES = {
    "qwen3_14b_p2": 1.00,
    "qwen3_14b_inhouse": 1.00,
    "qwen3_14b_inhouse_v2": 1.00,
    "qwen25_14b_p2": 0.85,
    "qwen25_14b_inhouse": 0.85,
    "deberta_p2": 0.80,
    "deberta_inhouse": 0.80,
    "deberta_gemini": 0.40,
    "deberta_glm": 0.35,
    "deberta_gpt": 0.35,
    "deberta_claude": 0.35,
    "deberta_claude_glm": 0.35,
    "deberta_gpt_qwen_kimi": 0.35,
    "qwen3_14b_p1": 0.50,
    "instyle": 0.45,
    "qwen3_8b_claude": 0.30,
    "qwen3_8b_inhouse": 0.30,
    "gemma2_9b_ts": 0.40,
    "llama31_8b_ts": 0.40,
    "mistral_7b_ts": 0.40,
    "bge_m3": 0.60,
}

def align_cols(arr: np.ndarray, src_labels: list) -> np.ndarray:
    """Aligns columns of array to canonical 18-class order."""
    src_list = [x.decode('utf-8') if isinstance(x, (bytes, np.bytes_)) else str(x) for x in src_labels]
    idx = [src_list.index(l) for l in LABELS]
    if arr.ndim == 1:
        return arr[idx]
    elif arr.ndim == 2:
        if arr.shape[0] == len(src_list) and arr.shape[1] != len(src_list):
            return arr[idx, :]
        elif arr.shape[1] == len(src_list):
            return arr[:, idx]
    return arr[..., idx]

def log_softmax(x: np.ndarray, axis: int = -1) -> np.ndarray:
    """Numerically stable log-softmax."""
    c = np.max(x, axis=axis, keepdims=True)
    return x - c - np.log(np.sum(np.exp(x - c), axis=axis, keepdims=True) + 1e-15)

def softmax(x: np.ndarray, axis: int = -1) -> np.ndarray:
    """Numerically stable softmax."""
    c = np.max(x, axis=axis, keepdims=True)
    e = np.exp(x - c)
    return e / (np.sum(e, axis=axis, keepdims=True) + 1e-15)

def resolve_dataset_paths(input_dir: Optional[Path] = None) -> Tuple[Path, Path, Optional[Path]]:
    """Resolves train.csv, test.csv, and pool_v4.parquet across local and Kaggle environments."""
    if input_dir is None:
        input_dir = Path('/kaggle/input')
        if not input_dir.exists():
            input_dir = Path('.')

    candidates = [
        input_dir / 'competitions' / 'name-that-llm',
        input_dir / 'name-that-llm',
        Path('local_data'),
        Path('.')
    ]
    train_path = None
    test_path = None
    for c in candidates:
        if (c / 'train.csv').exists() and train_path is None:
            train_path = c / 'train.csv'
        if (c / 'test.csv').exists() and test_path is None:
            test_path = c / 'test.csv'

    if train_path is None:
        train_path = Path('local_data') / 'train.csv'
    if test_path is None:
        test_path = Path('local_data') / 'test.csv'

    pool_files = resolve_all_pool_files(input_dir)
    pool_file = pool_files[0] if pool_files else None

    return train_path, test_path, pool_file

def resolve_all_pool_files(input_dir: Optional[Path] = None) -> List[Path]:
    """Discovers all pretraining/unlabeled pool parquet files in priority order."""
    if input_dir is None:
        input_dir = Path('/kaggle/input') if Path('/kaggle/input').exists() else Path('.')
    pool_files: List[Path] = []
    pool_patterns = ['clean_master_pool_v2.parquet', 'clean_master_pool.parquet', 'ntl_clean_master_pool.parquet', 'nlm_pool_v5.parquet', 'pool_v4.parquet', 'nlm_matched_clean.parquet', 'nlm_pool_v2.parquet']
    for pat in pool_patterns:
        for p in input_dir.rglob(pat):
            if p not in pool_files:
                pool_files.append(p)
    return pool_files

def find_base_model(input_dir: Path, model_family: str, hidden_sz: int, num_layers: int) -> Optional[Path]:
    """Discovers base model directories from Hugging Face config.json inspection."""
    candidates = []
    fam_target = model_family.lower()
    for cfg_p in input_dir.rglob('config.json'):
        try:
            with open(cfg_p, encoding='utf-8') as f:
                cfg = json.load(f)
            m_type = str(cfg.get('model_type', '')).lower()
            h_dim = cfg.get('hidden_size', 0)
            n_lay = cfg.get('num_hidden_layers', 0)
            text_cfg = cfg.get('text_config', {})
            if isinstance(text_cfg, dict):
                if not h_dim:
                    h_dim = text_cfg.get('hidden_size', 0)
                if not n_lay:
                    n_lay = text_cfg.get('num_hidden_layers', 0)
                if not m_type or m_type == 'none':
                    m_type = str(text_cfg.get('model_type', '')).lower()

            p_str = str(cfg_p.parent).lower().replace('\\', '/')
            fam_match = (
                fam_target in m_type or
                m_type in fam_target or
                ("gemma" in fam_target and ("gemma" in m_type or "gemma" in p_str)) or
                ("llama" in fam_target and ("llama" in m_type or "llama" in p_str)) or
                ("qwen" in fam_target and ("qwen" in m_type or "qwen" in p_str)) or
                ("mistral" in fam_target and ("mistral" in m_type or "mistral" in p_str))
            )
            if fam_match and h_dim == hidden_sz and n_lay == num_layers:
                p_str_norm = p_str.replace('-', '').replace('_', '')
                fam_target_norm = fam_target.replace('-', '').replace('_', '')
                if fam_target_norm in p_str_norm or fam_target in p_str or fam_target in m_type:
                    return cfg_p.parent
                candidates.append(cfg_p.parent)
        except Exception:
            continue
    return candidates[0] if candidates else None

def _inspect_checkpoint_dim_and_keep(ckpt_dir: Path) -> Tuple[int, int, str]:
    """Returns (head_feature_dim, keep_layers, backbone) without loading model weights."""
    dim = 0
    keep = 0
    backbone = ""
    meta_p = ckpt_dir / "meta.json"
    if meta_p.exists():
        try:
            with open(meta_p, encoding='utf-8') as f:
                m = json.load(f)
            cfg = m.get("cfg", m)
            keep = int(cfg.get("keep", 0))
            backbone = str(m.get("backbone", cfg.get("backbone", ""))).lower()
        except Exception:
            pass

    adapter_p = ckpt_dir / "adapter_config.json"
    if adapter_p.exists():
        try:
            with open(adapter_p, encoding='utf-8') as f:
                ac = json.load(f)
            base_model = str(ac.get("base_model_name_or_path", "")).lower()
            if not backbone and base_model:
                backbone = base_model
        except Exception:
            pass

    heads_p = ckpt_dir / "heads.safetensors"
    if heads_p.exists():
        try:
            from safetensors import safe_open
            with safe_open(str(heads_p), framework="pt", device="cpu") as f:
                for k in f.keys():
                    shape = f.get_tensor(k).shape
                    if len(shape) == 2 and (shape[0] == 18 or shape[1] == 18):
                        dim = int(shape[1] if shape[0] == 18 else shape[0])
                        break
        except Exception:
            pass
    elif (ckpt_dir / "heads.pt").exists():
        try:
            import torch
            sd = torch.load(ckpt_dir / "heads.pt", map_location="cpu", weights_only=False)
            if isinstance(sd, dict):
                for k, v in sd.items():
                    if hasattr(v, "shape") and len(v.shape) == 2 and (v.shape[0] == 18 or v.shape[1] == 18):
                        dim = int(v.shape[1] if v.shape[0] == 18 else v.shape[0])
                        break
        except Exception:
            pass
    return dim, keep, backbone

_FOLD_MARKER_TOKENS = ("_f0", "_f1", "_f2", "_f3", "_f4")
_INHOUSE_TOKENS = ("inhouse", "ntl-inhouse", "export_lora24", "export_champion", "lora-peft-training", "peft_training")
_FOLD_PRIORITY_RULES = (
    (("q14v5",), "qwen3_14b_v5"),
    (("ts_q3b14",), "qwen3_14b_ts"),
    (("q8v5",), "qwen3_8b_v5"),
    (("ts_q3b8", "ts8_s7", "l24x_q3b8"), "qwen3_8b_ts"),
    (("ll8v5",), "llama3_8b_v5"),
    (("ts_llama8",), "llama3_8b_ts"),
    (("ts_q25_7b", "l24x_q25_7b"), "qwen25_7b_ts"),
    (("gemma2", "gemma_2", "gemma-2", "ts_gemma", "ts-gemma"), "gemma2_9b_ts"),
    (("llama31", "llama_31", "llama-3.1", "ts_llama31", "ts-llama31"), "llama31_8b_ts"),
    (("mistral", "ts_mistral", "ts-mistral"), "mistral_7b_ts"),
    (("bge_m3", "bge-m3", "bgem3"), "bge_m3"),
)


def _classify_inhouse_checkpoint(ckpt_dir: Path, p_str: str) -> Optional[str]:
    """Routes in-house fine-tuned checkpoints using dim/keep/backbone metadata."""
    if any(skip in p_str for skip in ("family_gate", "/gate", "\\gate", "stage1_best", "stage1_last")):
        return None
    if any(tok in p_str for tok in ("bge_m3", "bge-m3", "bgem3")):
        return "bge_m3"
    dim, keep, backbone = _inspect_checkpoint_dim_and_keep(ckpt_dir)
    if "2.5" in backbone or "qwen25" in backbone or (dim == 40960 and (keep >= 30 or "25" in p_str or "qwen25" in p_str)):
        return "qwen25_14b_inhouse"
    if dim == 40960 or "14b" in p_str or "14b" in backbone:
        if any(v2_tok in p_str for v2_tok in ("inhouse_v2", "inhouse-v2", "14b_v2", "14b-v2")):
            return "qwen3_14b_inhouse_v2"
        return "qwen3_14b_inhouse"
    if dim == 8192 or "deberta" in p_str or "deberta" in backbone:
        return "deberta_inhouse"
    if dim in (32768, 28672) or ("8b" in p_str or "8b" in backbone) or ("14b" not in p_str and "14b" not in backbone):
        return "qwen3_8b_inhouse"
    return None


def find_checkpoints(input_dir: Path) -> Dict[str, Path]:
    """Finds all fine-tuned LoRA and head checkpoints across safetensors and PyTorch formats."""
    ckpts: Dict[str, Path] = {}

    head_candidates = list(input_dir.rglob("heads.safetensors")) + list(input_dir.rglob("heads.pt"))
    for p in head_candidates:
        p_str = str(p).lower().replace("\\", "/")
        is_fold = any(tok in p_str for tok in _FOLD_MARKER_TOKENS)

        if "lora_qwen3_14b" in p_str or "qwen3_14b_part2" in p_str or ("14b" in p_str and "part2" in p_str and "qwen3" in p_str):
            ckpts["qwen3_14b_p2"] = p.parent
        elif "lora_qwen25_14b" in p_str or "qwen25_14b_part2" in p_str or ("14b" in p_str and "part2" in p_str and "qwen2" in p_str):
            ckpts["qwen25_14b_p2"] = p.parent
        elif "lora14b_v1" in p_str:
            ckpts["qwen3_14b_p1"] = p.parent
        elif "lora8b_v1" in p_str:
            ckpts["qwen3_8b_p1"] = p.parent
        elif any(token in p_str for token in _INHOUSE_TOKENS):
            key = _classify_inhouse_checkpoint(p.parent, p_str)
            if key:
                ckpts[key] = p.parent
        elif "instyle" in p_str:
            ckpts["instyle"] = p.parent
        else:
            for tokens, key in _FOLD_PRIORITY_RULES:
                if any(tok in p_str for tok in tokens):
                    if key not in ckpts or not is_fold:
                        ckpts[key] = p.parent
                    break

    # 2. Check for Claude specialist head
    for p in input_dir.rglob("claude_head.safetensors"):
        ckpts["qwen3_8b_claude"] = p.parent
        break
    if "qwen3_8b_claude" not in ckpts:
        for p in input_dir.rglob("readouts.npz"):
            if "claude" in str(p).lower():
                ckpts["qwen3_8b_claude"] = p.parent
                break

    # 3. Check for DeBERTa sequence classification head
    for p in list(input_dir.rglob("head.safetensors")) + list(input_dir.rglob("heads.safetensors")):
        if "deberta" in str(p).lower():
            ckpts["deberta_p2"] = p.parent
            break

    # 4. Fallback adapter/config discovery
    if "instyle" not in ckpts:
        for p in input_dir.rglob("*"):
            p_s = str(p).lower().replace("\\", "/")
            if "instyle" in p_s and any(
                (p / rel).exists()
                for rel in ("adapter_model.safetensors", "adapter/adapter_model.safetensors", "heads.safetensors", "heads.pt")
            ):
                ckpts["instyle"] = p
                break

    if "qwen3_14b_ts" not in ckpts:
        for p in input_dir.rglob("adapter_model.safetensors"):
            p_s = str(p).lower().replace("\\", "/")
            if "q14v5" in p_s or "ts_q3b14" in p_s:
                ckpts["qwen3_14b_ts"] = p.parent
                break

    if "bge_m3" not in ckpts:
        for p in input_dir.rglob("config.json"):
            p_s = str(p.parent).lower().replace("\\", "/")
            if any(t in p_s for t in ("bge-m3", "bge_m3", "bgem3")):
                ckpts["bge_m3"] = p.parent
                break

    if "gemma2_9b_ts" not in ckpts:
        for fname in ("heads.safetensors", "adapter_model.safetensors"):
            for p in input_dir.rglob(fname):
                p_s = str(p.parent).lower().replace("\\", "/")
                if any(t in p_s for t in ("gemma2", "gemma_2", "gemma-2", "ts_gemma", "ts-gemma")):
                    ckpts["gemma2_9b_ts"] = p.parent.parent if p.parent.name == "adapter" else p.parent
                    break
            if "gemma2_9b_ts" in ckpts:
                break

    return ckpts

def find_fold_adapters(input_dir: Optional[Path], prefix: str) -> List[Path]:
    """Scans input_dir for all adapter/head directories matching prefix, prefix_f*, prefix_fold*, prefix-f*."""
    if input_dir is None:
        input_dir = Path('/kaggle/input') if Path('/kaggle/input').exists() else Path('.')

    matches: set[Path] = set()
    prefix_lower = prefix.lower()

    candidate_files = (
        list(input_dir.rglob('heads.safetensors')) +
        list(input_dir.rglob('heads.pt')) +
        list(input_dir.rglob('adapter_model.safetensors'))
    )
    for f in candidate_files:
        p = f.parent
        p_name = p.name.lower()
        if p_name in ('adapter', 'checkpoint'):
            p = p.parent
            p_name = p.name.lower()

        p_str = str(p).lower().replace('\\', '/')
        if prefix_lower in p_name or prefix_lower in p_str:
            # Check pattern: exact match or prefix followed by _ or - or fold
            tokens = p_name.replace('-', '_').split('_')
            matched = False
            for t in tokens:
                if t == prefix_lower or t.startswith(prefix_lower):
                    matched = True
                    break
            if not matched and prefix_lower in p_str:
                matched = True

            if matched:
                matches.add(p)

    return sorted(list(matches), key=lambda x: str(x).lower())

def validate_production_assets(
    input_dir: Path,
    train_path: Path,
    test_path: Path,
    pool_files: List[Path],
    ckpts: Dict[str, Path],
    cfg: Optional[Dict[str, Any]] = None,
) -> Dict[str, Path]:
    """
    Fail-closed pre-flight validation for production inference runs (when NTL_SMOKE != '1').
    Raises FileNotFoundError immediately at startup if any required dataset, pool parquet,
    base model trunk, or checkpoint is missing.
    Returns a dict of verified base_model paths keyed by name.
    """
    if cfg is None:
        cfg = {}
    req = cfg.get("required_assets", {})
    missing: List[str] = []

    # 1. Verify competition CSVs
    if not train_path or not Path(train_path).exists():
        missing.append(f"Competition train.csv not found at: {train_path}")
    if not test_path or not Path(test_path).exists():
        missing.append(f"Competition test.csv not found at: {test_path}")

    # 2. Verify public pool parquet(s)
    require_pool = req.get("require_pool_parquet", True)
    valid_pools = [p for p in pool_files if p is not None and Path(p).exists()]
    if require_pool and len(valid_pools) == 0:
        missing.append(f"No public pool parquet (pool_v4.parquet / nlm_pool_v5.parquet) found under {input_dir}")

    # 3. Verify all required base model trunks
    default_bases = [
        {"name": "qwen3_14b", "family": "qwen3", "hidden_size": 5120, "num_layers": 40},
        {"name": "qwen25_14b", "family": "qwen2", "hidden_size": 5120, "num_layers": 48},
        {"name": "qwen3_8b", "family": "qwen3", "hidden_size": 4096, "num_layers": 36},
        {"name": "deberta_v3_large", "family": "deberta", "hidden_size": 1024, "num_layers": 24},
    ]
    if "required_base_models" in req:
        base_specs = req["required_base_models"]
    elif "checkpoints" in cfg or "checkpoints" in req:
        # Dynamically infer required base trunks from active checkpoints
        active_cks = cfg.get("checkpoints", req.get("checkpoints", []))
        inferred_bases = []
        if any("qwen3_14b" in str(c) for c in active_cks):
            inferred_bases.append({"name": "qwen3_14b", "family": "qwen3", "hidden_size": 5120, "num_layers": 40})
        if any("qwen25_14b" in str(c) for c in active_cks):
            inferred_bases.append({"name": "qwen25_14b", "family": "qwen2", "hidden_size": 5120, "num_layers": 48})
        if any("qwen3_8b" in str(c) for c in active_cks):
            inferred_bases.append({"name": "qwen3_8b", "family": "qwen3", "hidden_size": 4096, "num_layers": 36})
        if any("deberta" in str(c) for c in active_cks):
            inferred_bases.append({"name": "deberta_v3_large", "family": "deberta", "hidden_size": 1024, "num_layers": 24})
        if any("gemma2" in str(c) or "gemma" in str(c) for c in active_cks):
            inferred_bases.append({"name": "gemma2_9b", "family": "gemma2", "hidden_size": 3584, "num_layers": 42})
        if any("llama" in str(c) for c in active_cks):
            inferred_bases.append({"name": "llama3_8b", "family": "llama", "hidden_size": 4096, "num_layers": 32})
        if any("mistral" in str(c) for c in active_cks):
            inferred_bases.append({"name": "mistral_7b", "family": "mistral", "hidden_size": 4096, "num_layers": 32})
        base_specs = inferred_bases
    else:
        base_specs = default_bases
    verified_bases: Dict[str, Path] = {}
    for b_spec in base_specs:
        b_path = find_base_model(
            input_dir,
            b_spec["family"],
            int(b_spec["hidden_size"]),
            int(b_spec["num_layers"]),
        )
        if b_path is None or not Path(b_path).exists():
            missing.append(
                f"Required base model '{b_spec['name']}' (family={b_spec['family']}, "
                f"hidden={b_spec['hidden_size']}, layers={b_spec['num_layers']}) not found under {input_dir}"
            )
        else:
            verified_bases[b_spec["name"]] = b_path

    # 4. Verify fine-tuned checkpoints
    default_req_ckpts = [
        "qwen3_14b_p2", "qwen25_14b_p2",
        "deberta_p2", "qwen3_8b_claude"
    ]
    if "checkpoints" in req or "checkpoints" in cfg:
        # Unified single checkpoints list: ensure valid checkpoints were discovered under input_dir
        if len(ckpts) == 0:
            missing.append(f"No fine-tuned checkpoints found under {input_dir}")
    else:
        req_ckpts = req.get("required_checkpoints", default_req_ckpts)
        for ck_name in req_ckpts:
            if ck_name not in ckpts or not Path(ckpts[ck_name]).exists():
                missing.append(f"Required checkpoint '{ck_name}' not found under {input_dir} (discovered: {list(ckpts.keys())})")

    if missing:
        error_msg = (
            "[PRE-FLIGHT ASSET AUDIT FAILED] Missing required inference assets:\n  - "
            + "\n  - ".join(missing)
        )
        raise FileNotFoundError(error_msg)

    print(
        f"[Pre-Flight Audit PASSED] Verified {len(valid_pools)} pool file(s), "
        f"{len(verified_bases)} base trunks, and {len(ckpts)} checkpoints: {list(ckpts.keys())}"
    )
    return verified_bases


In [ ]:
# === MODULE: src/namethatllm/lexical_engine.py ===
"""
Subprocess Dual-Output Lexical Engine
====================================
Subprocess memory isolation running 78,482-row 4-View LinearSVC (Output 1)
and 5-View BPE + SGD + Style Twin + Specialist Complementary Ensemble (Output 2).
"""

import os
import gc
import re
import sys
import time
import hashlib
import unicodedata
import collections
import subprocess
from pathlib import Path
from typing import List, Tuple, Optional, Dict, Any

import joblib
import numpy as np
import pandas as pd
from scipy import sparse
from scipy.spatial.distance import squareform
from scipy.cluster.hierarchy import linkage, fcluster
from sklearn.svm import LinearSVC
from sklearn.linear_model import SGDClassifier, LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler

LABELS = [
    "gpt-5.5", "gpt-5.6-luna", "gpt-5.6-sol", "gpt-5.6-terra", "gpt-6-astra",
    "gemini-3.1-pro-preview", "gemini-3.6-flash", "gemini-3.7-flash", "gemini-3.8-flash",
    "claude-opus-4-6", "claude-opus-4-8", "claude-opus-5", "claude-sonnet-5", "claude-fable-5",
    "qwen3.8-max", "glm-5.3", "glm-5.3-flash", "kimi-k3"
]
K = len(LABELS)
LAB2I = {l: i for i, l in enumerate(LABELS)}

FAMILIES = {
    "gpt": [0, 1, 2, 3, 4],
    "gemini": [5, 6, 7, 8],
    "claude": [9, 10, 11, 12, 13],
    "glm": [15, 16],
}

def softmax(x: np.ndarray, axis: int = -1) -> np.ndarray:
    """Numerically stable softmax."""
    c = np.max(x, axis=axis, keepdims=True)
    e = np.exp(x - c)
    return e / (np.sum(e, axis=axis, keepdims=True) + 1e-15)

def log_softmax(x: np.ndarray, axis: int = -1) -> np.ndarray:
    """Numerically stable log-softmax."""
    c = np.max(x, axis=axis, keepdims=True)
    return x - c - np.log(np.sum(np.exp(x - c), axis=axis, keepdims=True) + 1e-15)

def _text_hash(text: str) -> str:
    """Normalized SHA1 hash of whitespace-collapsed text to guarantee zero exact/near-exact leakage."""
    norm = " ".join(str(text).strip().split())
    return hashlib.sha1(norm.encode("utf-8", errors="ignore")).hexdigest()

def _norm_text_key(value: Any) -> str:
    """Normalized NFKC string key for exact deduplication."""
    return re.sub(r'\s+', ' ', unicodedata.normalize('NFKC', str(value or ''))).strip()

FUNCTION_WORDS = set(
    "a an the and or but if while although because as so than that which who whom whose "
    "this these those it its they them their theirs he him his she her hers we us our ours "
    "you your yours i me my mine in on at to from with without for of by over under between "
    "through into about is are was were be been being have has had do does did can could "
    "will would shall should may might must not no nor all any some each every both either "
    "neither other another such only also just very too more most less least here there "
    "when where why how what whether then now still yet even much many few little "
    "am having s t re ve ll d m don doesn didn isn aren wasn weren hasn haven hadn won wouldn couldn shouldn".split()
)
STOP_WORDS = FUNCTION_WORDS

def shape_text(t: str) -> str:
    """Replaces alphabetic character blocks with X (uppercase) and x (lowercase)."""
    return re.sub(r"[^\W\d_]+", lambda m: "X" if m.group(0)[0].isupper() else "x", str(t))

def function_text(t: str) -> str:
    """Masks non-stopwords and non-punctuation to CONTENT, preserving function and punctuation syntax."""
    tokens = re.findall(r"(?u)\b\w+\b|[^\w\s]", str(t))
    return " ".join([w.lower() if (w.lower() in STOP_WORDS or not w.isalnum()) else "CONTENT" for w in tokens])

_RE_CODE_BLOCK = re.compile(r"```[\s\S]*?```|`[^`\n]+`")
_RE_STRAIGHT_CONTR = re.compile(r"[A-Za-z]'[A-Za-z]")
_RE_CURLY_CONTR = re.compile(r"[A-Za-z]\u2019[A-Za-z]")

def extract_handwriting_and_orthographic_features(texts: List[str]) -> Tuple[np.ndarray, np.ndarray]:
    """
    Single-pass extractor returning:
    - feats: (N, 26) float32 continuous punctuation, contraction, and structural densities
    - rules: (N, 4) float32 binary indicators for the 4 discrete orthographic invariants:
        col 0: any curly quote (\u2018, \u2019, \u201c, \u201d)
        col 1: >= 2 em-dashes (\u2014) in non-code prose
        col 2: >= 2 straight contractions [A-Za-z]'[A-Za-z] in non-code prose AND 0 curly \u2019
        col 3: >= 1 curly contraction [A-Za-z]\u2019[A-Za-z] in non-code prose
    """
    N = len(texts)
    feats = np.zeros((N, 26), dtype=np.float32)
    rules = np.zeros((N, 4), dtype=np.float32)

    for i, raw_t in enumerate(texts):
        t = str(raw_t)
        prose = _RE_CODE_BLOCK.sub(" ", t)
        t_len = max(len(t), 1)
        words = t.split()
        n_words = max(len(words), 1)

        raw_curly_sq = t.count("\u2018") + t.count("\u2019")
        raw_curly_dq = t.count("\u201c") + t.count("\u201d")
        prose_em_dash = prose.count("\u2014")
        n_str_c = len(_RE_STRAIGHT_CONTR.findall(prose))
        n_cur_c = len(_RE_CURLY_CONTR.findall(prose))

        # 4 Discrete Structural Rule Flags
        rules[i, 0] = 1.0 if (raw_curly_sq + raw_curly_dq) > 0 else 0.0
        rules[i, 1] = 1.0 if prose_em_dash >= 2 else 0.0
        rules[i, 2] = 1.0 if (n_str_c >= 2 and raw_curly_sq == 0) else 0.0
        rules[i, 3] = 1.0 if n_cur_c >= 1 else 0.0

        # 26 Continuous Density Features
        c_colon = t.count(":") / n_words
        c_semi = t.count(";") / n_words
        c_em_dash = t.count("\u2014") / n_words
        c_double_hyph = t.count("--") / n_words
        c_curly_sq = raw_curly_sq / n_words
        c_straight_sq = t.count("'") / n_words
        c_curly_dq = raw_curly_dq / n_words
        c_straight_dq = t.count('"') / n_words
        c_paren = (t.count("(") + t.count(")")) / n_words
        c_comma = t.count(",") / n_words
        c_excl = t.count("!") / n_words
        c_ques = t.count("?") / n_words
        c_ellipsis = (t.count("...") + t.count("\u2026")) / n_words
        c_hyphen = t.count("-") / n_words
        n_straight_contr = n_str_c / n_words
        n_curly_contr = n_cur_c / n_words

        c_newline = t.count("\n") / n_words
        c_bullet = (t.count("\n- ") + t.count("\n* ")) / n_words
        c_upper = sum(1 for ch in t if ch.isupper()) / t_len
        c_digits = sum(1 for ch in t if ch.isdigit()) / t_len

        w_lens = [len(w) for w in words]
        mean_w_len = float(np.mean(w_lens)) if w_lens else 0.0
        std_w_len = float(np.std(w_lens)) if len(w_lens) > 1 else 0.0

        sents = [s.strip() for s in t.replace("!", ".").replace("?", ".").split(".") if s.strip()]
        s_lens = [len(s.split()) for s in sents]
        mean_s_len = float(np.mean(s_lens)) if s_lens else 0.0
        std_s_len = float(np.std(s_lens)) if len(s_lens) > 1 else 0.0
        burstiness = float(std_s_len / max(mean_s_len, 1e-5))
        ttr = float(len(set(words)) / n_words)

        feats[i] = [
            c_colon, c_semi, c_em_dash, c_double_hyph, c_curly_sq, c_straight_sq,
            c_curly_dq, c_straight_dq, c_paren, c_comma, c_excl, c_ques,
            c_ellipsis, c_hyphen, n_straight_contr, n_curly_contr, c_newline,
            c_bullet, c_upper, c_digits, mean_w_len, std_w_len, mean_s_len,
            std_s_len, burstiness, ttr
        ]
    return feats, rules

def extract_handwriting_features(texts: List[str]) -> np.ndarray:
    """Backward-compatible wrapper returning only the (N, 26) continuous matrix."""
    feats, _ = extract_handwriting_and_orthographic_features(texts)
    return feats

def fit_handwriting_model(
    hw_tr: np.ndarray,
    fit_y: np.ndarray,
    hw_te: np.ndarray,
    df_tr: pd.DataFrame,
    groups_085: np.ndarray,
    twin_rows_085: np.ndarray,
    len_tr_raw: int
) -> np.ndarray:
    """
    Fits dedicated LogisticRegression plain (C=0.25) and prompt-centered (C=0.15)
    classifiers strictly on the 26 continuous handwriting features.
    Returns log_p_handwriting (N_test, 18).
    """
    clf_hw_plain = LogisticRegression(C=0.25, max_iter=1000, random_state=42, class_weight="balanced")
    clf_hw_plain.fit(hw_tr, fit_y)
    lp_hw = log_softmax(clf_hw_plain.decision_function(hw_te))

    if twin_rows_085.any() and len_tr_raw >= 18:
        hw_tr_c = hw_tr[:len_tr_raw].copy()
        for p in np.unique(df_tr["prompt_id"].values):
            m_p = (df_tr["prompt_id"].values == p)
            hw_tr_c[m_p] -= hw_tr_c[m_p].mean(axis=0, keepdims=True)
        clf_hw_twin = LogisticRegression(C=0.15, max_iter=1000, random_state=42, class_weight="balanced")
        clf_hw_twin.fit(hw_tr_c, fit_y[:len_tr_raw])

        hw_te_c = hw_te.copy()
        for g in np.unique(groups_085[twin_rows_085]):
            m_g = (groups_085 == g)
            hw_te_c[m_g] -= hw_te_c[m_g].mean(axis=0, keepdims=True)
        lp_hw_twin = log_softmax(clf_hw_twin.decision_function(hw_te_c[twin_rows_085]))
        lp_hw[twin_rows_085] = log_softmax(0.70 * lp_hw[twin_rows_085] + 0.30 * lp_hw_twin)

    return lp_hw

def train_lexical_engine(
    fit_texts: List[str],
    fit_y: np.ndarray,
    fit_sw: Optional[np.ndarray] = None,
    df_tr: Optional[pd.DataFrame] = None,
    qwen_tok_dir: Optional[str] = None
) -> Dict[str, Any]:
    """
    Trains all TF-IDF vectorizers, linear classifiers, handwriting density models,
    and specialist refiners offline on train + synthetic pool data.
    Returns a self-contained bundle dictionary that can be saved with save_lexical_bundle.
    """
    if fit_sw is None:
        fit_sw = np.ones(len(fit_texts), dtype=np.float32)
    len_tr_raw = len(df_tr) if df_tr is not None else 0

    VIEWS_4V = [
        ("char", 3, 5, 250000, 1.0),
        ("word", 1, 3, 180000, 0.7),
        ("shape", 2, 6, 60000, 0.5),
        ("function", 1, 3, 50000, 0.5),
    ]

    fit_sh = [shape_text(t) for t in fit_texts]
    fit_fn = [function_text(t) for t in fit_texts]

    vectorizers_4v = []
    tr_mats_4v = []
    for mode, mn, mx, mf, sc in VIEWS_4V:
        analyzer = "char" if mode == "shape" else ("word" if mode == "function" else mode)
        opts = dict(
            analyzer=analyzer,
            ngram_range=(mn, mx),
            lowercase=False,
            sublinear_tf=True,
            min_df=2 if len(fit_texts) >= 50 else 1,
            max_features=mf,
            dtype=np.float32,
        )
        if analyzer == "word":
            opts["token_pattern"] = r"(?u)\b\w+\b|[^\w\s]"

        v = TfidfVectorizer(**opts)
        if mode == "shape":
            tr_m = v.fit_transform(fit_sh) * np.float32(sc)
        elif mode == "function":
            tr_m = v.fit_transform(fit_fn) * np.float32(sc)
        else:
            tr_m = v.fit_transform(fit_texts) * np.float32(sc)
        vectorizers_4v.append(v)
        tr_mats_4v.append(tr_m)

    X_tr_4v = sparse.hstack(tr_mats_4v, format="csr")

    # OUTPUT 1: Pure 4-View LinearSVC (C=0.5, class_weight='balanced', source sample weights)
    svc_main = LinearSVC(C=0.5, random_state=42, max_iter=5000, tol=1e-3, class_weight="balanced")
    svc_main.fit(X_tr_4v, fit_y, sample_weight=fit_sw)

    # OUTPUT 2: Stamatatos Text Distortion Model (keeps top-100 words, masks rest with '*')
    wre = re.compile(r"[A-Za-z’']+")
    cnt = collections.Counter(w.lower() for t in fit_texts for w in wre.findall(t))
    distort_keep = {w for w, _ in cnt.most_common(100)}
    dist_fn = lambda arr: [wre.sub(lambda m_: m_.group(0) if m_.group(0).lower() in distort_keep else '*', t) for t in arr]

    D_tr = dist_fn(fit_texts)
    vs_dist = [
        TfidfVectorizer(sublinear_tf=True, lowercase=False, analyzer='char', ngram_range=(2, 6), min_df=2 if len(fit_texts) >= 50 else 1),
        TfidfVectorizer(sublinear_tf=True, lowercase=True, token_pattern=r"[^\s]+", ngram_range=(1, 3), min_df=2 if len(fit_texts) >= 50 else 1)
    ]
    Xa_dist = sparse.hstack([v_.fit_transform(D_tr) for v_ in vs_dist], format="csr")

    lr_distort = LogisticRegression(C=10.0, max_iter=3000, random_state=42)
    lr_distort.fit(Xa_dist, fit_y, sample_weight=fit_sw)

    # OUTPUT 3: Continuous Handwriting Density Model
    hw_tr_raw, _ = extract_handwriting_and_orthographic_features(fit_texts)
    scaler = StandardScaler()
    hw_tr = np.clip(scaler.fit_transform(hw_tr_raw), -5.0, 5.0).astype(np.float32)

    clf_hw_plain = LogisticRegression(C=0.25, max_iter=1000, random_state=42, class_weight="balanced")
    clf_hw_plain.fit(hw_tr, fit_y)

    clf_hw_twin = None
    if df_tr is not None and len_tr_raw >= 18 and "prompt_id" in df_tr.columns:
        hw_tr_c = hw_tr[:len_tr_raw].copy()
        for p in np.unique(df_tr["prompt_id"].values):
            m_p = (df_tr["prompt_id"].values == p)
            hw_tr_c[m_p] -= hw_tr_c[m_p].mean(axis=0, keepdims=True)
        clf_hw_twin = LogisticRegression(C=0.15, max_iter=1000, random_state=42, class_weight="balanced")
        clf_hw_twin.fit(hw_tr_c, fit_y[:len_tr_raw])

    # 5th View: BPE or Character-n-gram
    tok = None
    if qwen_tok_dir and os.path.exists(qwen_tok_dir):
        try:
            from transformers import AutoTokenizer
            tok = AutoTokenizer.from_pretrained(qwen_tok_dir, use_fast=True)
        except Exception:
            tok = None

    if tok is not None:
        def batch_bpe_strings(texts: List[str], batch_sz: int = 4000) -> List[str]:
            out_strs = []
            for s in range(0, len(texts), batch_sz):
                batch_ids = tok(
                    texts[s : s + batch_sz],
                    add_special_tokens=False,
                    truncation=True,
                    max_length=512,
                )["input_ids"]
                out_strs.extend(" ".join(map(str, ids)) for ids in batch_ids)
            return out_strs

        fit_bpe_strs = batch_bpe_strings(fit_texts)
        v_5th = TfidfVectorizer(
            analyzer="word", token_pattern=r"\S+", lowercase=False,
            ngram_range=(1, 3), max_features=75000, sublinear_tf=True, dtype=np.float32
        )
        tr_5th = v_5th.fit_transform(fit_bpe_strs) * 0.8
        is_bpe = True
        del fit_bpe_strs
        gc.collect()
    else:
        v_5th = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), max_features=75000, sublinear_tf=True, dtype=np.float32)
        tr_5th = v_5th.fit_transform(fit_texts) * 0.8
        is_bpe = False

    X_tr_5v = sparse.hstack([X_tr_4v, tr_5th], format="csr")

    svc_comp = LinearSVC(C=0.35, random_state=42, max_iter=3000, tol=1e-3, class_weight="balanced")
    svc_comp.fit(X_tr_5v, fit_y, sample_weight=fit_sw)

    sgd_comp = SGDClassifier(loss='log_loss', alpha=1e-4, max_iter=1500, random_state=42)
    sgd_comp.fit(X_tr_5v, fit_y, sample_weight=fit_sw)

    svc_twin = None
    if df_tr is not None and len_tr_raw >= 18 and "prompt_id" in df_tr.columns:
        X_train_only = X_tr_5v[:len_tr_raw]
        u_tr, inv_tr = np.unique(df_tr['prompt_id'].values, return_inverse=True)
        counts_tr = np.bincount(inv_tr)
        vals_tr = (1.0 / counts_tr[inv_tr]).astype(np.float32)
        G = sparse.csr_matrix((vals_tr, (np.arange(len(inv_tr)), inv_tr)), shape=(len(inv_tr), len(u_tr)), dtype=np.float32)
        X_train_c = X_train_only - G @ (G.T @ X_train_only)

        svc_twin = LinearSVC(C=0.30, random_state=42, max_iter=2500)
        svc_twin.fit(X_train_c, fit_y[:len_tr_raw])
        del X_train_only, G, X_train_c
        gc.collect()

    specialist_svcs = {}
    for fam_name, fam_cols in FAMILIES.items():
        fam_mask = np.isin(fit_y, fam_cols)
        if fam_mask.sum() > 50:
            sub_y = fit_y[fam_mask]
            sub_lab2i = {c: i for i, c in enumerate(fam_cols)}
            sub_y_mapped = np.array([sub_lab2i[y] for y in sub_y])

            spec_svc = LinearSVC(C=0.40, random_state=42, max_iter=2000, tol=1e-3, class_weight="balanced")
            spec_svc.fit(X_tr_5v[fam_mask], sub_y_mapped)
            specialist_svcs[fam_name] = spec_svc

    bundle = {
        "views_4v_specs": VIEWS_4V,
        "vectorizers_4v": vectorizers_4v,
        "svc_main": svc_main,
        "distort_keep": sorted(list(distort_keep)),
        "vs_dist": vs_dist,
        "lr_distort": lr_distort,
        "hw_scaler": scaler,
        "clf_hw_plain": clf_hw_plain,
        "clf_hw_twin": clf_hw_twin,
        "is_bpe": is_bpe,
        "v_5th": v_5th,
        "svc_comp": svc_comp,
        "sgd_comp": sgd_comp,
        "svc_twin": svc_twin,
        "specialist_svcs": specialist_svcs,
        "families": FAMILIES,
        "labels": LABELS,
    }
    return bundle

def save_lexical_bundle(bundle: Dict[str, Any], path: Path | str) -> None:
    """Serializes the lexical bundle to disk using compressed joblib."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    joblib.dump(bundle, path, compress=3)
    sz_mb = path.stat().st_size / (1024 * 1024)
    print(f"[Lexical] Saved lexical bundle to {path} ({sz_mb:.2f} MB)")

def load_lexical_bundle(path: Path | str) -> Dict[str, Any]:
    """Loads a pre-trained lexical bundle from disk."""
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Lexical bundle not found at {path}")
    t0 = time.time()
    bundle = joblib.load(path)
    print(f"[Lexical] Loaded lexical bundle from {path} in {time.time() - t0:.2f}s")
    return bundle

def predict_lexical_engine(
    bundle: Dict[str, Any],
    test_texts: List[str],
    unseen_texts: Optional[List[str]] = None,
    qwen_tok_dir: Optional[str] = None,
    working_dir: Optional[Path | str] = None,
) -> Dict[str, np.ndarray]:
    """
    Executes fast inference using a pre-trained lexical bundle on test_texts and optional unseen_texts.
    Takes ~10-20 seconds on CPU.
    """
    t0 = time.time()
    unseen_texts = unseen_texts or []

    # 1. Compute stopword TF-IDF distance matrix and topic clusters (t=0.85) on test
    if len(test_texts) >= 2:
        Xt_tfidf = TfidfVectorizer(stop_words='english', sublinear_tf=True, max_features=100000, dtype=np.float32).fit_transform(test_texts)
        D = np.clip(1.0 - (Xt_tfidf @ Xt_tfidf.T).toarray(), 0.0, None)
        np.fill_diagonal(D, 0.0)
        cond_d = squareform(D, checks=False)
        groups_085 = fcluster(linkage(cond_d, method='average'), 0.85, criterion='distance')
    else:
        D = np.zeros((len(test_texts), len(test_texts)), dtype=np.float32)
        groups_085 = np.ones(len(test_texts), dtype=int)

    sz_085 = np.bincount(groups_085)[groups_085] if len(groups_085) > 0 else np.array([], dtype=int)
    twin_rows_085 = (sz_085 >= 2) & (sz_085 <= 18) if len(sz_085) > 0 else np.zeros(0, dtype=bool)

    # Stopword TF-IDF and clusters on unseen holdout benchmark
    if len(unseen_texts) >= 2:
        Xu_tfidf = TfidfVectorizer(stop_words='english', sublinear_tf=True, max_features=100000, dtype=np.float32).fit_transform(unseen_texts)
        D_u = np.clip(1.0 - (Xu_tfidf @ Xu_tfidf.T).toarray(), 0.0, None)
        np.fill_diagonal(D_u, 0.0)
        cond_du = squareform(D_u, checks=False)
        groups_085_u = fcluster(linkage(cond_du, method='average'), 0.85, criterion='distance')
    else:
        D_u = np.zeros((len(unseen_texts), len(unseen_texts)), dtype=np.float32)
        groups_085_u = np.ones(len(unseen_texts), dtype=int)
    sz_085_u = np.bincount(groups_085_u)[groups_085_u] if len(groups_085_u) > 0 else np.array([], dtype=int)
    twin_rows_085_u = (sz_085_u >= 2) & (sz_085_u <= 18) if len(sz_085_u) > 0 else np.zeros(0, dtype=bool)

    # 2. 4-View Transform
    test_sh = [shape_text(t) for t in test_texts]
    unseen_sh = [shape_text(t) for t in unseen_texts] if unseen_texts else []
    test_fn = [function_text(t) for t in test_texts]
    unseen_fn = [function_text(t) for t in unseen_texts] if unseen_texts else []

    te_mats_4v = []
    unseen_mats_4v = []
    for (mode, mn, mx, mf, sc), v in zip(bundle["views_4v_specs"], bundle["vectorizers_4v"]):
        if mode == "shape":
            te_m = v.transform(test_sh) * np.float32(sc)
            u_m = v.transform(unseen_sh) * np.float32(sc) if unseen_texts else None
        elif mode == "function":
            te_m = v.transform(test_fn) * np.float32(sc)
            u_m = v.transform(unseen_fn) * np.float32(sc) if unseen_texts else None
        else:
            te_m = v.transform(test_texts) * np.float32(sc)
            u_m = v.transform(unseen_texts) * np.float32(sc) if unseen_texts else None
        te_mats_4v.append(te_m)
        if u_m is not None:
            unseen_mats_4v.append(u_m)

    X_te_4v = sparse.hstack(te_mats_4v, format="csr")
    X_unseen_4v = sparse.hstack(unseen_mats_4v, format="csr") if unseen_mats_4v else None

    # OUTPUT 1: Pure 4-View LinearSVC
    svc_main = bundle["svc_main"]
    logits_svc_main = svc_main.decision_function(X_te_4v)
    log_p_svc_main = log_softmax(logits_svc_main)

    if X_unseen_4v is not None and X_unseen_4v.shape[0] > 0:
        unseen_log_p_svc_main = log_softmax(svc_main.decision_function(X_unseen_4v))
    else:
        unseen_log_p_svc_main = np.zeros((0, K), dtype=np.float32)

    # OUTPUT 2: Text Distortion Model
    wre = re.compile(r"[A-Za-z’']+")
    distort_keep = set(bundle["distort_keep"])
    dist_fn = lambda arr: [wre.sub(lambda m_: m_.group(0) if m_.group(0).lower() in distort_keep else '*', t) for t in arr]

    D_te = dist_fn(test_texts)
    D_unseen = dist_fn(unseen_texts) if unseen_texts else []
    vs_dist = bundle["vs_dist"]
    lr_distort = bundle["lr_distort"]

    Xb_dist = sparse.hstack([v_.transform(D_te) for v_ in vs_dist], format="csr")
    Xu_dist = sparse.hstack([v_.transform(D_unseen) for v_ in vs_dist], format="csr") if unseen_texts else None

    P_distort = lr_distort.predict_proba(Xb_dist)
    log_p_distort = lr_distort.predict_log_proba(Xb_dist)
    if Xu_dist is not None and Xu_dist.shape[0] > 0:
        P_unseen_distort = lr_distort.predict_proba(Xu_dist)
        log_p_unseen_distort = lr_distort.predict_log_proba(Xu_dist)
    else:
        P_unseen_distort = None
        log_p_unseen_distort = None

    # OUTPUT 3: Continuous Handwriting & Orthographic Features
    scaler = bundle["hw_scaler"]
    clf_hw_plain = bundle["clf_hw_plain"]
    clf_hw_twin = bundle.get("clf_hw_twin")

    hw_te_raw, rule_flags_te = extract_handwriting_and_orthographic_features(test_texts)
    hw_te = np.clip(scaler.transform(hw_te_raw), -5.0, 5.0).astype(np.float32)
    lp_hw = log_softmax(clf_hw_plain.decision_function(hw_te))

    if clf_hw_twin is not None and twin_rows_085.any():
        hw_te_c = hw_te.copy()
        for g in np.unique(groups_085[twin_rows_085]):
            m_g = (groups_085 == g)
            hw_te_c[m_g] -= hw_te_c[m_g].mean(axis=0, keepdims=True)
        lp_hw_twin = log_softmax(clf_hw_twin.decision_function(hw_te_c[twin_rows_085]))
        lp_hw[twin_rows_085] = log_softmax(0.70 * lp_hw[twin_rows_085] + 0.30 * lp_hw_twin)

    if unseen_texts:
        hw_u_raw, rule_flags_u = extract_handwriting_and_orthographic_features(unseen_texts)
        hw_u = np.clip(scaler.transform(hw_u_raw), -5.0, 5.0).astype(np.float32)
        lp_hw_u = log_softmax(clf_hw_plain.decision_function(hw_u))
        if clf_hw_twin is not None and twin_rows_085_u.any():
            hw_u_c = hw_u.copy()
            for g in np.unique(groups_085_u[twin_rows_085_u]):
                m_g = (groups_085_u == g)
                hw_u_c[m_g] -= hw_u_c[m_g].mean(axis=0, keepdims=True)
            lp_hw_u_twin = log_softmax(clf_hw_twin.decision_function(hw_u_c[twin_rows_085_u]))
            lp_hw_u[twin_rows_085_u] = log_softmax(0.70 * lp_hw_u[twin_rows_085_u] + 0.30 * lp_hw_u_twin)
    else:
        rule_flags_u = np.zeros((0, 4), dtype=np.float32)
        lp_hw_u = np.zeros((0, K), dtype=np.float32)

    # 5th View Transform
    is_bpe = bundle.get("is_bpe", False)
    v_5th = bundle["v_5th"]

    if is_bpe:
        tok = None
        if qwen_tok_dir and os.path.exists(qwen_tok_dir):
            try:
                from transformers import AutoTokenizer
                tok = AutoTokenizer.from_pretrained(qwen_tok_dir, use_fast=True)
            except Exception:
                tok = None

        if tok is not None:
            def batch_bpe_strings(texts: List[str], batch_sz: int = 4000) -> List[str]:
                out_strs = []
                for s in range(0, len(texts), batch_sz):
                    batch_ids = tok(
                        texts[s : s + batch_sz],
                        add_special_tokens=False,
                        truncation=True,
                        max_length=512,
                    )["input_ids"]
                    out_strs.extend(" ".join(map(str, ids)) for ids in batch_ids)
                return out_strs

            test_bpe_strs = batch_bpe_strings(test_texts)
            unseen_bpe_strs = batch_bpe_strings(unseen_texts) if unseen_texts else []
            te_5th = v_5th.transform(test_bpe_strs) * 0.8
            unseen_5th = v_5th.transform(unseen_bpe_strs) * 0.8 if unseen_texts else None
            del test_bpe_strs, unseen_bpe_strs
        else:
            te_5th = v_5th.transform(test_texts) * 0.8
            unseen_5th = v_5th.transform(unseen_texts) * 0.8 if unseen_texts else None
    else:
        te_5th = v_5th.transform(test_texts) * 0.8
        unseen_5th = v_5th.transform(unseen_texts) * 0.8 if unseen_texts else None

    X_te_5v = sparse.hstack([X_te_4v, te_5th], format="csr")
    X_unseen_5v = sparse.hstack([X_unseen_4v, unseen_5th], format="csr") if unseen_5th is not None else None

    # Complementary Ensemble
    svc_comp = bundle["svc_comp"]
    sgd_comp = bundle["sgd_comp"]
    svc_twin = bundle.get("svc_twin")

    P_svc_comp = softmax(svc_comp.decision_function(X_te_5v), axis=1)
    P_sgd_comp = sgd_comp.predict_proba(X_te_5v)
    P_twin_comp = P_svc_comp.copy()

    if svc_twin is not None and twin_rows_085.any():
        X_te_twin = X_te_5v[twin_rows_085]
        u_t, inv_t = np.unique(groups_085[twin_rows_085], return_inverse=True)
        counts_t = np.bincount(inv_t)
        vals_t = (1.0 / counts_t[inv_t]).astype(np.float32)
        G_t = sparse.csr_matrix((vals_t, (np.arange(len(inv_t)), inv_t)), shape=(len(inv_t), len(u_t)), dtype=np.float32)
        X_te_twin_c = X_te_twin - G_t @ (G_t.T @ X_te_twin)
        P_twin_comp[twin_rows_085] = softmax(svc_twin.decision_function(X_te_twin_c), axis=1)
        del X_te_twin, G_t, X_te_twin_c

    P_lex_comp = 0.40 * P_svc_comp + 0.25 * P_sgd_comp + 0.15 * P_twin_comp + 0.20 * P_distort
    log_p_lex_comp = np.log(np.clip(P_lex_comp, 1e-9, 1.0))

    P_unseen_comp = None
    log_p_unseen_comp = None
    if X_unseen_5v is not None and X_unseen_5v.shape[0] > 0:
        P_unseen_svc = softmax(svc_comp.decision_function(X_unseen_5v), axis=1)
        P_unseen_sgd = sgd_comp.predict_proba(X_unseen_5v)
        P_twin_comp_u = P_unseen_svc.copy()
        if svc_twin is not None and len(twin_rows_085_u) > 0 and twin_rows_085_u.any():
            X_u_twin = X_unseen_5v[twin_rows_085_u]
            u_u, inv_u = np.unique(groups_085_u[twin_rows_085_u], return_inverse=True)
            counts_u = np.bincount(inv_u)
            vals_u = (1.0 / counts_u[inv_u]).astype(np.float32)
            G_u = sparse.csr_matrix((vals_u, (np.arange(len(inv_u)), inv_u)), shape=(len(inv_u), len(u_u)), dtype=np.float32)
            X_u_twin_c = X_u_twin - G_u @ (G_u.T @ X_u_twin)
            P_twin_comp_u[twin_rows_085_u] = softmax(svc_twin.decision_function(X_u_twin_c), axis=1)
            del X_u_twin, G_u, X_u_twin_c

        u_twin = P_twin_comp_u if P_twin_comp_u is not None else P_unseen_svc
        if P_unseen_distort is not None:
            P_unseen_comp = 0.40 * P_unseen_svc + 0.25 * P_unseen_sgd + 0.15 * u_twin + 0.20 * P_unseen_distort
        else:
            P_unseen_comp = 0.55 * P_unseen_svc + 0.45 * P_unseen_sgd
        log_p_unseen_comp = np.log(np.clip(P_unseen_comp, 1e-9, 1.0))

    # 4 Within-Family Specialist Refiners
    beta = 0.25
    specialist_svcs = bundle.get("specialist_svcs", {})
    families = bundle.get("families", FAMILIES)

    for fam_name, spec_svc in specialist_svcs.items():
        if fam_name not in families:
            continue
        fam_cols = families[fam_name]
        df_spec = spec_svc.decision_function(X_te_5v)
        logits_spec = np.column_stack([-df_spec, df_spec]) if df_spec.ndim == 1 else df_spec
        P_spec = softmax(logits_spec, axis=1)

        P_prior = P_lex_comp[:, fam_cols]
        P_prior_norm = P_prior / (P_prior.sum(axis=1, keepdims=True) + 1e-12)
        P_refined = (1.0 - beta) * P_prior_norm + beta * P_spec

        log_ratio = np.log(np.clip(P_refined, 1e-9, 1.0)) - np.log(np.clip(P_prior_norm, 1e-9, 1.0))
        log_p_lex_comp[:, fam_cols] += log_ratio

        if log_p_unseen_comp is not None and X_unseen_5v is not None and X_unseen_5v.shape[0] > 0:
            df_spec_u = spec_svc.decision_function(X_unseen_5v)
            logits_spec_u = np.column_stack([-df_spec_u, df_spec_u]) if df_spec_u.ndim == 1 else df_spec_u
            P_spec_u = softmax(logits_spec_u, axis=1)
            P_prior_u = P_unseen_comp[:, fam_cols]
            P_prior_u_norm = P_prior_u / (P_prior_u.sum(axis=1, keepdims=True) + 1e-12)
            P_refined_u = (1.0 - beta) * P_prior_u_norm + beta * P_spec_u
            log_ratio_u = np.log(np.clip(P_refined_u, 1e-9, 1.0)) - np.log(np.clip(P_prior_u_norm, 1e-9, 1.0))
            log_p_unseen_comp[:, fam_cols] += log_ratio_u

    P_lex_comp_final = softmax(log_p_lex_comp, axis=1) if len(test_texts) > 0 else np.zeros((0, K), dtype=np.float32)
    unseen_p_lex_comp_final = softmax(log_p_unseen_comp, axis=1) if (log_p_unseen_comp is not None and len(unseen_texts) > 0) else np.zeros((0, K), dtype=np.float32)

    # Save to disk if working_dir provided
    if working_dir is not None:
        working_dir = Path(working_dir)
        os.makedirs(working_dir, exist_ok=True)
        np.save(working_dir / 'log_p_svc_main.npy', log_p_svc_main)
        np.save(working_dir / 'p_lex_comp.npy', P_lex_comp_final)
        np.save(working_dir / 'D_tfidf.npy', D)
        np.save(working_dir / 'groups_085.npy', groups_085)
        np.save(working_dir / 'log_p_handwriting.npy', lp_hw)
        np.save(working_dir / 'rule_flags_test.npy', rule_flags_te)
        np.save(working_dir / 'unseen_log_p_svc_main.npy', unseen_log_p_svc_main)
        np.save(working_dir / 'unseen_p_lex_comp.npy', unseen_p_lex_comp_final)
        if 'D_u' in locals() and D_u is not None:
            np.save(working_dir / 'D_tfidf_unseen.npy', D_u)
        if 'groups_085_u' in locals() and groups_085_u is not None:
            np.save(working_dir / 'groups_085_unseen.npy', groups_085_u)
        if 'lp_hw_u' in locals() and lp_hw_u is not None:
            np.save(working_dir / 'log_p_handwriting_unseen.npy', lp_hw_u)
        if 'rule_flags_u' in locals() and rule_flags_u is not None:
            np.save(working_dir / 'rule_flags_unseen.npy', rule_flags_u)
        if log_p_distort is not None:
            np.save(working_dir / 'log_p_distort.npy', log_p_distort)
        if log_p_unseen_distort is not None:
            np.save(working_dir / 'unseen_log_p_distort.npy', log_p_unseen_distort)

    print(f"[Lexical] predict_lexical_engine completed in {time.time() - t0:.2f}s")
    return {
        "log_p_svc_main": log_p_svc_main,
        "p_lex_comp": P_lex_comp_final,
        "D_tfidf": D,
        "groups_085": groups_085,
        "log_p_handwriting": lp_hw,
        "rule_flags_test": rule_flags_te,
        "unseen_log_p_svc_main": unseen_log_p_svc_main,
        "unseen_p_lex_comp": unseen_p_lex_comp_final,
        "D_tfidf_unseen": D_u if ('D_u' in locals() and D_u is not None) else None,
        "groups_085_unseen": groups_085_u if ('groups_085_u' in locals() and groups_085_u is not None) else None,
        "log_p_handwriting_unseen": lp_hw_u if ('lp_hw_u' in locals() and lp_hw_u is not None) else None,
        "rule_flags_unseen": rule_flags_u if ('rule_flags_u' in locals() and rule_flags_u is not None) else None,
        "log_p_distort": log_p_distort,
        "unseen_log_p_distort": log_p_unseen_distort,
    }

def run_lexical_engine(
    train_path: Optional[Path] = None,
    test_path: Optional[Path] = None,
    pool_file: Optional[Path] = None,
    qwen_tok_dir: str = "",
    working_dir: Optional[Path] = None,
    bundle_path: Optional[Path | str] = None,
) -> Dict[str, np.ndarray]:
    """
    Runs the lexical engine with automatic pre-trained bundle auto-discovery.
    If a pre-trained bundle exists, executes pure inference in ~15-30s.
    Otherwise fits models from scratch on train + pool data.
    """
    working_dir = Path(working_dir) if working_dir else Path(".")
    working_dir.mkdir(parents=True, exist_ok=True)

    # 1. Search for pre-trained bundle
    bundle = None
    bundle_candidates = []
    if bundle_path:
        bundle_candidates.append(Path(bundle_path))
    if working_dir:
        bundle_candidates.append(working_dir / "lexical_bundle.joblib")
    bundle_candidates.extend([
        Path("local_data/statistical_heads/lexical_bundle.joblib"),
        Path("/kaggle/input/ntl-statistical-heads/lexical_bundle.joblib"),
    ])
    if Path("/kaggle/input").exists():
        for p in Path("/kaggle/input").rglob("lexical_bundle.joblib"):
            if p not in bundle_candidates:
                bundle_candidates.append(p)

    for cand in bundle_candidates:
        if cand.exists():
            try:
                bundle = load_lexical_bundle(cand)
                print(f"[Lexical] Using pre-trained bundle from {cand}")
                break
            except Exception as e:
                print(f"[Lexical] Could not load candidate bundle {cand}: {e}")

    # 2. Extract test and unseen texts
    if test_path and str(test_path).endswith('.parquet') and Path(test_path).exists():
        df_te = pd.read_parquet(test_path)
    elif test_path and Path(test_path).exists():
        df_te = pd.read_csv(test_path)
    else:
        df_te = pd.DataFrame({"text": []})
    test_texts = df_te['text'].fillna('').astype(str).tolist()

    unseen_parquet = working_dir / "unseen_holdout.parquet"
    unseen_texts: List[str] = []
    unseen_hashes = set()
    if unseen_parquet.exists():
        try:
            df_unseen_holdout = pd.read_parquet(unseen_parquet)
            if len(df_unseen_holdout) > 0 and "text" in df_unseen_holdout.columns:
                unseen_texts = df_unseen_holdout["text"].fillna("").astype(str).tolist()
                if "_hash" in df_unseen_holdout.columns:
                    unseen_hashes = set(df_unseen_holdout["_hash"].tolist())
                else:
                    unseen_hashes = set(_text_hash(t) for t in unseen_texts)
                print(f"[Lexical] Found unseen holdout set: {len(unseen_texts)} rows quarantined from training.")
        except Exception as e:
            print(f"[Lexical] Note: Could not read {unseen_parquet}: {e}")

    # 3. If pre-trained bundle found, predict immediately!
    if bundle is not None:
        return predict_lexical_engine(
            bundle=bundle,
            test_texts=test_texts,
            unseen_texts=unseen_texts,
            qwen_tok_dir=qwen_tok_dir,
            working_dir=working_dir,
        )

    # 4. Fall back to training from scratch
    print("[Lexical] No pre-trained bundle found. Training lexical engine from scratch...")
    if train_path is None or not Path(train_path).exists():
        raise FileNotFoundError(f"Training path {train_path} not found and no pre-trained bundle available.")

    df_tr = pd.read_csv(train_path)
    fit_texts = df_tr['text'].fillna('').astype(str).tolist()
    fit_y = np.array([LAB2I[m] for m in df_tr['model']])
    fit_sw = np.ones(len(df_tr), dtype=np.float32)  # Official train rows get 1.00
    seen_texts = {_norm_text_key(t) for t in fit_texts}

    # Load pool parquet files if present with source weights
    pool_candidates = [pool_file] if isinstance(pool_file, (str, Path)) else (pool_file if isinstance(pool_file, list) else [])
    for pf in pool_candidates:
        if pf and os.path.exists(pf):
            try:
                p_df = pd.read_parquet(pf, columns=['text', 'model'])
                p_df = p_df[p_df['model'].isin(LABELS)].dropna(subset=['text', 'model']).copy()
                pf_name = str(pf).lower().replace('\\', '/')
                # Matched clean gets 0.50, general scraped pool gets 0.35
                src_w = 0.50 if ('matched' in pf_name or 'clean' in pf_name) else 0.35

                keep_mask = []
                for t in p_df['text'].values:
                    k = _norm_text_key(t)
                    h = _text_hash(t)
                    ok = bool(k) and (k not in seen_texts) and (h not in unseen_hashes)
                    keep_mask.append(ok)
                    if ok:
                        seen_texts.add(k)
                p_df = p_df.loc[keep_mask].reset_index(drop=True)
                if len(p_df) > 0:
                    p_texts = p_df['text'].fillna('').astype(str).tolist()
                    p_y = p_df['model'].map(LAB2I).values
                    p_sw = np.full(len(p_df), src_w, dtype=np.float32)
                    fit_texts.extend(p_texts)
                    fit_y = np.r_[fit_y, p_y]
                    fit_sw = np.r_[fit_sw, p_sw]
                    print(f"[Lexical] Added {len(p_df)} rows from {Path(pf).name} with source weight {src_w}")
            except Exception as e:
                print(f"[Lexical] Note: Could not read pool {pf}: {e}")

    bundle = train_lexical_engine(
        fit_texts=fit_texts,
        fit_y=fit_y,
        fit_sw=fit_sw,
        df_tr=df_tr,
        qwen_tok_dir=qwen_tok_dir,
    )

    # Optional: save the newly trained bundle to working_dir
    try:
        save_lexical_bundle(bundle, working_dir / "lexical_bundle.joblib")
    except Exception as e:
        print(f"[Lexical] Note: Could not save bundle: {e}")

    return predict_lexical_engine(
        bundle=bundle,
        test_texts=test_texts,
        unseen_texts=unseen_texts,
        qwen_tok_dir=qwen_tok_dir,
        working_dir=working_dir,
    )

if __name__ == "__main__":
    working_dir = Path("/kaggle/working")
    if not working_dir.exists():
        working_dir = Path("local_data")
    working_dir.mkdir(parents=True, exist_ok=True)

    input_dir = Path("/kaggle/input")
    if not input_dir.exists():
        input_dir = Path(".")

    candidates = [
        input_dir / 'competitions' / 'name-that-llm',
        input_dir / 'name-that-llm',
        Path('local_data'),
        Path('.')
    ]
    train_path = None
    test_path = None
    for c in candidates:
        if (c / 'train.csv').exists() and train_path is None:
            train_path = c / 'train.csv'
        if (c / 'test.csv').exists() and test_path is None:
            test_path = c / 'test.csv'
    if train_path is None:
        train_path = Path('local_data') / 'train.csv'
    if test_path is None:
        test_path = Path('local_data') / 'test.csv'

    pool_files: List[Path] = []
    for pat in ['nlm_pool_v5.parquet', 'pool_v4.parquet', 'nlm_matched_clean.parquet', 'nlm_pool_v2.parquet']:
        for p in input_dir.rglob(pat):
            if p not in pool_files:
                pool_files.append(p)

    tok_paths = list(input_dir.rglob("vocab.json"))
    qwen_tok_dir = str(tok_paths[0].parent) if tok_paths else ""

    print(f"[Module 1 CLI] Running lexical engine: train={train_path}, test={test_path}, pools={[p.name for p in pool_files]}")
    run_lexical_engine(train_path, test_path, pool_files, qwen_tok_dir, working_dir)
    print("[Module 1 CLI] Lexical engine finished successfully.")


In [ ]:
# === MODULE: src/namethatllm/wpca_gaussian.py ===
"""
Full-Rank wPCA & ExactAuthorPosterior (2^18 Bitmask DP)
======================================================
Two-way variance decomposition, Ledoit-Wolf shrinkage floors,
crash-proof cluster splitting (max_size <= 14), and exact forward-backward
bitmask DP author posteriors under the unique-authorship constraint.
"""

from functools import lru_cache
from typing import Tuple, List, Dict, Optional, Any
import numpy as np
from scipy.spatial.distance import squareform
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.special import logsumexp
from sklearn.covariance import ledoit_wolf_shrinkage


def _two_way(Z: np.ndarray, y: np.ndarray, pinv: np.ndarray, G: int, K_classes: int = 18, iters: int = 4) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Iterative two-way ANOVA decomposition separating author means and prompt random effects."""
    N, d = Z.shape
    mu = np.zeros((K_classes, d), dtype=np.float64)
    P = np.zeros((G, d), dtype=np.float64)
    ng = np.bincount(pinv, minlength=G).clip(min=1)
    for _ in range(iters):
        R = Z - P[pinv]
        for k in range(K_classes):
            idx = (y == k)
            if idx.any():
                mu[k] = R[idx].mean(axis=0)
        R = Z - mu[y]
        P[:] = 0.0
        np.add.at(P, pinv, R)
        P /= ng[:, None]
    return mu, P, ng

def _shrunk(S: np.ndarray, alpha: float) -> np.ndarray:
    """Shrinks covariance matrix S towards spherical target."""
    d = S.shape[0]
    tr = np.trace(S) / d
    return (1.0 - alpha) * S + alpha * tr * np.eye(d)

def fit_shared_gaussian_lowram(X_tr_f32: np.ndarray, y: np.ndarray, prompts: np.ndarray,
                               d_target: int = 256, se_floor: float = 0.10) -> dict:
    """
    Fits shared Gaussian whitener with within-prompt centering and Ledoit-Wolf shrinkage floor.
    Returns unsupervised top-64 within-prompt principal components V_top64 (zero label leakage!).
    """
    N, D = X_tr_f32.shape
    pu, pinv = np.unique(prompts, return_inverse=True)
    G = len(pu)

    mx = X_tr_f32.mean(axis=0, dtype=np.float64).astype(np.float32)
    sx = X_tr_f32.std(axis=0, dtype=np.float64).astype(np.float32)
    sx = np.where(sx > 1e-6, sx, 1.0).astype(np.float32)

    # Standardize and subtract prompt means in-place (Within-Prompt Whitened PCA!)
    R = (X_tr_f32 - mx) / sx
    Pm = np.zeros((G, D), dtype=np.float32)
    np.add.at(Pm, pinv, R)
    Pm /= np.bincount(pinv, minlength=G)[:, None].clip(min=1).astype(np.float32)
    R -= Pm[pinv]
    del Pm

    d = int(min(d_target, max(N - G - 1, 1), D))
    Gram = (R @ R.T).astype(np.float64)
    lam_r, U = np.linalg.eigh(Gram)
    idx_top = np.argsort(lam_r)[::-1][:d]
    lam_top = np.clip(lam_r[idx_top], 1e-10, None)
    U_f32 = U[:, idx_top].astype(np.float32)
    V_f32 = (R.T @ U_f32) / np.sqrt(lam_top).astype(np.float32)[None, :]
    del R, Gram, U, U_f32

    # Unsupervised within-prompt top-64 principal components (zero label leakage)
    V_top64 = V_f32[:, :min(64, d)].copy()

    Z = (((X_tr_f32 - mx) / sx) @ V_f32).astype(np.float64)

    mu, P, ng = _two_way(Z, y, pinv, G, K_classes=K, iters=4)
    E = Z - mu[y] - P[pinv]
    dof = max(N - G - len(np.unique(y)) + 1, 1)
    Se_raw = (E.T @ E) / dof
    ae = max(float(ledoit_wolf_shrinkage(E, assume_centered=True)), se_floor)
    Se = _shrunk(Se_raw, ae)

    Pc = P - P.mean(0)
    Sp_raw = (Pc.T @ Pc) / max(G - 1, 1) - np.mean(1.0 / ng) * Se_raw
    ap = float(ledoit_wolf_shrinkage(Pc, assume_centered=True))
    tr = max(np.trace(Sp_raw), 1e-8 * np.trace(Se))
    Sp = (1.0 - ap) * Sp_raw + ap * (tr / d) * np.eye(d)

    L = np.linalg.cholesky(Se)
    Li = np.linalg.inv(L)
    M = 0.5 * ((Li @ Sp @ Li.T) + (Li @ Sp @ Li.T).T)
    lam, Q = np.linalg.eigh(M)
    lam = np.clip(lam, 1e-4, None)
    A = Q.T @ Li
    W_f32 = (V_f32 @ A.T.astype(np.float32)).astype(np.float32)
    m_whitened = mu @ A.T

    return {
        "mx": mx, "sx": sx, "W": W_f32, "m": m_whitened, "lam": lam,
        "V_top64": V_top64, "T1": 2.40, "Tg": 2.35, "ae": ae
    }

def transform_lowram(X_f32: np.ndarray, prm: dict, batch_rows: int = 1000) -> np.ndarray:
    """Projects representations into whitened space."""
    N = len(X_f32)
    d = prm["W"].shape[1]
    U_out = np.empty((N, d), dtype=np.float64)
    mx, sx, W = prm["mx"], prm["sx"], prm["W"]
    for s in range(0, N, batch_rows):
        chunk = ((X_f32[s : s + batch_rows] - mx) / sx) @ W
        U_out[s : s + batch_rows] = chunk.astype(np.float64)
    return U_out

def split_oversized_clusters_dist(D: np.ndarray, groups: np.ndarray, max_cluster_size: int = 14) -> np.ndarray:
    """Recursively splits clusters with size > max_cluster_size to prevent bitmask DP overflow."""
    if len(groups) < 2:
        return groups.copy()
    new_groups = np.zeros(len(groups), dtype=np.int64)
    next_id = 1
    for g in np.unique(groups):
        rows = np.flatnonzero(groups == g)
        if len(rows) <= max_cluster_size:
            new_groups[rows] = next_id
            next_id += 1
        else:
            n_sub = int(np.ceil(len(rows) / max_cluster_size))
            D_sub = D[np.ix_(rows, rows)].copy()
            np.fill_diagonal(D_sub, 0.0)
            D_sub = np.clip(0.5 * (D_sub + D_sub.T), 0.0, None)
            d_condensed = squareform(D_sub, checks=False)
            sub_labels = fcluster(linkage(d_condensed, method='average'), t=n_sub, criterion='maxclust')
            for sg in np.unique(sub_labels):
                s_rows = rows[sub_labels == sg]
                for start in range(0, len(s_rows), max_cluster_size):
                    new_groups[s_rows[start : start + max_cluster_size]] = next_id
                    next_id += 1
    return new_groups

@lru_cache(maxsize=4)
def _bitmask_levels(k: int = 18):
    """Precomputes bitmasks partitioned by population count (number of set bits)."""
    masks = np.arange(1 << k, dtype=np.int64)
    counts = np.fromiter((int(x).bit_count() for x in masks), dtype=np.uint8, count=len(masks))
    return tuple(masks[counts == n] for n in range(k + 1))

class ExactAuthorPosterior:
    """Exact shared-prompt Gaussian posteriors under author uniqueness via 2^18 bitmask DP."""
    def __init__(self, m: np.ndarray, lam: np.ndarray):
        self.m = np.asarray(m, dtype=np.float64)
        self.lam = np.asarray(lam, dtype=np.float64)
        self.K, self.d = self.m.shape
        self.levels = _bitmask_levels(self.K)
        self._subset_means = {}

    def single(self, u: np.ndarray, temperature: float = 2.40) -> np.ndarray:
        diff = u[None, :] - self.m
        dist = np.sum((diff * diff) / (1.0 + self.lam)[None, :], axis=1)
        log_p = -0.5 * dist / max(float(temperature), 1e-4)
        return log_p - logsumexp(log_p)

    def group(self, U: np.ndarray, temperature: float = 2.35) -> np.ndarray:
        U = np.asarray(U, dtype=np.float64)
        n = len(U)
        if n <= 1 or n > self.K:
            return np.vstack([self.single(U[i], temperature=temperature) for i in range(n)])

        if n not in self._subset_means:
            if len(self._subset_means) >= 3:
                self._subset_means.clear()
            masks = self.levels[n]
            membership = ((masks[:, None] >> np.arange(self.K)) & 1).astype(np.float32)
            self._subset_means[n] = membership @ self.m.astype(np.float32)

        terminal = self.levels[n]
        gamma = self.lam / (1.0 + n * self.lam)
        residual = U.sum(axis=0, keepdims=True) - self._subset_means[n].astype(np.float64)
        correction = 0.5 * np.einsum("nd,d,nd->n", residual, gamma, residual) / max(float(temperature), 1e-4)
        correction -= correction.max()

        # Passage-specific unary log-likelihoods A[i, author]
        A = -0.5 * np.sum((U[:, None, :] - self.m[None, :, :]) ** 2, axis=2) / max(float(temperature), 1e-4)
        A -= A.max(axis=1, keepdims=True)

        forward = np.full(1 << self.K, -np.inf, dtype=np.float64)
        forward[0] = 0.0
        for count in range(1, n + 1):
            masks = self.levels[count]
            for author in range(self.K):
                bit = 1 << author
                chosen = masks[(masks & bit) != 0]
                forward[chosen] = np.logaddexp(forward[chosen], forward[chosen ^ bit] + A[count - 1, author])

        backward = np.full_like(forward, -np.inf)
        backward[terminal] = correction
        for count in range(n - 1, -1, -1):
            masks = self.levels[count]
            for author in range(self.K):
                bit = 1 << author
                chosen = masks[(masks & bit) == 0]
                backward[chosen] = np.logaddexp(backward[chosen], A[count, author] + backward[chosen | bit])

        result = np.empty((n, self.K), dtype=np.float64)
        for row in range(n):
            masks = self.levels[row]
            for author in range(self.K):
                bit = 1 << author
                chosen = masks[(masks & bit) == 0]
                result[row, author] = logsumexp(forward[chosen] + A[row, author] + backward[chosen | bit]) - backward[0]

        result -= logsumexp(result, axis=1, keepdims=True)
        return result

def scores_exact_dp(U_test: np.ndarray, groups: np.ndarray, prm: dict, D_dist: np.ndarray) -> np.ndarray:
    """Wraps ExactAuthorPosterior with oversized cluster splitting guardrail."""
    N = len(U_test)
    if N == 0:
        return np.zeros((0, K), dtype=np.float64)
    if N == 1:
        solver = ExactAuthorPosterior(prm["m"], prm["lam"])
        return solver.single(U_test[0], temperature=prm.get("T1", 2.40))[None, :]

    safe_groups = split_oversized_clusters_dist(D_dist, groups, max_cluster_size=14)
    solver = ExactAuthorPosterior(prm["m"], prm["lam"])
    T1 = prm.get("T1", 2.40)
    Tg = prm.get("Tg", 2.35)

    lp_dp = np.zeros((N, K), dtype=np.float64)
    for g in np.unique(safe_groups):
        idx_g = np.where(safe_groups == g)[0]
        if len(idx_g) == 1:
            lp_dp[idx_g] = solver.single(U_test[idx_g[0]], temperature=T1)
        else:
            lp_dp[idx_g] = solver.group(U_test[idx_g], temperature=Tg)
    return lp_dp


def save_wpca_params(prm: dict, path: Any):
    """Serializes fitted wPCA Gaussian whitener parameters to safetensors."""
    from pathlib import Path
    import torch
    from safetensors.torch import save_file

    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    sd = {
        "mx": torch.from_numpy(np.asarray(prm["mx"], dtype=np.float32)),
        "sx": torch.from_numpy(np.asarray(prm["sx"], dtype=np.float32)),
        "W": torch.from_numpy(np.asarray(prm["W"], dtype=np.float32)),
        "m": torch.from_numpy(np.asarray(prm["m"], dtype=np.float32)),
        "lam": torch.from_numpy(np.asarray(prm["lam"], dtype=np.float32)),
        "V_top64": torch.from_numpy(np.asarray(prm["V_top64"], dtype=np.float32)),
        "scalars": torch.tensor([float(prm.get("T1", 2.40)), float(prm.get("Tg", 2.35)), float(prm.get("ae", 0.10))], dtype=torch.float32),
    }
    save_file(sd, str(path))


def load_wpca_params(path: Any) -> dict:
    """Loads fitted wPCA parameters from safetensors."""
    from pathlib import Path
    from safetensors.torch import load_file

    sd = load_file(str(path))
    sc = sd["scalars"].cpu().numpy()
    return {
        "mx": sd["mx"].cpu().numpy(),
        "sx": sd["sx"].cpu().numpy(),
        "W": sd["W"].cpu().numpy(),
        "m": sd["m"].cpu().numpy().astype(np.float64),
        "lam": sd["lam"].cpu().numpy().astype(np.float64),
        "V_top64": sd["V_top64"].cpu().numpy(),
        "T1": float(sc[0]),
        "Tg": float(sc[1]),
        "ae": float(sc[2]),
    }


def train_and_save_wpca(
    Z_train: np.ndarray,
    y_train: np.ndarray,
    prompts_train: np.ndarray,
    output_path: Any,
    d_target: int = 256,
    se_floor: float = 0.10,
) -> dict:
    """
    Fits within-prompt whitened PCA (wPCA) on pre-extracted feature matrix and saves to safetensors.
    Zero GPU requirements; executes on CPU in seconds.
    """
    print(f"[wPCA Trainer] Fitting wPCA on shape {Z_train.shape} with {len(np.unique(prompts_train))} prompt groups...")
    prm = fit_shared_gaussian_lowram(
        X_tr_f32=np.asarray(Z_train, dtype=np.float32),
        y=np.asarray(y_train, dtype=int),
        prompts=np.asarray(prompts_train),
        d_target=d_target,
        se_floor=se_floor,
    )
    save_wpca_params(prm, output_path)
    print(f"[wPCA Trainer] Saved fitted wPCA parameters to {output_path}")
    return prm


def save_wpca_bundle(bundle: Dict[str, dict], path: Any):
    """Serializes a dictionary of {model_key: wpca_params} to a single safetensors file."""
    from pathlib import Path
    import torch
    from safetensors.torch import save_file

    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    sd = {}
    for model_key, prm in bundle.items():
        sd[f"{model_key}__mx"] = torch.from_numpy(np.asarray(prm["mx"], dtype=np.float32))
        sd[f"{model_key}__sx"] = torch.from_numpy(np.asarray(prm["sx"], dtype=np.float32))
        sd[f"{model_key}__W"] = torch.from_numpy(np.asarray(prm["W"], dtype=np.float32))
        sd[f"{model_key}__m"] = torch.from_numpy(np.asarray(prm["m"], dtype=np.float32))
        sd[f"{model_key}__lam"] = torch.from_numpy(np.asarray(prm["lam"], dtype=np.float32))
        sd[f"{model_key}__V_top64"] = torch.from_numpy(np.asarray(prm["V_top64"], dtype=np.float32))
        sd[f"{model_key}__scalars"] = torch.tensor(
            [float(prm.get("T1", 2.40)), float(prm.get("Tg", 2.35)), float(prm.get("ae", 0.10))],
            dtype=torch.float32
        )
    save_file(sd, str(path))
    sz_mb = path.stat().st_size / (1024 * 1024)
    print(f"[wPCA] Saved {len(bundle)} models to wPCA bundle at {path} ({sz_mb:.2f} MB)")


def load_wpca_bundle(path: Any) -> Dict[str, dict]:
    """Loads all model wPCA parameters from a single safetensors file."""
    from safetensors.torch import load_file
    sd = load_file(str(path))
    model_keys = set()
    for k in sd.keys():
        if "__" in k:
            model_keys.add(k.split("__")[0])
    bundle = {}
    for mk in sorted(model_keys):
        sc = sd[f"{mk}__scalars"].cpu().numpy()
        bundle[mk] = {
            "mx": sd[f"{mk}__mx"].cpu().numpy(),
            "sx": sd[f"{mk}__sx"].cpu().numpy(),
            "W": sd[f"{mk}__W"].cpu().numpy(),
            "m": sd[f"{mk}__m"].cpu().numpy().astype(np.float64),
            "lam": sd[f"{mk}__lam"].cpu().numpy().astype(np.float64),
            "V_top64": sd[f"{mk}__V_top64"].cpu().numpy(),
            "T1": float(sc[0]),
            "Tg": float(sc[1]),
            "ae": float(sc[2]),
        }
    print(f"[wPCA] Loaded {len(bundle)} models from wPCA bundle at {path}: {list(bundle.keys())}")
    return bundle


In [ ]:
# === MODULE: src/namethatllm/neural_extractor.py ===
"""
Universal Neural Trunk Loader & Feature Extractor
=================================================
Manages multi-GPU sharding (2xT4 GPUs, max 13.5 GiB), layer truncation (keep_layers),
dynamic bfloat16 row-level retry for 14B fp16 overflow rescue, LayerNorm heads,
and bit-exact CPU fp16 snapshot/restore for shared trunks.
"""

import gc
import os
import re
import json
import time
import shutil
from pathlib import Path
from typing import Dict, List, Optional, Tuple, Any

import numpy as np
import torch
import torch.nn as nn
from transformers import AutoConfig, AutoModel, AutoTokenizer
from safetensors.torch import load_file


def infer_adapter_max_layer(ckpt_dir: Path) -> Optional[int]:
    """Returns 1-indexed max transformer layer modified by the LoRA adapter (e.g. 24 or 27)."""
    ckpt_dir = Path(ckpt_dir)
    for rel in (
        "adapter_model.safetensors",
        "adapter/adapter_model.safetensors",
        "adapter_model.bin",
        "adapter/adapter_model.bin",
        "adapter_model.pt",
        "adapter/adapter_model.pt",
    ):
        p = ckpt_dir / rel
        if p.exists():
            try:
                if p.suffix == ".safetensors":
                    sd = load_file(str(p))
                else:
                    sd = torch.load(str(p), map_location="cpu", weights_only=False)
                    if hasattr(sd, "state_dict") and callable(sd.state_dict):
                        sd = sd.state_dict()
                layer_nums = []
                for k in sd.keys():
                    m = re.search(r"layers\.(\d+)\.", str(k))
                    if m:
                        layer_nums.append(int(m.group(1)) + 1)
                if layer_nums:
                    return max(layer_nums)
            except Exception:
                continue
    return None

def in_place_copy(dst: torch.Tensor, src: torch.Tensor):
    """In-place weight copy."""
    dst.data.copy_(src.to(device=dst.device, dtype=dst.dtype))

def _resolve_module_name(k: str, state_dict: dict) -> Optional[str]:
    """Resolves module name across various AutoModel and PEFT naming schemes."""
    raw = k.split(".lora_A")[0]
    candidates = [raw, raw.replace("base_model.model.", "")]
    if "layers." in raw:
        rel = raw[raw.index("layers."):]
        candidates.extend([rel, f"model.{rel}"])
    for c in candidates:
        if c in state_dict and hasattr(state_dict[c], "weight"):
            return c
    return None

def merge_lora(model: nn.Module, adapter_path: Path, sign: float = +1.0) -> bool:
    """
    Atomically merges or unmerges (+1.0 or -1.0) a LoRA adapter into trunk weights.
    Pre-validates all tensor shapes BEFORE mutating any weights so an incompatible
    architecture (e.g. Llama vs Qwen) never partially corrupts the trunk.
    """
    weights_path = adapter_path / "adapter_model.safetensors"
    if not weights_path.exists():
        weights_path = adapter_path / "adapter" / "adapter_model.safetensors"
    if not weights_path.exists():
        return False

    try:
        sd = load_file(str(weights_path))
    except Exception as e:
        print(f"[merge_lora] Could not load {weights_path}: {e}")
        return False

    cfg_p = adapter_path / "adapter_config.json"
    if not cfg_p.exists():
        cfg_p = adapter_path / "adapter" / "adapter_config.json"
    scale = 1.0
    if cfg_p.exists():
        try:
            c = json.load(open(cfg_p))
            scale = float(c.get("lora_alpha", 16)) / float(c.get("r", 16))
        except Exception:
            scale = 1.0

    state_dict = dict(model.named_modules())
    pairs = {}
    for k in sd.keys():
        if "lora_A" in k:
            b_key = k.replace("lora_A", "lora_B")
            if b_key in sd:
                m_name = _resolve_module_name(k, state_dict)
                if m_name is not None:
                    pairs[m_name] = (sd[k], sd[b_key])

    if not pairs:
        return False

    # Pass 1: Pre-validate all shapes before modifying any weight in `model`
    for m_name, (A, B) in pairs.items():
        mod = state_dict.get(m_name)
        if mod is None or not hasattr(mod, "weight"):
            continue
        expected_shape = (B.shape[0], A.shape[1])
        if tuple(mod.weight.shape) != expected_shape:
            print(
                f"[merge_lora] Architecture mismatch on {m_name} in {adapter_path.name}: "
                f"trunk weight {tuple(mod.weight.shape)} != adapter delta {expected_shape}. Aborting merge."
            )
            return False

    # Pass 2: All shapes verified — apply in-place update
    applied = 0
    with torch.no_grad():
        for m_name, (A, B) in pairs.items():
            mod = state_dict.get(m_name)
            if mod is not None and hasattr(mod, "weight"):
                A_d = A.to(mod.weight.device, dtype=torch.float32)
                B_d = B.to(mod.weight.device, dtype=torch.float32)
                delta = (B_d @ A_d) * (scale * sign)
                mod.weight.data.add_(delta.to(mod.weight.dtype))
                applied += 1
    return applied > 0

@torch.no_grad()
def forward_and_pool_batch(model: nn.Module, enc: dict, layers: List[int], norms: nn.ModuleList,
                           dtype: torch.dtype = torch.float16, drop_first_token: bool = True):
    """
    Runs forward pass in `dtype` (float16 on T4).
    CRITICAL: If any target layer overflows fp16 (Inf/NaN), recomputes ONLY the overflowed
    rows in `torch.bfloat16` and pools directly in float32 before restoring `dtype`!
    """
    hs = model(**enc, output_hidden_states=True, use_cache=False).hidden_states

    B = enc["input_ids"].shape[0]
    first_norm_dev = norms[0].weight.device

    # Check if any selected layer has Inf/NaN per row
    bad_mask = torch.zeros(B, dtype=torch.bool, device="cpu")
    for l in layers:
        bad_mask |= ~torch.isfinite(hs[l]).all(dim=(1, 2)).cpu()

    n_bad = int(bad_mask.sum().item())
    m = enc["attention_mask"].unsqueeze(-1).float().to(first_norm_dev)
    if drop_first_token:
        m[:, 0, :] = 0.0
    den = m.sum(dim=1).clamp(min=1.0)

    # Pool finite rows first in float32
    pooled_layers = [torch.zeros((B, model.config.hidden_size), dtype=torch.float32, device=first_norm_dev) for _ in layers]
    good_rows = torch.where(~bad_mask)[0].to(first_norm_dev)
    if len(good_rows) > 0:
        for idx_l, l in enumerate(layers):
            h_good = hs[l][good_rows.to(hs[l].device)].to(first_norm_dev).float()
            pooled_layers[idx_l][good_rows] = (h_good * m[good_rows]).sum(dim=1) / den[good_rows]

    # Rescue overflowed rows in bfloat16 (exact exponent range as float32, NEVER overflows!)
    if n_bad > 0:
        bad_rows = torch.where(bad_mask)[0]
        del hs
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        model.to(torch.bfloat16)
        setattr(model, "_in_bf16", True)
        enc_bad = {k: v[bad_rows] for k, v in enc.items()}
        hs_bad = model(**enc_bad, output_hidden_states=True, use_cache=False).hidden_states
        bad_rows_dev = bad_rows.to(first_norm_dev)
        for idx_l, l in enumerate(layers):
            h_b = hs_bad[l].to(first_norm_dev).float()
            h_b = torch.nan_to_num(h_b, nan=0.0, posinf=1e4, neginf=-1e4)
            pooled_layers[idx_l][bad_rows_dev] = (h_b * m[bad_rows_dev]).sum(dim=1) / den[bad_rows_dev]
        del hs_bad

    out = torch.cat([nm(pooled_layers[idx_l]) for idx_l, nm in enumerate(norms)], dim=-1)
    return out.cpu().numpy(), n_bad

def _flatten_state_dict(obj: Any, prefix: str = "", out: Optional[dict] = None) -> dict:
    if out is None:
        out = {}
    if hasattr(obj, "state_dict") and callable(obj.state_dict):
        obj = obj.state_dict()
    if isinstance(obj, dict):
        for k, v in obj.items():
            new_k = f"{prefix}.{k}" if prefix else str(k)
            if isinstance(v, (dict, list, tuple)) or (hasattr(v, "state_dict") and callable(v.state_dict)):
                _flatten_state_dict(v, new_k, out)
            else:
                out[new_k] = v
                if prefix in ("state_dict", "model", "heads", "head") and str(k) not in out:
                    out[str(k)] = v
    elif isinstance(obj, (list, tuple)):
        for idx, item in enumerate(obj):
            new_k = f"{prefix}.{idx}" if prefix else str(idx)
            _flatten_state_dict(item, new_k, out)
    return out

def load_head_file(path_or_dir: Path) -> dict:
    """Unified loader for classification head state dicts (.safetensors, .pt, .bin, .npz)."""
    p = Path(path_or_dir)
    res = {}
    if p.is_file():
        if p.suffix == ".safetensors":
            res = load_file(str(p))
        elif p.suffix in (".pt", ".bin"):
            raw = torch.load(str(p), map_location="cpu", weights_only=False)
            res = _flatten_state_dict(raw)
        elif p.suffix == ".npz":
            npz = np.load(str(p), allow_pickle=True)
            res = {k: npz[k] for k in npz.files}
        if res:
            print(f"[Head File] {p.name} loaded ({len(res)} keys): {list(res.keys())[:15]}")
        return res
    for fname in ("heads.safetensors", "head.safetensors", "heads.pt", "head.pt", "readouts.npz"):
        cand = p / fname
        if cand.exists():
            return load_head_file(cand)
    return {}

def resolve_head_tensors(heads_dict: dict) -> Tuple[Optional[np.ndarray], Optional[np.ndarray], Optional[np.ndarray], Optional[np.ndarray]]:
    """Resolves plain and centered head weights and biases matching American/British naming and handles transposed weights."""
    W_p, b_p, W_c, b_c = None, None, None, None
    for wk, bk in [
        ("plain.weight", "plain.bias"),
        ("plain_coef", "plain_intercept"),
        ("head.weight", "head.bias"),
        ("classifier.weight", "classifier.bias"),
        ("fc.weight", "fc.bias"),
        ("W_plain", "b_plain"),
        ("weight", "bias"),
        ("W", "b"),
    ]:
        if wk in heads_dict and bk in heads_dict:
            W_p, b_p = heads_dict[wk], heads_dict[bk]
            break
        elif wk in heads_dict and bk not in heads_dict:
            W_p = heads_dict[wk]
            b_p = np.zeros(K, dtype=np.float32)
            break

    for wk, bk in [
        ("centered.weight", "centered.bias"),
        ("centred.weight", "centred.bias"),
        ("center.weight", "center.bias"),
        ("twin.weight", "twin.bias"),
        ("center_coef", "center_intercept"),
        ("centred_coef", "centred_intercept"),
        ("W_center", "b_center"),
        ("W_twin", "b_twin"),
    ]:
        if wk in heads_dict and bk in heads_dict:
            W_c, b_c = heads_dict[wk], heads_dict[bk]
            break
        elif wk in heads_dict and bk not in heads_dict:
            W_c = heads_dict[wk]
            b_c = np.zeros(K, dtype=np.float32)
            break

    # Suffix / Keyword Matching fallback if W_p or W_c still None
    if W_p is None or W_c is None:
        for k, v in heads_dict.items():
            k_lower = str(k).lower()
            shape = getattr(v, "shape", None)
            if shape is not None and len(shape) == 2 and (shape[0] == K or shape[1] == K):
                # Check for center / twin
                if any(tag in k_lower for tag in ("center", "centred", "twin")):
                    if W_c is None:
                        W_c = v
                        base_prefix = k.rsplit(".", 1)[0] if "." in k else k.rsplit("_", 1)[0]
                        for bk in [f"{base_prefix}.bias", f"{base_prefix}_bias", f"{base_prefix}.intercept", f"{base_prefix}_intercept", "bias", "centered.bias", "b"]:
                            if bk in heads_dict and getattr(heads_dict[bk], "shape", None) == (K,):
                                b_c = heads_dict[bk]
                                break
                        if b_c is None:
                            b_c = np.zeros(K, dtype=np.float32)
                # Check for plain / head / clf / classifier
                elif any(tag in k_lower for tag in ("plain", "head", "clf", "classifier")):
                    if W_p is None:
                        W_p = v
                        base_prefix = k.rsplit(".", 1)[0] if "." in k else k.rsplit("_", 1)[0]
                        for bk in [f"{base_prefix}.bias", f"{base_prefix}_bias", f"{base_prefix}.intercept", f"{base_prefix}_intercept", "bias", "plain.bias", "b"]:
                            if bk in heads_dict and getattr(heads_dict[bk], "shape", None) == (K,):
                                b_p = heads_dict[bk]
                                break
                        if b_p is None:
                            b_p = np.zeros(K, dtype=np.float32)

    if W_p is not None and hasattr(W_p, "cpu"):
        W_p = W_p.cpu().numpy()
    if b_p is not None and hasattr(b_p, "cpu"):
        b_p = b_p.cpu().numpy()
    if W_c is not None and hasattr(W_c, "cpu"):
        W_c = W_c.cpu().numpy()
    if b_c is not None and hasattr(b_c, "cpu"):
        b_c = b_c.cpu().numpy()

    # Transpose Guard: if shape is (D, K) where K == 18, transpose to canonical (18, D)
    if W_p is not None and W_p.ndim == 2:
        if W_p.shape[1] == K and W_p.shape[0] != K:
            W_p = W_p.T
    if W_c is not None and W_c.ndim == 2:
        if W_c.shape[1] == K and W_c.shape[0] != K:
            W_c = W_c.T

    return W_p, b_p, W_c, b_c

def compute_centered_logits(Z_test: np.ndarray, groups: np.ndarray, W_c: np.ndarray, b_c: np.ndarray, src_labels: list) -> Tuple[np.ndarray, np.ndarray]:
    """Computes prompt-topic centered twin logits for valid multi-sample clusters."""
    N = len(Z_test)
    out_lp = np.zeros((N, K), dtype=np.float64)
    if N < 2 or W_c is None:
        return out_lp, np.zeros(N, dtype=bool)
    _, inv_g = np.unique(np.asarray(groups, dtype=str), return_inverse=True)
    safe_g = inv_g + 1
    sz = np.bincount(safe_g)[safe_g]
    valid = (sz >= 2) & (sz <= 18)
    if not valid.any():
        return out_lp, valid
    Zc = Z_test.copy()
    for g in np.unique(safe_g[valid]):
        mask = (safe_g == g)
        Zc[mask] -= Zc[mask].mean(axis=0, keepdims=True)
    raw_c = Zc[valid] @ W_c.T + b_c
    out_lp[valid] = log_softmax(align_cols(raw_c, src_labels))
    return out_lp, valid

class UniversalLoRATrunkLoader:
    def __init__(self, base_dir: Path, keep_layers: int, layers: List[int], max_len: int = 447):
        if base_dir is not None and not (base_dir / "config.json").exists():
            for sub in base_dir.rglob("config.json"):
                base_dir = sub.parent
                break
        self.base_dir = base_dir
        self.keep_layers = keep_layers
        self.layers = layers
        self.max_len = max_len
        self.total_rescued = 0
        self.tok = AutoTokenizer.from_pretrained(str(base_dir), trust_remote_code=True)
        if self.tok.pad_token is None:
            self.tok.pad_token = self.tok.eos_token or "<|endoftext|>"

        cfg = AutoConfig.from_pretrained(str(base_dir), trust_remote_code=True)
        cfg.num_hidden_layers = keep_layers
        if getattr(cfg, "layer_types", None) is not None:
            cfg.layer_types = cfg.layer_types[:keep_layers]
        cfg.use_cache = False

        n_gpus = torch.cuda.device_count()
        if n_gpus >= 2:
            max_memory = {0: "13.5GiB", 1: "13.5GiB"}
            self.model = AutoModel.from_pretrained(
                str(base_dir), config=cfg, device_map="auto", max_memory=max_memory,
                torch_dtype=torch.float16, trust_remote_code=True, low_cpu_mem_usage=True
            )
        elif n_gpus == 1:
            self.model = AutoModel.from_pretrained(
                str(base_dir), config=cfg, device_map="auto",
                torch_dtype=torch.float16, trust_remote_code=True, low_cpu_mem_usage=True
            )
        else:
            self.model = AutoModel.from_pretrained(str(base_dir), config=cfg, torch_dtype=torch.float32, low_cpu_mem_usage=True)

        # Preserve original final trunk norm before optionally disabling
        self.orig_final_norm = getattr(self.model, "norm", None) or getattr(getattr(self.model, "model", None), "norm", None)

        # Default: disable final trunk norm for two-stage models
        if hasattr(self.model, "norm"):
            self.model.norm = nn.Identity()
        elif hasattr(self.model, "model") and hasattr(self.model.model, "norm"):
            self.model.model.norm = nn.Identity()

        self.model.eval()
        self.device_norms = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

        # nn.LayerNorm with both weight and bias matching heads.safetensors
        hidden_dim = self.model.config.hidden_size
        self.norms = nn.ModuleList([
            nn.LayerNorm(hidden_dim)
            for _ in layers
        ]).to(self.device_norms).float()

        self.norms.eval()
        self.first_dev = next(self.model.parameters()).device

    def set_final_norm(self, enabled: bool):
        """Restores original trunk norm when enabled=True, disables with nn.Identity() when False."""
        target = self.orig_final_norm if enabled and self.orig_final_norm is not None else nn.Identity()
        if hasattr(self.model, "norm"):
            self.model.norm = target
        elif hasattr(self.model, "model") and hasattr(self.model.model, "norm"):
            self.model.model.norm = target

    def snapshot_weights(self, adapter_path: Path) -> dict:
        """Saves CPU fp16 copies of trunk modules modified by adapter."""
        weights_path = adapter_path / "adapter_model.safetensors"
        if not weights_path.exists():
            weights_path = adapter_path / "adapter" / "adapter_model.safetensors"
        if not weights_path.exists():
            return {}
        sd = load_file(str(weights_path))
        state_dict = dict(self.model.named_modules())
        backup = {}
        for k in sd.keys():
            if "lora_A" in k:
                m_name = _resolve_module_name(k, state_dict)
                if m_name is not None and m_name in state_dict and hasattr(state_dict[m_name], "weight"):
                    backup[m_name] = state_dict[m_name].weight.detach().cpu().half().clone()
        return backup

    def restore_weights(self, backup: dict):
        """Restores base trunk weights bit-for-bit without FP16 subtraction drift."""
        state_dict = dict(self.model.named_modules())
        with torch.no_grad():
            for m_name, orig in backup.items():
                if m_name in state_dict and hasattr(state_dict[m_name], "weight"):
                    state_dict[m_name].weight.data.copy_(orig.to(state_dict[m_name].weight.device))

    def set_active_layers(self, layers: List[int], max_len: Optional[int] = None):
        """Updates active pooled layers and instantiates LayerNorms matching the checkpoint."""
        self.layers = list(layers)
        if max_len is not None:
            self.max_len = max_len
        hidden_dim = self.model.config.hidden_size
        self.norms = nn.ModuleList([
            nn.LayerNorm(hidden_dim)
            for _ in self.layers
        ]).to(self.device_norms).float()
        self.norms.eval()

    def load_norm_weights(self, heads_sd: dict):
        """Loads LayerNorm weights and biases from head state dict after resetting to identity."""
        with torch.no_grad():
            # First reset every ln in self.norms to identity so weights/biases from previous checkpoint never bleed
            for ln in self.norms:
                ln.weight.data.fill_(1.0)
                if hasattr(ln, "bias") and ln.bias is not None:
                    ln.bias.data.zero_()

            # Support all norm key prefixes: norms.{i}, layer_norms.{i}, ln.{i}, norm.{i}
            for i, ln in enumerate(self.norms):
                for pfx in ("norms", "layer_norms", "ln", "norm"):
                    w_k = f"{pfx}.{i}.weight"
                    b_k = f"{pfx}.{i}.bias"
                    if w_k in heads_sd:
                        ln.weight.copy_(heads_sd[w_k].to(ln.weight.device, ln.weight.dtype))
                    elif f"{pfx}.{i}" in heads_sd and hasattr(heads_sd[f"{pfx}.{i}"], "shape") and heads_sd[f"{pfx}.{i}"].shape == ln.weight.shape:
                        ln.weight.copy_(heads_sd[f"{pfx}.{i}"].to(ln.weight.device, ln.weight.dtype))

                    if b_k in heads_sd and hasattr(ln, "bias") and ln.bias is not None:
                        ln.bias.copy_(heads_sd[b_k].to(ln.bias.device, ln.bias.dtype))

            # Check for stacked 2D tensor: norms_weight or norms.weight of shape (len(self.layers), H)
            for stk in ("norms_weight", "norms.weight", "layer_norms.weight", "ln.weight"):
                if stk in heads_sd and hasattr(heads_sd[stk], "ndim") and heads_sd[stk].ndim == 2:
                    stk_t = heads_sd[stk]
                    if stk_t.shape[0] == len(self.norms):
                        for i, ln in enumerate(self.norms):
                            ln.weight.copy_(stk_t[i].to(ln.weight.device, ln.weight.dtype))

    def _extract_batch_ids(self, id_list: List[List[int]], batch_size: int = 8) -> np.ndarray:
        order = np.argsort([len(x) for x in id_list], kind="stable")
        total_dim = len(self.layers) * self.model.config.hidden_size
        Z = np.zeros((len(id_list), total_dim), dtype=np.float32)

        for s in range(0, len(order), batch_size):
            idx = order[s : s + batch_size]
            enc = self.tok.pad({"input_ids": [id_list[i] for i in idx]}, padding=True, return_tensors="pt")
            enc = {k: v.to(self.first_dev) for k, v in enc.items()}
            out_np, n_bad = forward_and_pool_batch(self.model, enc, self.layers, self.norms, torch.float16, drop_first_token=True)
            self.total_rescued += n_bad
            Z[idx] = out_np

        if getattr(self.model, "_in_bf16", False):
            self.model.to(torch.float16)
            self.model._in_bf16 = False
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        return Z

    @torch.no_grad()
    def extract_exact(self, texts: List[str], batch_size: int = 8, end_id: Optional[int] = None) -> np.ndarray:
        t0 = time.time()
        self.total_rescued = 0
        raw_ids = self.tok(list(texts), add_special_tokens=False, truncation=True, max_length=self.max_len)["input_ids"]
        eff_end = end_id if end_id is not None else getattr(self.tok, "eos_token_id", 151643)
        if eff_end is not None:
            ids = [x + [eff_end] for x in raw_ids]
        else:
            ids = raw_ids
        Z = self._extract_batch_ids(ids, batch_size=batch_size)
        print(f"[Extractor] Extracted {len(texts)} samples in {time.time() - t0:.2f}s (Total BF16 Rescued Rows: {self.total_rescued}).")
        return Z

    def free_memory(self):
        del self.model, self.norms
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

def read_checkpoint_meta(ckpt_dir: Path, default_keep: int, default_layers: List[int], default_max_len: int = 447) -> dict:
    """Reads keep, layers, max_len, end_id, and labels dynamically from meta.json or infers from adapter weights."""
    ckpt_dir = Path(ckpt_dir)
    meta_p = ckpt_dir / "meta.json"
    labels = list(LABELS)
    keep = default_keep
    layers = list(default_layers)
    max_len = default_max_len
    end_id = None
    if meta_p.exists():
        try:
            m = json.load(open(meta_p))
            labels = m.get("labels", m.get("classes", labels))
            cfg = m.get("cfg", m)
            keep = int(cfg.get("keep", keep))
            layers = list(cfg.get("layers", layers))
            max_len = int(cfg.get("max_len", max_len))
            for ek in ("end_token_id", "eos_id", "eos_token_id", "end_token"):
                if ek in cfg and isinstance(cfg[ek], int):
                    end_id = int(cfg[ek])
                    break
        except Exception:
            pass
    else:
        # Infer exact layer depth from adapter weights (e.g. l24 vs l27)
        max_l = infer_adapter_max_layer(ckpt_dir)
        if max_l == 24:
            keep = 24
            layers = [2, 4, 6, 8, 12, 16, 20, 24]
        elif max_l == 27:
            keep = 27
            layers = [2, 4, 6, 9, 13, 17, 22, 27]
        elif max_l == 32:
            keep = 32
            layers = [3, 5, 8, 11, 16, 21, 27, 32]
        elif max_l is not None:
            keep = max_l
        else:
            dir_str = str(ckpt_dir).lower().replace("\\", "/")
            if any(x in dir_str for x in ("l24", "q14v5", "ts_q3b14", "ts_q3b8", "q8v5", "ll8v5", "ts_llama8", "l24x")):
                keep = 24
                layers = [2, 4, 6, 8, 12, 16, 20, 24]

    dir_str = str(ckpt_dir).lower().replace("\\", "/")
    is_dhruv = any(x in dir_str for x in ("l24", "q14v5", "ts_q3b14", "ts_q3b8", "q8v5", "ll8v5", "ts_llama8", "l24x"))
    no_final_norm = False if is_dhruv else True
    if is_dhruv:
        max_len = 384
        end_id = None
        labels = sorted(LABELS)

    return {"keep": keep, "layers": layers, "max_len": max_len, "end_id": end_id, "no_final_norm": no_final_norm, "labels": labels}

def extract_deberta_checkpoint(
    ckpt_dir: Path,
    base_deb_dir: Optional[Path],
    texts_test: List[str],
    texts_train: List[str],
    y_train: np.ndarray,
    max_len: int = 512,
    batch_size: int = 8
) -> Tuple[np.ndarray, np.ndarray, Optional[np.ndarray], Optional[np.ndarray], Optional[np.ndarray], Optional[np.ndarray], list]:
    """
    Dedicated feature extractor for DeBERTa-v3-large sequence classification checkpoints.
    Supports both single-layer encoder (Layer 24 float32 last_hidden_state) and multi-layer heads.
    """
    dev = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

    heads_sd = load_head_file(ckpt_dir)
    W_p, b_p, W_c, b_c = resolve_head_tensors(heads_sd)

    # Load tokenizer from encoder directory if available, matching exact training setup
    if (ckpt_dir / "encoder").exists():
        tok_dir = str(ckpt_dir / "encoder")
    elif base_deb_dir and (base_deb_dir / "config.json").exists():
        tok_dir = str(base_deb_dir)
    else:
        tok_dir = str(ckpt_dir)
    tok = AutoTokenizer.from_pretrained(tok_dir, use_fast=True)

    meta = read_checkpoint_meta(ckpt_dir, default_keep=24, default_layers=[3, 6, 9, 12, 15, 18, 21, 24], default_max_len=max_len)
    ckpt_labels = meta["labels"]
    layers = meta["layers"]
    max_len = meta["max_len"]

    if (ckpt_dir / "encoder" / "config.json").exists():
        model_src = ckpt_dir / "encoder"
    elif (ckpt_dir / "config.json").exists() and ((ckpt_dir / "model.safetensors").exists() or (ckpt_dir / "pytorch_model.bin").exists()):
        model_src = ckpt_dir
    elif base_deb_dir and (base_deb_dir / "config.json").exists():
        model_src = base_deb_dir
    else:
        model_src = ckpt_dir

    stage_dir = Path("/kaggle/temp/deberta_p2_stage")
    staged = False
    load_src = model_src
    if Path("/kaggle/temp").exists() and os.access("/kaggle/temp", os.W_OK):
        try:
            if stage_dir.exists():
                shutil.rmtree(stage_dir, ignore_errors=True)
            shutil.copytree(model_src, stage_dir)
            load_src = stage_dir
            staged = True
        except Exception:
            load_src = model_src

    # CRITICAL: Always use torch.float32 for DeBERTa-v3 to avoid FP16 overflow!
    try:
        model = AutoModel.from_pretrained(str(load_src), torch_dtype=torch.float32).to(dev)
    finally:
        if staged and stage_dir.exists():
            shutil.rmtree(stage_dir, ignore_errors=True)

    model.eval()

    hidden_dim = model.config.hidden_size
    has_norm_keys = any(k.startswith("norms.") and k.endswith(".weight") for k in heads_sd)
    is_single_last_layer = (W_p is not None and W_p.shape[1] == hidden_dim and not has_norm_keys)

    if not is_single_last_layer:
        if W_p is not None and W_p.shape[1] > 0:
            expected_layers = W_p.shape[1] // hidden_dim
            if len(layers) != expected_layers:
                norm_keys = [k for k in heads_sd if k.startswith("norms.") and k.endswith(".weight")]
                if len(norm_keys) == expected_layers:
                    layers = layers[-expected_layers:] if len(layers) > expected_layers else list(range(expected_layers))
                elif len(layers) > expected_layers:
                    layers = layers[-expected_layers:]

        norms = nn.ModuleList([nn.LayerNorm(hidden_dim) for _ in layers]).to(dev).float()
        with torch.no_grad():
            for i, ln in enumerate(norms):
                w_k = f"norms.{i}.weight"
                b_k = f"norms.{i}.bias"
                if w_k in heads_sd:
                    ln.weight.copy_(heads_sd[w_k].to(dev, ln.weight.dtype))
                if b_k in heads_sd:
                    ln.bias.copy_(heads_sd[b_k].to(dev, ln.bias.dtype))
        norms.eval()
    else:
        norms = None

    def pool_texts(txt_list: List[str], drop_first: bool = False) -> np.ndarray:
        out_dim = hidden_dim if is_single_last_layer else (len(layers) * hidden_dim)
        if len(txt_list) == 0:
            return np.zeros((0, out_dim), dtype=np.float32)
        order = np.argsort([len(t) for t in txt_list], kind="stable")
        all_vecs = np.zeros((len(txt_list), out_dim), dtype=np.float32)
        for s in range(0, len(order), batch_size):
            idx = order[s : s + batch_size]
            chunk = [txt_list[i] for i in idx]
            enc = tok(chunk, padding=True, truncation=True, max_length=max_len, return_tensors="pt")
            enc = {k: v.to(dev) for k, v in enc.items()}
            with torch.no_grad():
                if is_single_last_layer:
                    out_hs = model(**enc).last_hidden_state
                    m = enc["attention_mask"].unsqueeze(-1).float()
                    if drop_first:
                        m[:, 0, :] = 0.0
                    den = m.sum(dim=1).clamp(min=1.0)
                    pooled = (torch.nan_to_num(out_hs.float(), nan=0.0, posinf=1e4, neginf=-1e4) * m).sum(dim=1) / den
                    all_vecs[idx] = pooled.cpu().numpy()
                else:
                    hs = model(**enc, output_hidden_states=True).hidden_states
                    m = enc["attention_mask"].unsqueeze(-1).float()
                    if drop_first:
                        m[:, 0, :] = 0.0
                    den = m.sum(dim=1).clamp(min=1.0)
                    pooled = [((torch.nan_to_num(hs[l].float(), nan=0.0, posinf=1e4, neginf=-1e4) * m).sum(dim=1) / den) for l in layers]
                    out = torch.cat([norms[i](pooled[i]) for i in range(len(layers))], dim=-1)
                    all_vecs[idx] = out.cpu().numpy()
        return all_vecs

    # Probe drop_first_token on train.csv[:180]
    probe_n = min(180, len(texts_train))
    best_drop = False if is_single_last_layer else True
    if probe_n > 0 and W_p is not None:
        best_acc = -1.0
        for cand_drop in ([False, True] if is_single_last_layer else [True, False]):
            z_pr = pool_texts(texts_train[:probe_n], drop_first=cand_drop)
            preds = align_cols(z_pr @ W_p.T + b_p, ckpt_labels).argmax(axis=1)
            acc = float((preds == y_train[:probe_n]).mean())
            if acc > best_acc:
                best_acc = acc
                best_drop = cand_drop
        print(f"[DeBERTa Extractor] Probed drop_first={best_drop} (accuracy = {best_acc:.4f})")

    out_dim = hidden_dim if is_single_last_layer else (len(layers) * hidden_dim)
    Z_test = pool_texts(texts_test, drop_first=best_drop)
    Z_train = pool_texts(texts_train, drop_first=best_drop) if len(texts_train) > 0 else np.zeros((0, out_dim), dtype=np.float32)

    del model
    if norms is not None:
        del norms
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return Z_test, Z_train, W_p, b_p, W_c, b_c, ckpt_labels


In [ ]:
# === MODULE: src/namethatllm/postprocess.py ===
"""
Postprocessing, Fusion & Calibration Engine
===========================================
This module handles all post-ensemble refinement stages:
1. Two-tier convex mixture (85% Main Stack + 15% Complementary Stack)
2. Mass-preserving Specialist Neural Gates (Claude head, Gemini LoRA, GLM LoRA, GPT LoRA)
3. Confidence-aware Within-Family Spectral Gates (dynamic margin tie-breakers)
4. Unified Handwriting & Orthographic Shield (discrete Bayesian tokenization rules + continuous density)
5. Cavity Mean-Field Cluster Uniqueness Repulsion (soft at-most-once constraint on prompt clusters)
6. Multiplicative Sinkhorn-Knopp Column Marginal Balancing (global class prior equalization)
7. Hierarchical Beta-Bernoulli TypoBayes Postprocessor
8. Final Kaggle submission generation (submission.parquet and submission.csv)
"""

import re
from pathlib import Path

import numpy as np
import pandas as pd


# =============================================================================
# 1. WITHIN-FAMILY CONFIDENCE GATES (SPECTRAL / TIE-BREAKER HEURISTICS)
# =============================================================================


def apply_all_family_confidence_gates(
    P_mix: np.ndarray,
    family_gates: dict[str, np.ndarray] | None,
    max_strength: float = 0.20,
    margin_thr: float = 0.45,
) -> np.ndarray:
    """
    Confidence-aware within-family and cross-family specialist reranker.

    Concept:
    --------
    When an ensemble predicts probabilities across 18 classes, sister models within
    the same family (e.g., Gemini 3.7 vs 3.8, or GLM-5.3 vs GLM-5.3-flash) can have
    extremely close probabilities (low margin). This function uses auxiliary gate
    logits (from spectral/TF-IDF classifiers) to break ties in low-margin cases.

    Key Invariant - Mass Conservation:
    ----------------------------------
    The total probability assigned to the family is strictly preserved!
    Only the internal relative distribution among the sisters is adjusted.
    Probabilities for classes outside the family are completely untouched.

    Dynamic Confidence Gating:
    --------------------------
    - Margin = (Top-1 prob within family) - (Top-2 prob within family).
    - If margin >= margin_thr: Model is already confident; gate strength = 0 (no intervention).
    - If margin < margin_thr: Low confidence; gate strength scales up to max_strength.

    Parameters:
    -----------
    P_mix : np.ndarray of shape (N, 18)
        Current blended prediction probabilities (summing to 1.0 per row).
    family_gates : dict mapping group_name -> log_probabilities of shape (N, K_group)
        Pre-computed log-probabilities from the auxiliary family classifiers.
    max_strength : float
        Maximum interpolation weight for the gate (0.0 to 1.0). If <= 0, gates are bypassed.
    margin_thr : float
        Confidence margin threshold above which the gate does not intervene.

    Returns:
    --------
    P_out : np.ndarray of shape (N, 18)
        Calibrated probabilities with sister-class confusions resolved.
    """
    # Early exit: If gates are disabled or weight is zero, return immediately
    if not family_gates or float(max_strength) <= 0.0:
        return P_mix

    P_out = P_mix.astype(np.float64).copy()

    for fam_name, gate_lp in family_gates.items():
        # Identify the column indices belonging to this family/specialist group
        cols = SPECIALIST_GROUPS.get(fam_name, FAMILIES.get(fam_name))
        if cols is None:
            continue
        if gate_lp is None or gate_lp.shape != (len(P_out), len(cols)):
            continue

        # Determine which samples are eligible for this gate:
        # Cross-family triads only apply when top-1 prediction is in the triad.
        # Single-family gates apply across all rows where that family has mass.
        if fam_name in ("frontier_and_fable", "kimi_gpt55_qwen", "glm_kimi_triad"):
            row_mask = np.isin(P_out.argmax(axis=1), cols)
            if not row_mask.any():
                continue
        elif fam_name == "gemini_37_38_pair":
            row_mask = np.isin(P_out.argmax(axis=1), FAMILIES.get("gemini", [5, 6, 7, 8]))
            if not row_mask.any():
                continue
        else:
            row_mask = np.ones(len(P_out), dtype=bool)

        # Hard-sister groups (e.g. Gemini 3.7 vs 3.8, GLM) use wider margin threshold
        eff_margin_thr = 0.65 if fam_name in ("gemini_37_38_pair", "glm", "glm_kimi_triad") else margin_thr
        eff_max_strength = (
            max(float(max_strength), 0.35) if fam_name in ("gemini_37_38_pair", "glm", "glm_kimi_triad") else float(max_strength)
        )

        # Step 1: Extract sub-probabilities and preserve total marginal mass
        sub_p = np.clip(P_out[np.ix_(row_mask, cols)], 1e-12, 1.0)
        tot_mass = sub_p.sum(axis=1, keepdims=True)
        base_rel_p = sub_p / tot_mass  # Relative distribution within the family (sums to 1.0)

        # Step 2: Compute margin between top-1 and top-2 candidates within family
        sp = np.sort(base_rel_p, axis=1)
        margin = sp[:, -1] - sp[:, -2]

        # Step 3: Compute dynamic intervention strength (higher when margin is small)
        strength = np.clip((eff_margin_thr - margin) / max(eff_margin_thr, 1e-6), 0.0, 1.0)[:, None] * eff_max_strength

        # Step 4: Blend base probabilities with gate probabilities in log-space
        base_lp = np.log(np.clip(base_rel_p, 1e-12, 1.0))
        gate_p = softmax(gate_lp[row_mask], axis=1)
        gate_lp_norm = np.log(np.clip(gate_p, 1e-12, 1.0))
        mixed_rel_p = softmax((1.0 - strength) * base_lp + strength * gate_lp_norm, axis=1)

        # Step 5: Restore original total family mass, leaving external classes unaffected
        P_out[np.ix_(row_mask, cols)] = tot_mass * mixed_rel_p

    # Final safety re-normalization to guarantee row sums == 1.0
    return P_out / P_out.sum(axis=1, keepdims=True).clip(min=1e-12)


# =============================================================================
# 2. PROMPT-CLUSTER UNIQUENESS REPULSION (CAVITY MEAN-FIELD)
# =============================================================================


def apply_cluster_uniqueness_repulsion(
    P_mix: np.ndarray,
    groups: np.ndarray | None,
    strength: float = 0.35,
    iters: int = 3,
) -> np.ndarray:
    """
    Enforces soft at-most-once author uniqueness within test prompt clusters.

    Domain Insight:
    ---------------
    In LLM benchmarking, multiple completions are generated by presenting the
    EXACT SAME PROMPT to different LLM models. Hence, when two or more test passages
    share an identical prompt (forming a 'prompt cluster'), competition rules imply
    that each completion came from a DIFFERENT model (at most 1 completion per model).

    Mathematical Formulation (Cavity Mean-Field):
    ---------------------------------------------
    For each text i in cluster g, the 'cavity field' for class k is the probability
    that no other text j in cluster g (j != i) was generated by class k:
        log P(other != k) = sum_{j in g, j != i} log(1 - q_{j, k})

    We update the unnormalized log-posterior:
        L_{i, k} = log(P0_{i, k}) + strength * sum_{j != i} log(1 - q_{j, k})
    where P0 is the original prediction prior, and q is the current cluster belief.
    Softmax is applied over k, iterating 3 times to reach a self-consistent equilibrium.

    Parameters:
    -----------
    P_mix : np.ndarray of shape (N, 18)
        Input prediction probabilities.
    groups : np.ndarray of shape (N,)
        Cluster IDs assigned to each test sample (e.g. from TF-IDF prompt clustering).
    strength : float
        Repulsion strength factor (typically 0.35).
    iters : int
        Number of mean-field coordinate ascent iterations (typically 3).

    Returns:
    --------
    q : np.ndarray of shape (N, 18)
        Repulsed probabilities penalizing multiple identical predictions in one prompt cluster.
    """
    N = len(P_mix)
    # Require at least 18 rows and valid cluster assignments
    if N < 18 or groups is None or len(groups) != N or strength <= 0.0:
        return P_mix

    # Map cluster identifiers to dense integer labels 1, 2, ...
    _, inv_g = np.unique(np.asarray(groups, dtype=str), return_inverse=True)
    safe_g = inv_g + 1
    sz = np.bincount(safe_g)[safe_g]

    # Only operate on multi-sample clusters between 2 and 18 texts (size <= 18 since there are 18 classes)
    valid_g = np.unique(safe_g[(sz >= 2) & (sz <= 18)])
    if len(valid_g) == 0:
        return P_mix

    log_P0 = np.log(np.clip(P_mix.astype(np.float64), 1e-12, 1.0))
    q = P_mix.astype(np.float64).copy()

    # Mean-field relaxation loop
    for _ in range(iters):
        L = log_P0.copy()
        for g in valid_g:
            m = safe_g == g
            q_g = q[m]  # Shape: (cluster_size, 18)

            # lo[j, k] = log(1 - q_{j, k}): log-probability that sample j did NOT pick class k
            lo = np.log(np.clip(1.0 - q_g, 1e-4, 1.0))

            # Cavity sum: sum of lo over all samples in cluster minus sample i's own term
            # Vectorized identity: sum_{j} lo[j] - lo[i] equals sum_{j != i} lo[j]
            cavity_penalty = lo.sum(axis=0, keepdims=True) - lo
            L[m] += float(strength) * cavity_penalty

        # Re-normalize across the 18 classes for next iteration
        q = softmax(L, axis=1)

    return q


# =============================================================================
# 3. TWO-TIER CONVEX ENSEMBLE BLENDING
# =============================================================================


def blend_main_and_complementary(
    LOGIT_MAIN: np.ndarray, COMP_LIST: list[tuple[float, np.ndarray]], main_ratio: float = 0.85, comp_ratio: float = 0.15
) -> tuple[np.ndarray, np.ndarray]:
    """
    Blends the Main Stack and Complementary Stack in probability space.

    Architecture:
    -------------
    - Main Stack (85% weight): High-capacity deep readers (Qwen3-14B Part 2,
      in-house champion LoRA, etc.) providing top semantic representations.
    - Complementary Stack (15% weight): Fast, diverse, non-neural or shallow readers
      (Style SVC 4-view, Lexical 5-view, DeBERTa, 8B patches) providing orthographic
      and stylistic diversity that regularizes neural hallucinations.

    Formula:
    --------
        R_main = softmax(LOGIT_MAIN)
        P_comp = sum(w_i * P_i) / sum(w_i)
        P_mix  = main_ratio * R_main + comp_ratio * P_comp

    Parameters:
    -----------
    LOGIT_MAIN : np.ndarray of shape (N, 18)
        Raw logit sum from the primary deep models.
    COMP_LIST : list of tuples (weight, probability_array)
        List of complementary models and their assigned ensemble weights.
    main_ratio : float
        Convex weight for Main Stack (default 0.85).
    comp_ratio : float
        Convex weight for Complementary Stack (default 0.15).

    Returns:
    --------
    P_mix : np.ndarray of shape (N, 18)
        Blended ensemble probabilities.
    R_main : np.ndarray of shape (N, 18)
        Main stack probabilities alone (used for routing/gating decisions).
    """
    R_main = softmax(LOGIT_MAIN, axis=1)
    if len(COMP_LIST) > 0:
        comp_weight_total = sum(w for w, _ in COMP_LIST)
        P_comp = sum(w * P_c for w, P_c in COMP_LIST) / max(1e-8, comp_weight_total)
        P_mix = main_ratio * R_main + comp_ratio * P_comp
    else:
        P_mix = R_main
    return P_mix, R_main


# =============================================================================
# 4. NEURAL SPECIALIST GATES (CLAUDE HEAD & FAMILY LORAS)
# =============================================================================


def apply_claude_specialist_gate(
    P_mix: np.ndarray,
    R_main: np.ndarray,
    claude_spec_logits: np.ndarray | None,
    claude_classes: list | None = None,
    labels: list = LABELS,
    claude_cols: list = FAMILIES["claude"],
    gate_weight: float = 0.40,
    weight: float | None = None,
) -> np.ndarray:
    """
    Refines passages predicted in the Claude family using a specialist head
    while strictly conserving total Claude family probability mass.
    """
    if claude_spec_logits is None:
        return P_mix
    if weight is not None:
        gate_weight = weight
    if float(gate_weight) <= 0.0:
        return P_mix

    P_out = P_mix.copy()
    P_claude_head = softmax(claude_spec_logits, axis=-1)

    if claude_classes is not None:
        src_c = [x.decode("utf-8") if isinstance(x, (bytes, np.bytes_)) else str(x) for x in claude_classes]
        if all(labels[c] in src_c for c in claude_cols):
            c_idx = [src_c.index(labels[c]) for c in claude_cols]
            P_claude_head = P_claude_head[:, c_idx]

    fam_mass = P_out[:, claude_cols].sum(axis=1)
    claude_mask = np.isin(R_main.argmax(axis=1), claude_cols) | (fam_mass >= 0.10)
    if claude_mask.any():
        for idx in np.flatnonzero(claude_mask):
            orig_mass = float(P_out[idx, claude_cols].sum())
            if orig_mass > 1e-6:
                gw_eff = float(gate_weight) * min(orig_mass / 0.50, 1.0)
                main_norm = P_out[idx, claude_cols] / orig_mass
                fused_fam = (1.0 - gw_eff) * main_norm + gw_eff * P_claude_head[idx]
                P_out[idx, claude_cols] = fused_fam * orig_mass

    return P_out


def apply_family_specialist_lora_gate(
    P_mix: np.ndarray,
    R_main: np.ndarray,
    spec_logits: np.ndarray | None,
    spec_classes: list | None = None,
    labels: list = LABELS,
    family_cols: list = FAMILIES["gemini"],
    gate_weight: float = 0.40,
) -> np.ndarray:
    """
    Refines intra-family predictions using fine-tuned Qwen3-14B Family-Specialist LoRAs
    while strictly preserving 100% of total marginal family probability mass.
    """
    if spec_logits is None or len(spec_logits) != len(P_mix) or float(gate_weight) <= 0.0:
        return P_mix

    P_out = P_mix.astype(np.float64).copy()
    spec_arr = np.asarray(spec_logits, dtype=np.float64)
    if spec_arr.ndim != 2 or spec_arr.shape[1] != len(family_cols):
        return P_mix

    P_spec = softmax(spec_arr, axis=-1)

    if spec_classes is not None and len(spec_classes) == len(family_cols):
        src_c = [x.decode("utf-8") if isinstance(x, (bytes, np.bytes_)) else str(x) for x in spec_classes]
        if all(labels[c] in src_c for c in family_cols):
            c_idx = [src_c.index(labels[c]) for c in family_cols]
            P_spec = P_spec[:, c_idx]

    fam_mass = P_out[:, family_cols].sum(axis=1)
    has_signal = np.ptp(spec_arr, axis=1) > 1e-6
    fam_mask = (np.isin(R_main.argmax(axis=1), family_cols) | (fam_mass >= 0.10)) & has_signal

    if fam_mask.any():
        for idx in np.flatnonzero(fam_mask):
            orig_mass = float(P_out[idx, family_cols].sum())
            if orig_mass > 1e-6:
                gw_eff = float(gate_weight) * min(orig_mass / 0.50, 1.0)
                main_norm = P_out[idx, family_cols] / orig_mass
                fused_fam = (1.0 - gw_eff) * main_norm + gw_eff * P_spec[idx]
                P_out[idx, family_cols] = fused_fam * orig_mass

    return P_out


# =============================================================================
# 5. ORTHOGRAPHIC RULES & HANDWRITING SHIELD
# =============================================================================


def build_orthographic_penalty_matrix(
    curly_quote_log_penalty: float = -1.00,  # Rule 1: Claude & GLM never emit curly quotes
    multi_em_dash_log_penalty: float = 0.0,  # Rule 2: Multiple prose em-dashes (rare in Gemini)
    straight_contr_log_penalty: float = 0.0,  # Rule 3: Straight contractions
    curly_contr_log_penalty: float = 0.0,  # Rule 4: Curly contractions
) -> np.ndarray:
    """
    Builds the (4, 18) log-penalty matrix W_pen mapping discrete orthographic
    rule indicators to the 18 canonical model classes.

    Empirical Foundations:
    ----------------------
    - Rule 1 (Curly Quotes): In the competition training corpus, Claude (0 / 1049)
      and GLM (0 / 300) NEVER produce typographic curly quotes (‘, ’, “, ”) in standard prose.
      They consistently output straight quotes (', "). If a passage contains curly quotes,
      the probability of it being Claude or GLM is astronomically low.
      Log penalty of -1.0 reduces their odds by a factor of e^(-1) ≈ 0.368.
    - Rule 2 (Multiple Em-Dashes): Gemini models rarely use two or more prose em-dashes (—).
    - Rule 3 (Straight Contractions): Distinctive signature for gpt-5.6-sol vs sisters.
    - Rule 4 (Curly Contractions): Reinforces rejection of Claude & GLM.
    """
    W_pen = np.zeros((4, K), dtype=np.float64)
    claude_and_glm_idx = FAMILIES["claude"] + FAMILIES["glm"]
    gemini_idx = FAMILIES["gemini"]
    sol_idx = LAB2I["gpt-5.6-sol"]

    # Rule 1: Any curly quote -> penalize Claude (5 models) and GLM (2 models)
    W_pen[0, claude_and_glm_idx] = float(curly_quote_log_penalty)
    # Rule 2: >= 2 prose em-dashes -> penalize Gemini (4 models)
    W_pen[1, gemini_idx] = float(multi_em_dash_log_penalty)
    # Rule 3: >= 2 prose straight contractions (with 0 curly) -> penalize gpt-5.6-sol
    W_pen[2, sol_idx] = float(straight_contr_log_penalty)
    # Rule 4: >= 1 prose curly contraction -> reinforce penalty on Claude & GLM
    W_pen[3, claude_and_glm_idx] = float(curly_contr_log_penalty)

    return W_pen


def _extract_rules_fallback(texts: list[str]) -> np.ndarray:
    """
    Fallback discrete rules extractor using regex when lexical_engine is unavailable.
    Detects curly quotes, em-dashes, and straight/curly prose contractions.
    """
    re_code = re.compile(r"```[\s\S]*?```|`[^`\n]+`")
    re_str = re.compile(r"[A-Za-z]'[A-Za-z]")
    re_cur = re.compile(r"[A-Za-z]\u2019[A-Za-z]")
    rules = np.zeros((len(texts), 4), dtype=np.float32)

    for i, raw_t in enumerate(texts):
        t = str(raw_t)
        prose = re_code.sub(" ", t)  # Strip inline and block code before checking prose
        raw_curly = t.count("\u2018") + t.count("\u2019") + t.count("\u201c") + t.count("\u201d")
        rules[i, 0] = 1.0 if raw_curly > 0 else 0.0
        rules[i, 1] = 1.0 if prose.count("\u2014") >= 2 else 0.0
        rules[i, 2] = 1.0 if (len(re_str.findall(prose)) >= 2 and (t.count("\u2018") + t.count("\u2019")) == 0) else 0.0
        rules[i, 3] = 1.0 if len(re_cur.findall(prose)) >= 1 else 0.0

    return rules


def apply_unified_handwriting_shield(
    P_mix: np.ndarray,
    test_texts: list[str],
    log_p_handwriting: np.ndarray | None = None,
    rule_flags: np.ndarray | None = None,
    curly_quote_multiplier: float = 0.36787944117144233,  # exp(-1.0)
    multi_em_dash_log_penalty: float = 0.0,
    straight_contr_log_penalty: float = 0.0,
    curly_contr_log_penalty: float = 0.0,
    continuous_hw_weight: float = 0.0,
) -> np.ndarray:
    """
    Single unified point of application for ALL handwriting and orthographic rules:
    1. Computes vectorized discrete log-penalties via matrix product: rule_flags @ W_pen
    2. Adds continuous 26-feature handwriting Gaussian log-likelihoods
    3. Re-normalizes rows to the probability simplex via softmax.
    """
    if rule_flags is None:
        if "extract_handwriting_and_orthographic_features" in globals():
            _, rule_flags = extract_handwriting_and_orthographic_features(test_texts)
        else:
            rule_flags = _extract_rules_fallback(test_texts)

    curly_log_pen = float(np.log(max(curly_quote_multiplier, 1e-12)))
    W_pen = build_orthographic_penalty_matrix(
        curly_quote_log_penalty=curly_log_pen,
        multi_em_dash_log_penalty=multi_em_dash_log_penalty,
        straight_contr_log_penalty=straight_contr_log_penalty,
        curly_contr_log_penalty=curly_contr_log_penalty,
    )

    log_P = np.log(np.clip(P_mix.astype(np.float64), 1e-12, 1.0))

    # 1. Discrete Bayesian rule subtraction (vectorized matrix multiply)
    log_P += rule_flags.astype(np.float64) @ W_pen

    # 2. Continuous 26-feature handwriting log-likelihood contribution
    if log_p_handwriting is not None and continuous_hw_weight > 0.0:
        log_P += float(continuous_hw_weight) * log_p_handwriting.astype(np.float64)

    return softmax(log_P, axis=1)


def apply_curly_quote_shield(P_mix: np.ndarray, test_texts: list[str], multiplier: float = 0.36787944117144233) -> np.ndarray:
    """Backward-compatible wrapper invoking the unified handwriting shield."""
    return apply_unified_handwriting_shield(
        P_mix=P_mix,
        test_texts=test_texts,
        log_p_handwriting=None,
        rule_flags=None,
        curly_quote_multiplier=multiplier,
        multi_em_dash_log_penalty=0.0,
        straight_contr_log_penalty=0.0,
        curly_contr_log_penalty=0.0,
        continuous_hw_weight=0.0,
    )


# =============================================================================
# 6. SINKHORN-KNOPP COLUMN MASS BALANCING
# =============================================================================


def damped_sinkhorn(P: np.ndarray, alpha: float = 0.50, iters: int = 100, eps_val: float = 1e-12) -> np.ndarray:
    """
    Damped Sinkhorn-Knopp column marginal projection onto a smoothed target prior:
        c_target = (1 - alpha) * mean(P, axis=0) + alpha * (1 / K)
    Guarantees strict row normalization (row sums == 1.0) while preventing uniform collapse
    when test class priors diverge from uniform.
    """
    if len(P) < 2:
        return P
    P_eps = np.clip(P.astype(np.float64), eps_val, 1.0)
    n_rows, n_cols = P_eps.shape
    p_bar = P_eps.mean(axis=0)
    c_target = (1.0 - float(alpha)) * p_bar + float(alpha) * (1.0 / n_cols)
    c_target = c_target / max(float(c_target.sum()), eps_val)
    r = np.ones(n_rows, dtype=np.float64)
    c = n_rows * c_target
    u = np.ones(n_rows, dtype=np.float64)
    for _ in range(int(iters)):
        v = c / np.maximum(np.dot(P_eps.T, u), eps_val)
        u = r / np.maximum(np.dot(P_eps, v), eps_val)
    S = u[:, None] * P_eps * v[None, :]
    return S / np.maximum(S.sum(axis=1, keepdims=True), eps_val)


def sinkhorn_balancing(
    P: np.ndarray,
    iters: int = 100,
    sinkhorn_strength: float = 0.65,
    target_mix: float = 0.35,
    train_prior: np.ndarray | None = None,
    allow_pure: bool = False,
) -> np.ndarray:
    """
    Soft Multiplicative Sinkhorn-Knopp Column Marginal Balancing.
    =============================================================
    Equalizes global class marginals using Soft Sinkhorn shrinkage.
    Pure Sinkhorn (1.0 / 1.0) is disallowed by default as it destroys test performance
    by hard-forcing uniform marginals on batches with natural class variance.

    Parameters:
    -----------
    P : np.ndarray of shape (N, 18)
        Probability matrix to balance.
    iters : int
        Number of alternating row/column scaling steps (default 100).
    sinkhorn_strength : float
        Soft shrinkage interpolation between original P and balanced Q (default 0.65).
        Capped at 0.75 unless allow_pure=True.
    target_mix : float
        Convex blend between predicted marginals and prior (default 0.35).
        Capped at 0.50 unless allow_pure=True.
    train_prior : np.ndarray of shape (18,), optional
        Empirical training class frequencies (used when target_mix < 1.0).
    allow_pure : bool, optional
        Whether to allow 100% hard pure Sinkhorn (default False).

    Returns:
    --------
    Balanced probability matrix Q of shape (N, 18).
    """
    if len(P) < 18 or sinkhorn_strength <= 0.0:
        return P

    P_work = np.clip(P.astype(np.float64), 1e-12, 1.0)
    K_cols = P_work.shape[1]

    # Target column marginal
    if train_prior is not None:
        p_train = np.clip(train_prior.astype(np.float64), 1e-6, None)
        p_train /= p_train.sum()
    else:
        p_train = np.full(K_cols, 1.0 / K_cols, dtype=np.float64)

    eff_mix = target_mix if allow_pure else min(target_mix, 0.50)
    if eff_mix < 1.0:
        pred_marg = np.clip(P_work.mean(axis=0), 1e-6, None)
        pred_marg /= pred_marg.sum()
        rho = float(np.clip(eff_mix, 0.0, 1.0))
        target_col = (1.0 - rho) * pred_marg + rho * p_train
        target_col = np.clip(target_col, 1e-6, None)
        target_col /= target_col.sum()
    else:
        target_col = p_train

    # Alternating row and column projections
    Q = P_work.copy()
    for _ in range(iters):
        # Row normalization: sum across classes = 1.0
        Q /= Q.sum(axis=1, keepdims=True).clip(min=1e-12)
        # Column scaling: sum across samples for class k = N * target_col[k]
        Q *= (len(Q) * target_col[None, :]) / Q.sum(axis=0, keepdims=True).clip(min=1e-12)

    # Final row normalization
    Q /= Q.sum(axis=1, keepdims=True).clip(min=1e-12)

    if allow_pure and sinkhorn_strength >= 1.0:
        return Q

    # Soft logarithmic mixture between original P and balanced Q (pure Sinkhorn disallowed)
    eff_strength = float(np.clip(sinkhorn_strength, 0.0, 0.75 if not allow_pure else 1.0))
    if eff_strength <= 0.0:
        return P_work

    log_P = (1.0 - eff_strength) * np.log(np.clip(P_work, 1e-12, None)) + eff_strength * np.log(np.clip(Q, 1e-12, None))
    return softmax(log_P, axis=1)


def apply_prior_and_sinkhorn(
    logits: np.ndarray,
    temperature: float = 1.5,
    iters: int = 80,
    sinkhorn_strength: float = 0.85,
    target_mix: float = 0.15,
) -> np.ndarray:
    """
    Applies per-sample standardization, iterative column-wise logit prior centering (20 steps),
    and Sinkhorn marginal probability balancing to ensure unbiased class priors.
    """
    if len(logits) < 18 or sinkhorn_strength <= 0.0:
        return softmax(logits, axis=1)

    row_mean = logits.mean(axis=1, keepdims=True)
    row_std = np.clip(logits.std(axis=1, keepdims=True), 1e-4, None)
    norm_logits = (logits - row_mean) / row_std

    adj_logits = norm_logits / max(temperature, 1e-3)
    num_classes = logits.shape[1]
    for _ in range(20):
        probs = softmax(adj_logits, axis=1)
        col_mean = np.clip(probs.mean(axis=0, keepdims=True), 1e-6, None)
        adj_logits = adj_logits - 0.5 * np.log(col_mean * num_classes)

    return sinkhorn_balancing(
        softmax(adj_logits, axis=1),
        iters=iters,
        sinkhorn_strength=sinkhorn_strength,
        target_mix=target_mix,
    )


# =============================================================================
# 7. TYPOBAYES POSTPROCESSOR (HIERARCHICAL BETA-BERNOULLI SHRINKAGE)
# =============================================================================


def apply_typobayes_postprocess(
    P_or_logP: np.ndarray,
    train_texts: list[str],
    y_train: np.ndarray,
    test_texts: list[str],
    labels: list = LABELS,
    kappa: float = 24.0,
    evidence_scale: float = 1.2,
    evidence_cap: float = 0.30,
) -> np.ndarray:
    """
    Applies Hierarchical Beta-Bernoulli 37-Marker TypoBayes postprocessing.

    Concept:
    --------
    Extracts 37 orthographic, punctuation, and structural markdown markers
    (e.g., ' — ', '`', '###', 'However,', '\n- ') across training texts and models.
    Uses Beta-Bernoulli empirical Bayes with prior strength kappa to calculate
    smoothed class-conditional emission rates, deriving log-likelihood ratio evidence.
    Applies conservative confidence gating so high-confidence predictions are not overridden.
    """
    if len(train_texts) == 0 or len(test_texts) == 0 or len(y_train) == 0:
        return P_or_logP

    y_tr = np.array([LAB2I[y] if isinstance(y, str) else int(y) for y in y_train], dtype=int)
    marks = [
        " \u2014 ",
        "\u2014",
        "\u2013",
        "\u2019",
        "\u2018",
        "\u201c",
        "\u201d",
        "\u00ab",
        "\u00bb",
        '"',
        ";",
        ":",
        "(",
        ")",
        "[",
        "]",
        "**",
        "`",
        "\u2026",
        "\u2022",
        "\n",
        "\n\n",
        "e.g.",
        "i.e.",
        "However,",
        "Overall,",
        "In conclusion",
        "Therefore,",
        "Moreover,",
        "\t",
        " - ",
        " = ",
        "\n- ",
        "\n1.",
        "###",
        "```",
        "> ",
    ]
    K_cls = len(labels)

    def _extract_marker_row(t: str) -> list[float]:
        # Mutually exclusive decoupling for overlapping substrings:
        # 1. ' — ' vs unspaced '—'
        has_spaced_em = " \u2014 " in t
        t_no_spaced = t.replace(" \u2014 ", "")
        has_unspaced_em = "\u2014" in t_no_spaced

        # 2. '\n\n', '\n- ', '\n1.' vs standalone '\n'
        has_double_nl = "\n\n" in t
        has_bullet = "\n- " in t
        has_num_list = "\n1." in t
        t_no_compound_nl = t.replace("\n\n", "").replace("\n- ", "").replace("\n1.", "")
        has_single_nl = "\n" in t_no_compound_nl

        row = []
        for m in marks:
            if m == " \u2014 ":
                row.append(float(has_spaced_em))
            elif m == "\u2014":
                row.append(float(has_unspaced_em))
            elif m == "\n\n":
                row.append(float(has_double_nl))
            elif m == "\n- ":
                row.append(float(has_bullet))
            elif m == "\n1.":
                row.append(float(has_num_list))
            elif m == "\n":
                row.append(float(has_single_nl))
            else:
                row.append(float(m in t))
        return row

    Xtr = np.array([_extract_marker_row(t) for t in train_texts], dtype=np.float32)
    Xte = np.array([_extract_marker_row(t) for t in test_texts], dtype=np.float32)
    freq = np.bincount(y_tr, minlength=K_cls).astype(np.float64)
    hits = np.stack([Xtr[y_tr == k].sum(axis=0) for k in range(K_cls)], axis=0).astype(np.float64)

    # Hierarchical Beta-Bernoulli smoothing: class rates shrink toward global rate
    global_rate = (Xtr.sum(axis=0) + 2.0) / (len(Xtr) + 4.0)
    rate = (hits + kappa * global_rate[None, :]) / (freq[:, None] + kappa)
    rate = np.clip(rate, 0.01, 0.99)
    global_rate = np.clip(global_rate, 0.01, 0.99)

    log_rate = np.log(rate)
    log_not_rate = np.log1p(-rate)
    base_log_rate = np.log(global_rate)
    base_log_not_rate = np.log1p(-global_rate)

    # Weight markers by discriminative power across classes
    signal = np.sqrt(np.mean((rate - global_rate[None, :]) ** 2, axis=0))
    med_signal = float(np.median(signal[signal > 0])) if np.any(signal > 0) else 1.0
    fw = signal / max(med_signal, 1e-6)
    fw = np.clip(fw, 0.25, 2.0)

    # Matrix form: weighted Bernoulli log-likelihood ratio for each class
    evidence = (Xte * fw[None, :]) @ log_rate.T + ((1.0 - Xte) * fw[None, :]) @ log_not_rate.T
    evidence -= (((Xte * fw[None, :]) @ base_log_rate) + (((1.0 - Xte) * fw[None, :]) @ base_log_not_rate))[:, None]
    evidence -= evidence.mean(axis=1, keepdims=True)
    evidence = np.clip(evidence_scale * evidence / max(float(fw.sum()), 1e-6), -evidence_cap, evidence_cap)

    # Convert inputs to probabilities and log-probabilities
    if np.all(P_or_logP <= 0.0) and not np.all(P_or_logP == 0.0):
        raw_logP = P_or_logP.copy()
        probs = softmax(raw_logP, axis=1)
    else:
        probs = P_or_logP.copy()
        raw_logP = np.log(np.clip(probs, 1e-12, 1.0))

    # Gating: only apply evidence when model is uncertain, text is long, and markers are present
    sorted_p = np.sort(probs, axis=1)
    margin = sorted_p[:, -1] - sorted_p[:, -2]
    confidence_gate = np.clip((0.60 - margin) / 0.60, 0.0, 1.0)
    word_count = np.array([len(re.findall(r"\b\w+\b", str(t))) for t in test_texts], dtype=np.float32)
    length_gate = np.clip(np.sqrt(np.maximum(word_count, 1.0) / 80.0), 0.40, 1.0)
    active_markers = Xte.sum(axis=1)
    coverage_gate = np.clip(np.sqrt((active_markers + 1.0) / 4.0), 0.45, 1.0)
    gate = confidence_gate * length_gate * coverage_gate

    safe_logits = raw_logP + gate[:, None] * evidence
    return softmax(safe_logits, axis=1)


# =============================================================================
# 7.5. UNIFIED END-TO-END INFERENCE POSTPROCESSING PIPELINE
# =============================================================================


def run_full_inference_pipeline(
    P_mix: np.ndarray,
    texts: list[str],
    post_cfg: dict,
    R_main: np.ndarray | None = None,
    claude_spec_logits: np.ndarray | None = None,
    claude_classes: list | None = None,
    log_p_handwriting: np.ndarray | None = None,
    rule_flags: np.ndarray | None = None,
    train_texts: list[str] | None = None,
    y_train: np.ndarray | None = None,
    train_prior: np.ndarray | None = None,
) -> np.ndarray:
    """
    Unified end-to-end postprocessing pipeline executing all downstream stages:
    1. Claude Specialist Gate (Vectorized, mass-conserving)
    2. Continuous Handwriting & Orthographic Shield (Bayesian log-penalty + continuous density)
    3. Sinkhorn Mass Balancing (Iterative row/column equalization)
    4. TypoBayes 37-Marker Hierarchical Beta-Bernoulli Postprocessing

    Guarantees 100% mathematical parity between test inference and unseen benchmark evaluation.
    """
    if len(P_mix) == 0:
        return P_mix

    P_curr = P_mix.copy()
    R_ref = R_main if R_main is not None else P_curr

    # 1. Claude Specialist Gate
    gate_weight = float(post_cfg.get("claude_specialist_weight", 0.15))
    if claude_spec_logits is not None:
        P_curr = apply_claude_specialist_gate(
            P_mix=P_curr,
            R_main=R_ref,
            claude_spec_logits=claude_spec_logits,
            claude_classes=claude_classes,
            labels=LABELS,
            claude_cols=FAMILIES["claude"],
            gate_weight=gate_weight,
        )

    # 2. Unified Handwriting & Orthographic Shield (defaults to 0.0 uncalibrated extra penalties)
    default_curly_multiplier = float(np.exp(-1.0))
    multiplier = float(post_cfg.get("curly_quote_multiplier", default_curly_multiplier))
    multi_em_dash_log_pen = float(post_cfg.get("multi_em_dash_log_penalty", 0.0))
    straight_contr_log_pen = float(post_cfg.get("straight_contr_log_penalty", 0.0))
    curly_contr_log_pen = float(post_cfg.get("curly_contr_log_penalty", 0.0))
    continuous_hw_weight = float(post_cfg.get("continuous_hw_weight", 0.0))

    P_curr = apply_unified_handwriting_shield(
        P_mix=P_curr,
        test_texts=texts,
        log_p_handwriting=log_p_handwriting,
        rule_flags=rule_flags,
        curly_quote_multiplier=multiplier,
        multi_em_dash_log_penalty=multi_em_dash_log_pen,
        straight_contr_log_penalty=straight_contr_log_pen,
        curly_contr_log_penalty=curly_contr_log_pen,
        continuous_hw_weight=continuous_hw_weight,
    )

    # 3. Sinkhorn Mass Balancing (canonical damped_sinkhorn default when sinkhorn_alpha configured)
    sinkhorn_iters = int(post_cfg.get("sinkhorn_iterations", 100))
    sinkhorn_mode = str(post_cfg.get("sinkhorn_mode", "damped" if "sinkhorn_alpha" in post_cfg else "soft")).lower()
    if sinkhorn_mode == "damped":
        sinkhorn_alpha = float(post_cfg.get("sinkhorn_alpha", 0.50))
        P_curr = damped_sinkhorn(P_curr, alpha=sinkhorn_alpha, iters=sinkhorn_iters)
    else:
        sinkhorn_strength = float(post_cfg.get("sinkhorn_strength", 0.65))
        sinkhorn_target_mix = float(post_cfg.get("sinkhorn_target_mix", 0.35))
        P_curr = sinkhorn_balancing(
            P_curr,
            iters=sinkhorn_iters,
            sinkhorn_strength=sinkhorn_strength,
            target_mix=sinkhorn_target_mix,
            train_prior=train_prior,
        )

    # 4. TypoBayes 37-Marker Postprocessing
    if post_cfg.get("enable_typobayes", True) and train_texts and y_train is not None and len(train_texts) > 0:
        P_curr = apply_typobayes_postprocess(
            P_curr,
            train_texts=train_texts,
            y_train=y_train,
            test_texts=texts,
            labels=LABELS,
            kappa=float(post_cfg.get("typobayes_kappa", 24.0)),
            evidence_scale=float(post_cfg.get("typobayes_evidence_scale", 1.2)),
            evidence_cap=float(post_cfg.get("typobayes_evidence_cap", 0.30)),
        )

    return P_curr


# =============================================================================
# 8. FINAL SUBMISSION EXPORT
# =============================================================================


def generate_submission(P_final: np.ndarray, test_ids: pd.Series, working_dir: Path) -> tuple[pd.DataFrame, Path, Path]:
    """
    Generates submission.parquet and submission.csv with exact Kaggle competition schema:
    - id: test sample identifier string
    - model: predicted model class name string (argmax over calibrated probabilities)
    """
    if len(P_final) == 0 or len(test_ids) == 0:
        if len(test_ids) > 0:
            preds = [LABELS[0]] * len(test_ids)
            df_sub = pd.DataFrame({"id": test_ids, "model": preds})
        else:
            df_sub = pd.DataFrame(columns=["id", "model"])
    elif len(P_final) != len(test_ids):
        if len(P_final) < len(test_ids):
            preds = [LABELS[idx] for idx in P_final.argmax(axis=1)] + [LABELS[0]] * (len(test_ids) - len(P_final))
        else:
            preds = [LABELS[idx] for idx in P_final[:len(test_ids)].argmax(axis=1)]
        df_sub = pd.DataFrame({"id": test_ids, "model": preds})
    else:
        preds = [LABELS[idx] for idx in P_final.argmax(axis=1)]
        df_sub = pd.DataFrame({"id": test_ids, "model": preds})

    sub_parquet_path = working_dir / "submission.parquet"
    sub_csv_path = working_dir / "submission.csv"

    df_sub.to_parquet(sub_parquet_path, index=False)
    df_sub.to_csv(sub_csv_path, index=False)
    return df_sub, sub_parquet_path, sub_csv_path


In [ ]:
# === MODULE: src/namethatllm/ablation.py ===
"""
Zero-Leakage Unseen Holdout Benchmark & Forward Stepwise Ablation Engine
========================================================================
Constructs a strictly unseen evaluation dataset from mounted parquet/CSV corpora
by subtracting all passages present in `train.csv` and `pool_v4.parquet`
(the training sets of the pre-trained Stage-1/Stage-2 checkpoints), quarantines
those holdout passages from `lexical_engine.py` and `trainer.py`, and evaluates:
1. Standalone (Solo) Unseen Accuracy of each model.
2. Forward Stepwise Cumulative Accuracy (1 model -> +2nd model -> ... -> Full Ensemble).
"""

import os
import gc
import hashlib
import warnings
from itertools import combinations
from pathlib import Path
from typing import Dict, List, Tuple, Optional, Any, Set
import numpy as np
import pandas as pd
from scipy.optimize import minimize, minimize_scalar
from scipy.spatial.distance import squareform
from scipy.cluster.hierarchy import linkage, fcluster
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import GroupKFold, KFold



def _ablation_sinkhorn(P: np.ndarray, iters: int = 200, eps: float = 1e-12, **kwargs) -> np.ndarray:
    """Fast uniform Sinkhorn mass balancing delegating to canonical damped_sinkhorn."""
    if len(P) < 18:
        return P
    return damped_sinkhorn(P, alpha=1.0, iters=iters, eps_val=eps)


def _clip_and_log_probs(raw_lps_list: List[np.ndarray], clip_min: float = 1e-4) -> np.ndarray:
    """Converts (M, N, K) raw logits or log-probs into probability-clipped log probabilities."""
    raw_arr = np.asarray(raw_lps_list, dtype=np.float64)
    M = raw_arr.shape[0]
    P_clipped = np.zeros_like(raw_arr, dtype=np.float64)
    for m in range(M):
        p_m = softmax(raw_arr[m], axis=1)
        p_m = np.clip(p_m, clip_min, 1.0)
        p_m /= np.maximum(p_m.sum(axis=1, keepdims=True), 1e-12)
        P_clipped[m] = p_m
    return np.log(P_clipped)


def _lbfgs_loss_and_grad(theta: np.ndarray, Z_tr: np.ndarray, Y_smooth: np.ndarray, lam: float, M: int) -> Tuple[float, np.ndarray]:
    """Label-smoothed NLL + uniform L2 simplex regularization objective and analytic gradient."""
    N_tr = len(Y_smooth)
    w = softmax(theta)
    U = np.tensordot(w, Z_tr, axes=(0, 0))
    P = softmax(U, axis=1)
    nll = float(-np.mean(np.sum(Y_smooth * np.log(np.clip(P, 1e-12, 1.0)), axis=1)))
    reg = float(lam) * float(np.sum((w - 1.0 / M) ** 2))
    loss = nll + reg
    G = (P - Y_smooth) / max(1, N_tr)
    grad_w_nll = np.tensordot(Z_tr, G, axes=((1, 2), (0, 1)))
    grad_w_reg = 2.0 * float(lam) * (w - 1.0 / M)
    d_w = grad_w_nll + grad_w_reg
    grad_theta = w * (d_w - np.dot(w, d_w))
    return loss, grad_theta


def pool_calibrated_log_probs(
    raw_lps_list: List[np.ndarray],
    model_names: List[str],
    mean_T_m: Dict[str, float],
    mean_w_m: Dict[str, float],
    mean_T_global: float = 1.0,
    clip_min: float = 1e-4,
) -> np.ndarray:
    """Applies pre-calibrated per-model temperatures T_m, simplex weights w_m, and global T_global."""
    if not raw_lps_list:
        return np.zeros((0, K), dtype=np.float64)
    n_rows = len(raw_lps_list[0])
    if n_rows == 0:
        return np.zeros((0, K), dtype=np.float64)

    log_P = _clip_and_log_probs(raw_lps_list, clip_min=clip_min)
    weights = np.array([max(0.0, float(mean_w_m.get(name, 0.0))) for name in model_names], dtype=np.float64)
    w_sum = float(weights.sum())
    if w_sum <= 1e-12:
        weights = np.full(len(model_names), 1.0 / max(1, len(model_names)), dtype=np.float64)
    else:
        weights = weights / w_sum

    U = np.zeros((n_rows, K), dtype=np.float64)
    for m, name in enumerate(model_names):
        if weights[m] <= 0.0:
            continue
        T_m = max(1e-3, float(mean_T_m.get(name, 1.0)))
        Z_m = log_softmax(log_P[m] / T_m, axis=1)
        U += weights[m] * Z_m

    T_g = max(1e-3, float(mean_T_global))
    return softmax(U / T_g, axis=1)


def fit_lbfgs_log_linear_pooling(
    raw_lps_list: List[np.ndarray],
    y: np.ndarray,
    groups: Optional[np.ndarray],
    model_names: List[str],
    spec_logits: Optional[np.ndarray] = None,
    claude_classes: Optional[list] = None,
    lam: float = 0.20,
    eps: float = 0.05,
    sinkhorn_alpha: float = 0.50,
    clip_min: float = 1e-4,
    n_splits: int = 5,
) -> Dict[str, Any]:
    """Fits 5-fold GroupKFold L-BFGS log-linear pooling with per-model T_m, global T_g, and optional Claude gate."""
    log_P = _clip_and_log_probs(raw_lps_list, clip_min=clip_min)
    M, N, K_cls = log_P.shape
    y_arr = np.asarray(y, dtype=int)

    if groups is not None and len(groups) == N and len(np.unique(groups)) >= 2:
        k_folds = min(n_splits, len(np.unique(groups)))
        splits = list(GroupKFold(n_splits=k_folds).split(log_P[0], y_arr, groups=groups))
    else:
        k_folds = min(n_splits, max(2, N // max(1, K_cls)))
        splits = list(KFold(n_splits=k_folds, shuffle=True, random_state=42).split(log_P[0]))

    trial_gws = [0.0, 0.02, 0.04, 0.06, 0.08, 0.10, 0.15, 0.20]
    oof_blended_lp = np.zeros((N, K_cls), dtype=np.float64)
    oof_post_probs = np.zeros((N, K_cls), dtype=np.float64)
    fold_weights, fold_model_temps, fold_global_temps, fold_stds, fold_gws = [], [], [], [], []

    for tr_idx, val_idx in splits:
        N_tr = len(tr_idx)
        y_tr = y_arr[tr_idx]
        Z_fold = np.zeros_like(log_P)
        model_temps = np.ones(M, dtype=np.float64)
        model_stds = np.zeros(M, dtype=np.float64)

        for m in range(M):
            lpm_tr = log_P[m, tr_idx, :]
            res_t = minimize_scalar(
                lambda T: float(-np.mean(log_softmax(lpm_tr / T, axis=1)[np.arange(N_tr), y_tr])),
                bounds=(0.05, 10.0),
                method="bounded",
            )
            T_opt = float(res_t.x)
            model_temps[m] = T_opt
            Z_fold[m] = log_softmax(log_P[m] / T_opt, axis=1)
            model_stds[m] = float(np.std(Z_fold[m, tr_idx, :]))

        fold_model_temps.append(model_temps)
        fold_stds.append(model_stds)

        Y_smooth = np.full((N_tr, K_cls), float(eps) / K_cls, dtype=np.float64)
        Y_smooth[np.arange(N_tr), y_tr] += 1.0 - float(eps)
        Z_tr = Z_fold[:, tr_idx, :]

        if M == 1:
            w_k = np.array([1.0], dtype=np.float64)
        else:
            res_w = minimize(
                fun=lambda th: _lbfgs_loss_and_grad(th, Z_tr, Y_smooth, lam, M),
                x0=np.zeros(M, dtype=np.float64),
                method="L-BFGS-B",
                jac=True,
                options={"maxiter": 500, "ftol": 1e-9},
            )
            w_k = softmax(res_w.x)
        fold_weights.append(w_k)

        U_tr = np.tensordot(w_k, Z_tr, axes=(0, 0))
        res_tg = minimize_scalar(
            lambda T_g: float(-np.mean(log_softmax(U_tr / T_g, axis=1)[np.arange(N_tr), y_tr])),
            bounds=(0.1, 5.0),
            method="bounded",
        )
        T_global_opt = float(res_tg.x)
        fold_global_temps.append(T_global_opt)

        P_tr = softmax(U_tr / T_global_opt, axis=1)
        best_gw_fold = 0.0
        if spec_logits is not None and claude_classes is not None and len(spec_logits) == N:
            best_bacc_tr = float(balanced_accuracy_score(y_tr, damped_sinkhorn(P_tr, alpha=sinkhorn_alpha).argmax(axis=1)))
            for trial_gw in trial_gws:
                P_tr_g = apply_claude_specialist_gate(
                    P_mix=P_tr,
                    R_main=P_tr,
                    claude_spec_logits=spec_logits[tr_idx],
                    claude_classes=claude_classes,
                    gate_weight=trial_gw,
                )
                b_tr = float(balanced_accuracy_score(y_tr, damped_sinkhorn(P_tr_g, alpha=sinkhorn_alpha).argmax(axis=1)))
                if b_tr > best_bacc_tr:
                    best_bacc_tr = b_tr
                    best_gw_fold = trial_gw
        fold_gws.append(best_gw_fold)

        U_val = np.tensordot(w_k, Z_fold[:, val_idx, :], axes=(0, 0)) / T_global_opt
        P_val = softmax(U_val, axis=1)
        oof_blended_lp[val_idx] = log_softmax(U_val, axis=1)
        if best_gw_fold > 0.0 and spec_logits is not None and claude_classes is not None:
            oof_post_probs[val_idx] = apply_claude_specialist_gate(
                P_mix=P_val,
                R_main=P_val,
                claude_spec_logits=spec_logits[val_idx],
                claude_classes=claude_classes,
                gate_weight=best_gw_fold,
            )
        else:
            oof_post_probs[val_idx] = P_val

    P_oof_raw = softmax(oof_blended_lp, axis=1)
    oof_bacc_raw = float(balanced_accuracy_score(y_arr, P_oof_raw.argmax(axis=1)))
    oof_nll_raw = float(-np.mean(oof_blended_lp[np.arange(N), y_arr]))
    P_oof_sink = damped_sinkhorn(P_oof_raw, alpha=sinkhorn_alpha)
    oof_bacc_sink = float(balanced_accuracy_score(y_arr, P_oof_sink.argmax(axis=1)))
    P_oof_sink_cl = damped_sinkhorn(oof_post_probs, alpha=sinkhorn_alpha)
    oof_bacc_nested_claude = float(balanced_accuracy_score(y_arr, P_oof_sink_cl.argmax(axis=1)))

    avg_w = np.mean(fold_weights, axis=0)
    avg_temps = np.mean(fold_model_temps, axis=0)
    avg_stds = np.mean(fold_stds, axis=0)
    avg_tg = float(np.mean(fold_global_temps))
    avg_gw = float(np.mean(fold_gws))

    use_claude_score = spec_logits is not None and claude_classes is not None and len(spec_logits) == N
    primary_bacc = oof_bacc_nested_claude if use_claude_score else oof_bacc_sink
    composite_oof_score = primary_bacc - 0.05 * oof_nll_raw

    return {
        "model_names": list(model_names),
        "lam": float(lam),
        "eps": float(eps),
        "clip_min": float(clip_min),
        "sinkhorn_alpha": float(sinkhorn_alpha),
        "mean_T_m": {name: float(avg_temps[i]) for i, name in enumerate(model_names)},
        "mean_w_m": {name: float(avg_w[i]) for i, name in enumerate(model_names)},
        "mean_stds": {name: float(avg_stds[i]) for i, name in enumerate(model_names)},
        "fold_weights": np.array(fold_weights),
        "mean_T_global": avg_tg,
        "mean_gw": avg_gw,
        "oof_bacc_raw": oof_bacc_raw,
        "oof_nll_raw": oof_nll_raw,
        "oof_bacc_sink": oof_bacc_sink,
        "oof_bacc_nested_claude": oof_bacc_nested_claude,
        "composite_oof_score": float(composite_oof_score),
        "P_oof_raw": P_oof_raw,
        "P_oof_sink": P_oof_sink,
        "P_oof_sink_claude": P_oof_sink_cl,
    }


def auto_prune_and_fit_lbfgs(
    raw_lps_list: List[np.ndarray],
    y: np.ndarray,
    groups: Optional[np.ndarray],
    model_names: List[str],
    spec_logits: Optional[np.ndarray] = None,
    claude_classes: Optional[list] = None,
    lambdas: Tuple[float, ...] = (0.05, 0.10, 0.20),
    eps: float = 0.05,
    sinkhorn_alpha: float = 0.50,
    clip_min: float = 1e-4,
    pruning_tol: float = 1e-4,
) -> Tuple[Dict[str, Any], List[str]]:
    """
    Automatically selects active models and regularization strength lambda via 5-fold GroupKFold L-BFGS
    pooling + OOF backward pruning optimizing combined OOF BAcc + NLL (without any hardcoded model count).
    Returns (best_fit_dict, dropped_model_names).
    """
    active_idx = list(range(len(model_names)))
    dropped_names: List[str] = []

    def _fit_best_lam(indices: List[int]) -> Dict[str, Any]:
        sub_names = [model_names[i] for i in indices]
        sub_lps = [raw_lps_list[i] for i in indices]
        fits = [
            fit_lbfgs_log_linear_pooling(
                sub_lps, y, groups, sub_names,
                spec_logits=spec_logits, claude_classes=claude_classes,
                lam=lam, eps=eps, sinkhorn_alpha=sinkhorn_alpha, clip_min=clip_min,
            )
            for lam in lambdas
        ]
        return max(fits, key=lambda d: d["composite_oof_score"])

    cur_fit = _fit_best_lam(active_idx)
    while len(active_idx) > 1:
        cur_score = cur_fit["composite_oof_score"]
        cur_names = [model_names[i] for i in active_idx]
        zero_drop = [i for i, nm in zip(active_idx, cur_names) if cur_fit["mean_w_m"].get(nm, 0.0) < 1e-3]

        best_cand_fit = None
        best_rm_idx = None
        best_cand_score = -1e9
        candidates_to_test = zero_drop[:1] if zero_drop else active_idx
        for rm_i in candidates_to_test:
            trial_idx = [j for j in active_idx if j != rm_i]
            trial_fit = _fit_best_lam(trial_idx)
            if trial_fit["composite_oof_score"] > best_cand_score:
                best_cand_score = trial_fit["composite_oof_score"]
                best_rm_idx = rm_i
                best_cand_fit = trial_fit

        should_drop = (best_rm_idx is not None) and (
            (bool(zero_drop) and best_cand_score >= cur_score - pruning_tol)
            or (best_cand_score > cur_score + 1e-6)
        )
        if should_drop:
            dropped_names.append(model_names[best_rm_idx])
            active_idx.remove(best_rm_idx)
            cur_fit = best_cand_fit
        else:
            break

    return cur_fit, dropped_names

def _fast_sinkhorn_bacc(lp_mat: np.ndarray, y_true: np.ndarray, iters: int = 40) -> float:
    """Computes balanced accuracy after fast Sinkhorn mass equalization."""
    P = softmax(lp_mat, axis=1)
    if len(P) >= 18:
        P = _ablation_sinkhorn(P, iters=iters)
    return float(balanced_accuracy_score(y_true, P.argmax(axis=1)))

def _text_hash(text: str) -> str:
    """Normalized SHA1 hash of whitespace-collapsed text to guarantee zero exact/near-exact leakage."""
    norm = " ".join(str(text).strip().split())
    return hashlib.sha1(norm.encode("utf-8", errors="ignore")).hexdigest()

def build_unseen_holdout_dataset(
    input_dir: Path,
    train_path: Optional[Path] = None,
    samples_per_class: int = 100,
    seed: int = 42,
) -> Tuple[pd.DataFrame, Set[str]]:
    """
    Builds a balanced 18-class unseen holdout dataset (`df_unseen`) by:
    1. Hashing all texts in `train.csv` and `pool_v4.parquet` (and `pool_v1.parquet`) as `contaminated_hashes`.
    2. Scanning `nlm_matched_clean.parquet`, `nlm_pool_v5.parquet`, and `openrouter-harvest`
       for passages whose text hash is strictly NOT in `contaminated_hashes`.
    3. Sampling up to `samples_per_class` passages per class (e.g., 18 * 100 = 1,800 unseen passages)
       and returning `(df_unseen, holdout_hashes)` so `lexical_engine.py` and `trainer.py`
       exclude `holdout_hashes` from training!
    """
    input_dir = Path(input_dir) if input_dir else Path(".")

    # 0. Check for verified clean benchmark 1800 rows first
    bench_patterns = [
        "clean_benchmark_1800_rows.parquet",
        "*clean_benchmark*.parquet",
    ]
    for pat in bench_patterns:
        for bench_p in input_dir.rglob(pat):
            try:
                df_bench = pd.read_parquet(bench_p)
                if "text" in df_bench.columns and "model" in df_bench.columns:
                    df_bench = df_bench.dropna(subset=["text", "model"]).copy()
                    df_bench = df_bench[df_bench["model"].isin(LABELS)]
                    if len(df_bench) >= 180:
                        needs_clustering = False
                        if "prompt_id" not in df_bench.columns and "prompt_canon" in df_bench.columns:
                            df_bench["prompt_id"] = df_bench["prompt_canon"]

                        if "prompt_id" not in df_bench.columns:
                            needs_clustering = True
                        else:
                            p_col = df_bench["prompt_id"]
                            is_int = pd.api.types.is_integer_dtype(p_col) or (
                                p_col.dropna().apply(lambda x: isinstance(x, (int, np.integer))).all()
                                if len(p_col) > 0 else False
                            )
                            n_unique = p_col.nunique()
                            if (not is_int) or (n_unique > 0.5 * len(df_bench)):
                                needs_clustering = True

                        if needs_clustering:
                            try:
                                tfidf = TfidfVectorizer(stop_words="english", max_features=10000, sublinear_tf=True)
                                X_tfidf = tfidf.fit_transform(df_bench["text"].fillna("").tolist())
                                D = 1.0 - (X_tfidf @ X_tfidf.T).toarray().clip(0.0, 1.0)
                                np.fill_diagonal(D, 0.0)
                                cond_d = squareform(D, checks=False)
                                clusters = fcluster(linkage(cond_d, method="average"), t=0.85, criterion="distance")
                            except Exception:
                                clusters = np.arange(1, len(df_bench) + 1, dtype=int)
                            df_bench["cluster_id"] = clusters
                            df_bench["prompt_id"] = clusters
                        else:
                            if "cluster_id" not in df_bench.columns:
                                df_bench["cluster_id"] = df_bench["prompt_id"]

                        df_bench["_hash"] = [_text_hash(t) for t in df_bench["text"]]
                        holdout_hashes = set(df_bench["_hash"])
                        print(f"[Unseen Benchmark] Loaded verified clean holdout benchmark: {len(df_bench)} rows across {len(set(df_bench['model']))} classes from {bench_p}")
                        return df_bench, holdout_hashes
            except Exception as e:
                print(f"[Unseen Benchmark Warning] Failed to load {bench_p}: {e}")

    contaminated_hashes: Set[str] = set()

    # 1. Mark all train.csv rows as contaminated (seen by all Stage-2 LoRAs)
    if train_path and Path(train_path).exists():
        try:
            df_tr = pd.read_csv(train_path, usecols=["text"])
            contaminated_hashes.update(_text_hash(t) for t in df_tr["text"].fillna(""))
        except Exception:
            try:
                df_tr = pd.read_csv(train_path)
                if "text" in df_tr.columns:
                    contaminated_hashes.update(_text_hash(t) for t in df_tr["text"].fillna(""))
            except Exception:
                pass

    # 2. Mark all training pools as contaminated:
    # - pool_v4.parquet, pool_v1.parquet (seen by Part-2 LoRAs, DeBERTa, inhouse)
    # - nlm_pool_v5.parquet, nlm_matched_clean.parquet, nlm_pool_v2.parquet (seen by v5 and ts LoRAs)
    for pat in [
        "pool_v4.parquet",
        "pool_v1.parquet",
        "nlm_pool_v5.parquet",
        "nlm_matched_clean.parquet",
        "nlm_pool_v2.parquet",
    ]:
        for p in input_dir.rglob(pat):
            try:
                df_p = pd.read_parquet(p, columns=["text"])
                contaminated_hashes.update(_text_hash(t) for t in df_p["text"].fillna(""))
            except Exception:
                pass

    # 3. Collect strictly unseen candidate passages strictly from disjoint corpora (e.g. openrouter-harvest)
    candidate_dfs = []
    seen_paths = set()
    for pat in [
        "*openrouter*.parquet",
        "*openrouter*.csv",
        "*harvest*.parquet",
        "*harvest*.csv",
    ]:
        for p in input_dir.rglob(pat):
            if p in seen_paths:
                continue
            seen_paths.add(p)
            try:
                if p.suffix == ".parquet":
                    df_c = pd.read_parquet(p)
                elif p.suffix == ".csv":
                    df_c = pd.read_csv(p)
                else:
                    continue

                if "text" in df_c.columns and "model" in df_c.columns:
                    keep_cols = [c for c in ["id", "prompt_id", "text", "model"] if c in df_c.columns]
                    df_c = df_c[keep_cols].dropna(subset=["text", "model"]).copy()
                    df_c = df_c[df_c["model"].isin(LABELS)]
                    df_c["_hash"] = [_text_hash(t) for t in df_c["text"]]
                    df_c = df_c[~df_c["_hash"].isin(contaminated_hashes)]
                    if len(df_c) > 0:
                        candidate_dfs.append(df_c)
            except Exception:
                pass

    # Fallback to other non-training parquet/csv files if no openrouter files are discovered
    if not candidate_dfs:
        for pat in ["*.parquet", "*.csv"]:
            for p in input_dir.rglob(pat):
                p_name = p.name.lower()
                if p in seen_paths or any(x in p_name for x in ["train", "test", "submission", "pool", "matched"]):
                    continue
                seen_paths.add(p)
                try:
                    df_c = pd.read_parquet(p) if p.suffix == ".parquet" else pd.read_csv(p)
                    if "text" in df_c.columns and "model" in df_c.columns:
                        keep_cols = [c for c in ["id", "prompt_id", "text", "model"] if c in df_c.columns]
                        df_c = df_c[keep_cols].dropna(subset=["text", "model"]).copy()
                        df_c = df_c[df_c["model"].isin(LABELS)]
                        df_c["_hash"] = [_text_hash(t) for t in df_c["text"]]
                        df_c = df_c[~df_c["_hash"].isin(contaminated_hashes)]
                        if len(df_c) > 0:
                            candidate_dfs.append(df_c)
                except Exception:
                    pass

    if not candidate_dfs:
        return pd.DataFrame(columns=["id", "prompt_id", "text", "model", "_hash"]), set()

    df_pool_unseen = pd.concat(candidate_dfs, ignore_index=True).drop_duplicates(subset=["_hash"])

    # Stratified balanced sampling across all 18 classes
    rng = np.random.RandomState(seed)
    sampled_parts = []

    # Prompt-grouped sampling: if prompt_id exists with multi-model prompts, sample by prompt_id first
    if "prompt_id" in df_pool_unseen.columns:
        prompt_counts = df_pool_unseen.groupby("prompt_id")["model"].nunique()
        multi_prompts = prompt_counts[prompt_counts > 1].index.tolist()
        rng.shuffle(multi_prompts)

        class_counts = {m: 0 for m in LABELS}
        chosen_indices = []
        for pid in multi_prompts:
            sub_p = df_pool_unseen[df_pool_unseen["prompt_id"] == pid]
            sub_m_counts = sub_p["model"].value_counts().to_dict()
            if all(class_counts.get(m, 0) + c <= samples_per_class for m, c in sub_m_counts.items()):
                chosen_indices.extend(sub_p.index.tolist())
                for m, c in sub_m_counts.items():
                    class_counts[m] = class_counts.get(m, 0) + c

        if chosen_indices:
            sampled_parts.append(df_pool_unseen.loc[chosen_indices])
            rem_pool = df_pool_unseen.drop(index=chosen_indices)
        else:
            rem_pool = df_pool_unseen
    else:
        rem_pool = df_pool_unseen
        class_counts = {m: 0 for m in LABELS}

    # Fill remaining class quotas up to samples_per_class
    for m in LABELS:
        curr_cnt = class_counts.get(m, 0)
        needed = samples_per_class - curr_cnt
        if needed > 0:
            sub = rem_pool[rem_pool["model"] == m]
            if len(sub) > 0:
                n_take = min(needed, len(sub))
                idx = rng.choice(sub.index.values, size=n_take, replace=False)
                sampled_parts.append(sub.loc[idx])

    if not sampled_parts:
        return pd.DataFrame(columns=["id", "prompt_id", "text", "model", "_hash"]), set()

    df_unseen = pd.concat(sampled_parts, ignore_index=True).sample(frac=1.0, random_state=seed).reset_index(drop=True)
    if "id" not in df_unseen.columns:
        df_unseen["id"] = [f"unseen_{i}" for i in range(len(df_unseen))]

    # Compute valid stopword TF-IDF average-linkage clusters (t=0.85) so prompt_id is positive integer 1..G
    if len(df_unseen) >= 2:
        try:
            tfidf = TfidfVectorizer(stop_words="english", max_features=10000, sublinear_tf=True)
            X_tfidf = tfidf.fit_transform(df_unseen["text"].fillna("").tolist())
            D = 1.0 - (X_tfidf @ X_tfidf.T).toarray().clip(0.0, 1.0)
            np.fill_diagonal(D, 0.0)
            cond_d = squareform(D, checks=False)
            clusters = fcluster(linkage(cond_d, method="average"), t=0.85, criterion="distance")
        except Exception:
            clusters = np.arange(1, len(df_unseen) + 1, dtype=int)
    else:
        clusters = np.ones(len(df_unseen), dtype=int)

    df_unseen["cluster_id"] = clusters
    df_unseen["prompt_id"] = clusters

    holdout_hashes = set(df_unseen["_hash"].tolist())
    print(
        f"[Unseen Benchmark] Built strictly unseen holdout set: {len(df_unseen)} rows "
        f"across {df_unseen['model'].nunique()} classes (0% overlap with train.csv & pool_v4.parquet)."
    )
    return df_unseen, holdout_hashes

class UnseenStepwiseAuditor:
    """
    Evaluates every model on the strictly unseen holdout dataset (`df_unseen`),
    reporting:
    1. Table 1 — Standalone (Solo) Unseen Balanced Accuracy (sorted descending).
    2. Table 2 — Greedy Forward Stepwise Selection (optimal combinations, stopping when lift < 0.0).
    3. Table 3 — Exhaustive All-2^M Subset Search & Leave-One-Out (LOO) Attribution.
    """
    def __init__(self, y_unseen: np.ndarray, groups_unseen: np.ndarray, N_test: int):
        self.y_unseen = np.asarray(y_unseen, dtype=int) if len(y_unseen) > 0 else np.array([], dtype=int)
        if groups_unseen is not None and len(groups_unseen) > 0:
            try:
                self.groups_unseen = np.asarray(groups_unseen, dtype=int)
            except (ValueError, TypeError):
                codes, _ = pd.factorize(groups_unseen)
                self.groups_unseen = (codes + 1).astype(int)
        else:
            self.groups_unseen = np.array([], dtype=int)
        self.N_test = int(N_test)
        self.models: List[Dict[str, Any]] = []
        self.greedy_model_indices: List[int] = []
        self.greedy_model_names: List[str] = []
        self.greedy_unseen_lp: np.ndarray = np.zeros((len(self.y_unseen), K), dtype=np.float64)
        self.greedy_test_lp: np.ndarray = np.zeros((self.N_test, K), dtype=np.float64)

    def add_model(
        self,
        name: str,
        *args,
        tier: str = "Flat",
        weight: Optional[float] = None,
        unseen_log_probs: Optional[np.ndarray] = None,
        test_log_probs: Optional[np.ndarray] = None,
        **kwargs,
    ):
        if len(args) == 4:
            tier, weight, unseen_log_probs, test_log_probs = args
        elif len(args) == 3:
            weight, unseen_log_probs, test_log_probs = args
        elif len(args) == 1:
            if isinstance(args[0], (int, float)):
                weight = float(args[0])
            elif isinstance(args[0], str):
                tier = args[0]

        if weight is None:
            weight = float(kwargs.get("weight", 1.0))
        if unseen_log_probs is None:
            unseen_log_probs = kwargs.get("unseen_log_probs", np.zeros((0, K), dtype=np.float64))
        if test_log_probs is None:
            test_log_probs = kwargs.get("test_log_probs", np.zeros((self.N_test, K), dtype=np.float64))

        if len(self.y_unseen) > 0 and len(unseen_log_probs) != len(self.y_unseen):
            return
        u_lp = log_softmax(np.asarray(unseen_log_probs, dtype=np.float64), axis=1) if len(self.y_unseen) > 0 and len(unseen_log_probs) == len(self.y_unseen) else np.zeros((0, K), dtype=np.float64)
        t_lp = log_softmax(np.asarray(test_log_probs, dtype=np.float64), axis=1) if len(test_log_probs) == self.N_test else np.zeros((self.N_test, K), dtype=np.float64)
        solo_raw = float(balanced_accuracy_score(self.y_unseen, u_lp.argmax(axis=1))) if len(self.y_unseen) > 0 else 0.0
        solo_sink = _fast_sinkhorn_bacc(u_lp, self.y_unseen, iters=200) if len(self.y_unseen) >= 18 else solo_raw
        self.models.append({
            "name": name,
            "tier": tier if tier else "Flat",
            "weight": float(weight),
            "unseen_lp": u_lp,
            "test_lp": t_lp,
            "solo_unseen_bacc": solo_raw,
            "solo_unseen_sinkhorn_bacc": solo_sink,
        })

    def get_greedy_ensemble(self) -> Tuple[np.ndarray, np.ndarray, List[str]]:
        """Returns (greedy_test_lp, greedy_unseen_lp, greedy_model_names)."""
        return (
            getattr(self, "greedy_test_lp", np.zeros((self.N_test, K), dtype=np.float64)),
            getattr(self, "greedy_unseen_lp", np.zeros((len(self.y_unseen), K), dtype=np.float64)),
            getattr(self, "greedy_model_names", []),
        )

    def optimize_weights_and_blend(
        self,
        working_dir: Path,
        extra_test_log_probs: Optional[List[Tuple[str, float, np.ndarray]]] = None,
        optimize_on_unseen: bool = True,
        lock_prior_weights: bool = False,
        extra_unseen_log_probs: Optional[List[Tuple[str, float, np.ndarray]]] = None,
        calibration_cfg: Optional[Dict[str, Any]] = None,
        claude_spec_logits_u: Optional[np.ndarray] = None,
        claude_classes: Optional[list] = None,
        **kwargs,
    ) -> Tuple[np.ndarray, Dict[str, float], pd.DataFrame]:
        """
        Calibrated Log-Linear Pooling & Automated OOF Selection Engine.
        1. When `lock_prior_weights` is True (or `optimize_on_unseen` is False / N < 36):
           applies pre-calibrated per-model temperatures `mean_T_m`, weights `mean_w_m`, and
           `mean_T_global` if present in `calibration_cfg`; otherwise blends using model weights.
        2. When `optimize_on_unseen` is True and `len(y_unseen) >= 36`:
           runs 5-fold GroupKFold L-BFGS log-linear pooling with per-model T_m calibration,
           global T_g, and automated OOF backward pruning over all candidate models (no fixed count).
        """
        if lock_prior_weights:
            optimize_on_unseen = False

        if len(self.models) == 0:
            self.P_mix_unseen = np.zeros((0, K), dtype=np.float64)
            return np.ones((self.N_test, K), dtype=np.float64) / float(K), {}, pd.DataFrame()

        for m in self.models:
            m["prior_weight"] = m.get("prior_weight", m["weight"])

        extra_u_lp_base = np.zeros((len(self.y_unseen), K), dtype=np.float64)
        extra_u_weight_tot = 0.0
        if extra_unseen_log_probs and len(self.y_unseen) > 0:
            for _, w_ex, lp_ex in extra_unseen_log_probs:
                if len(lp_ex) == len(self.y_unseen) and w_ex > 0.0:
                    extra_u_lp_base += float(w_ex) * log_softmax(lp_ex, axis=1)
                    extra_u_weight_tot += float(w_ex)

        # Branch A: Pre-calibrated config mode or small/frozen holdout fallback
        if not optimize_on_unseen or len(self.y_unseen) < 36:
            alias_to_disp = {
                "svc_4view": "LinearSVC (4-View)",
                "lexical_5v": "Lexical Comp (5-View)",
                "qwen3_14b_p2": "Qwen3-14B Part-2",
                "qwen3_14b_p1": "Qwen3-14B Part-1",
                "qwen25_14b_p2": "Qwen2.5-14B Part-2",
                "deberta_p2": "DeBERTa-v3-Large",
                "bge_m3": "BGE-M3-v2",
            }
            disp_to_canon = {v: k for k, v in alias_to_disp.items()}

            def _resolve_cal_entry(name: str, mapping: Dict[str, Any]) -> Optional[str]:
                for cand in (name, alias_to_disp.get(name), disp_to_canon.get(name), "BGE-M3" if name in ("bge_m3", "BGE-M3-v2") else None):
                    if cand and cand in mapping:
                        return cand
                return None

            mean_T_m = calibration_cfg.get("mean_T_m", {}) if calibration_cfg else {}
            mean_w_m = calibration_cfg.get("mean_w_m", {}) if calibration_cfg else {}
            clip_min = float(calibration_cfg.get("clip_min", 1e-4)) if calibration_cfg else 1e-4
            resolved_keys = {
                m["name"]: (_resolve_cal_entry(m["name"], mean_T_m), _resolve_cal_entry(m["name"], mean_w_m))
                for m in self.models
            }
            matched_models = [m for m in self.models if all(resolved_keys[m["name"]])]

            if calibration_cfg and "mean_T_global" in calibration_cfg and len(matched_models) > 0:
                m_names = [m["name"] for m in matched_models]
                raw_w = np.array([max(0.0, float(mean_w_m[resolved_keys[n][1]])) for n in m_names], dtype=np.float64)
                w_norm = raw_w / max(float(raw_w.sum()), 1e-12)
                norm_map = {n: float(w_norm[i]) for i, n in enumerate(m_names)}
                t_map = {n: float(mean_T_m[resolved_keys[n][0]]) for n in m_names}
                weights_map: Dict[str, float] = {}
                for m in self.models:
                    w_val = norm_map.get(m["name"], 0.0)
                    m["learned_weight"] = w_val
                    m["weight"] = w_val
                    weights_map[m["name"]] = w_val

                P_mix = (
                    pool_calibrated_log_probs(
                        [m["test_lp"] for m in matched_models],
                        m_names,
                        t_map,
                        norm_map,
                        mean_T_global=float(calibration_cfg["mean_T_global"]),
                        clip_min=clip_min,
                    )
                    if self.N_test > 0
                    else np.zeros((0, K), dtype=np.float64)
                )
                self.P_mix_unseen = (
                    pool_calibrated_log_probs(
                        [m["unseen_lp"] for m in matched_models],
                        m_names,
                        t_map,
                        norm_map,
                        mean_T_global=float(calibration_cfg["mean_T_global"]),
                        clip_min=clip_min,
                    )
                    if len(self.y_unseen) > 0
                    else np.zeros((0, K), dtype=np.float64)
                )
                self.normalized_weights_map = dict(weights_map)
                self.fitted_artifact = {
                    "model_names": m_names,
                    "mean_T_m": dict(t_map),
                    "mean_w_m": dict(norm_map),
                    "mean_T_global": float(calibration_cfg["mean_T_global"]),
                    "clip_min": clip_min,
                }
            else:
                weights_map = {m["name"]: float(m["weight"]) for m in self.models}
                logit_flat = np.zeros((self.N_test, K), dtype=np.float64)
                logit_flat_u = extra_u_lp_base.copy() if len(self.y_unseen) > 0 else np.zeros((0, K), dtype=np.float64)
                for m in self.models:
                    logit_flat += m["weight"] * m["test_lp"]
                    if len(m["unseen_lp"]) == len(self.y_unseen):
                        logit_flat_u += m["weight"] * m["unseen_lp"]
                if extra_test_log_probs:
                    for ex_name, w_ex, lp_ex in extra_test_log_probs:
                        if len(lp_ex) == self.N_test and w_ex > 0.0:
                            logit_flat += float(w_ex) * log_softmax(lp_ex, axis=1)
                            weights_map[ex_name] = float(w_ex)
                P_mix = softmax(logit_flat, axis=1)
                self.P_mix_unseen = softmax(logit_flat_u, axis=1) if len(self.y_unseen) > 0 else np.zeros((0, K), dtype=np.float64)

            df_summary = pd.DataFrame([
                {"model": m["name"], "tier": m["tier"], "prior_weight": m["prior_weight"], "learned_weight": m["weight"]}
                for m in self.models
            ])
            return P_mix, weights_map, df_summary

        # Branch B: Dynamic 5-Fold GroupKFold L-BFGS Pooling + Automated OOF Backward Pruning
        chance_floor = 1.0 / float(K)
        valid_pool = [m for m in self.models if m["solo_unseen_bacc"] > chance_floor]
        if not valid_pool:
            valid_pool = list(self.models)
        cand_names = [m["name"] for m in valid_pool]
        cand_u_lps = [m["unseen_lp"] for m in valid_pool]
        pre_pruned = [m["name"] for m in self.models if m not in valid_pool]
        clip_min = float(calibration_cfg.get("clip_min", 1e-4)) if calibration_cfg else 1e-4
        eps = float(calibration_cfg.get("label_smoothing_eps", 0.05)) if calibration_cfg else 0.05
        sink_alpha = float(calibration_cfg.get("sinkhorn_alpha", 0.50)) if calibration_cfg else 0.50

        best_fit, dropped_names = auto_prune_and_fit_lbfgs(
            raw_lps_list=cand_u_lps,
            y=self.y_unseen,
            groups=self.groups_unseen,
            model_names=cand_names,
            spec_logits=claude_spec_logits_u,
            claude_classes=claude_classes,
            lambdas=(0.05, 0.10, 0.20),
            eps=eps,
            sinkhorn_alpha=sink_alpha,
            clip_min=clip_min,
        )
        dropped_names = pre_pruned + dropped_names

        kept_names = best_fit["model_names"]
        kept_weights = best_fit["mean_w_m"]
        kept_temps = best_fit["mean_T_m"]
        T_global_opt = float(best_fit["mean_T_global"])
        self.fitted_artifact = best_fit

        # Map kept weights across registered models
        weights_map = {}
        normalized_weights_map = {}
        chosen_indices = []
        for idx, m in enumerate(self.models):
            w_val = float(kept_weights.get(m["name"], 0.0)) if m["name"] in kept_names else 0.0
            m["learned_weight"] = w_val
            m["weight"] = w_val
            weights_map[m["name"]] = w_val
            normalized_weights_map[m["name"]] = w_val
            if w_val > 0.0:
                chosen_indices.append(idx)

        kept_models = [m for m in self.models if m["name"] in kept_names and weights_map[m["name"]] > 0.0]
        kept_m_names = [m["name"] for m in kept_models]
        P_mix_base = (
            pool_calibrated_log_probs(
                [m["test_lp"] for m in kept_models],
                kept_m_names,
                kept_temps,
                kept_weights,
                mean_T_global=T_global_opt,
                clip_min=clip_min,
            )
            if self.N_test > 0
            else np.zeros((0, K), dtype=np.float64)
        )
        P_u_base = pool_calibrated_log_probs(
            [m["unseen_lp"] for m in kept_models],
            kept_m_names,
            kept_temps,
            kept_weights,
            mean_T_global=T_global_opt,
            clip_min=clip_min,
        )

        # Incorporate auxiliary external streams (e.g., wpca_dp / spectral_bank) cleanly if passed
        if extra_test_log_probs or (extra_unseen_log_probs and extra_u_weight_tot > 0.0):
            base_te_lp = np.log(np.clip(P_mix_base, 1e-12, 1.0)) if self.N_test > 0 else np.zeros((0, K), dtype=np.float64)
            base_u_lp = np.log(np.clip(P_u_base, 1e-12, 1.0))
            ext_te_tot = 0.0
            ext_te_lp = np.zeros((self.N_test, K), dtype=np.float64)
            if extra_test_log_probs:
                for ex_name, w_ex, lp_ex in extra_test_log_probs:
                    if len(lp_ex) == self.N_test and w_ex > 0.0:
                        ext_te_lp += float(w_ex) * log_softmax(lp_ex, axis=1)
                        ext_te_tot += float(w_ex)
                        weights_map[ex_name] = float(w_ex)
                        normalized_weights_map[ex_name] = float(w_ex) / max(1.0 + float(w_ex), 1e-8)
            P_mix = softmax((base_te_lp + ext_te_lp) / max(1.0 + ext_te_tot, 1.0), axis=1) if self.N_test > 0 else P_mix_base
            self.P_mix_unseen = softmax((base_u_lp + extra_u_lp_base) / max(1.0 + extra_u_weight_tot, 1.0), axis=1)
        else:
            P_mix = P_mix_base
            self.P_mix_unseen = P_u_base

        # Populate report state & log concise 5-fold OOF diagnostics
        self.greedy_model_indices = chosen_indices
        self.greedy_model_names = kept_m_names
        self.greedy_raw_weights = {n: float(kept_weights[n]) for n in kept_m_names}
        self.greedy_norm_weights = {n: float(kept_weights[n]) for n in kept_m_names}
        self.greedy_unseen_lp = np.log(np.clip(self.P_mix_unseen, 1e-12, 1.0))
        self.greedy_test_lp = np.log(np.clip(P_mix, 1e-12, 1.0)) if self.N_test > 0 else np.zeros((0, K), dtype=np.float64)
        self.greedy_unseen_bacc_raw = float(best_fit["oof_bacc_raw"])
        self.greedy_unseen_bacc_sink = float(best_fit["oof_bacc_sink"])
        self.normalized_weights_map = normalized_weights_map

        # Build clean step records in descending weight order for reporting
        step_records = []
        for s_i, n in enumerate(sorted(kept_m_names, key=lambda x: -kept_weights[x]), 1):
            step_records.append({
                "step": s_i,
                "model": n,
                "raw_weight": float(kept_weights[n]),
                "cumul_raw": float(best_fit["oof_bacc_raw"]),
                "cumul_sink": float(best_fit["oof_bacc_sink"]),
                "marginal_lift": 0.0 if s_i == 1 else float(best_fit["oof_bacc_sink"] - self.models[0]["solo_unseen_sinkhorn_bacc"]),
                "test_rows_flipped": 0,
            })
        self.greedy_step_records = step_records

        print(
            f"[Ablation L-BFGS] lam={best_fit['lam']:.2f} | Pruned={dropped_names or 'None'} | "
            f"OOF Raw={best_fit['oof_bacc_raw']:.4f} (NLL={best_fit['oof_nll_raw']:.4f}) | "
            f"+Sink={best_fit['oof_bacc_sink']:.4f} | Tg={T_global_opt:.3f}"
        )

        df_summary = pd.DataFrame([
            {"model": m["name"], "tier": m["tier"], "prior_weight": m["prior_weight"], "learned_weight": m["weight"]}
            for m in self.models
        ])
        return P_mix, weights_map, df_summary

    def evaluate_marginal_lift(
        self,
        candidate_unseen_lp: np.ndarray,
        weight: float = 1.0,
    ) -> float:
        """
        Multi-scale marginal lift search: tests adding `candidate_unseen_lp` across
        multipliers [0.05, 0.10, 0.15, 0.25, 0.35, 0.50, 0.75, 1.00] * weight
        and returns the maximum Delta BAcc achieved at any positive scale.
        """
        if len(self.y_unseen) == 0 or len(candidate_unseen_lp) != len(self.y_unseen) or len(self.models) == 0:
            return 0.0

        cum_unseen_lp = np.zeros((len(self.y_unseen), K), dtype=np.float64)
        for m in self.models:
            w_m = m["weight"] if m["weight"] > 0 else m.get("prior_weight", 0.15)
            cum_unseen_lp += w_m * m["unseen_lp"]

        cur_bacc = _fast_sinkhorn_bacc(cum_unseen_lp, self.y_unseen, iters=40)
        cand_lp = log_softmax(np.asarray(candidate_unseen_lp, dtype=np.float64), axis=1)

        best_lift = -1.0
        full_weight_lift = 0.0
        for mult in (0.05, 0.10, 0.15, 0.25, 0.35, 0.50, 0.75, 1.00):
            cand_w = float(weight) * mult
            new_unseen_lp = cum_unseen_lp + cand_w * cand_lp
            new_bacc = _fast_sinkhorn_bacc(new_unseen_lp, self.y_unseen, iters=40)
            lift = new_bacc - cur_bacc
            if mult == 1.00:
                full_weight_lift = lift
            if lift > best_lift:
                best_lift = lift

        if best_lift <= 0.0:
            return float(full_weight_lift)
        return float(best_lift)

    def print_and_save_report(self, working_dir: Path, final_test_preds: Optional[np.ndarray] = None) -> pd.DataFrame:
        if not self.models or len(self.y_unseen) == 0:
            print("[Ablation] No models or unseen holdout samples available; skipping report.")
            return pd.DataFrame()

        M = len(self.models)

        # Part 1: Standalone (Solo) Accuracy, sorted descending
        print("\n" + "=" * 105)
        print(f" PART 1: STANDALONE (SOLO) ACCURACY ON STRICTLY UNSEEN HOLDOUT ({len(self.y_unseen)} Disjoint Passages)")
        print("=" * 105)
        print(f"{'Model / Checkpoint':<28} | {'Tier':<6} | {'Unseen Raw BAcc':<16} | {'Unseen+Sinkhorn BAcc':<20} | {'Agrees w/ Final Test'}")
        print("-" * 105)
        sorted_solo = sorted(self.models, key=lambda m: m["solo_unseen_sinkhorn_bacc"], reverse=True)
        for m in sorted_solo:
            agree_str = "N/A"
            if final_test_preds is not None and self.N_test > 0 and len(final_test_preds) == self.N_test:
                pct = 100.0 * float((m["test_lp"].argmax(axis=1) == final_test_preds).mean())
                agree_str = f"{pct:5.1f}%"
            print(
                f"{m['name']:<28} | {m['tier']:<6} | "
                f"{m['solo_unseen_bacc']:<16.4f} | {m['solo_unseen_sinkhorn_bacc']:<20.4f} | {agree_str}"
            )

        # Part 2: Greedy Forward Stepwise Selection (Single Truth from Caruana selection)
        print("\n" + "=" * 115)
        print(" PART 2: GREEDY FORWARD STEPWISE SELECTION ON STRICTLY UNSEEN HOLDOUT (Optimal Combinations)")
        print("=" * 115)
        print(f"{'Step':<5} | {'Added Model':<26} | {'Raw Wt':<7} | {'Share (%)':<10} | {'Cumul Raw':<10} | {'Cumul+Sink':<11} | {'Marginal Lift':<13} | {'Test Rows Flipped'}")
        print("-" * 115)

        records = list(getattr(self, "greedy_step_records", []))
        if records:
            tot_gw = sum(r.get("raw_weight", r.get("weight", 0.0)) for r in records)
            for r in records:
                w_val = r.get("raw_weight", r.get("weight", 0.0))
                pct = (w_val / tot_gw * 100.0) if tot_gw > 0 else 0.0
                lift_str = f"{r['marginal_lift']:+.4f}" if r["step"] > 1 else "BASELINE"
                flips = r.get("test_rows_flipped", 0)
                flip_str = f"{flips} (Init)" if r["step"] == 1 else f"{flips:4d} ({100.0 * flips / max(1, self.N_test):4.1f}%)"
                c_raw = r.get("cumul_raw", r.get("cumul_unseen_raw_bacc", 0.0))
                c_sink = r.get("cumul_sink", r.get("cumul_unseen_sinkhorn_bacc", 0.0))
                print(
                    f"{r['step']:<5} | + {r['model']:<24} | {w_val:<7.2f} | {pct:>7.1f}%   | "
                    f"{c_raw:<10.4f} | {c_sink:<11.4f} | {lift_str:<13} | {flip_str}"
                )
            print("-" * 115)
            print(f"Total | {len(getattr(self, 'greedy_model_indices', []))} Active Models{'':<13} | {tot_gw:<7.2f} |   100.0%   | Final Greedy Optimal Subset (+Sinkhorn BAcc: {getattr(self, 'greedy_unseen_bacc_sink', 0.0):.4f})")
            print("=" * 115 + "\n")
        else:
            M = len(self.models)
            remaining = list(range(M))
            cum_unseen_lp = np.zeros((len(self.y_unseen), K), dtype=np.float64)
            cum_test_lp = np.zeros((self.N_test, K), dtype=np.float64)
            prev_bacc = 0.0
            prev_test_preds = None
            chosen_indices = []

            step_idx = 1
            while remaining:
                best_cand_idx = None
                best_cand_bacc = -1.0
                best_cand_raw = 0.0

                if step_idx == 1:
                    for idx in remaining:
                        cand = self.models[idx]
                        cand_w = cand["weight"] if cand["weight"] > 0 else cand.get("prior_weight", 0.15)
                        cand_lp = cand_w * cand["unseen_lp"]
                        cand_bacc = _fast_sinkhorn_bacc(cand_lp, self.y_unseen, iters=40)
                        if cand_bacc > best_cand_bacc:
                            best_cand_bacc = cand_bacc
                            best_cand_idx = idx
                            best_cand_raw = float(balanced_accuracy_score(self.y_unseen, cand_lp.argmax(axis=1)))
                else:
                    for idx in remaining:
                        cand = self.models[idx]
                        cand_w = cand["weight"] if cand["weight"] > 0 else cand.get("prior_weight", 0.15)
                        cand_lp = cum_unseen_lp + cand_w * cand["unseen_lp"]
                        cand_bacc = _fast_sinkhorn_bacc(cand_lp, self.y_unseen, iters=40)
                        if cand_bacc > best_cand_bacc:
                            best_cand_bacc = cand_bacc
                            best_cand_idx = idx
                            best_cand_raw = float(balanced_accuracy_score(self.y_unseen, cand_lp.argmax(axis=1)))

                if best_cand_idx is None:
                    break

                lift = best_cand_bacc - prev_bacc if step_idx > 1 else 0.0
                if step_idx > 1 and lift < 0.0:
                    print(f"[Greedy Forward Selection] Best remaining candidate '{self.models[best_cand_idx]['name']}' gives negative lift ({lift:+.4f}); stopping forward selection.")
                    break

                m = self.models[best_cand_idx]
                remaining.remove(best_cand_idx)
                chosen_indices.append(best_cand_idx)

                w = m["weight"] if m["weight"] > 0 else m.get("prior_weight", 0.15)
                cum_unseen_lp = cum_unseen_lp + w * m["unseen_lp"]
                cum_test_lp = cum_test_lp + w * m["test_lp"]

                lift_str = f"{lift:+.4f}" if step_idx > 1 else "BASELINE"
                cur_test_preds = _ablation_sinkhorn(softmax(cum_test_lp, axis=1), iters=40).argmax(axis=1) if self.N_test > 0 else np.array([])
                flips = self.N_test if prev_test_preds is None else int((cur_test_preds != prev_test_preds).sum())
                flip_str = f"{flips} (Init)" if prev_test_preds is None else f"{flips:4d} ({100.0 * flips / max(1, self.N_test):4.1f}%)"

                print(
                    f"{step_idx:<5} | + {m['name']:<24} | {w:<7.2f} |       -    | "
                    f"{best_cand_raw:<10.4f} | {best_cand_bacc:<11.4f} | {lift_str:<13} | {flip_str}"
                )
                prev_bacc = best_cand_bacc
                prev_test_preds = cur_test_preds
                records.append({
                    "step": step_idx,
                    "model": m["name"],
                    "added_model": m["name"],
                    "tier": m["tier"],
                    "weight": w,
                    "raw_weight": w,
                    "solo_unseen_bacc": m["solo_unseen_bacc"],
                    "solo_unseen_sinkhorn_bacc": m["solo_unseen_sinkhorn_bacc"],
                    "cumul_unseen_raw_bacc": best_cand_raw,
                    "cumul_unseen_sinkhorn_bacc": best_cand_bacc,
                    "marginal_lift": lift,
                    "test_rows_flipped": flips,
                })
                step_idx += 1

            total_gw = sum(max(0.0, float(self.models[i]["weight"] if self.models[i]["weight"] > 0 else self.models[i].get("prior_weight", 0.15))) for i in chosen_indices)
            gw_scale = (1.0 / total_gw) if total_gw > 0.0 else 1.0
            self.greedy_model_indices = chosen_indices
            self.greedy_model_names = [self.models[i]["name"] for i in chosen_indices]
            self.greedy_unseen_lp = cum_unseen_lp * gw_scale
            self.greedy_test_lp = cum_test_lp * gw_scale
            self.greedy_unseen_bacc_raw = float(balanced_accuracy_score(self.y_unseen, cum_unseen_lp.argmax(axis=1))) if len(self.y_unseen) > 0 else 0.0
            self.greedy_unseen_bacc_sink = prev_bacc
            print("=" * 115 + "\n")

        # Part 3: Exhaustive All 2^M Subset Search & LOO Attribution (for M <= 12)
        if 2 <= M <= 12:
            print("=" * 105)
            print(f" PART 3: GLOBALLY OPTIMAL SUBSET & LEAVE-ONE-OUT (LOO) ATTRIBUTION (All {2**M - 1} Non-Empty Subsets)")
            print("=" * 105)

            best_subset = None
            best_subset_bacc = -1.0

            for r in range(1, M + 1):
                for sub in combinations(range(M), r):
                    sub_unseen_lp = sum((self.models[i]["weight"] if self.models[i]["weight"] > 0 else self.models[i].get("prior_weight", 0.15)) * self.models[i]["unseen_lp"] for i in sub)
                    bacc = _fast_sinkhorn_bacc(sub_unseen_lp, self.y_unseen, iters=30)
                    if bacc > best_subset_bacc:
                        best_subset_bacc = bacc
                        best_subset = sub

            opt_names = [self.models[i]["name"] for i in best_subset]
            print(f"Globally Optimal Ensemble BAcc: {best_subset_bacc:.4f} (Uses {len(best_subset)} / {M} models)")
            print(f"Optimal Set: {', '.join(opt_names)}")
            print("-" * 105)
            print(f"{'Model / Checkpoint':<28} | {'In Optimal?':<12} | {'Weight':<7} | {'Delta_LOO (BAcc Drop If Excluded)'}")
            print("-" * 105)

            opt_set = set(best_subset)
            for i, m in enumerate(self.models):
                in_opt = i in opt_set
                in_str = "YES" if in_opt else "NO (Pruned)"
                if in_opt:
                    if len(best_subset) == 1:
                        delta_loo = best_subset_bacc
                    else:
                        loo_sub = [j for j in best_subset if j != i]
                        loo_lp = sum((self.models[j]["weight"] if self.models[j]["weight"] > 0 else self.models[j].get("prior_weight", 0.15)) * self.models[j]["unseen_lp"] for j in loo_sub)
                        loo_bacc = _fast_sinkhorn_bacc(loo_lp, self.y_unseen, iters=30)
                        delta_loo = best_subset_bacc - loo_bacc
                    loo_str = f"-{delta_loo:.4f}" if delta_loo >= 0 else f"+{abs(delta_loo):.4f}"
                else:
                    with_m_sub = list(best_subset) + [i]
                    with_m_lp = sum((self.models[j]["weight"] if self.models[j]["weight"] > 0 else self.models[j].get("prior_weight", 0.15)) * self.models[j]["unseen_lp"] for j in with_m_sub)
                    with_m_bacc = _fast_sinkhorn_bacc(with_m_lp, self.y_unseen, iters=30)
                    delta_add = with_m_bacc - best_subset_bacc
                    loo_str = f"{delta_add:+.4f} (Degrades if added)"

                print(f"{m['name']:<28} | {in_str:<12} | {m['weight']:<7.2f} | {loo_str}")

                for r in records:
                    if r["model"] == m["name"]:
                        r["in_optimal_subset"] = in_opt
                        r["delta_loo"] = delta_loo if in_opt else 0.0

            print("=" * 105 + "\n")

        df_rep = pd.DataFrame(records)
        working_dir = Path(working_dir)
        working_dir.mkdir(parents=True, exist_ok=True)
        df_rep.to_csv(working_dir / "unseen_stepwise_ablation.csv", index=False)
        return df_rep


In [ ]:
# === MODULE: src/namethatllm/layer_probing.py ===
"""
Layer-by-Layer Probing, Pairwise Sister Fisher Subspace, and Multi-View Feature Engineering
===========================================================================================
Provides:
1. build_supervised_family_subspace:
   Constructs a whitened linear projection matrix C_eff (subspace_dim, 40960) and mean vector
   mu_fam (1, 40960) combining:
     - Per-layer Pairwise Sister Fisher Discriminant directions: (Sigma_W + reg * I)^-1 (mu_a - mu_b)
     - Inter-layer Residual Velocity Fisher directions on Delta h^(l) = h^(l) - h^(l-1)
     - Per-layer multi-class Ridge probe weights
     - Orthogonalized intra-family residual PCA components
   Because C_eff is a linear operator on 40,960-dim Z_cat, it folds with 100% mathematical
   exactness into mlp_w1 / mlp_b1 and plain.weight / plain.bias for zero-cost inference.

2. run_all_layers_probe_sweep:
   Automated transformer-heads style probe across all L layers ([N, L, hidden_size]),
   reporting overall family Balanced Accuracy and hardest-sister-pair accuracy per layer,
   plus learned ScalarMix layer aggregation weights.

3. engineer_multiview_features:
   Extracts rich tabular/neural features from exported Kaggle Dataset bundles
   (head-tail cosine drift, inter-layer velocity norms, token dispersion ratios, and OOF log-odds).

4. LayerScalarMix & LayerScalarMixWithHead:
   Differentiable convex combination of layer representations with learnable softmax weights.

5. fit_trace_scaled_ridge_probe:
   Closed-form trace-scaled Ridge regression in primal (N >= D) or dual (N < D) regimes.
"""

from __future__ import annotations

import json
from itertools import combinations
from pathlib import Path
from typing import Any

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import GroupKFold, StratifiedKFold


# ============================================================================
# Layer ScalarMix Modules & Ridge Probing
# ============================================================================

class LayerScalarMix(nn.Module):
    """
    Differentiable convex combination of layer-normalized hidden states.

    Formula:
        h_mix = gamma * sum_{l=0}^{L-1} softmax(w)_l * LayerNorm_l(h_l)
    """

    def __init__(self, num_layers: int, hidden_dim: int):
        super().__init__()
        self.num_layers = num_layers
        self.hidden_dim = hidden_dim
        self.layer_norms = nn.ModuleList([
            nn.LayerNorm(hidden_dim, elementwise_affine=False) for _ in range(num_layers)
        ])
        self.scalar_weights = nn.Parameter(torch.zeros(num_layers))
        self.gamma = nn.Parameter(torch.ones(1))

    def forward(self, layer_pooled_states: torch.Tensor | list[torch.Tensor]) -> torch.Tensor:
        if isinstance(layer_pooled_states, torch.Tensor):
            normed = [
                self.layer_norms[l](layer_pooled_states[:, l, :])
                for l in range(layer_pooled_states.shape[1])
            ]
            stacked = torch.stack(normed, dim=1)
        else:
            normed = [self.layer_norms[l](h) for l, h in enumerate(layer_pooled_states)]
            stacked = torch.stack(normed, dim=1)

        probs = F.softmax(self.scalar_weights, dim=0).view(1, -1, 1)
        mixed = self.gamma * torch.sum(stacked * probs, dim=1)
        return mixed

    def get_layer_importance(self) -> dict[int, float]:
        probs = F.softmax(self.scalar_weights.detach(), dim=0).cpu().tolist()
        return {idx: round(p, 4) for idx, p in enumerate(probs)}


class LayerScalarMixWithHead(nn.Module):
    """
    ScalarMix layer aggregation combined with an end-to-end linear classification probe.
    """

    def __init__(self, num_layers: int, hidden_dim: int, num_classes: int, dropout: float = 0.10):
        super().__init__()
        self.mix = LayerScalarMix(num_layers, hidden_dim)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_dim, num_classes)

    def forward(self, layer_pooled_states: torch.Tensor | list[torch.Tensor]) -> torch.Tensor:
        mixed = self.mix(layer_pooled_states)
        return self.classifier(self.dropout(mixed))

    def get_layer_importance(self) -> dict[int, float]:
        return self.mix.get_layer_importance()


def fit_trace_scaled_ridge_probe(
    X_train: np.ndarray,
    y_train: np.ndarray,
    num_classes: int,
    alpha: float = 1.0,
    label_smoothing: float = 0.05,
    device: torch.device | None = None,
) -> tuple[np.ndarray, np.ndarray]:
    """
    Fits closed-form trace-scaled Ridge regression in PyTorch.
    Returns (W, b) where W is [D, num_classes] and b is [num_classes].
    """
    if device is None:
        device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

    N, D = X_train.shape
    Y_onehot = np.zeros((N, num_classes), dtype=np.float32)
    for i, lab in enumerate(y_train):
        if 0 <= lab < num_classes:
            Y_onehot[i, lab] = 1.0
    Y_smooth = (1.0 - label_smoothing) * Y_onehot + (label_smoothing / float(num_classes))

    X_t = torch.from_numpy(X_train).to(device, dtype=torch.float32)
    Y_t = torch.from_numpy(Y_smooth).to(device, dtype=torch.float32)

    mu_X = X_t.mean(dim=0, keepdim=True)
    mu_Y = Y_t.mean(dim=0, keepdim=True)
    X_c = X_t - mu_X
    Y_c = Y_t - mu_Y

    scale = float(torch.sum(X_c ** 2).item() / max(N, 1))
    eff_alpha = max(alpha * scale, 1e-6)

    use_primal = (N >= D)
    if use_primal:
        Cov = X_c.T @ X_c
        reg = eff_alpha * torch.eye(D, device=device, dtype=torch.float32)
        W = torch.linalg.solve(Cov + reg, X_c.T @ Y_c)
    else:
        K_mat = X_c @ X_c.T
        reg = eff_alpha * torch.eye(N, device=device, dtype=torch.float32)
        A = torch.linalg.solve(K_mat + reg, Y_c)
        W = X_c.T @ A

    b = (mu_Y - mu_X @ W).squeeze(0)
    W_np = W.cpu().numpy().astype(np.float32)
    b_np = b.cpu().numpy().astype(np.float32)

    del X_t, Y_t, X_c, Y_c, W, b
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return W_np, b_np


# ============================================================================
# Supervised Within-Class Fisher Discriminant & Sister Decoupling Subspace
# ============================================================================

def _solve_woodbury_direction(
    R_sub: torch.Tensor,
    Delta_sub: torch.Tensor,
    reg_scale: float,
) -> torch.Tensor:
    N_s, D_s = R_sub.shape
    trace_val = float(torch.sum(R_sub * R_sub).item() / max(N_s, 1))
    lam = max(reg_scale * (trace_val / max(D_s, 1)), 1e-4)
    if N_s >= D_s:
        Cov_W = (R_sub.T @ R_sub) / float(max(N_s, 1))
        Cov_W.diagonal().add_(lam)
        W_sol = torch.linalg.solve(Cov_W, Delta_sub)
    else:
        K_gram = R_sub @ R_sub.T
        K_gram.diagonal().add_(float(max(N_s, 1)) * lam)
        R_Delta = R_sub @ Delta_sub
        Dual_sol = torch.linalg.solve(K_gram, R_Delta)
        W_sol = (Delta_sub - R_sub.T @ Dual_sol) / lam
    return torch.nn.functional.normalize(W_sol, p=2, dim=0, eps=1e-6)


def _solve_within_class_fisher_gpu(
    X_fam: np.ndarray,
    y_fam: np.ndarray,
    num_family_classes: int,
    reg_ratio: float = 0.35,
    reg_scales: tuple[float, ...] = (0.20, 0.60, 1.50),
    device: torch.device | None = None,
) -> tuple[np.ndarray, np.ndarray]:
    """
    100% Pure Supervised Multi-Scale Fisher Solver (Zero PCA):
    1. Solves multi-class OVR Fisher directions on pooled intra-family residuals.
    2. Solves pairwise sister Fisher directions on pair-specific residuals R_ab
       across multiple spectral regularization scales (reg_scales).
    """
    if device is None:
        device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

    N, D = X_fam.shape
    X_t = torch.from_numpy(X_fam).to(device, dtype=torch.float32)
    mu_all = X_t.mean(dim=0, keepdim=True)

    centroids = []
    R_t = X_t.clone()
    for c in range(num_family_classes):
        mask = (y_fam == c)
        if mask.sum() > 0:
            idx_t = torch.from_numpy(np.flatnonzero(mask)).to(device)
            mu_c = X_t[idx_t].mean(dim=0, keepdim=True)
            centroids.append(mu_c.squeeze(0))
            R_t[idx_t] -= mu_c
        else:
            centroids.append(mu_all.squeeze(0))

    # 1. Multi-class OVR Fisher directions
    rhs_list = [centroids[c] - mu_all.squeeze(0) for c in range(num_family_classes)]
    Delta_ovr = torch.stack(rhs_list, dim=1) if rhs_list else torch.zeros((D, 1), device=device, dtype=torch.float32)
    W_ovr_t = _solve_woodbury_direction(R_t, Delta_ovr, reg_scale=reg_ratio)
    W_ovr = W_ovr_t.cpu().numpy().T  # (K, D)

    # 2. Pairwise Sister Fisher directions on pair-specific covariance across reg_scales
    pair_dirs = []
    for a, b in combinations(range(num_family_classes), 2):
        delta_ab = (centroids[a] - centroids[b]).unsqueeze(1)  # (D, 1)
        mask_ab = (y_fam == a) | (y_fam == b)
        if mask_ab.sum() >= 4:
            idx_ab = torch.from_numpy(np.flatnonzero(mask_ab)).to(device)
            R_ab = R_t[idx_ab]
        else:
            R_ab = R_t
        for r_scale in reg_scales:
            w_ab = _solve_woodbury_direction(R_ab, delta_ab, reg_scale=r_scale)
            pair_dirs.append(w_ab.squeeze(1).cpu().numpy())

    W_pairs = np.vstack(pair_dirs).astype(np.float32) if pair_dirs else np.zeros((0, D), dtype=np.float32)

    del X_t, R_t, Delta_ovr, W_ovr_t
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return W_ovr, W_pairs


def _compute_layer_oof_scores(
    Z_fam_c: np.ndarray,
    y_fam: np.ndarray,
    num_tapped_layers: int,
    num_family_classes: int,
    hard_sister_pair: tuple[int, int] | None = None,
    prompt_ids: list[Any] | np.ndarray | None = None,
    alpha: float = 5.0,
    n_splits: int = 5,
    device: torch.device | None = None,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Computes 5-fold OOF Ridge probe scores s_l = 0.5 * family_oof_bacc + 0.5 * hard_pair_oof_acc
    for each tapped layer block X_l in Z_fam_c.
    Returns (layer_weights, oof_baccs, composite_scores).
    """
    if device is None:
        device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    else:
        device = torch.device(device)

    N, D_total = Z_fam_c.shape
    D_layer = D_total // max(num_tapped_layers, 1)

    min_class_count = min(int((y_fam == c).sum()) for c in range(num_family_classes)) if num_family_classes > 0 else 0
    if N < 5 or min_class_count < 2:
        uniform_w = np.full(num_tapped_layers, 1.0 / max(num_tapped_layers, 1), dtype=np.float32)
        ones_bacc = np.ones(num_tapped_layers, dtype=np.float32)
        return uniform_w, ones_bacc, ones_bacc

    n_splits_eff = max(min(n_splits, min_class_count), 2)
    p_arr = np.asarray(prompt_ids) if prompt_ids is not None else None
    use_groups = False
    if p_arr is not None and len(p_arr) == N:
        u_p = np.unique(p_arr)
        if len(u_p) >= n_splits_eff and len(u_p) < 0.8 * N:
            use_groups = True

    if use_groups:
        splitter = GroupKFold(n_splits=n_splits_eff).split(Z_fam_c[:, :D_layer], y_fam, groups=p_arr)
    else:
        splitter = StratifiedKFold(n_splits=n_splits_eff, shuffle=True, random_state=42).split(Z_fam_c[:, :D_layer], y_fam)
    folds = list(splitter)

    Y_oh = np.eye(num_family_classes, dtype=np.float32)[y_fam]
    Y_oh = 0.95 * Y_oh + (0.05 / float(max(num_family_classes, 1)))

    scores = []
    bacc_list = []
    for l_idx in range(num_tapped_layers):
        s_col = l_idx * D_layer
        e_col = (l_idx + 1) * D_layer
        X_l = Z_fam_c[:, s_col:e_col]

        oof_l = np.zeros((N, num_family_classes), dtype=np.float32)
        for tr_idx, val_idx in folds:
            X_tr_t = torch.from_numpy(X_l[tr_idx]).float().to(device)
            Y_tr_t = torch.from_numpy(Y_oh[tr_idx]).float().to(device)
            X_val_t = torch.from_numpy(X_l[val_idx]).float().to(device)

            mu_tr = X_tr_t.mean(dim=0, keepdim=True)
            X_tr_c = X_tr_t - mu_tr
            X_val_c = X_val_t - mu_tr

            N_tr, D_l = X_tr_c.shape
            scale = float((X_tr_c * X_tr_c).sum(dim=1).mean().item())
            eff_alpha = float(alpha * max(scale / 512.0, 1e-3))
            Y_tr_c = Y_tr_t - Y_tr_t.mean(dim=0, keepdim=True)

            if N_tr >= D_l:
                Cov = X_tr_c.T @ X_tr_c
                Cov.diagonal().add_(eff_alpha)
                W_l = torch.linalg.solve(Cov, X_tr_c.T @ Y_tr_c)
                b_l = Y_tr_t.mean(dim=0)
            else:
                K_mat = X_tr_c @ X_tr_c.T
                K_mat.diagonal().add_(eff_alpha)
                A = torch.linalg.solve(K_mat, Y_tr_c)
                W_l = X_tr_c.T @ A
                b_l = Y_tr_t.mean(dim=0)

            oof_val = X_val_c @ W_l + b_l
            oof_l[val_idx] = oof_val.cpu().numpy()
            del X_tr_t, Y_tr_t, X_val_t, X_tr_c, X_val_c, W_l, b_l

        pred_labels = oof_l.argmax(axis=1)
        fam_bacc = float(balanced_accuracy_score(y_fam, pred_labels))
        bacc_list.append(fam_bacc)

        if hard_sister_pair is not None:
            s_a, s_b = hard_sister_pair
            pair_mask = (y_fam == s_a) | (y_fam == s_b)
            if pair_mask.sum() > 0 and len(np.unique(y_fam[pair_mask])) == 2:
                pair_preds = np.where(oof_l[pair_mask, s_a] >= oof_l[pair_mask, s_b], s_a, s_b)
                pair_acc = float(balanced_accuracy_score(y_fam[pair_mask], pair_preds))
            else:
                pair_acc = fam_bacc
        else:
            pair_acc = fam_bacc

        s_l = 0.5 * fam_bacc + 0.5 * pair_acc
        scores.append(s_l)

    scores_arr = np.array(scores, dtype=np.float32)
    shifted = (scores_arr - scores_arr.max()) * 12.0
    exp_s = np.exp(shifted)
    w_layers = exp_s / np.sum(exp_s)
    return w_layers, np.array(bacc_list, dtype=np.float32), scores_arr


def build_supervised_family_subspace(
    Z_train: np.ndarray,
    y_train: np.ndarray,
    num_family_classes: int,
    num_tapped_layers: int = 8,
    subspace_dim: int | None = 96,
    reg_ratio: float = 0.35,
    device: torch.device | None = None,
    include_velocity: bool = True,
    layer_weighting: bool = True,
    hard_sister_pair: tuple[int, int] | None = None,
    prompt_ids: list[Any] | np.ndarray | None = None,
    **kwargs: Any,
) -> tuple[np.ndarray, np.ndarray, dict[str, Any]]:
    """
    Constructs a 100% Pure Supervised Fisher Subspace (Zero Unsupervised PCA):
      1. Part 1: Per-Layer Multi-Scale Pairwise Sister + OVR Fisher directions across all tapped layers,
         optionally weighted by layer importance w_l = softmax(12 * s_l).
      2. Part 2: Inter-Layer Residual Velocity Fisher directions on Delta X^(l) = X^(l) - X^(l-1),
         optionally enabled via include_velocity.
      3. Part 3: Economic QR orthonormalization (filtering collinear directions via diag_r > 5e-2 * max).
      4. Part 4: Unit-variance coordinate whitening by intra-family training std.
    """
    if subspace_dim is None:
        subspace_dim = 96
    if device is None:
        device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

    N_total, D_total = Z_train.shape
    D_layer = D_total // max(num_tapped_layers, 1)
    assert D_layer * num_tapped_layers == D_total, (
        f"D_total ({D_total}) must be divisible by num_tapped_layers ({num_tapped_layers})"
    )

    fam_mask = (y_train < num_family_classes)
    if fam_mask.sum() == 0:
        fam_mask = np.ones(len(y_train), dtype=bool)

    Z_fam = Z_train[fam_mask]
    y_fam = y_train[fam_mask]
    fam_pids = (
        np.asarray(prompt_ids)[fam_mask]
        if (prompt_ids is not None and len(prompt_ids) == len(y_train))
        else prompt_ids
    )

    mu_fam = Z_fam.mean(axis=0, keepdims=True).astype(np.float32)
    Z_fam_c = (Z_fam - mu_fam).astype(np.float32)

    # Resolve default hard_sister_pair if None
    if hard_sister_pair is None:
        if num_family_classes == 4:
            hard_sister_pair = (2, 3)
        elif num_family_classes == 5:
            hard_sister_pair = (1, 2)
        elif num_family_classes == 2:
            hard_sister_pair = (0, 1)

    # Compute Layer-Probing Importance Weights if layer_weighting is enabled
    if layer_weighting:
        w_layers, oof_baccs, comp_scores = _compute_layer_oof_scores(
            Z_fam_c=Z_fam_c,
            y_fam=y_fam,
            num_tapped_layers=num_tapped_layers,
            num_family_classes=num_family_classes,
            hard_sister_pair=hard_sister_pair,
            prompt_ids=fam_pids,
        )
    else:
        w_layers = np.full(num_tapped_layers, 1.0 / max(num_tapped_layers, 1), dtype=np.float32)
        oof_baccs = np.zeros(num_tapped_layers, dtype=np.float32)
        comp_scores = np.zeros(num_tapped_layers, dtype=np.float32)

    basis_rows: list[np.ndarray] = []
    layer_fisher_norms: dict[int, float] = {}

    # Part 1: Per-Layer Multi-Scale Fisher Directions (unscaled before QR)
    for l_idx in range(num_tapped_layers):
        s_col = l_idx * D_layer
        e_col = (l_idx + 1) * D_layer
        X_l = Z_fam_c[:, s_col:e_col]

        W_ovr, W_pairs = _solve_within_class_fisher_gpu(
            X_l, y_fam, num_family_classes=num_family_classes, reg_ratio=reg_ratio,
            reg_scales=(0.20, 0.60, 1.50), device=device
        )
        stacked_dirs = np.vstack([W_ovr, W_pairs]) if len(W_pairs) > 0 else W_ovr
        for row in stacked_dirs:
            full_vec = np.zeros(D_total, dtype=np.float32)
            full_vec[s_col:e_col] = row
            basis_rows.append(full_vec)

        layer_fisher_norms[l_idx] = float(np.linalg.norm(W_pairs)) if len(W_pairs) > 0 else 0.0

    # Part 2: Inter-Layer Residual Velocity Fisher Directions: Delta X^(l) = X^(l) - X^(l-1)
    if include_velocity:
        for l_idx in range(1, num_tapped_layers):
            s_prev, e_prev = (l_idx - 1) * D_layer, l_idx * D_layer
            s_curr, e_curr = l_idx * D_layer, (l_idx + 1) * D_layer
            Delta_X = Z_fam_c[:, s_curr:e_curr] - Z_fam_c[:, s_prev:e_prev]

            _, W_vel_pairs = _solve_within_class_fisher_gpu(
                Delta_X, y_fam, num_family_classes=num_family_classes, reg_ratio=reg_ratio * 1.5,
                reg_scales=(0.30, 0.80, 2.00), device=device
            )
            for row in W_vel_pairs:
                full_vec = np.zeros(D_total, dtype=np.float32)
                full_vec[s_curr:e_curr] = row / np.sqrt(2.0)
                full_vec[s_prev:e_prev] = -row / np.sqrt(2.0)
                basis_rows.append(full_vec)

    W_sup = np.vstack(basis_rows)  # (D_sup, 40960)

    # Part 3: Orthonormalize pure Fisher directions via reduced QR (rejecting float32 noise)
    Q_sup, R_sup = np.linalg.qr(W_sup.T, mode="reduced")
    diag_r = np.abs(np.diag(R_sup))
    r_thr = max(float(diag_r.max()) * 2e-2, 1e-4) if len(diag_r) > 0 else 1e-4
    valid_sup = diag_r > r_thr
    V_basis = Q_sup[:, valid_sup].T.astype(np.float32)  # (D_fisher, 40960)
    if subspace_dim is not None and subspace_dim > 0 and V_basis.shape[0] > subspace_dim:
        V_basis = V_basis[:subspace_dim]

    # Part 4: Whiten each Fisher coordinate to std=1.0 first, THEN apply post-whitening layer volume boost
    Z_fam_proj = Z_fam_c @ V_basis.T  # (N_fam, D_fisher)
    sd_proj = Z_fam_proj.std(axis=0, keepdims=True).clip(min=1e-3).astype(np.float32)
    C_white = (V_basis / sd_proj.T).astype(np.float32)

    if layer_weighting:
        V_blocks = V_basis.reshape(V_basis.shape[0], num_tapped_layers, D_layer)
        block_energy = np.sum(V_blocks * V_blocks, axis=2)  # (D_fisher, num_tapped_layers)
        layer_mult = float(num_tapped_layers) * w_layers[None, :]
        dir_scales = np.sqrt(np.sum(block_energy * layer_mult, axis=1, keepdims=True)).astype(np.float32)
        C_eff = (C_white * dir_scales).astype(np.float32)
    else:
        C_eff = C_white

    diagnostics = {
        "n_supervised_dirs": int(V_basis.shape[0]),
        "n_residual_pca_dirs": 0,
        "total_subspace_dim": int(V_basis.shape[0]),
        "subspace_dim": int(V_basis.shape[0]),
        "layer_fisher_norms": layer_fisher_norms,
        "layer_scalar_weights": [float(w) for w in w_layers],
        "per_layer_oof_bacc": [float(b) for b in oof_baccs],
        "layer_composite_scores": [float(s) for s in comp_scores],
        "include_velocity": include_velocity,
        "layer_weighting": layer_weighting,
    }
    return C_eff, mu_fam, diagnostics


def build_oof_supervised_family_subspace(
    Z_train: np.ndarray,
    y_train: np.ndarray,
    Z_eval: np.ndarray | None = None,
    num_family_classes: int = 4,
    num_tapped_layers: int = 8,
    subspace_dim: int | None = 96,
    reg_ratio: float = 0.35,
    device: torch.device | None = None,
    include_velocity: bool = True,
    layer_weighting: bool = True,
    hard_sister_pair: tuple[int, int] | None = None,
    prompt_ids: list[Any] | np.ndarray | None = None,
    n_splits: int = 5,
    random_state: int = 42,
    **kwargs: Any,
) -> tuple[np.ndarray, np.ndarray | None, np.ndarray, np.ndarray, dict[str, Any]]:
    """
    Out-of-Fold (OOF) Supervised Fisher Subspace Stacking:
    Eliminates 99%+ training/OOF label leakage by fitting Fisher bases strictly on (K-1) folds
    and projecting held-out validation folds, aligned to the full-dataset Fisher basis
    via orthogonal Procrustes / least-squares alignment.

    Returns: (Z_tr_oof, Z_ev_active, C_full, mu_full, diagnostics)
    """
    if device is None:
        device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

    # 1. Fits C_full, mu_full, diag on the full training set to serve as the target basis
    C_full, mu_full, diag = build_supervised_family_subspace(
        Z_train=Z_train,
        y_train=y_train,
        num_family_classes=num_family_classes,
        num_tapped_layers=num_tapped_layers,
        subspace_dim=subspace_dim,
        reg_ratio=reg_ratio,
        device=device,
        include_velocity=include_velocity,
        layer_weighting=layer_weighting,
        hard_sister_pair=hard_sister_pair,
        prompt_ids=prompt_ids,
        **kwargs,
    )
    Z_ev_active = ((Z_eval - mu_full) @ C_full.T).astype(np.float32) if Z_eval is not None else None

    N_samples = Z_train.shape[0]
    D_full = C_full.shape[0]
    Z_tr_oof = np.zeros((N_samples, D_full), dtype=np.float32)

    pids_arr = np.asarray(prompt_ids) if prompt_ids is not None else None
    unique_groups = len(set(prompt_ids)) if prompt_ids is not None else 0
    max_group_frac = 1.0
    if pids_arr is not None and len(pids_arr) > 0:
        _, counts = np.unique(pids_arr, return_counts=True)
        max_group_frac = float(counts.max() / len(pids_arr))

    # Fix GroupKFold "-1" collapse: only use GroupKFold if no group holds > 5% of dataset
    if pids_arr is not None and unique_groups >= 10 and max_group_frac < 0.05:
        splitter = GroupKFold(n_splits=n_splits)
        splits = list(splitter.split(Z_train, y_train, groups=pids_arr))
    else:
        try:
            splitter = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
            splits = list(splitter.split(Z_train, y_train))
        except Exception:
            from sklearn.model_selection import KFold
            splitter = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)
            splits = list(splitter.split(Z_train, y_train))

    for tr_idx, val_idx in splits:
        pids_fold = pids_arr[tr_idx] if pids_arr is not None else None
        C_fold, mu_fold, _ = build_supervised_family_subspace(
            Z_train=Z_train[tr_idx],
            y_train=y_train[tr_idx],
            num_family_classes=num_family_classes,
            num_tapped_layers=num_tapped_layers,
            subspace_dim=subspace_dim,
            reg_ratio=reg_ratio,
            device=device,
            include_velocity=include_velocity,
            layer_weighting=layer_weighting,
            hard_sister_pair=hard_sister_pair,
            prompt_ids=pids_fold,
            **kwargs,
        )
        # Least-squares Procrustes alignment to target basis C_full on tr_idx:
        P_target = (Z_train[tr_idx] - mu_full) @ C_full.T
        P_fold_tr = (Z_train[tr_idx] - mu_fold) @ C_fold.T
        R_align, _, _, _ = np.linalg.lstsq(P_fold_tr, P_target, rcond=1e-3)

        P_fold_val = (Z_train[val_idx] - mu_fold) @ C_fold.T
        Z_tr_oof[val_idx] = (P_fold_val @ R_align).astype(np.float32)

    return Z_tr_oof, Z_ev_active, C_full, mu_full, diag


# ============================================================================
# Automated Layer-by-Layer Probing Sweep
# ============================================================================

def run_all_layers_probe_sweep(
    features_by_layer: np.ndarray,
    y: np.ndarray | None = None,
    num_family_classes: int = 4,
    hard_sister_pair: tuple[int, int] = (2, 3),
    prompts: np.ndarray | None = None,
    alpha: float = 5.0,
    n_splits: int = 5,
    output_dir: Path | str | None = None,
    labels: np.ndarray | list[int] | None = None,
    prompt_ids: list[Any] | None = None,
    output_json: Path | str | None = None,
    **kwargs: Any,
) -> dict[str, Any]:
    """
    Automated layer-by-layer linear probing sweep across all L layers ([N, L, hidden_size]).
    Evaluates:
      1. Per-layer 5-fold OOF Balanced Accuracy across all family classes.
      2. Per-layer Binary Pairwise Accuracy on the hardest sister pair (default: indices (2, 3)
         corresponding to gemini-3.7-flash vs gemini-3.8-flash).
      3. Learned ScalarMix / Precision-Weighted Layer Ensemble accuracy.
    """
    if y is None and labels is not None:
        y = np.asarray(labels)
    if y is None:
        raise ValueError("Either `y` or `labels` must be provided.")

    if prompts is None and prompt_ids is not None:
        prompts = np.asarray(prompt_ids)

    if output_dir is None and output_json is not None:
        output_dir = Path(output_json).parent

    fam_mask = (y < num_family_classes)
    if fam_mask.sum() == 0:
        fam_mask = np.ones(len(y), dtype=bool)

    X_layers = features_by_layer[fam_mask]
    y_fam = y[fam_mask]
    N, L, D = X_layers.shape

    if prompts is not None:
        p_fam = prompts[fam_mask]
        p_str = np.asarray([str(p) for p in p_fam])
        valid_grp_mask = ~((p_str == "-1") | np.char.startswith(p_str, "pool_") | np.char.startswith(p_str, "neg_"))
        if valid_grp_mask.sum() > 0:
            u_p, counts = np.unique(p_str[valid_grp_mask], return_counts=True)
            use_groups = int((counts > 1).sum()) >= 5
        else:
            use_groups = False
    else:
        p_fam = None
        use_groups = False

    device = kwargs.get("device")
    if device is None:
        device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    else:
        device = torch.device(device)

    n_splits_eff = max(min(n_splits, N // max(num_family_classes, 1)), 2)
    if use_groups:
        splitter = GroupKFold(n_splits=n_splits_eff).split(X_layers[:, 0, :], y_fam, groups=p_fam)
    else:
        splitter = StratifiedKFold(n_splits=n_splits_eff, shuffle=True, random_state=42).split(X_layers[:, 0, :], y_fam)
    folds = list(splitter)

    Y_oh = np.eye(num_family_classes, dtype=np.float32)[y_fam]
    Y_oh = 0.95 * Y_oh + 0.05 / max(num_family_classes, 1)

    layer_oof_logits = np.zeros((L, N, num_family_classes), dtype=np.float32)
    layer_metrics = []

    s_a, s_b = hard_sister_pair
    pair_mask = (y_fam == s_a) | (y_fam == s_b)

    print("\n================================================================================")
    print(f" AUTOMATED LAYER-BY-LAYER PROBING SWEEP ({L} Layers x {D} Dim | N={N})")
    print("================================================================================")
    print(f" Layer | Family OOF BAcc | Hard Sister Pair ({s_a} vs {s_b}) Acc | Signal Weight")
    print("--------------------------------------------------------------------------------")

    for l_idx in range(L):
        X_l = X_layers[:, l_idx, :]
        oof_l = np.zeros((N, num_family_classes), dtype=np.float32)

        for tr_idx, val_idx in folds:
            X_tr_t = torch.from_numpy(X_l[tr_idx]).float().to(device)
            Y_tr_t = torch.from_numpy(Y_oh[tr_idx]).float().to(device)
            X_val_t = torch.from_numpy(X_l[val_idx]).float().to(device)

            mu_tr = X_tr_t.mean(dim=0, keepdim=True)
            X_tr_c = X_tr_t - mu_tr
            X_val_c = X_val_t - mu_tr

            N_tr, D_l = X_tr_c.shape
            scale = float((X_tr_c * X_tr_c).sum(dim=1).mean().item())
            eff_alpha = float(alpha * max(scale / 512.0, 1e-3))
            Y_tr_c = Y_tr_t - Y_tr_t.mean(dim=0, keepdim=True)

            if N_tr >= D_l:
                Cov = X_tr_c.T @ X_tr_c
                Cov.diagonal().add_(eff_alpha)
                W_l = torch.linalg.solve(Cov, X_tr_c.T @ Y_tr_c)
                b_l = Y_tr_t.mean(dim=0)
            else:
                K_mat = X_tr_c @ X_tr_c.T
                K_mat.diagonal().add_(eff_alpha)
                A = torch.linalg.solve(K_mat, Y_tr_c)
                W_l = X_tr_c.T @ A
                b_l = Y_tr_t.mean(dim=0)

            oof_val = X_val_c @ W_l + b_l
            oof_l[val_idx] = oof_val.cpu().numpy()
            del X_tr_t, Y_tr_t, X_val_t, X_tr_c, X_val_c, W_l, b_l

        layer_oof_logits[l_idx] = oof_l
        fam_bacc = float(balanced_accuracy_score(y_fam, oof_l.argmax(axis=1)))

        if pair_mask.sum() > 0:
            pair_preds = np.where(oof_l[pair_mask, s_a] >= oof_l[pair_mask, s_b], s_a, s_b)
            pair_acc = float(balanced_accuracy_score(y_fam[pair_mask], pair_preds))
        else:
            pair_acc = fam_bacc

        layer_metrics.append({
            "layer": int(l_idx),
            "layer_idx": int(l_idx),
            "family_oof_bacc": fam_bacc,
            "hard_pair_oof_acc": pair_acc,
            "oof_balanced_acc": fam_bacc,
        })
        print(
            f" L{l_idx:02d}   | {fam_bacc * 100:6.2f}%         | {pair_acc * 100:6.2f}%                    | (solving...)",
            flush=True,
        )

    # Compute precision-weighted ScalarMix weights across layers based on log-odds margin
    raw_scores = np.array([0.5 * m["family_oof_bacc"] + 0.5 * m["hard_pair_oof_acc"] for m in layer_metrics])
    centered_scores = (raw_scores - raw_scores.max()) * 15.0
    scalar_mix_weights = np.exp(centered_scores) / np.sum(np.exp(centered_scores))

    print("\n--- Final Layer Probing Weights ---", flush=True)
    for l_idx, m in enumerate(layer_metrics):
        m["scalar_mix_weight"] = float(scalar_mix_weights[l_idx])
        print(
            f" L{l_idx:02d}   | {m['family_oof_bacc'] * 100:6.2f}%         | "
            f"{m['hard_pair_oof_acc'] * 100:6.2f}%                    | {scalar_mix_weights[l_idx]:6.4f}",
            flush=True,
        )

    # Evaluate ScalarMix Ensemble across all layers
    mix_oof = np.tensordot(scalar_mix_weights.astype(np.float32), layer_oof_logits, axes=(0, 0))
    mix_bacc = float(balanced_accuracy_score(y_fam, mix_oof.argmax(axis=1)))
    if pair_mask.sum() > 0:
        mix_pair_preds = np.where(mix_oof[pair_mask, s_a] >= mix_oof[pair_mask, s_b], s_a, s_b)
        mix_pair_acc = float(balanced_accuracy_score(y_fam[pair_mask], mix_pair_preds))
    else:
        mix_pair_acc = mix_bacc

    ranked_layers = sorted(
        layer_metrics,
        key=lambda d: 0.5 * d["family_oof_bacc"] + 0.5 * d["hard_pair_oof_acc"],
        reverse=True,
    )
    top8_layers = sorted([d["layer"] for d in ranked_layers[:8]])

    # -------------------------------------------------------------------------
    # Layer-to-Layer Cosine Similarity / Correlation Matrix Analysis
    # -------------------------------------------------------------------------
    X_f = X_layers.astype(np.float32)
    norms = np.linalg.norm(X_f, axis=-1, keepdims=True) + 1e-7
    X_norm = X_f / norms

    layer_corr = np.zeros((L, L), dtype=np.float32)
    chunk_sz = 2000
    for c_start in range(0, N, chunk_sz):
        c_end = min(c_start + chunk_sz, N)
        sub_x = X_norm[c_start:c_end]
        layer_corr += np.einsum("bld,bkd->lk", sub_x, sub_x)
    layer_corr /= float(max(N, 1))

    print("\n--- Layer-to-Layer Representation Similarity ---")
    print(" Adjacent Layer Cosine Similarities (Trajectory Continuity):")
    for l_i in range(L - 1):
        print(f"   L{l_i:02d} <-> L{l_i+1:02d} : {layer_corr[l_i, l_i+1]:.4f}")

    sampled_indices = [int(i) for i in np.linspace(0, L - 1, min(8, L)).round()]
    if len(sampled_indices) > 1:
        hdr = "       " + " ".join(f"L{idx:02d}  " for idx in sampled_indices)
        print(f"\n Sampled Inter-Layer Cosine Correlation Matrix:\n{hdr}")
        for r_idx in sampled_indices:
            row_str = f" L{r_idx:02d}  " + " ".join(f"{layer_corr[r_idx, c_idx]:.3f} " for c_idx in sampled_indices)
            print(row_str)

    print("--------------------------------------------------------------------------------")
    print(f" ScalarMix All-{L}-Layer OOF BAcc: {mix_bacc * 100:.2f}% | Hard Pair ({s_a} vs {s_b}): {mix_pair_acc * 100:.2f}%")
    print(f" Empirically Optimal Top-8 Layers: {top8_layers}")
    print("================================================================================\n")

    report = {
        "num_layers": int(L),
        "num_samples": int(N),
        "scalar_mix_oof_bacc": mix_bacc,
        "scalar_mix_hard_pair_acc": mix_pair_acc,
        "recommended_top8_layers": top8_layers,
        "layers": layer_metrics,
        "ranked_layers": ranked_layers,
        "top_4_layers": [d["layer"] for d in ranked_layers[:4]],
        "top_8_layers": top8_layers,
        "best_single_layer": ranked_layers[0],
        "layer_correlation_matrix": layer_corr.tolist(),
        "adjacent_layer_similarities": [float(layer_corr[i, i + 1]) for i in range(L - 1)],
    }

    if output_dir is not None:
        out_p = Path(output_dir)
        out_p.mkdir(parents=True, exist_ok=True)
        with open(out_p / "layer_probe_report.json", "w", encoding="utf-8") as f:
            json.dump(report, f, indent=2)

    if output_json is not None:
        out_f = Path(output_json)
        out_f.parent.mkdir(parents=True, exist_ok=True)
        with open(out_f, "w", encoding="utf-8") as f:
            json.dump(report, f, indent=2)

    return report


# ============================================================================
# Offline Multi-View Feature Engineering
# ============================================================================

def engineer_multiview_features(
    features_concat: np.ndarray,
    features_by_layer: np.ndarray | None = None,
    head_tail_cos: np.ndarray | None = None,
    token_std_norms: np.ndarray | None = None,
    num_tapped_layers: int = 8,
) -> np.ndarray:
    """
    Engineers explicit structural & trajectory features for offline Kaggle Dataset modeling:
    1. Per-layer L2 activation norms across tapped layers [N, 8].
    2. Inter-layer cosine similarity & L2 velocity ||h^(l) - h^(l-1)||_2 across consecutive layers [N, 14].
    3. Head-to-Tail positional cosine drift [N, 8] (opening framing vs closing summary divergence).
    4. Token-level activation dispersion norms & ratios [N, 16] (distilled Flash vs full Pro burstiness).
    """
    N, D_total = features_concat.shape
    D_layer = D_total // max(num_tapped_layers, 1)
    feats: list[np.ndarray] = []

    # Reshape tapped layers: (N, 8, D_layer)
    blocks = features_concat.reshape(N, num_tapped_layers, D_layer).astype(np.float32)
    block_norms = np.linalg.norm(blocks, axis=2).clip(min=1e-6)  # (N, 8)
    feats.append(block_norms)

    # Inter-layer cosine similarity and L2 transition velocity between consecutive tapped layers
    blocks_unit = blocks / block_norms[:, :, None]
    inter_cos = np.sum(blocks_unit[:, 1:, :] * blocks_unit[:, :-1, :], axis=2)  # (N, 7)
    inter_vel = np.linalg.norm(blocks[:, 1:, :] - blocks[:, :-1, :], axis=2)    # (N, 7)
    feats.extend([inter_cos, inter_vel])

    # If full all-28-layer tensor is provided, compute per-layer norms and adjacent cosine similarities layer-by-layer
    if features_by_layer is not None and features_by_layer.ndim == 3:
        L_all = features_by_layer.shape[1]
        f_norms_list = []
        adj_cos_list = []
        prev_unit = None
        for l_idx in range(L_all):
            layer_f = features_by_layer[:, l_idx, :].astype(np.float32)
            l_norm = np.linalg.norm(layer_f, axis=1, keepdims=True).clip(min=1e-6)
            f_norms_list.append(l_norm)
            curr_unit = layer_f / l_norm
            if prev_unit is not None:
                adj_cos_list.append(np.sum(curr_unit * prev_unit, axis=1, keepdims=True))
            prev_unit = curr_unit
        feats.append(np.hstack(f_norms_list))
        if adj_cos_list:
            feats.append(np.hstack(adj_cos_list))

    if head_tail_cos is not None:
        feats.append(head_tail_cos.astype(np.float32))

    if token_std_norms is not None:
        std_f = token_std_norms.astype(np.float32)
        feats.append(std_f)
        # Coefficient of variation (dispersion relative to layer norm)
        if std_f.shape[1] == block_norms.shape[1]:
            feats.append(std_f / block_norms)

    return np.hstack(feats).astype(np.float32)


In [ ]:
# === MODULE: src/namethatllm/experiment_logger.py ===
"""
Unified Multi-Backend Experiment Tracker (CSV/Markdown/SQLite + MLflow + W&B)
=============================================================================
Provides a unified experiment ledger that simultaneously logs runs to:
  1. Local storage & SQLite (Always Active, zero external dependencies):
     - experiments/experiment_ledger.csv
     - experiments/experiment_ledger.md
     - experiments/experiment_ledger.db (table: experiments)
  2. MLflow Backend (optional, auto-initialized if installed):
     - Logs parameters, accuracy metrics, and artifacts to SQLite/local URI.
  3. Weights & Biases Backend (optional, auto-enabled if WANDB_API_KEY is in env):
     - Logs parameters and metrics live to cloud W&B project dashboard.
     - Automatically guarded against offline Kaggle competition rerun environments.
"""

from __future__ import annotations

import csv
import os
import shutil
import sqlite3
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import pandas as pd


LEDGER_COLUMNS = [
    "exp_id",
    "timestamp",
    "phase",
    "target_family",
    "feature_set",
    "subspace_dim",
    "head_type",
    "gate_weight",
    "oof_family_bacc",
    "holdout_family_bacc",
    "hard_sister_acc",
    "holdout_18cls_raw_bacc",
    "holdout_18cls_sinkhorn_bacc",
    "delta_vs_baseline_bacc",
    "notes",
]


class ExperimentLedger:
    """
    Unified Multi-Backend Experiment Tracker:
    - Zero external dependencies for CSV, Markdown, and SQLite storage.
    - Safe optional integration with MLflow and Weights & Biases.
    """

    def __init__(
        self,
        ledger_dir: Path | str = "experiments",
        baseline_bacc: float | None = None,
        use_mlflow: bool = True,
        use_wandb: bool | None = None,
    ):
        self.ledger_dir = Path(ledger_dir)
        self.ledger_dir.mkdir(parents=True, exist_ok=True)
        self.csv_path = self.ledger_dir / "experiment_ledger.csv"
        self.md_path = self.ledger_dir / "experiment_ledger.md"
        self.db_path = self.ledger_dir / "experiment_ledger.db"
        self.baseline_bacc = baseline_bacc

        # 1. Initialize CSV
        if not self.csv_path.exists():
            with open(self.csv_path, "w", newline="", encoding="utf-8") as f:
                writer = csv.writer(f)
                writer.writerow(LEDGER_COLUMNS)

        # 2. Initialize SQLite table
        self._init_sqlite()

        # 3. Optional MLflow backend initialization
        self.use_mlflow = use_mlflow
        if self.use_mlflow:
            self._init_mlflow()

        # 4. Optional W&B backend initialization
        if use_wandb is None:
            # Auto-enable if WANDB_API_KEY is present and not in competition rerun
            self.use_wandb = bool(os.environ.get("WANDB_API_KEY")) and (
                os.environ.get("KAGGLE_IS_COMPETITION_RERUN") != "1"
            )
        else:
            self.use_wandb = use_wandb and (os.environ.get("KAGGLE_IS_COMPETITION_RERUN") != "1")

    # -------------------------------------------------------------------------
    # Core Local Storage (CSV, Markdown, SQLite)
    # -------------------------------------------------------------------------

    def _init_sqlite(self) -> None:
        """Initializes the local SQLite database and table schema."""
        try:
            with sqlite3.connect(self.db_path) as conn:
                cursor = conn.cursor()
                cursor.execute("""
                    CREATE TABLE IF NOT EXISTS experiments (
                        exp_id TEXT PRIMARY KEY,
                        timestamp TEXT,
                        phase TEXT,
                        target_family TEXT,
                        feature_set TEXT,
                        subspace_dim INTEGER,
                        head_type TEXT,
                        gate_weight REAL,
                        oof_family_bacc REAL,
                        holdout_family_bacc REAL,
                        hard_sister_acc REAL,
                        holdout_18cls_raw_bacc REAL,
                        holdout_18cls_sinkhorn_bacc REAL,
                        delta_vs_baseline_bacc REAL,
                        notes TEXT
                    )
                """)
                conn.commit()
        except Exception as e:
            print(f"[ExperimentLedger] SQLite init warning: {e}")

    def _log_sqlite(self, entry: dict[str, Any]) -> None:
        """Inserts or replaces an experiment entry in SQLite."""
        try:
            def _to_num(val: Any, typ: type) -> Any:
                if val is None or val == "" or pd.isna(val):
                    return None
                try:
                    return typ(val)
                except (ValueError, TypeError):
                    return None

            row = (
                str(entry.get("exp_id", "")),
                str(entry.get("timestamp", "")),
                str(entry.get("phase", "")),
                str(entry.get("target_family", "")),
                str(entry.get("feature_set", "")),
                _to_num(entry.get("subspace_dim"), int),
                str(entry.get("head_type", "")),
                _to_num(entry.get("gate_weight"), float),
                _to_num(entry.get("oof_family_bacc"), float),
                _to_num(entry.get("holdout_family_bacc"), float),
                _to_num(entry.get("hard_sister_acc"), float),
                _to_num(entry.get("holdout_18cls_raw_bacc"), float),
                _to_num(entry.get("holdout_18cls_sinkhorn_bacc"), float),
                _to_num(entry.get("delta_vs_baseline_bacc"), float),
                str(entry.get("notes", "")),
            )
            with sqlite3.connect(self.db_path) as conn:
                cursor = conn.cursor()
                cursor.execute("""
                    INSERT OR REPLACE INTO experiments (
                        exp_id, timestamp, phase, target_family, feature_set,
                        subspace_dim, head_type, gate_weight, oof_family_bacc,
                        holdout_family_bacc, hard_sister_acc, holdout_18cls_raw_bacc,
                        holdout_18cls_sinkhorn_bacc, delta_vs_baseline_bacc, notes
                    ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
                """, row)
                conn.commit()
        except Exception as e:
            print(f"[ExperimentLedger] SQLite log warning: {e}")

    # -------------------------------------------------------------------------
    # Optional Cloud & Local UI Backends (MLflow & W&B)
    # -------------------------------------------------------------------------

    def _init_mlflow(self) -> None:
        """Configures MLflow tracking URI and active experiment."""
        try:
            import mlflow

            tracking_uri = os.environ.get("MLFLOW_TRACKING_URI")
            if not tracking_uri:
                mlflow_db = (self.ledger_dir / "mlflow.db").resolve()
                tracking_uri = f"sqlite:///{mlflow_db.as_posix()}"
            mlflow.set_tracking_uri(tracking_uri)
            mlflow.set_experiment("NameThatLLM_Gates_and_Ensemble")
        except Exception:
            self.use_mlflow = False

    def _log_mlflow(self, entry: dict[str, Any], artifacts: list[Path | str] | None = None) -> None:
        """Logs parameters, metrics, and artifacts to MLflow."""
        if not self.use_mlflow:
            return
        try:
            import mlflow

            exp_id = str(entry.get("exp_id", "EXP"))
            target_fam = str(entry.get("target_family", ""))
            feat = str(entry.get("feature_set", ""))
            head = str(entry.get("head_type", ""))
            run_name = f"{exp_id}_{target_fam}_{feat}_{head}".strip("_")

            with mlflow.start_run(run_name=run_name):
                # 1. Parameters
                param_keys = [
                    "exp_id", "phase", "target_family", "feature_set",
                    "subspace_dim", "head_type", "gate_weight",
                    "select_layers", "include_velocity", "layer_weighting"
                ]
                params = {}
                for k in param_keys:
                    if k in entry and entry[k] != "" and entry[k] is not None:
                        params[k] = str(entry[k])[:250]
                if params:
                    mlflow.log_params(params)

                # 2. Metrics
                metric_keys = [
                    "oof_family_bacc", "holdout_family_bacc", "hard_sister_acc",
                    "holdout_18cls_raw_bacc", "holdout_18cls_sinkhorn_bacc",
                    "delta_vs_baseline_bacc"
                ]
                metrics = {}
                for k in metric_keys:
                    v = entry.get(k)
                    if v is not None and v != "" and not pd.isna(v):
                        try:
                            metrics[k] = float(v)
                        except (ValueError, TypeError):
                            pass
                if metrics:
                    mlflow.log_metrics(metrics)

                # 3. Artifacts
                if artifacts:
                    for art in artifacts:
                        art_p = Path(art)
                        if art_p.exists():
                            if art_p.is_file():
                                mlflow.log_artifact(str(art_p))
                            elif art_p.is_dir():
                                mlflow.log_artifacts(str(art_p))
        except Exception as e:
            print(f"[ExperimentLedger] MLflow logging note: {e}")

    def _log_wandb(self, entry: dict[str, Any]) -> None:
        """Logs run to Weights & Biases if enabled."""
        if not self.use_wandb:
            return
        if os.environ.get("KAGGLE_IS_COMPETITION_RERUN") == "1":
            return
        try:
            import wandb

            exp_id = str(entry.get("exp_id", "EXP"))
            target_fam = str(entry.get("target_family", ""))
            feat = str(entry.get("feature_set", ""))
            head = str(entry.get("head_type", ""))
            run_name = f"{exp_id}_{target_fam}_{feat}_{head}".strip("_")

            param_keys = [
                "exp_id", "phase", "target_family", "feature_set",
                "subspace_dim", "head_type", "gate_weight",
                "select_layers", "include_velocity", "layer_weighting"
            ]
            params = {k: entry[k] for k in param_keys if k in entry and entry[k] != "" and entry[k] is not None}

            metric_keys = [
                "oof_family_bacc", "holdout_family_bacc", "hard_sister_acc",
                "holdout_18cls_raw_bacc", "holdout_18cls_sinkhorn_bacc",
                "delta_vs_baseline_bacc"
            ]
            metrics = {}
            for k in metric_keys:
                v = entry.get(k)
                if v is not None and v != "" and not pd.isna(v):
                    try:
                        metrics[k] = float(v)
                    except (ValueError, TypeError):
                        pass

            wandb.init(
                project="NameThatLLM",
                name=run_name,
                config=params,
                reinit=True,
            )
            if metrics:
                wandb.log(metrics)
            wandb.finish()
        except Exception as e:
            print(f"[ExperimentLedger] W&B logging note: {e}")

    # -------------------------------------------------------------------------
    # Public Entrypoints
    # -------------------------------------------------------------------------

    def add_entry(
        self,
        record: dict[str, Any],
        artifacts: list[Path | str] | None = None,
    ) -> dict[str, Any]:
        """
        Appends an experiment record to CSV, SQLite, and Markdown table,
        and optionally streams to MLflow and W&B backends.
        """
        entry = dict(record)

        if "timestamp" not in entry or not entry["timestamp"]:
            entry["timestamp"] = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")

        # Automatically record baseline if this is EXP-00 or F0_plain_solo
        if entry.get("feature_set") == "F0_plain_solo" or entry.get("exp_id") == "EXP-00":
            if self.baseline_bacc is None and "holdout_18cls_sinkhorn_bacc" in entry:
                self.baseline_bacc = float(entry["holdout_18cls_sinkhorn_bacc"])

        if "delta_vs_baseline_bacc" not in entry or entry["delta_vs_baseline_bacc"] is None:
            if self.baseline_bacc is not None and "holdout_18cls_sinkhorn_bacc" in entry:
                try:
                    entry["delta_vs_baseline_bacc"] = float(entry["holdout_18cls_sinkhorn_bacc"]) - float(self.baseline_bacc)
                except (TypeError, ValueError):
                    entry["delta_vs_baseline_bacc"] = 0.0
            else:
                entry["delta_vs_baseline_bacc"] = 0.0

        # Standardize missing columns for CSV
        for col in LEDGER_COLUMNS:
            if col not in entry or entry[col] is None:
                entry[col] = ""

        # 1. Append to CSV
        with open(self.csv_path, "a", newline="", encoding="utf-8") as f:
            writer = csv.writer(f)
            writer.writerow([entry.get(c, "") for c in LEDGER_COLUMNS])

        # 2. Log to SQLite
        self._log_sqlite(entry)

        # 3. Log to MLflow
        self._log_mlflow(entry, artifacts=artifacts)

        # 4. Log to W&B
        self._log_wandb(entry)

        # 5. Regenerate Markdown table
        self._regenerate_markdown()
        return entry

    def record_run(self, **kwargs) -> dict[str, Any]:
        """Convenience method accepting keyword arguments for add_entry."""
        artifacts = kwargs.pop("artifacts", None)
        return self.add_entry(kwargs, artifacts=artifacts)

    def get_dataframe(self) -> pd.DataFrame:
        """Loads all logged experiments as a pandas DataFrame."""
        if self.db_path.exists():
            try:
                with sqlite3.connect(self.db_path) as conn:
                    return pd.read_sql_query("SELECT * FROM experiments ORDER BY timestamp ASC", conn)
            except Exception:
                pass

        if self.csv_path.exists():
            try:
                return pd.read_csv(self.csv_path)
            except Exception:
                pass
        return pd.DataFrame(columns=LEDGER_COLUMNS)

    def get_best_run(
        self,
        target_family: str | None = None,
        metric: str = "holdout_18cls_sinkhorn_bacc",
    ) -> dict[str, Any] | None:
        """Finds the best run matching criteria by given metric."""
        df = self.get_dataframe()
        if df.empty:
            return None
        if target_family is not None:
            df = df[df["target_family"].str.lower() == target_family.lower()]
        if df.empty or metric not in df.columns:
            return None

        valid_rows = df.dropna(subset=[metric])
        if valid_rows.empty:
            return None
        best_idx = pd.to_numeric(valid_rows[metric], errors="coerce").idxmax()
        if pd.isna(best_idx):
            return None
        return df.loc[best_idx].to_dict()

    def export_copy(self, dest_dir: Path | str) -> None:
        """Copies the CSV, SQLite DB, and Markdown files to dest_dir (for Kaggle output artifacts)."""
        d = Path(dest_dir)
        d.mkdir(parents=True, exist_ok=True)
        if self.csv_path.exists():
            shutil.copy2(str(self.csv_path), str(d / "experiment_ledger.csv"))
        if self.md_path.exists():
            shutil.copy2(str(self.md_path), str(d / "experiment_ledger.md"))
        if self.db_path.exists():
            shutil.copy2(str(self.db_path), str(d / "experiment_ledger.db"))

    def _regenerate_markdown(self) -> None:
        """Formats the entire experiment history into a GitHub-Flavored Markdown table."""
        df = self.get_dataframe()
        if df.empty:
            self.md_path.write_text("# NameThatLLM Experiment Ledger\n\nNo experiments logged yet.\n", encoding="utf-8")
            return

        lines = [
            "# NameThatLLM Gate & Ensemble Experiment Ledger",
            f"*Last Updated: {datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S UTC')}*",
            "",
            "| Exp ID | Phase | Family | Feature Set | D_fish | Head | GW | Family OOF | Family Holdout | Sister Acc | 18c Raw | 18c Sinkhorn | Δ vs Base | Notes |",
            "|---|---|---|---|---:|---|---:|---:|---:|---:|---:|---:|---:|---|",
        ]

        def _fmt_pct(v: Any) -> str:
            if v is None or v == "" or pd.isna(v):
                return "-"
            try:
                val = float(v)
                return f"{val * 100:.2f}%" if val <= 1.0 else f"{val:.2f}%"
            except (ValueError, TypeError):
                return str(v)

        def _fmt_delta(v: Any) -> str:
            if v is None or v == "" or pd.isna(v):
                return "+0.00%"
            try:
                val = float(v)
                pct = val * 100 if abs(val) <= 1.0 else val
                return f"{pct:+.2f}%"
            except (ValueError, TypeError):
                return str(v)

        for _, row in df.iterrows():
            exp_id = str(row.get("exp_id", ""))
            phase = str(row.get("phase", ""))
            fam = str(row.get("target_family", ""))
            feat = str(row.get("feature_set", ""))
            sub_d = str(row.get("subspace_dim", "")) or "-"
            head = str(row.get("head_type", ""))
            gw = f"{float(row['gate_weight']):.2f}" if row.get("gate_weight") != "" and not pd.isna(row.get("gate_weight")) else "-"
            oof_fam = _fmt_pct(row.get("oof_family_bacc"))
            hld_fam = _fmt_pct(row.get("holdout_family_bacc"))
            sis_acc = _fmt_pct(row.get("hard_sister_acc"))
            raw_18 = _fmt_pct(row.get("holdout_18cls_raw_bacc"))
            sk_18 = _fmt_pct(row.get("holdout_18cls_sinkhorn_bacc"))
            delta = _fmt_delta(row.get("delta_vs_baseline_bacc"))
            notes = str(row.get("notes", "")).replace("|", "\\|")

            line = (
                f"| `{exp_id}` | {phase} | {fam} | `{feat}` | {sub_d} | {head} | {gw} | "
                f"{oof_fam} | **{hld_fam}** | {sis_acc} | {raw_18} | **{sk_18}** | **{delta}** | {notes} |"
            )
            lines.append(line)

        lines.append("")
        self.md_path.write_text("\n".join(lines), encoding="utf-8")


In [ ]:
# === MODULE: src/namethatllm/training/backbones/base.py ===
"""
Backbone Adapter Base Contracts, Precision/Tokenization Specs, and Registry.
"""

from abc import ABC, abstractmethod
from collections.abc import Callable, Iterator
from contextlib import contextmanager
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import torch
from torch import nn


@dataclass(frozen=True)
class TokenizationSpec:
    """Normalized tokenization and sequence boundary specification for a backbone family."""

    max_len: int
    pad_id: int
    cls_id: int | None
    end_id: int | None
    drop_first_token: bool


@dataclass(frozen=True)
class PrecisionSpec:
    """Resolved compute dtype and hardware capability profile."""

    compute_dtype: torch.dtype
    use_bf16: bool
    use_fa2: bool
    total_vram_gb: float


class BaseBackboneAdapter(ABC):
    """Strategy contract encapsulating backbone-specific loading, precision, tokenization, and LoRA rules."""

    family_name: str = "base"
    default_backbone: str = ""
    default_output_dir: str = "export_checkpoint"
    default_keep_layers: int = 24
    default_layers: tuple[int, ...] = (3, 6, 9, 12, 15, 18, 21, 24)
    default_max_len: int = 384
    default_end_id: int = 0
    preserves_working_checkpoint: bool = False

    def default_lora_targets(self) -> list[str]:
        return []

    @abstractmethod
    def resolve_model_source(self, cfg: dict[str, Any], input_dir: Path) -> str:
        """Resolves a local cached backbone path under `input_dir` or falls back to HF ID."""

    def resolve_precision(self, cfg: dict[str, Any], device: torch.device) -> PrecisionSpec:
        is_ampere_or_newer = False
        total_vram_gb = 0.0
        if torch.cuda.is_available():
            major_cap = torch.cuda.get_device_capability(0)[0]
            is_ampere_or_newer = major_cap >= 8
            total_vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
        use_bf16 = bool(cfg.get("bf16", False) or is_ampere_or_newer)
        compute_dtype = torch.bfloat16 if use_bf16 else (torch.float16 if torch.cuda.is_available() else torch.float32)
        return PrecisionSpec(
            compute_dtype=compute_dtype,
            use_bf16=use_bf16,
            use_fa2=False,
            total_vram_gb=total_vram_gb,
        )

    @abstractmethod
    def load_tokenizer(self, model_source: str, cfg: dict[str, Any], is_smoke: bool = False) -> tuple[Any, TokenizationSpec]:
        """Loads tokenizer (or mock/fallback) and constructs its TokenizationSpec."""

    def tok_spec_from_tokenizer(
        self, tok: Any, max_len: int | None = None, end_id: int | None = None
    ) -> TokenizationSpec:
        """Builds a TokenizationSpec from an already-instantiated tokenizer (fallback for legacy raw mocks)."""
        tok_pad = getattr(tok, "pad_token_id", None)
        pad_id = int(tok_pad) if tok_pad is not None else 0
        raw_cls = getattr(tok, "cls_token_id", None)
        cls_id = int(raw_cls) if raw_cls is not None else None
        if end_id is not None:
            raw_end = end_id
        else:
            tok_sep = getattr(tok, "sep_token_id", None)
            raw_end = tok_sep if tok_sep is not None else getattr(tok, "eos_token_id", None)
        eff_end_id = int(raw_end) if raw_end is not None else None
        return TokenizationSpec(
            max_len=int(max_len if max_len is not None else self.default_max_len),
            pad_id=pad_id,
            cls_id=cls_id,
            end_id=eff_end_id,
            drop_first_token=cls_id is None,
        )

    @abstractmethod
    def load_backbone(
        self, model_source: str, cfg: dict[str, Any], precision: PrecisionSpec, device: torch.device
    ) -> tuple[nn.Module, int, int, list[int]]:
        """Loads backbone weights and returns `(base_model, hidden_size, keep_layers, layers)`."""

    def prepare_backbone(
        self, base_model: nn.Module, precision: PrecisionSpec, cfg: dict[str, Any], is_eval_only: bool
    ) -> nn.Module:
        """Applies optional model-specific modifications prior to wrapping with LoRA."""
        return base_model

    def apply_lora(self, base_model: nn.Module, cfg: dict[str, Any], is_eval_only: bool) -> nn.Module:
        """Wraps backbone with PEFT LoRA, enforces FP32 trainable params, and sets checkpointing/eval mode."""
        from peft import LoraConfig, get_peft_model

        try:
            import peft.import_utils
            import peft.tuners.lora.torchao

            peft.import_utils.is_torchao_available = lambda: False
            peft.tuners.lora.torchao.is_torchao_available = lambda: False
        except Exception:
            pass

        if hasattr(base_model, "enable_input_require_grads"):
            base_model.enable_input_require_grads()

        linear_mod_names = {
            name.split(".")[-1]
            for name, module in base_model.named_modules()
            if isinstance(module, nn.Linear)
        }
        configured_targets = list(cfg.get("lora_target_modules", [])) or self.default_lora_targets()
        active_targets = [m for m in configured_targets if m in linear_mod_names]
        if not active_targets:
            active_targets = list(linear_mod_names)[:7] if linear_mod_names else configured_targets
            print(f"[Trainer] Configured LoRA targets not found; falling back to: {active_targets}")
        else:
            print(f"[Trainer] Active LoRA target modules: {active_targets}")

        lora_cfg = LoraConfig(
            r=int(cfg.get("lora_r", 32)),
            lora_alpha=int(cfg.get("lora_alpha", 64)),
            lora_dropout=float(cfg.get("lora_dropout", 0.05)),
            target_modules=active_targets,
            bias="none",
            task_type="FEATURE_EXTRACTION",
        )
        peft_model = get_peft_model(base_model, lora_cfg)
        for p in peft_model.parameters():
            if p.requires_grad and p.dtype != torch.float32:
                p.data = p.data.float()

        if not is_eval_only:
            if hasattr(peft_model, "gradient_checkpointing_enable"):
                try:
                    peft_model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
                except TypeError:
                    peft_model.gradient_checkpointing_enable()
            for mod in peft_model.modules():
                if hasattr(mod, "gradient_checkpointing"):
                    mod.gradient_checkpointing = True
        else:
            peft_model.eval()
        return peft_model

    @abstractmethod
    def build_scalers(self, precision: PrecisionSpec, device: torch.device) -> tuple[Any | None, Any | None]:
        """Returns `(grad_scaler, manual_scaler)` appropriate for this backbone and precision."""

    def adjust_stage1_batch_config(
        self, stage1_cfg: dict[str, Any], precision: PrecisionSpec
    ) -> tuple[int, int, int]:
        """Returns `(batch_size, accum_steps, eval_batch_size)`."""
        bs1 = int(stage1_cfg.get("batch_size", 2))
        accum1 = int(stage1_cfg.get("gradient_accumulation_steps", 8))
        eval_bs = int(stage1_cfg.get("eval_batch_size", 32))
        return bs1, accum1, eval_bs

    def clamp_extraction_batch_size(self, batch_size: int) -> int:
        return batch_size

    def stage2_eval_batch_size(self) -> int:
        return 4

    def specialist_phase_tag(self, target_family: str) -> str:
        return f"P3_{target_family}_finetuned"

    def specialist_feature_set_tag(self, target_family: str) -> str:
        return f"F6_{target_family}_tapped"

    def specialist_notes_label(self) -> str:
        return self.family_name.capitalize()

    def maybe_run_pipeline(
        self,
        train_cfg: dict[str, Any],
        step_callback: Any | None = None,
        **kwargs: Any,
    ) -> Any | None:
        """Optional hook allowing a backbone adapter to handle specialized pipeline modes."""
        return None

    @contextmanager
    def export_context(self, backbone: nn.Module) -> Iterator[None]:
        """Context manager bracketing checkpoint export (default no-op)."""
        yield

    def default_layer_count_for_disk_estimate(self, model: nn.Module) -> int:
        """Dynamically infers total hidden state layer count for disk estimation."""
        bb = getattr(model, "backbone", model)
        cfg = getattr(bb, "config", None)
        n_hidden = getattr(cfg, "num_hidden_layers", None) if cfg is not None else None
        if n_hidden is not None:
            return int(n_hidden) + 1
        tapped = getattr(model, "layers", None)
        if tapped:
            return max(len(tapped), 1)
        return 1


_BACKBONE_REGISTRY: dict[str, type[BaseBackboneAdapter]] = {}


def register_backbone(name: str) -> Callable[[type[BaseBackboneAdapter]], type[BaseBackboneAdapter]]:
    """Class decorator registering a backbone adapter by canonical family name."""

    def _decorator(cls: type[BaseBackboneAdapter]) -> type[BaseBackboneAdapter]:
        _BACKBONE_REGISTRY[name.lower().strip()] = cls
        return cls

    return _decorator


def resolve_backbone_adapter(cfg: dict[str, Any] | None = None) -> BaseBackboneAdapter:
    """
    Resolves the concrete BaseBackboneAdapter for a configuration dict.
    Prefers explicit `backbone_family`; falls back ONLY here to inspecting `backbone`.
    """
    cfg = cfg or {}
    fam = cfg.get("backbone_family")
    if fam:
        key = str(fam).lower().strip()
        if key in _BACKBONE_REGISTRY:
            return _BACKBONE_REGISTRY[key]()

    b_name = str(cfg.get("backbone", "")).lower()
    if "deberta" in b_name and "deberta" in _BACKBONE_REGISTRY:
        return _BACKBONE_REGISTRY["deberta"]()
    if "bge" in b_name and "bge" in _BACKBONE_REGISTRY:
        return _BACKBONE_REGISTRY["bge"]()
    return _BACKBONE_REGISTRY["qwen"]()


In [ ]:
# === MODULE: src/namethatllm/training/losses.py ===
"""
Training Loss Functions: Family-Margin Cross-Entropy, Supervised Contrastive Loss, and Dual-Head Stage-2 Loss.
"""

from __future__ import annotations

from typing import Any

import numpy as np
import torch
import torch.nn.functional as F



def _build_sister_mask(device: torch.device, boosted: bool = False) -> torch.Tensor:
    """
    Builds an (18, 18) indicator matrix M where M[i, j] > 0.0 if class i and
    class j belong to the same LLM provider family (sisters), and 0.0 otherwise.
    When boosted=True, applies empirically validated sibling and cross-family sink margin penalties.
    """
    M = torch.zeros((K, K), dtype=torch.float32, device=device)
    for _, members in FAMILIES.items():
        for m1 in members:
            for m2 in members:
                if m1 != m2:
                    M[m1, m2] = 1.0

    if boosted:
        boost_pairs = {
            (7, 8): 1.9, (8, 7): 1.9,
            (6, 7): 1.6, (7, 6): 1.6,
            (6, 8): 1.6, (8, 6): 1.6,
            (6, 5): 1.5, (5, 6): 1.5,
            (7, 5): 1.5, (5, 7): 1.5,
            (8, 5): 1.4, (5, 8): 1.4,
            (1, 2): 1.85, (2, 1): 1.85,
            (1, 3): 1.7, (3, 1): 1.7,
            (2, 3): 1.7, (3, 2): 1.7,
            (3, 0): 1.3, (0, 3): 1.2,
            (2, 0): 1.3, (0, 2): 1.2,
            (4, 2): 1.5, (2, 4): 1.4,
            (4, 1): 1.5, (1, 4): 1.4,
            (4, 3): 1.4, (3, 4): 1.4,
            (9, 10): 1.6, (10, 9): 1.6,
            (9, 12): 1.5, (12, 9): 1.5,
            (9, 13): 1.6, (13, 9): 1.6,
            (9, 11): 1.3, (11, 9): 1.3,
            (13, 10): 1.5, (10, 13): 1.5,
            (15, 16): 1.8, (16, 15): 1.8,
            (17, 16): 1.85, (16, 17): 1.75,
            (17, 15): 1.6, (15, 17): 1.5,
            (17, 13): 1.6, (13, 17): 1.6,
            (16, 13): 1.7, (13, 16): 1.7,
            (15, 13): 1.6, (13, 15): 1.6,
            (17, 9): 1.5, (9, 17): 1.4,
            (15, 9): 1.5, (9, 15): 1.4,
            (16, 9): 1.5, (9, 16): 1.4,
            (8, 9): 1.3, (9, 8): 1.2,
            (15, 12): 1.5, (12, 15): 1.4,
            (15, 11): 1.4, (11, 15): 1.3,
            (16, 10): 1.4, (10, 16): 1.3,
            (14, 0): 1.3, (0, 14): 1.2,
            (14, 1): 1.4, (1, 14): 1.3,
        }
        for (i, j), val in boost_pairs.items():
            M[i, j] = float(val)

    return M


def family_margin_cross_entropy(
    logits: torch.Tensor,
    labels: torch.Tensor,
    family_margin: float = 0.35,
    label_smoothing: float = 0.0,
    weight: torch.Tensor | None = None,
    boost_hard_sisters: bool = True,
    sample_weight: torch.Tensor | None = None,
) -> torch.Tensor:
    """Computes Cross-Entropy with Additive Sibling Margin Penalty."""
    labels_dev = labels.to(logits.device)
    if family_margin > 0.0:
        num_cls = logits.shape[1]
        if num_cls == K:
            mask = _build_sister_mask(logits.device, boosted=boost_hard_sisters)
        else:
            mask = 1.0 - torch.eye(num_cls, dtype=torch.float32, device=logits.device)
            if num_cls == 4:
                mask[2, 3] = 1.8
                mask[3, 2] = 1.8
            elif num_cls == 5:
                for i in (1, 2, 3):
                    for j in (1, 2, 3):
                        if i != j:
                            mask[i, j] = 1.6
        boost = mask[labels_dev] * family_margin
        logits = logits + boost
    w_dev = weight.to(logits.device, dtype=torch.float32) if weight is not None else None
    if sample_weight is not None:
        per_sample_loss = F.cross_entropy(logits, labels_dev, weight=w_dev, label_smoothing=label_smoothing, reduction="none")
        sw = sample_weight.to(logits.device, dtype=torch.float32)
        eff_w = sw * w_dev[labels_dev] if w_dev is not None else sw
        return (per_sample_loss * sw).sum() / eff_w.sum().clamp(min=1e-6)
    return F.cross_entropy(logits, labels_dev, weight=w_dev, label_smoothing=label_smoothing)


def compute_supcon_loss(z: torch.Tensor, labels: torch.Tensor, temperature: float = 0.10, **kwargs) -> torch.Tensor:
    """Supervised contrastive loss with hinge-repulsion fallback for unique-class prompt groups."""
    if z.shape[0] < 2:
        return (z * 0.0).sum()

    dev = z.device
    z_norm = F.normalize(z.float(), p=2, dim=-1, eps=1e-6)
    raw_cos = torch.matmul(z_norm, z_norm.T)
    diag_mask = torch.eye(z.shape[0], device=dev, dtype=torch.bool)

    labels_dev = labels.to(dev)
    label_mat = labels_dev.unsqueeze(0) == labels_dev.unsqueeze(1)
    pos_mask = label_mat & ~diag_mask

    prototypes = kwargs.get("prototypes")
    if prototypes is None:
        prototypes = kwargs.get("centered_head_weight")

    if pos_mask.sum() == 0:
        off_diag = ~diag_mask
        if off_diag.sum() > 0:
            loss = torch.clamp(raw_cos[off_diag] + 0.05, min=0.0).pow(2).mean()
        else:
            loss = (z * 0.0).sum()

        if prototypes is not None:
            w_norm = F.normalize(prototypes.float().to(dev), p=2, dim=-1, eps=1e-6)
            sim_proto = (z_norm @ w_norm.T) / max(float(temperature), 1e-3)
            loss = loss + 0.5 * F.cross_entropy(sim_proto, labels_dev)

        Z_group = kwargs.get("Z_group_centered")
        if Z_group is not None and Z_group.shape[0] > z.shape[0]:
            g_tensor = Z_group.detach().float().to(dev)
            group_slice = kwargs.get("group_slice")
            if group_slice is not None:
                s_idx, e_idx = int(group_slice[0]), int(group_slice[1])
                keep_rows = torch.ones(g_tensor.shape[0], dtype=torch.bool, device=dev)
                keep_rows[max(0, s_idx) : min(g_tensor.shape[0], e_idx)] = False
                g_tensor = g_tensor[keep_rows]
            if g_tensor.shape[0] > 0:
                g_norm = F.normalize(g_tensor, p=2, dim=-1, eps=1e-6)
                sim_group = z_norm @ g_norm.T
                if group_slice is None:
                    sim_group = sim_group.masked_fill(sim_group > 0.995, 0.0)
                loss = loss + 0.10 * torch.clamp(sim_group + 0.05, min=0.0).pow(2).mean()
        return loss

    sim = raw_cos / max(float(temperature), 1e-4)
    sim = sim.masked_fill(diag_mask, -1e4)
    sim_max, _ = torch.max(sim, dim=1, keepdim=True)
    sim = sim - sim_max.detach()

    exp_sim = torch.exp(sim) * (~diag_mask).float()
    denom = exp_sim.sum(dim=1, keepdim=True).clamp(min=1e-8)
    log_prob = sim - torch.log(denom)
    mean_log_prob_pos = (pos_mask.float() * log_prob).sum(dim=1) / pos_mask.sum(dim=1).clamp(min=1)
    valid_anchors = pos_mask.sum(dim=1) > 0
    loss = -mean_log_prob_pos[valid_anchors].mean()

    if prototypes is not None:
        w_norm = F.normalize(prototypes.float().to(dev), p=2, dim=-1, eps=1e-6)
        sim_proto = (z_norm @ w_norm.T) / max(float(temperature), 1e-3)
        loss = loss + 0.5 * F.cross_entropy(sim_proto, labels_dev)

    return loss


def compute_two_stage_loss(
    model: Any,
    Z: torch.Tensor,
    labels: torch.Tensor,
    prompt_ids: list[Any] | None = None,
    centered_weight: float = 0.5,
    family_margin: float = 0.35,
    supcon_weight: float = 0.0,
    label_smoothing: float = 0.0,
    weight: torch.Tensor | None = None,
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor | None]:
    """
    Computes joint plain cross-entropy and within-prompt contrastive centered loss
    with intra-family margin penalties and optional supervised contrastive loss:
    L = L_plain + centered_weight * L_centered + supcon_weight * L_supcon
    """
    dev = Z.device
    labels_dev = labels.to(dev)

    if hasattr(model, "predict_plain"):
        logits_plain = model.predict_plain(Z)
    else:
        if next(model.plain_head.parameters()).device != dev:
            model.plain_head = model.plain_head.to(dev)
        logits_plain = model.plain_head(Z)

    loss_plain = family_margin_cross_entropy(
        logits_plain, labels_dev, family_margin=family_margin, label_smoothing=label_smoothing, weight=weight
    )

    if prompt_ids is None or centered_weight <= 0.0:
        return loss_plain, loss_plain, None

    Z_c = Z.clone()
    prompt_arr = np.array(prompt_ids)
    unique_prompts = list(dict.fromkeys(prompt_ids))
    for p in unique_prompts:
        mask = torch.tensor(prompt_arr == p, device=dev, dtype=torch.bool)
        if mask.sum() > 1:
            Z_c[mask] = Z[mask] - Z[mask].mean(dim=0, keepdim=True)
        else:
            Z_c[mask] = Z[mask] - Z[mask]

    if hasattr(model, "predict_centered"):
        logits_centered = model.predict_centered(Z_c)
    else:
        if next(model.centered_head.parameters()).device != dev:
            model.centered_head = model.centered_head.to(dev)
        logits_centered = model.centered_head(Z_c)

    loss_centered = family_margin_cross_entropy(
        logits_centered, labels_dev, family_margin=family_margin, label_smoothing=label_smoothing, weight=weight
    )

    total_loss = loss_plain + float(centered_weight) * loss_centered
    if supcon_weight > 0.0:
        loss_sup = compute_supcon_loss(Z_c, labels_dev, prototypes=model.centered_head.weight.detach())
        total_loss = total_loss + float(supcon_weight) * loss_sup

    return total_loss, loss_plain, loss_centered


In [ ]:
# === MODULE: src/namethatllm/training/modeling.py ===
"""
Shared Multi-Layer LoRA Architecture, FP16 Dynamic Loss Scaling, Optimizer Grouping, and Checkpoint I/O.
"""

from __future__ import annotations

import json
from pathlib import Path
from typing import Any

import numpy as np
import torch
import torch.nn.functional as F
from safetensors.torch import load_file, save_file
from torch import nn



class MultiLayerLoRAClassifier(nn.Module):
    """Universal Multi-Layer LoRA Sequence Classifier with Plain and Prompt-Centered Dual Heads."""

    def __init__(
        self,
        backbone: nn.Module,
        hidden_size: int,
        layers: list[int],
        num_classes: int = K,
        keep_layers: int | None = None,
        backbone_family: str = "unknown",
        tok_spec: TokenizationSpec | None = None,
    ):
        super().__init__()
        self.backbone = backbone
        self.layers = list(layers)
        self.num_classes = num_classes
        self.hidden_size = hidden_size
        self.total_dim = len(layers) * hidden_size
        self.backbone_family = str(backbone_family)
        self.tok_spec = tok_spec

        self.norms = nn.ModuleList([nn.LayerNorm(hidden_size) for _ in layers])
        self.plain_head = nn.Linear(self.total_dim, num_classes)
        self.centered_head = nn.Linear(self.total_dim, num_classes)
        self.dropouts = nn.ModuleList([nn.Dropout(p) for p in (0.1, 0.2, 0.3, 0.4, 0.5)])

    def predict_plain(self, Z: torch.Tensor) -> torch.Tensor:
        Z = Z.to(self.plain_head.weight.device)
        if self.training:
            return torch.mean(torch.stack([self.plain_head(d(Z)) for d in self.dropouts], dim=0), dim=0)
        return self.plain_head(Z)

    def predict_centered(self, Z: torch.Tensor) -> torch.Tensor:
        Z = Z.to(self.centered_head.weight.device)
        if self.training:
            return torch.mean(torch.stack([self.centered_head(d(Z)) for d in self.dropouts], dim=0), dim=0)
        return self.centered_head(Z)

    def forward_features(
        self,
        input_ids: torch.Tensor,
        attention_mask: torch.Tensor,
        drop_first_token: bool = True,
        return_by_layer: bool = False,
        extract_all_layers: bool = False,
        return_multiview: bool = False,
    ) -> torch.Tensor | tuple[torch.Tensor, ...]:
        m = attention_mask.clone().float()
        if drop_first_token and m.shape[1] > 1:
            m[:, 0] = 0.0
        den = m.sum(dim=1, keepdim=True).clamp(min=1e-5)
        first_device = input_ids.device

        if return_multiview:
            seq_idx = torch.cumsum(m, dim=1)
            half_len = (den * 0.5).clamp(min=1.0)
            m_head = m * (seq_idx <= half_len).float()
            m_tail = m * (seq_idx > half_len).float()
            den_head = m_head.sum(dim=1, keepdim=True).clamp(min=1e-5)
            den_tail = m_tail.sum(dim=1, keepdim=True).clamp(min=1e-5)

        try:
            outputs = self.backbone(
                input_ids=input_ids, attention_mask=attention_mask, output_hidden_states=True, return_dict=True, use_cache=False
            )
        except TypeError:
            outputs = self.backbone(input_ids=input_ids, attention_mask=attention_mask, output_hidden_states=True)

        hidden_states = getattr(outputs, "hidden_states", outputs)

        all_layers_pooled = []
        if extract_all_layers:
            for l_idx in range(len(hidden_states)):
                h = hidden_states[l_idx].float()
                layer_dev = h.device
                m_l = m.to(layer_dev)
                den_l = den.to(layer_dev)
                p_l = (h * m_l.unsqueeze(-1)).sum(dim=1) / den_l
                p_l = torch.nan_to_num(p_l, nan=0.0, posinf=65000.0, neginf=-65000.0).clamp(min=-65000.0, max=65000.0)
                all_layers_pooled.append(p_l.to(first_device))

        pooled_list = []
        head_tail_cos_list = []
        token_std_norm_list = []

        for idx, l in enumerate(self.layers):
            h = hidden_states[l].float()
            layer_dev = h.device
            m_l = m.to(layer_dev)
            den_l = den.to(layer_dev)
            pooled = (h * m_l.unsqueeze(-1)).sum(dim=1) / den_l
            pooled = torch.nan_to_num(pooled, nan=0.0, posinf=65000.0, neginf=-65000.0).clamp(min=-65000.0, max=65000.0)

            if return_multiview:
                mh_l = m_head.to(layer_dev).unsqueeze(-1)
                mt_l = m_tail.to(layer_dev).unsqueeze(-1)
                p_head = (h * mh_l).sum(dim=1) / den_head.to(layer_dev)
                p_tail = (h * mt_l).sum(dim=1) / den_tail.to(layer_dev)
                ht_cos = F.cosine_similarity(p_head, p_tail, dim=-1, eps=1e-6)
                head_tail_cos_list.append(ht_cos.to(first_device))

                diff_sq = ((h - pooled.unsqueeze(1)) ** 2) * m_l.unsqueeze(-1)
                tok_std = torch.sqrt((diff_sq.sum(dim=1) / den_l).clamp(min=1e-6))
                std_norm = torch.norm(tok_std, p=2, dim=-1)
                token_std_norm_list.append(std_norm.to(first_device))

            norm = self.norms[idx]
            norm_dev = next(norm.parameters()).device
            normed = norm(pooled.to(norm_dev))
            pooled_list.append(normed.to(first_device))

        Z_cat = torch.cat(pooled_list, dim=-1)

        if return_by_layer:
            Z_layers = torch.stack(all_layers_pooled if extract_all_layers else pooled_list, dim=1)
            if return_multiview:
                ht_cos_t = torch.stack(head_tail_cos_list, dim=1)
                std_norm_t = torch.stack(token_std_norm_list, dim=1)
                return Z_cat, Z_layers, ht_cos_t, std_norm_t
            return Z_cat, Z_layers

        return Z_cat

    def forward(
        self, input_ids: torch.Tensor, attention_mask: torch.Tensor, drop_first_token: bool = True
    ) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        Z = self.forward_features(input_ids, attention_mask, drop_first_token=drop_first_token)
        logits_plain = self.predict_plain(Z)
        logits_centered = self.predict_centered(Z)
        return Z, logits_plain, logits_centered


class _ManualFP16LossScaler:
    """Device-agnostic dynamic FP16 loss scaler safe across multi-GPU device_map='auto' pipelines."""

    def __init__(
        self,
        enabled: bool = True,
        scale: float = 256.0,
        min_scale: float = 1.0,
        max_scale: float = 4096.0,
        backoff_factor: float = 0.5,
        growth_factor: float = 2.0,
        growth_interval: int = 200,
    ) -> None:
        self.enabled = bool(enabled)
        self.scale = float(scale)
        self.min_scale = float(min_scale)
        self.max_scale = float(max_scale)
        self.backoff_factor = float(backoff_factor)
        self.growth_factor = float(growth_factor)
        self.growth_interval = int(growth_interval)
        self._growth_tracker = 0

    def is_enabled(self) -> bool:
        return self.enabled

    def scale_loss(self, loss: torch.Tensor) -> torch.Tensor:
        return loss * self.scale if self.enabled else loss

    def unscale_grads_(self, model: nn.Module) -> None:
        if not self.enabled or self.scale == 0.0:
            return
        inv_scale = 1.0 / self.scale
        for p in model.parameters():
            if p.requires_grad and p.grad is not None:
                p.grad.mul_(inv_scale)

    def update(self, found_inf: bool) -> None:
        if not self.enabled:
            return
        if found_inf:
            self.scale = max(self.scale * self.backoff_factor, self.min_scale)
            self._growth_tracker = 0
        else:
            self._growth_tracker += 1
            if self._growth_tracker >= self.growth_interval:
                self.scale = min(self.scale * self.growth_factor, self.max_scale)
                self._growth_tracker = 0


def _step_with_grad_guard(
    model: nn.Module,
    opt: torch.optim.Optimizer,
    sched: Any,
    scaler: Any | None = None,
    max_grad_norm: float = 1.0,
    manual_scaler: _ManualFP16LossScaler | None = None,
) -> bool:
    """Steps optimizer/scheduler safely, skipping update if any trainable gradient is non-finite."""
    use_manual = manual_scaler is not None and manual_scaler.is_enabled()
    use_scaler = (not use_manual) and scaler is not None and getattr(scaler, "is_enabled", lambda: False)()
    if use_manual:
        manual_scaler.unscale_grads_(model)
    elif use_scaler:
        scaler.unscale_(opt)
    for p in model.parameters():
        if p.requires_grad and p.grad is not None and not torch.isfinite(p.grad).all():
            if use_manual:
                manual_scaler.update(found_inf=True)
                print(
                    f"[Trainer Warning] Non-finite gradient detected; skipping optimizer step "
                    f"(new manual FP16 scale={manual_scaler.scale:.1f})."
                )
            else:
                print("[Trainer Warning] Non-finite gradient detected; skipping optimizer step to protect weights.")
                if use_scaler:
                    scaler.update()
            opt.zero_grad(set_to_none=True)
            return False
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
    if use_scaler:
        scaler.step(opt)
        scaler.update()
    else:
        opt.step()
    sched.step()
    if use_manual:
        manual_scaler.update(found_inf=False)
    opt.zero_grad(set_to_none=True)
    return True


def _get_loraplus_param_groups(
    model: MultiLayerLoRAClassifier,
    lr_lora: float,
    lr_head: float,
    lora_plus_ratio: float = 2.5,
    include_centered_head: bool = False,
) -> list[dict[str, Any]]:
    """Builds LoRA+ asymmetric parameter groups with higher learning rate on lora_B."""
    lora_a_params = []
    lora_b_params = []
    other_backbone_params = []
    for name, param in model.backbone.named_parameters():
        if not param.requires_grad:
            continue
        name_lower = name.lower()
        if "lora_b" in name_lower:
            lora_b_params.append(param)
        elif "lora_a" in name_lower:
            lora_a_params.append(param)
        else:
            other_backbone_params.append(param)

    head_params = list(model.norms.parameters()) + list(model.plain_head.parameters())
    if include_centered_head and hasattr(model, "centered_head"):
        head_params += list(model.centered_head.parameters())

    groups = []
    if lora_a_params:
        groups.append({"params": lora_a_params, "lr": float(lr_lora)})
    if lora_b_params:
        groups.append({"params": lora_b_params, "lr": float(lr_lora) * float(lora_plus_ratio)})
    if other_backbone_params:
        groups.append({"params": other_backbone_params, "lr": float(lr_lora)})
    if head_params:
        groups.append({"params": head_params, "lr": float(lr_head)})
    return groups


def _build_class_weight_tensor(
    active_labels: list[str],
    sub_lab2i: dict[str, int],
    stage_cfg: dict[str, Any],
    default_hard_weight: float,
    device: torch.device,
) -> torch.Tensor:
    """Builds per-class CE loss weights from explicit class_weights map and hard_class_weight fallback."""
    num_classes = len(active_labels)
    hard_set = set(HARD_CLASS_INDICES)
    base_hard = float(stage_cfg.get("hard_class_weight", default_hard_weight))
    cls_map = stage_cfg.get("class_weights") or {}
    weights = torch.ones(num_classes, device=device, dtype=torch.float32)
    for lbl, local_idx in sub_lab2i.items():
        if lbl in cls_map:
            weights[local_idx] = float(cls_map[lbl])
        elif LAB2I.get(lbl) in hard_set and base_hard != 1.0:
            weights[local_idx] = base_hard
    return weights


def export_compatible_checkpoint(
    model: MultiLayerLoRAClassifier,
    output_dir: Path,
    layers: list[int],
    keep: int,
    max_len: int = 447,
    end_id: int = 151643,
    backbone_name: str | None = None,
    Z_train: np.ndarray | None = None,
    y_train: np.ndarray | None = None,
    prompts_train: np.ndarray | None = None,
    class_labels: list[str] | None = None,
    target_family: str | None = None,
) -> Path:
    """Exports standard checkpoint artifacts: adapter/, heads.safetensors, meta.json, and optional wPCA_bank.npz."""
    out_p = Path(output_dir)
    out_p.mkdir(parents=True, exist_ok=True)

    adapter_dir = out_p / "adapter"
    adapter_dir.mkdir(parents=True, exist_ok=True)
    if hasattr(model.backbone, "save_pretrained"):
        try:
            model.backbone.save_pretrained(str(adapter_dir))
        except Exception as e:
            print(f"[Trainer] Note on saving adapter: {e}")
    elif hasattr(model, "save_pretrained"):
        try:
            model.save_pretrained(str(adapter_dir))
        except Exception as e:
            print(f"[Trainer] Note on saving model: {e}")

    heads_dict = {}
    for i, norm in enumerate(model.norms):
        heads_dict[f"norms.{i}.weight"] = norm.weight.detach().cpu().contiguous()
        if hasattr(norm, "bias") and norm.bias is not None:
            heads_dict[f"norms.{i}.bias"] = norm.bias.detach().cpu().contiguous()

    heads_dict["plain.weight"] = model.plain_head.weight.detach().cpu().contiguous()
    heads_dict["plain.bias"] = model.plain_head.bias.detach().cpu().contiguous()
    heads_dict["centered.weight"] = model.centered_head.weight.detach().cpu().contiguous()
    heads_dict["centered.bias"] = model.centered_head.bias.detach().cpu().contiguous()

    save_file(heads_dict, str(out_p / "heads.safetensors"))

    active_labels = list(class_labels) if class_labels is not None else list(LABELS)
    meta = {
        "labels": active_labels,
        "classes": active_labels,
        "target_family": str(target_family or ""),
        "backbone": str(backbone_name) if backbone_name else "",
        "cfg": {
            "keep": int(keep),
            "layers": [int(x) for x in layers],
            "max_len": int(max_len),
            "end_token_id": int(end_id),
            "backbone": str(backbone_name) if backbone_name else "",
            "target_family": str(target_family or ""),
        },
    }
    (out_p / "meta.json").write_text(json.dumps(meta, indent=2), encoding="utf-8")

    if len(active_labels) == K and Z_train is not None and y_train is not None and prompts_train is not None:
        d_target = min(256, Z_train.shape[1])
        res = fit_shared_gaussian_lowram(Z_train, y_train, prompts_train, d_target=d_target)
        np.savez_compressed(
            str(out_p / "wPCA_bank.npz"),
            mx=res["mx"].astype(np.float32),
            sx=res["sx"].astype(np.float32),
            mu=res["mx"].astype(np.float32),
            sd=res["sx"].astype(np.float32),
            W=res["W"].astype(np.float32),
            m=res["m"].astype(np.float32),
            lam=res["lam"].astype(np.float32),
            V_top64=res["V_top64"].astype(np.float32),
            layers=np.array(layers, dtype=np.int32),
        )

    return out_p


def find_compatible_champion_lora(
    input_dir: Path,
    backbone_name: str,
    lora_r: int = 32,
    target_family: str = "",
) -> Path | None:
    """Finds the matching fine-tuned LoRA adapter directory for the given backbone and rank."""
    if not input_dir or not Path(input_dir).exists():
        return None

    input_p = Path(input_dir)
    candidates = list(input_p.rglob("adapter_model.safetensors")) + list(input_p.rglob("adapter_model.bin"))
    if not candidates:
        return None

    b_lower = backbone_name.lower()
    is_qwen3 = "qwen3" in b_lower or ("qwen" in b_lower and "2" not in b_lower and "2.5" not in b_lower)
    is_qwen25 = "qwen2" in b_lower or "2.5" in b_lower
    is_deberta = "deberta" in b_lower
    is_bge = "bge" in b_lower or "xlm-roberta" in b_lower

    matching_dirs: list[tuple[int, Path]] = []

    for cand in candidates:
        p_dir = cand.parent
        p_str = str(p_dir).lower().replace("\\", "/")

        if target_family:
            other_fams = [f for f in FAMILIES.keys() if f != target_family.lower()]
            if any(f"export_{f}" in p_str or f"{f}_gate" in p_str or f"{f}_spec" in p_str for f in other_fams):
                continue

        if is_bge and any(x in p_str for x in ("qwen", "deberta", "llama")) and "bge" not in p_str and "xlm-roberta" not in p_str:
            continue
        if (is_qwen3 or is_qwen25 or is_deberta) and ("bge" in p_str or "xlm-roberta" in p_str):
            if not any(x in p_str for x in ("qwen", "deberta")):
                continue

        cfg_p = p_dir / "adapter_config.json"
        if not cfg_p.exists():
            cfg_p = p_dir.parent / "adapter_config.json"

        score = 0
        if cfg_p.exists():
            try:
                c_data = json.loads(cfg_p.read_text(encoding="utf-8"))
                r_found = int(c_data.get("r", 0))
                if lora_r > 0 and r_found > 0 and r_found != lora_r:
                    continue

                base_str = str(c_data.get("base_model_name_or_path", "")).lower()
                if is_qwen3 and ("qwen2" in base_str or "2.5" in base_str or "llama" in base_str or "deberta" in base_str or "bge" in base_str or "xlm-roberta" in base_str):
                    continue
                if is_qwen25 and ("qwen3" in base_str or "llama" in base_str or "deberta" in base_str or "bge" in base_str or "xlm-roberta" in base_str) and "2.5" not in base_str and "qwen2" not in base_str:
                    continue
                if is_deberta and base_str and "deberta" not in base_str:
                    continue
                if is_bge and base_str and "bge" not in base_str and "xlm-roberta" not in base_str:
                    continue

                targets = {str(t).lower() for t in (c_data.get("target_modules") or [])}
                if is_bge and ("q_proj" in targets or "query_proj" in targets):
                    continue

                if is_qwen3 and "qwen3" in base_str:
                    score += 10
                if is_qwen25 and ("qwen2" in base_str or "2.5" in base_str):
                    score += 10
                if is_deberta and "deberta" in base_str:
                    score += 10
                if is_bge and ("bge" in base_str or "xlm-roberta" in base_str):
                    score += 10
            except Exception:
                pass

        if "export_champion" in p_str:
            score += 20
        if "inhouse" in p_str:
            score += 15
        if is_qwen3 and "qwen3_14b" in p_str:
            score += 10
        if is_qwen25 and ("qwen25" in p_str or "qwen2" in p_str):
            score += 10
        if is_bge and ("bge" in p_str or "xlm-roberta" in p_str):
            score += 10

        if is_qwen3 and ("qwen2" in p_str or "2.5" in p_str or "8b" in p_str):
            continue
        if is_qwen25 and ("qwen3" in p_str or "8b" in p_str):
            continue

        matching_dirs.append((score, p_dir))

    if not matching_dirs:
        return None

    matching_dirs.sort(key=lambda x: x[0], reverse=True)
    return matching_dirs[0][1]


def load_and_slice_champion_heads(
    heads_path: Path,
    model: nn.Module,
    fam_cols: list[int] | None = None,
) -> bool:
    """Loads champion per-layer LayerNorms, plain_head, and centered_head after verifying shape compatibility."""
    if not Path(heads_path).exists():
        return False

    try:
        if str(heads_path).endswith(".safetensors"):
            h_sd = load_file(str(heads_path))
        else:
            h_sd = torch.load(str(heads_path), map_location="cpu")

        if not hasattr(model, "plain_head") or model.plain_head is None:
            return False

        w_ckpt = h_sd.get("plain.weight") if "plain.weight" in h_sd else h_sd.get("weight", None)
        b_ckpt = h_sd.get("plain.bias") if "plain.bias" in h_sd else h_sd.get("bias", None)
        if w_ckpt is None or b_ckpt is None:
            return False

        tgt_w = model.plain_head.weight
        tgt_b = model.plain_head.bias

        is_exact = tgt_w.shape == w_ckpt.shape and tgt_b.shape == b_ckpt.shape
        is_sliced = False
        sliced_w, sliced_b = None, None
        if not is_exact:
            if fam_cols is not None and w_ckpt.shape[0] >= max(fam_cols) + 1:
                sliced_w = w_ckpt[fam_cols]
                sliced_b = b_ckpt[fam_cols]
                if sliced_w.shape == tgt_w.shape and sliced_b.shape == tgt_b.shape:
                    is_sliced = True
            if not is_sliced:
                return False

        if is_exact:
            tgt_w.data.copy_(w_ckpt.to(tgt_w.device, dtype=tgt_w.dtype))
            tgt_b.data.copy_(b_ckpt.to(tgt_b.device, dtype=tgt_b.dtype))
        else:
            tgt_w.data.copy_(sliced_w.to(tgt_w.device, dtype=tgt_w.dtype))
            tgt_b.data.copy_(sliced_b.to(tgt_b.device, dtype=tgt_b.dtype))

        if hasattr(model, "norms") and model.norms is not None:
            for i, norm in enumerate(model.norms):
                wk, bk = f"norms.{i}.weight", f"norms.{i}.bias"
                if wk in h_sd and norm.weight.shape == h_sd[wk].shape:
                    norm.weight.data.copy_(h_sd[wk].to(norm.weight.device, dtype=norm.weight.dtype))
                if bk in h_sd and hasattr(norm, "bias") and norm.bias is not None and norm.bias.shape == h_sd[bk].shape:
                    norm.bias.data.copy_(h_sd[bk].to(norm.bias.device, dtype=norm.bias.dtype))

        if hasattr(model, "centered_head") and model.centered_head is not None:
            cw = h_sd.get("centered.weight", None)
            cb = h_sd.get("centered.bias", None)
            if cw is not None and cb is not None:
                tgt_cw = model.centered_head.weight
                tgt_cb = model.centered_head.bias
                if tgt_cw.shape == cw.shape and tgt_cb.shape == cb.shape:
                    tgt_cw.data.copy_(cw.to(tgt_cw.device, dtype=tgt_cw.dtype))
                    tgt_cb.data.copy_(cb.to(tgt_cb.device, dtype=tgt_cb.dtype))
                elif fam_cols is not None and cw.shape[0] >= max(fam_cols) + 1:
                    cw_sl = cw[fam_cols]
                    cb_sl = cb[fam_cols]
                    if cw_sl.shape == tgt_cw.shape and cb_sl.shape == tgt_cb.shape:
                        tgt_cw.data.copy_(cw_sl.to(tgt_cw.device, dtype=tgt_cw.dtype))
                        tgt_cb.data.copy_(cb_sl.to(tgt_cb.device, dtype=tgt_cb.dtype))

        return True
    except Exception as e:
        print(f"[Trainer] Note on loading/slicing champion head: {e}")
        return False


In [ ]:
# === MODULE: src/namethatllm/training/data.py ===
"""
Dataset Loading, Zero-Leak Quarantine, Conflict-Free Deduplication, Sample Weight Normalization, and Samplers.
"""

from __future__ import annotations

import hashlib
from collections.abc import Iterator
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, Sampler

try:
    import pyarrow.parquet as pq
except ImportError:
    pq = None



PROBLEM_WATCH_CLASSES: tuple[str, ...] = (
    "kimi-k3",
    "glm-5.3-flash",
    "glm-5.3",
    "gpt-5.6-sol",
    "claude-fable-5",
    "gpt-5.6-terra",
    "gemini-3.8-flash",
    "gemini-3.7-flash",
    LABELS[14],
)


def norm_text_hash(text: str) -> str:
    """Normalized whitespace SHA-1 hash for strict zero-leak deduplication."""
    norm = " ".join(str(text).strip().split())
    return hashlib.sha1(norm.encode("utf-8")).hexdigest()


def _select_stage1_pool_files(pool_list: list[Path], use_v2_only: bool = True) -> list[Path]:
    """Prefers clean_master_pool_v2 files exclusively when present so v1 rows are not mixed back in."""
    if use_v2_only:
        v2_files = [p for p in pool_list if "clean_master_pool_v2" in Path(p).name.lower()]
        if v2_files:
            return v2_files
    return list(pool_list)


def _normalize_match_weights(
    values: Any,
    weight_clip: tuple[float, float] = (0.5, 2.5),
    groups: pd.Series | np.ndarray | None = None,
) -> np.ndarray:
    """Clips numeric sample match weights into a safe range and normalizes mean to 1.0 (per group if provided)."""
    lo, hi = float(weight_clip[0]), float(weight_clip[1])
    raw_w = pd.to_numeric(values, errors="coerce")
    if isinstance(raw_w, pd.Series):
        raw_arr = raw_w.fillna(1.0).to_numpy(dtype=np.float32)
    else:
        raw_arr = np.nan_to_num(np.asarray(raw_w, dtype=np.float32), nan=1.0)
    clipped = np.clip(raw_arr, lo, hi).astype(np.float32)
    if groups is not None:
        grp_arr = np.asarray(groups)
        if len(grp_arr) == len(clipped) and len(clipped) > 0:
            out = clipped.copy()
            for g in np.unique(grp_arr):
                mask = grp_arr == g
                if mask.any():
                    out[mask] = clipped[mask] / max(float(clipped[mask].mean()), 1e-6)
            return out.astype(np.float32)
    return (clipped / max(float(clipped.mean()), 1e-6)).astype(np.float32)


def _load_stage1_pool_df(pool_list: list[Path]) -> tuple[pd.DataFrame, dict[str, int]]:
    """Loads Stage-1 pool files preserving optional match_weight, fidelity, and prompt metadata."""
    opt_cols = ["text", "model", "match_weight", "source_fidelity", "prompt_canon", "priority"]
    dfs = []
    pool_source_counts: dict[str, int] = {}
    for p in pool_list:
        try:
            p_str = str(p)
            if p_str.endswith(".parquet"):
                if pq is not None:
                    avail_cols = set(pq.read_schema(p).names)
                    cols = [c for c in opt_cols if c in avail_cols]
                    df_one = pd.read_parquet(p, columns=cols if cols else ["text", "model"])
                else:
                    df_one = pd.read_parquet(p)
                    cols = [c for c in opt_cols if c in df_one.columns]
                    if cols:
                        df_one = df_one[cols]
            else:
                hdr = pd.read_csv(p, nrows=0).columns
                cols = [c for c in opt_cols if c in hdr]
                df_one = pd.read_csv(p, usecols=cols if cols else ["text", "model"])
            pool_source_counts[Path(p).name] = len(df_one)
            dfs.append(df_one)
        except Exception as e:
            print(f"[Trainer] Could not read pool {p}: {e}")
    df_p = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame(columns=["text", "model"])
    return df_p, pool_source_counts


def load_clean_stage1_pool(
    pool_list: list[Path],
    input_dir: Path | None = None,
    train_path: Path | None = None,
    bench_bundle: tuple[list[str], np.ndarray, list[str], np.ndarray, np.ndarray] | None = None,
    active_labels: list[str] | None = None,
    exclude_train_overlap: bool = True,
    weight_clip: tuple[float, float] = (0.5, 2.5),
    use_match_weight: bool = True,
    min_char_len: int = 40,
    include_unseen_holdout: bool = True,
) -> tuple[pd.DataFrame, dict[str, int]]:
    """
    Unified Stage-1 pool loader: ingests pool files, filters valid rows, drops cross-label
    `_norm_hash` conflicts, deterministically deduplicates by `(priority asc, match_weight desc)`,
    quarantines all unseen holdout / Bench-1800 / train.csv overlaps, and normalizes `match_weight` per class.
    """
    df_p, pool_source_counts = _load_stage1_pool_df(pool_list)
    total_raw_rows = sum(pool_source_counts.values())

    allowed = set(active_labels) if active_labels is not None else set(LABELS)
    df_p = df_p.dropna(subset=["text", "model"])
    df_p["text"] = df_p["text"].astype(str)
    if min_char_len > 0:
        df_p = df_p[df_p["text"].str.strip().str.len() >= min_char_len]
    df_p = df_p[df_p["model"].isin(allowed)].reset_index(drop=True)

    df_p["_hash"] = [_text_hash(t) for t in df_p["text"]]
    df_p["_norm_hash"] = [norm_text_hash(t) for t in df_p["text"]]
    label_nunique = df_p.groupby("_norm_hash")["model"].transform("nunique")
    n_conflict_rows = int((label_nunique > 1).sum())
    if n_conflict_rows > 0:
        n_conflict_hashes = int(df_p.loc[label_nunique > 1, "_norm_hash"].nunique())
        print(
            f"[Trainer] Dropped {n_conflict_rows} rows across {n_conflict_hashes} cross-label _norm_hash conflicts before deduplication."
        )
        df_p = df_p[label_nunique == 1].reset_index(drop=True)

    sort_cols: list[str] = []
    sort_asc: list[bool] = []
    if "priority" in df_p.columns:
        sort_cols.append("priority")
        sort_asc.append(True)
    if "match_weight" in df_p.columns:
        sort_cols.append("match_weight")
        sort_asc.append(False)
    if sort_cols:
        df_p = df_p.sort_values(by=sort_cols, ascending=sort_asc, kind="mergesort")
    df_p = df_p.drop_duplicates(subset=["_norm_hash"], keep="first").reset_index(drop=True)
    dedup_rows = len(df_p)

    quarantine_dropped = 0
    try:
        holdout_raw: set[str] = set()
        holdout_norm: set[str] = set()
        if include_unseen_holdout and input_dir is not None:
            _, holdout_hashes = build_unseen_holdout_dataset(input_dir, train_path)
            if holdout_hashes:
                holdout_raw.update(holdout_hashes)
                holdout_norm.update(holdout_hashes)
        if bench_bundle is not None:
            for t_b in bench_bundle[0]:
                holdout_raw.add(_text_hash(t_b))
                holdout_norm.add(norm_text_hash(t_b))
        elif input_dir is not None and input_dir.exists():
            bench_paths = list(input_dir.rglob("clean_benchmark_1800_rows.parquet"))
            if bench_paths:
                df_bench_q = pd.read_parquet(bench_paths[0])
                for t_b in df_bench_q["text"].dropna():
                    holdout_raw.add(_text_hash(t_b))
                    holdout_norm.add(norm_text_hash(t_b))
        if train_path and Path(train_path).exists() and exclude_train_overlap:
            df_tr_q = pd.read_csv(train_path, usecols=["text"])
            for t_q in df_tr_q["text"].dropna():
                holdout_raw.add(_text_hash(t_q))
                holdout_norm.add(norm_text_hash(t_q))

        if holdout_raw or holdout_norm:
            mask_clean = (~df_p["_hash"].isin(holdout_raw)) & (~df_p["_norm_hash"].isin(holdout_norm))
            quarantine_dropped = int(len(df_p) - mask_clean.sum())
            if quarantine_dropped > 0:
                print(
                    f"[Trainer] Quarantined {quarantine_dropped} pool rows that overlapped with unseen holdout / clean benchmark / train."
                )
            df_p = df_p[mask_clean].reset_index(drop=True)
    except Exception as e:
        print(f"[Trainer] Note on holdout quarantine: {e}")

    if use_match_weight and ("match_weight" in df_p.columns):
        df_p["_sample_weight"] = _normalize_match_weights(
            df_p["match_weight"], weight_clip=weight_clip, groups=df_p["model"]
        )
        print(
            f"[Trainer] Loaded & clipped per-class Stage-1 match_weight (clip={weight_clip}, "
            f"mean={df_p['_sample_weight'].mean():.3f}, max={df_p['_sample_weight'].max():.3f})"
        )

    stats = {
        "source_counts": pool_source_counts,
        "total_raw_rows": int(total_raw_rows),
        "dedup_rows": int(dedup_rows),
        "quarantine_dropped": int(quarantine_dropped),
        "conflict_rows": int(n_conflict_rows),
    }
    return df_p, stats


def load_zero_leak_stage1_pool(
    datasets_dir: Path,
    weight_clip: tuple[float, float] = (0.5, 2.5),
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Loads clean_benchmark_1800_rows.parquet, train.csv, and builds the deduplicated
    Stage-1 pool strictly excluding all normalized text hashes in Bench-1800 or train.csv.
    """
    bench_path = datasets_dir / "clean_benchmark_1800_rows.parquet"
    df_bench = pd.read_parquet(bench_path)
    df_bench = df_bench[df_bench["model"].isin(LABELS)].reset_index(drop=True)
    bench_hashes = set(df_bench["text"].map(norm_text_hash))

    train_path = datasets_dir / "train.csv"
    df_train = pd.read_csv(train_path)
    df_train = df_train[df_train["model"].isin(LABELS)].reset_index(drop=True)
    df_train = df_train[~df_train["text"].map(norm_text_hash).isin(bench_hashes)].reset_index(drop=True)
    train_hashes = set(df_train["text"].map(norm_text_hash))

    excl_hashes = bench_hashes | train_hashes
    v2_path = datasets_dir / "clean_master_pool_v2.parquet"
    v1_path = datasets_dir / "clean_master_pool.parquet"
    master_path = v2_path if v2_path.exists() else v1_path

    df_master, _ = load_clean_stage1_pool(
        pool_list=[master_path],
        input_dir=datasets_dir,
        train_path=train_path,
        bench_bundle=(df_bench["text"].astype(str).tolist(), np.array([]), [], np.array([]), np.array([])),
        active_labels=list(LABELS),
        exclude_train_overlap=True,
        weight_clip=weight_clip,
        use_match_weight=False,
        min_char_len=0,
        include_unseen_holdout=False,
    )
    if "priority" not in df_master.columns:
        df_master["priority"] = 1
    if "match_weight" not in df_master.columns:
        df_master["match_weight"] = 1.0
    df_master["nh"] = df_master["_norm_hash"]

    master_hashes = set(df_master["nh"])
    parts = [df_master[["text", "model", "priority", "match_weight", "nh"]]]
    pool_v4_path = datasets_dir / "pool_v4.parquet"
    if not v2_path.exists() and pool_v4_path.exists():
        df_v4 = pd.read_parquet(pool_v4_path, columns=["text", "model"])
        df_v4 = df_v4[df_v4["model"].isin(LABELS)].copy()
        df_v4["nh"] = df_v4["text"].map(norm_text_hash)
        df_extra = df_v4[~df_v4["nh"].isin(excl_hashes | master_hashes)].drop_duplicates("nh").copy()
        if len(df_extra) > 0:
            df_extra["priority"] = 2
            df_extra["match_weight"] = 1.0
            parts.append(df_extra[["text", "model", "priority", "match_weight", "nh"]])

    df_pool = pd.concat(parts, ignore_index=True).drop(columns=["nh"]).reset_index(drop=True)
    df_pool["sample_weight"] = _normalize_match_weights(df_pool["match_weight"], weight_clip=weight_clip, groups=df_pool["model"])
    df_pool["label_idx"] = np.array([LAB2I[m] for m in df_pool["model"]], dtype=np.int64)

    pool_hashes = set(df_pool["text"].map(norm_text_hash))
    assert len(pool_hashes & bench_hashes) == 0, "Leak detected against Bench-1800!"
    assert len(pool_hashes & train_hashes) == 0, "Leak detected against train.csv!"
    return df_pool, df_train, df_bench


def build_targeted_epoch_indices(
    labels: np.ndarray,
    class_quotas: dict[str, int],
    epoch: int,
    seed: int = 42,
    class_labels: list[str] | None = None,
) -> np.ndarray:
    """Builds one epoch's index list matching per-class quotas via modular sliding slices and clean oversampling."""
    active_names = class_labels if class_labels is not None else list(LABELS)
    chosen_parts: list[np.ndarray] = []
    rng_ep = np.random.RandomState(seed + 1009 * (epoch + 1))
    for c_idx, name in enumerate(active_names):
        pool_idx = np.where(labels == c_idx)[0]
        n_avail = len(pool_idx)
        assert n_avail > 0, f"Class {name} has 0 examples in Stage-1 pool!"
        quota = int(class_quotas.get(name, n_avail))
        if quota <= 0:
            continue
        perm = pool_idx[np.random.RandomState(seed + 97 * c_idx).permutation(n_avail)]
        if quota <= n_avail:
            start = (epoch * quota) % n_avail
            if start + quota <= n_avail:
                sel = perm[start : start + quota]
            else:
                first = perm[start:]
                sel = np.concatenate([first, perm[: quota - len(first)]])
        else:
            reps = [perm] * (quota // n_avail)
            rem = quota % n_avail
            if rem > 0:
                reps.append(rng_ep.choice(perm, size=rem, replace=False))
            sel = np.concatenate(reps)
        chosen_parts.append(sel)

    out = np.concatenate(chosen_parts) if chosen_parts else np.array([], dtype=np.int64)
    rng_ep.shuffle(out)
    return out


def _load_benchmark_holdout(
    input_dir: Path, subset_size: int = 0, seed: int = 42
) -> tuple[list[str], np.ndarray, list[str], np.ndarray, np.ndarray] | None:
    """Loads clean_benchmark_1800_rows.parquet and optionally builds a deterministic stratified slice."""
    bench_paths = list(input_dir.rglob("clean_benchmark_1800_rows.parquet")) if input_dir.exists() else []
    if not bench_paths:
        for cand in (Path("clean_benchmark_1800_rows.parquet"), Path("local_data/clean_benchmark_1800_rows.parquet")):
            if cand.exists():
                bench_paths = [cand]
                break
    if not bench_paths:
        return None
    try:
        df_b = pd.read_parquet(bench_paths[0])
        df_b = df_b[df_b["model"].isin(LABELS)].reset_index(drop=True)
        if df_b.empty:
            return None
        full_texts = df_b["text"].astype(str).tolist()
        full_y = np.array([LAB2I[m] for m in df_b["model"]], dtype=np.int64)
        if subset_size > 0 and subset_size < len(full_y):
            rng = np.random.RandomState(seed)
            per_cls = max(1, subset_size // len(LABELS))
            idx_parts = []
            for c_i in range(len(LABELS)):
                c_idx = np.where(full_y == c_i)[0]
                if len(c_idx) > 0:
                    take = min(per_cls, len(c_idx))
                    idx_parts.append(rng.choice(c_idx, size=take, replace=False))
            sub_idx = np.sort(np.concatenate(idx_parts)).astype(np.int64)
            sub_texts = [full_texts[int(i)] for i in sub_idx]
            sub_y = full_y[sub_idx]
        else:
            sub_idx = np.arange(len(full_y), dtype=np.int64)
            sub_texts, sub_y = full_texts, full_y
        return full_texts, full_y, sub_texts, sub_y, sub_idx
    except Exception as e:
        print(f"[Trainer] Note loading benchmark holdout: {e}")
        return None


class PromptGroupBatchSampler(Sampler[list[int]]):
    """Yields entire prompt clusters per step so within-prompt centering matches inference."""

    def __init__(self, prompt_ids: list[Any], batch_size: int | None = None, shuffle: bool = True, seed: int = 42):
        super().__init__()
        self.groups: dict[Any, list[int]] = {}
        for idx, pid in enumerate(prompt_ids):
            self.groups.setdefault(pid, []).append(idx)
        self.group_keys = list(self.groups.keys())
        self.shuffle = shuffle
        self.seed = seed
        self.epoch = 0

    def set_epoch(self, ep: int) -> None:
        self.epoch = ep

    def __len__(self) -> int:
        return len(self.group_keys)

    def __iter__(self) -> Iterator[list[int]]:
        rng = np.random.RandomState(self.seed + self.epoch)
        keys = list(self.group_keys)
        if self.shuffle:
            rng.shuffle(keys)
        for k in keys:
            idxs = list(self.groups[k])
            if self.shuffle:
                rng.shuffle(idxs)
            yield idxs


class TextSequenceDataset(Dataset):
    """Dataset applying exact head+tail truncation matching neural_extractor.UniversalLoRATrunkLoader."""

    def __init__(
        self,
        texts: list[str],
        labels: list[int] | None,
        prompt_ids: list[Any] | None,
        tok: Any,
        max_len: int = 384,
        end_id: int | None = 151643,
        cls_id: int | None = None,
        sample_weights: list[float] | None = None,
        tok_spec: TokenizationSpec | None = None,
    ):
        self.texts = texts
        self.labels = labels
        self.prompt_ids = prompt_ids
        self.tok = tok
        self.tok_spec = tok_spec
        self.max_len = tok_spec.max_len if tok_spec is not None else max_len
        self.end_id = tok_spec.end_id if tok_spec is not None else end_id
        self.cls_id = tok_spec.cls_id if tok_spec is not None else cls_id
        self.sample_weights = sample_weights

    def __len__(self) -> int:
        return len(self.texts)

    def __getitem__(self, idx: int) -> dict[str, Any]:
        full_ids = self.tok(str(self.texts[idx]), add_special_tokens=False, truncation=False)["input_ids"]
        if len(full_ids) > self.max_len:
            head_len = self.max_len // 2
            tail_len = self.max_len - head_len
            raw_ids = full_ids[:head_len] + full_ids[-tail_len:]
        else:
            raw_ids = full_ids
        input_ids = ([self.cls_id] if self.cls_id is not None else []) + raw_ids
        if self.end_id is not None:
            input_ids = input_ids + [self.end_id]
        item: dict[str, Any] = {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "attention_mask": torch.ones(len(input_ids), dtype=torch.long),
        }
        if self.labels is not None:
            item["label"] = torch.tensor(self.labels[idx], dtype=torch.long)
        if self.prompt_ids is not None:
            item["prompt_id"] = self.prompt_ids[idx]
        if self.sample_weights is not None:
            item["sample_weight"] = torch.tensor(float(self.sample_weights[idx]), dtype=torch.float32)
        return item


def collate_text_batch(batch: list[dict[str, Any]], pad_token_id: int = 151643) -> dict[str, Any]:
    input_ids = [item["input_ids"] for item in batch]
    masks = [item.get("attention_mask", torch.ones_like(item["input_ids"])) for item in batch]
    padded_ids = torch.nn.utils.rnn.pad_sequence(input_ids, batch_first=True, padding_value=pad_token_id)
    padded_mask = torch.nn.utils.rnn.pad_sequence(masks, batch_first=True, padding_value=0)
    res: dict[str, Any] = {"input_ids": padded_ids, "attention_mask": padded_mask}
    if "label" in batch[0]:
        res["labels"] = torch.stack([item["label"] for item in batch])
    if "prompt_id" in batch[0]:
        res["prompt_ids"] = [item["prompt_id"] for item in batch]
    if "sample_weight" in batch[0]:
        res["sample_weights"] = torch.stack([item["sample_weight"] for item in batch])
    return res


class BucketedLengthBatchSampler(Sampler[list[int]]):
    """Groups indices into mega-buckets sorted by length with jittered length sorting to minimize padding."""

    def __init__(self, lengths: list[int], batch_size: int, bucket_size_multiplier: int = 50, shuffle: bool = True, seed: int = 42):
        super().__init__()
        self.lengths = lengths
        self.batch_size = batch_size
        self.bucket_size = batch_size * bucket_size_multiplier
        self.shuffle = shuffle
        self.seed = seed
        self.epoch = 0

    def set_epoch(self, ep: int) -> None:
        self.epoch = ep

    def __len__(self) -> int:
        return (len(self.lengths) + self.batch_size - 1) // self.batch_size

    def __iter__(self) -> Iterator[list[int]]:
        rng = np.random.RandomState(self.seed + self.epoch)
        indices = np.arange(len(self.lengths))
        if self.shuffle:
            rng.shuffle(indices)
        batches = []
        for s in range(0, len(indices), self.bucket_size):
            bucket = indices[s : s + self.bucket_size]
            lens = np.array([self.lengths[i] for i in bucket], dtype=np.float64)
            if self.shuffle and len(bucket) > 1:
                lens = lens + rng.uniform(-12.0, 12.0, size=len(bucket))
            sorted_bucket = bucket[np.argsort(lens)]
            for b in range(0, len(sorted_bucket), self.batch_size):
                batches.append(sorted_bucket[b : b + self.batch_size].tolist())
        if self.shuffle:
            rng.shuffle(batches)
        for batch in batches:
            yield batch


In [ ]:
# === MODULE: src/namethatllm/training/evaluation.py ===
"""
Evaluation, Cross-Validation, Ridge Head Solvers, Grouped Inference Logits, and Feature Extraction.
"""

from __future__ import annotations

import functools
import json
from pathlib import Path
from typing import Any

import numpy as np
import torch
from sklearn.metrics import balanced_accuracy_score, confusion_matrix, log_loss
from sklearn.model_selection import GroupKFold
from torch import nn
from torch.utils.data import DataLoader



def _solve_centered_dual_ridge(
    X: np.ndarray,
    Y: np.ndarray,
    alpha: float,
    scale_by_trace: bool = True,
) -> tuple[np.ndarray, np.ndarray]:
    """Solves centered float64 dual Ridge regression with exact unpenalized intercept."""
    N = X.shape[0]
    mu_x = X.mean(axis=0, keepdims=True).astype(np.float64)
    mu_y = Y.mean(axis=0, keepdims=True).astype(np.float64)
    X_c = X.astype(np.float64) - mu_x
    Y_c = Y.astype(np.float64) - mu_y
    K_mat = X_c @ X_c.T
    if scale_by_trace:
        mean_diag = float(np.mean(np.sum(X_c * X_c, axis=1)))
        eff_alpha = float(alpha) * max(mean_diag / 512.0, 1.0)
    else:
        eff_alpha = float(alpha)
    K_mat.flat[:: N + 1] += max(eff_alpha, 1e-6)
    A = np.linalg.solve(K_mat, Y_c)
    W = (A.T @ X_c).astype(np.float32)
    b = (mu_y.ravel() - (mu_x @ W.astype(np.float64).T).ravel()).astype(np.float32)
    return W, b


def fit_ridge_heads(
    Z: np.ndarray,
    y: np.ndarray,
    prompts: np.ndarray | None = None,
    alpha: float = 10.0,
    label_smoothing: float = 0.10,
    num_classes: int = K,
    scale_by_trace: bool = True,
) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """Fits closed-form dual Ridge regression heads for plain and centered representations."""
    N, _ = Z.shape
    Y = np.zeros((N, num_classes), dtype=np.float32)
    for i, lab in enumerate(y):
        Y[i, lab] = 1.0
    if label_smoothing > 0.0:
        Y = (1.0 - label_smoothing) * Y + (label_smoothing / num_classes)

    W_p, b_p = _solve_centered_dual_ridge(Z, Y, alpha=alpha, scale_by_trace=scale_by_trace)

    Z_c = Z.copy()
    if prompts is not None:
        for p in np.unique(prompts):
            mask = prompts == p
            if mask.sum() > 1:
                Z_c[mask] = Z[mask] - Z[mask].mean(axis=0, keepdims=True)
            else:
                Z_c[mask] = 0.0

    W_c, b_c = _solve_centered_dual_ridge(Z_c, Y, alpha=alpha, scale_by_trace=scale_by_trace)
    return W_p, b_p, W_c, b_c


def compute_grouped_inference_logits(
    Z: np.ndarray,
    plain_weight: np.ndarray,
    plain_bias: np.ndarray,
    centered_weight: np.ndarray,
    centered_bias: np.ndarray,
    groups: list[Any] | np.ndarray | None = None,
    blend_centered: float = 0.20,
) -> dict[str, np.ndarray]:
    """Computes plain, gated prompt-group centered, blended logits, and Sinkhorn-balanced probabilities."""
    logits_plain = Z @ plain_weight.T + plain_bias
    if groups is not None and len(groups) == len(Z):
        Z_c = Z.copy()
        singleton_mask = np.zeros(len(Z), dtype=bool)
        grps_arr = np.asarray(groups)
        plain_preds = logits_plain.argmax(axis=1)
        for g in np.unique(grps_arr):
            mask = grps_arr == g
            distinct_classes = len(np.unique(plain_preds[mask]))
            if mask.sum() >= 6 and distinct_classes >= 4:
                Z_c[mask] = Z[mask] - 0.75 * Z[mask].mean(axis=0, keepdims=True)
            else:
                singleton_mask[mask] = True
        logits_centered = Z_c @ centered_weight.T + centered_bias
        if np.any(singleton_mask):
            logits_centered[singleton_mask] = logits_plain[singleton_mask]
    elif groups is None and len(Z) >= 18:
        Z_c = Z - 0.75 * Z.mean(axis=0, keepdims=True)
        logits_centered = Z_c @ centered_weight.T + centered_bias
    else:
        logits_centered = logits_plain.copy()

    row_mean_p = logits_plain.mean(axis=1, keepdims=True)
    row_std_p = np.clip(logits_plain.std(axis=1, keepdims=True), 1e-4, None)
    norm_plain = (logits_plain - row_mean_p) / row_std_p

    row_mean_c = logits_centered.mean(axis=1, keepdims=True)
    row_std_c = np.clip(logits_centered.std(axis=1, keepdims=True), 1e-4, None)
    norm_centered = (logits_centered - row_mean_c) / row_std_c

    logits_blend = (1.0 - blend_centered) * norm_plain + blend_centered * norm_centered
    probs_plain_sinkhorn = apply_prior_and_sinkhorn(logits_plain)
    probs_sinkhorn = apply_prior_and_sinkhorn(logits_blend)

    return {
        "logits_plain": logits_plain,
        "logits_centered": logits_centered,
        "logits_blend": logits_blend,
        "probs_plain_sinkhorn": probs_plain_sinkhorn,
        "probs_sinkhorn": probs_sinkhorn,
    }


def _blend_head(head_linear: nn.Linear, W_r: np.ndarray, b_r: np.ndarray, w_neural: float = 0.65, w_ridge: float = 0.35) -> None:
    """Blends neural head with closed-form ridge head after matching Frobenius norm."""
    with torch.no_grad():
        W_n = head_linear.weight.detach().cpu().float().numpy()
        b_n = head_linear.bias.detach().cpu().float().numpy()
        norm_n = float(np.linalg.norm(W_n))
        norm_r = float(np.linalg.norm(W_r))
        scale = (norm_n / norm_r) if (norm_r > 1e-6 and norm_n > 1e-6) else 1.0
        W_r_scaled = W_r * scale
        b_r_centered = (b_r - b_r.mean()) * scale + b_n.mean()
        W_blend = w_neural * W_n + w_ridge * W_r_scaled
        b_blend = w_neural * b_n + w_ridge * b_r_centered
        head_linear.weight.copy_(torch.from_numpy(W_blend).to(head_linear.weight.device, dtype=head_linear.weight.dtype))
        head_linear.bias.copy_(torch.from_numpy(b_blend).to(head_linear.bias.device, dtype=head_linear.bias.dtype))


def evaluate_stage2_oof_cv(
    Z: np.ndarray,
    y: np.ndarray,
    prompts: np.ndarray,
    n_splits: int = 5,
    alpha: float = 10.0,
    label_smoothing: float = 0.10,
    num_classes: int = K,
    class_labels: list[str] | None = None,
    output_dir: Path | None = None,
) -> dict[str, Any]:
    """Executes GroupKFold CV on Stage 2 representations grouped by prompt_id to measure OOF BAcc & overfitting."""
    u_prompts = np.unique(prompts)
    actual_splits = min(n_splits, len(u_prompts))
    if actual_splits < 2:
        print(f"[Stage 2 CV] Insufficient prompt groups ({len(u_prompts)}) for cross-validation; skipping.")
        return {}

    labels = class_labels or list(LABELS[:num_classes])
    gkf = GroupKFold(n_splits=actual_splits)

    oof_plain_lp = np.zeros((len(y), num_classes), dtype=np.float32)
    oof_centered_lp = np.zeros((len(y), num_classes), dtype=np.float32)
    oof_blend_lp = np.zeros((len(y), num_classes), dtype=np.float32)
    fold_results = []

    print("\n==========================================================================================")
    print(f" STAGE 2: {actual_splits}-STAGE GROUP-KFOLD CV & OVERFITTING AUDIT (grouped by prompt_id)")
    print("==========================================================================================")
    print(" Fold   | Train Prompts | Val Prompts | Train BAcc | Val Plain  | Val Center | Val Sink   | Overfit Gap")
    print("------------------------------------------------------------------------------------------")

    for fold_idx, (tr_idx, val_idx) in enumerate(gkf.split(Z, y, groups=prompts)):
        Z_tr, y_tr, p_tr = Z[tr_idx], y[tr_idx], prompts[tr_idx]
        Z_val, y_val, p_val = Z[val_idx], y[val_idx], prompts[val_idx]

        W_p, b_p, W_c, b_c = fit_ridge_heads(Z_tr, y_tr, p_tr, alpha=alpha, label_smoothing=label_smoothing, num_classes=num_classes)

        tr_plain = Z_tr @ W_p.T + b_p
        Z_tr_c = Z_tr.copy()
        for p in np.unique(p_tr):
            m = p_tr == p
            if m.sum() > 1:
                Z_tr_c[m] -= Z_tr_c[m].mean(axis=0, keepdims=True)
            else:
                Z_tr_c[m] = 0.0
        tr_centered = Z_tr_c @ W_c.T + b_c
        tr_blend = 0.5 * tr_plain + 0.5 * tr_centered
        train_bacc = float(balanced_accuracy_score(y_tr, tr_blend.argmax(axis=1)))

        val_plain = Z_val @ W_p.T + b_p
        Z_val_c = Z_val.copy()
        singleton_mask = np.zeros(len(p_val), dtype=bool)
        for p in np.unique(p_val):
            m = p_val == p
            if m.sum() > 1:
                Z_val_c[m] -= Z_val_c[m].mean(axis=0, keepdims=True)
            else:
                Z_val_c[m] = 0.0
                singleton_mask[m] = True
        val_centered = Z_val_c @ W_c.T + b_c
        if np.any(singleton_mask):
            val_centered[singleton_mask] = val_plain[singleton_mask]
        val_blend = 0.5 * val_plain + 0.5 * val_centered

        val_plain_bacc = float(balanced_accuracy_score(y_val, val_plain.argmax(axis=1)))
        val_center_bacc = float(balanced_accuracy_score(y_val, val_centered.argmax(axis=1)))
        val_blend_bacc = float(balanced_accuracy_score(y_val, val_blend.argmax(axis=1)))

        val_std = float(np.std(val_blend, axis=1).mean())
        val_scale = (4.0 / val_std) if val_std > 1e-4 else 1.0
        P_val = softmax(val_blend * val_scale, axis=1)
        P_val_sk = sinkhorn_balancing(P_val, iters=60, sinkhorn_strength=0.65, target_mix=0.35)
        val_sink_bacc = float(balanced_accuracy_score(y_val, P_val_sk.argmax(axis=1)))

        gap = train_bacc - val_sink_bacc

        oof_plain_lp[val_idx] = val_plain
        oof_centered_lp[val_idx] = val_centered
        oof_blend_lp[val_idx] = val_blend

        fold_res = {
            "fold": fold_idx + 1,
            "train_prompts": int(len(np.unique(p_tr))),
            "val_prompts": int(len(np.unique(p_val))),
            "train_bacc": train_bacc,
            "val_plain_bacc": val_plain_bacc,
            "val_center_bacc": val_center_bacc,
            "val_blend_bacc": val_blend_bacc,
            "val_sink_bacc": val_sink_bacc,
            "overfit_gap": gap,
        }
        fold_results.append(fold_res)

        print(
            f" Fold {fold_idx + 1:<2} | {len(np.unique(p_tr)):<13} | {len(np.unique(p_val)):<11} | "
            f"{train_bacc * 100:5.2f}%    | {val_plain_bacc * 100:5.2f}%    | "
            f"{val_center_bacc * 100:5.2f}%    | {val_sink_bacc * 100:5.2f}%   | {gap * 100:+5.2f}%"
        )

    oof_plain_bacc = float(balanced_accuracy_score(y, oof_plain_lp.argmax(axis=1)))
    oof_center_bacc = float(balanced_accuracy_score(y, oof_centered_lp.argmax(axis=1)))
    oof_blend_bacc = float(balanced_accuracy_score(y, oof_blend_lp.argmax(axis=1)))

    oof_std = float(np.std(oof_blend_lp, axis=1).mean())
    oof_scale = (4.0 / oof_std) if oof_std > 1e-4 else 1.0
    P_oof = softmax(oof_blend_lp * oof_scale, axis=1)
    P_oof_sk = sinkhorn_balancing(P_oof, iters=60, sinkhorn_strength=0.65, target_mix=0.35)
    oof_sink_bacc = float(balanced_accuracy_score(y, P_oof_sk.argmax(axis=1)))

    mean_tr_bacc = float(np.mean([f["train_bacc"] for f in fold_results]))
    total_overfit_gap = mean_tr_bacc - oof_sink_bacc

    if total_overfit_gap <= 0.08:
        verdict = "LOW (Excellent Generalization across Unseen Prompts)"
    elif total_overfit_gap <= 0.18:
        verdict = "MODERATE (Normal Stylistic Domain Variance)"
    else:
        verdict = "HIGH (Prompt Memorization Risk - Increase supcon/family margin)"

    print("------------------------------------------------------------------------------------------")
    print(
        f" OOF   | {len(u_prompts):<13} | {len(u_prompts):<11} | "
        f"{mean_tr_bacc * 100:5.2f}%    | {oof_plain_bacc * 100:5.2f}%    | "
        f"{oof_center_bacc * 100:5.2f}%    | {oof_sink_bacc * 100:5.2f}%   | {total_overfit_gap * 100:+5.2f}%"
    )
    print("==========================================================================================")
    print(f" Overfitting Assessment: {verdict}")
    print("==========================================================================================\n")

    report = {
        "n_splits": actual_splits,
        "n_samples": int(len(y)),
        "n_prompts": int(len(u_prompts)),
        "mean_train_bacc": mean_tr_bacc,
        "oof_plain_bacc": oof_plain_bacc,
        "oof_center_bacc": oof_center_bacc,
        "oof_blend_bacc": oof_blend_bacc,
        "oof_sink_bacc": oof_sink_bacc,
        "overfit_gap": total_overfit_gap,
        "verdict": verdict,
        "folds": fold_results,
    }

    if output_dir is not None:
        out_p = Path(output_dir)
        out_p.mkdir(parents=True, exist_ok=True)
        rep_file = out_p / "stage2_cv_report.json"
        rep_file.write_text(json.dumps(report, indent=2), encoding="utf-8")
        print(f"[Stage 2 CV] Saved cross-validation and overfitting report to: {rep_file}")

        oof_npz = out_p / "stage2_oof_predictions.npz"
        np.savez_compressed(
            str(oof_npz),
            oof_plain_lp=oof_plain_lp,
            oof_centered_lp=oof_centered_lp,
            oof_blend_lp=oof_blend_lp,
            oof_sink_probs=P_oof_sk,
            y_true=y,
            prompts=prompts,
            classes=np.array(labels),
        )
        print(f"[Stage 2 CV] Saved out-of-fold predictions to: {oof_npz}")

    return report


def compute_detailed_metrics(logits: np.ndarray, y_true: np.ndarray) -> dict[str, Any]:
    """Computes BAcc, NLL, Gemini 4-class metrics, sister pair accuracy, and per-class recall/precision."""
    preds = logits.argmax(axis=1)
    bacc = float(balanced_accuracy_score(y_true, preds))
    shifted = logits - np.max(logits, axis=1, keepdims=True)
    probs = np.exp(shifted) / np.sum(np.exp(shifted), axis=1, keepdims=True)
    nll = float(log_loss(y_true, probs, labels=list(range(K))))

    gem_idx = FAMILIES["gemini"]
    gem_mask = np.isin(y_true, gem_idx)
    gem_sub_preds = logits[gem_mask][:, gem_idx].argmax(axis=1)
    gem_y = np.array([gem_idx.index(int(v)) for v in y_true[gem_mask]])
    gem_within_bacc = float(balanced_accuracy_score(gem_y, gem_sub_preds))
    gem_ret = float(np.mean(np.isin(preds[gem_mask], gem_idx)))
    gem_cm = confusion_matrix(gem_y, gem_sub_preds, labels=list(range(len(gem_idx))))

    pair_mask = (y_true == 7) | (y_true == 8)
    pair_preds = np.where(logits[pair_mask, 7] >= logits[pair_mask, 8], 7, 8)
    sister_acc = float(balanced_accuracy_score(y_true[pair_mask], pair_preds)) if pair_mask.any() else 0.0

    cm = confusion_matrix(y_true, preds, labels=list(range(K)))
    per_class = []
    zero_recall = []
    for i, name in enumerate(LABELS):
        true_n = int(cm[i, :].sum())
        pred_n = int(cm[:, i].sum())
        correct_n = int(cm[i, i])
        if correct_n == 0:
            zero_recall.append(name)
        per_class.append({
            "idx": i,
            "model": name,
            "true": true_n,
            "preds": pred_n,
            "correct": correct_n,
            "recall": float(correct_n / max(true_n, 1)),
            "precision": float(correct_n / max(pred_n, 1)) if pred_n > 0 else 0.0,
            "bias": float(pred_n / max(true_n, 1)),
        })

    return {
        "bacc": bacc,
        "nll": nll,
        "gem_within_bacc": gem_within_bacc,
        "gem_retention": gem_ret,
        "sister_37_38_acc": sister_acc,
        "zero_recall_classes": zero_recall,
        "per_class": per_class,
        "gemini_4x4": gem_cm.tolist(),
        "confusion_matrix": cm.tolist(),
    }


def print_prediction_table(metrics: dict[str, Any], header: str) -> None:
    """Prints a compact 18-class diagnostic breakdown table."""
    print(f"\n=== {header} ===")
    print(
        f"BAcc={metrics['bacc']*100:.2f}% | NLL={metrics['nll']:.4f} | "
        f"Gem4={metrics['gem_within_bacc']*100:.2f}% | GemRet={metrics['gem_retention']*100:.2f}% | "
        f"3.7v3.8={metrics['sister_37_38_acc']*100:.2f}% | ZeroRec={len(metrics['zero_recall_classes'])}"
    )
    print("  # | CLASS NAME               | PREDS | CORRECT | RECALL% |  PREC% |  BIAS")
    print("----+--------------------------+-------+---------+---------+--------+------")
    for row in metrics["per_class"]:
        print(
            f" {row['idx']:2d} | {row['model']:<24} | {row['preds']:5d} | {row['correct']:7d} | "
            f"{row['recall']*100:6.1f}% | {row['precision']*100:5.1f}% | {row['bias']:4.2f}x"
        )


@torch.no_grad()
def extract_features_and_logits(
    model: MultiLayerLoRAClassifier,
    texts: list[str],
    tok: Any,
    max_len: int,
    device: torch.device,
    batch_size: int = 64,
    drop_first_token: bool = True,
    compute_dtype: torch.dtype = torch.float16,
    end_id: int | None = None,
    cls_id: int | None = None,
) -> tuple[np.ndarray, np.ndarray]:
    """Length-sorted evaluation using TextSequenceDataset head+tail truncation returning float32 Z and plain logits."""
    model.eval()
    n = len(texts)
    order = np.argsort([len(str(t)) for t in texts], kind="stable")
    feats = np.zeros((n, model.total_dim), dtype=np.float32)
    logits_out = np.zeros((n, model.num_classes), dtype=np.float32)
    tok_spec = getattr(model, "tok_spec", None)
    raw_cls_id = cls_id if cls_id is not None else (tok_spec.cls_id if tok_spec is not None else getattr(tok, "cls_token_id", None))
    eff_cls_id = int(raw_cls_id) if raw_cls_id is not None else None
    if end_id is not None:
        raw_end_id = end_id
    elif tok_spec is not None:
        raw_end_id = tok_spec.end_id
    else:
        tok_sep = getattr(tok, "sep_token_id", None)
        raw_end_id = tok_sep if tok_sep is not None else getattr(tok, "eos_token_id", None)
    eff_end_id = int(raw_end_id) if raw_end_id is not None else None
    if tok_spec is not None:
        pad_id = tok_spec.pad_id
    else:
        tok_pad = getattr(tok, "pad_token_id", None)
        pad_id = int(tok_pad) if tok_pad is not None else 0

    for s in range(0, n, batch_size):
        idx = order[s : s + batch_size]
        chunk = [str(texts[i]) for i in idx]
        ds_chunk = TextSequenceDataset(
            chunk,
            None,
            None,
            tok,
            max_len=max_len,
            end_id=eff_end_id,
            cls_id=eff_cls_id,
        )
        batch_items = [ds_chunk[k] for k in range(len(chunk))]
        enc = collate_text_batch(batch_items, pad_token_id=pad_id)
        mb_ids = enc["input_ids"].to(device)
        mb_mask = enc["attention_mask"].to(device)
        max_l = max(int(mb_mask.sum(dim=1).max().item()), 2)
        with torch.amp.autocast("cuda", enabled=torch.cuda.is_available(), dtype=compute_dtype):
            Z = model.forward_features(mb_ids[:, :max_l], mb_mask[:, :max_l], drop_first_token=drop_first_token)
            lg = model.predict_plain(Z)
        feats[idx] = Z.float().cpu().numpy().astype(np.float32)
        logits_out[idx] = lg.float().cpu().numpy().astype(np.float32)

    return feats, logits_out


def _evaluate_bench_checkpoint(
    model: MultiLayerLoRAClassifier,
    tok: Any,
    texts: list[str],
    y_true: np.ndarray,
    max_len: int,
    device: torch.device,
    compute_dtype: torch.dtype,
    drop_first_token: bool,
    tag: str,
    end_id: int | None = 151643,
    cls_id: int | None = None,
    pad_id: int = 151643,
    batch_size: int = 8,
    blend_centered: float = 0.0,
    out_dir: Path | None = None,
    history: list[dict[str, Any]] | None = None,
    eval_extras_out: dict[str, Any] | None = None,
) -> tuple[dict[str, Any], np.ndarray]:
    """Evaluates Bench-1800 (or subset) with train/inference tokenization parity and logs problem-class metrics."""
    ds_eval = TextSequenceDataset(texts, None, None, tok, max_len=max_len, end_id=end_id, cls_id=cls_id)
    collate_eval = functools.partial(collate_text_batch, pad_token_id=pad_id)
    loader_eval = DataLoader(ds_eval, batch_size=batch_size, shuffle=False, collate_fn=collate_eval)

    model.eval()
    z_parts = []
    logits_p_parts = []
    with torch.no_grad():
        with torch.amp.autocast("cuda", enabled=torch.cuda.is_available(), dtype=compute_dtype):
            for batch in loader_eval:
                b_ids = batch["input_ids"].to(device)
                b_mask = batch["attention_mask"].to(device)
                max_l = max(int(b_mask.sum(dim=1).max().item()), 2)
                z_b = model.forward_features(b_ids[:, :max_l], b_mask[:, :max_l], drop_first_token=drop_first_token)
                lg_b = model.predict_plain(z_b)
                z_parts.append(z_b.float().cpu().numpy())
                logits_p_parts.append(lg_b.float().cpu().numpy())

    Z_eval = np.vstack(z_parts)
    logits = np.vstack(logits_p_parts)
    if blend_centered > 0.0 or eval_extras_out is not None:
        w_p = model.plain_head.weight.detach().float().cpu().numpy()
        b_p = model.plain_head.bias.detach().float().cpu().numpy()
        w_c = model.centered_head.weight.detach().float().cpu().numpy()
        b_c = model.centered_head.bias.detach().float().cpu().numpy()
        eff_bc = blend_centered if blend_centered > 0.0 else 0.20
        res_grp = compute_grouped_inference_logits(Z_eval, w_p, b_p, w_c, b_c, groups=None, blend_centered=eff_bc)
        if eval_extras_out is not None:
            eval_extras_out["plain_sinkhorn"] = compute_detailed_metrics(np.log(np.clip(res_grp["probs_plain_sinkhorn"], 1e-8, 1.0)), y_true)
            eval_extras_out["blend_global_approx"] = compute_detailed_metrics(res_grp["logits_blend"], y_true)
            eval_extras_out["blend_global_approx_sinkhorn"] = compute_detailed_metrics(np.log(np.clip(res_grp["probs_sinkhorn"], 1e-8, 1.0)), y_true)
        if blend_centered > 0.0:
            logits = res_grp["logits_blend"]

    metrics = compute_detailed_metrics(logits, y_true)
    pc_map = {r["model"]: r for r in metrics["per_class"]}
    print(
        f"[Milestone {tag}] N={len(y_true)} | BAcc={metrics['bacc']*100:.2f}% | "
        f"NLL={metrics['nll']:.4f} | Gem4={metrics['gem_within_bacc']*100:.2f}% | "
        f"3.7v3.8={metrics['sister_37_38_acc']*100:.2f}%"
    )
    watch_parts = []
    for name in PROBLEM_WATCH_CLASSES:
        if name in pc_map:
            r = pc_map[name]
            f1 = (2.0 * r["precision"] * r["recall"] / max(r["precision"] + r["recall"], 1e-8)) if (r["precision"] + r["recall"]) > 0 else 0.0
            r["f1"] = float(f1)
            watch_parts.append(f"{name}:R{r['recall']*100:.0f}/P{r['precision']*100:.0f}/F{f1*100:.0f}({r['preds']})")
    if watch_parts:
        print("  [Watch Classes] " + " | ".join(watch_parts))

    entry = {
        "tag": tag,
        "n_samples": int(len(y_true)),
        "bacc": metrics["bacc"],
        "nll": metrics["nll"],
        "gem_within_bacc": metrics["gem_within_bacc"],
        "gem_retention": metrics["gem_retention"],
        "sister_37_38_acc": metrics["sister_37_38_acc"],
        "watch": {k: pc_map[k] for k in PROBLEM_WATCH_CLASSES if k in pc_map},
    }
    if history is not None:
        history.append(entry)
    if out_dir is not None:
        try:
            out_dir.mkdir(parents=True, exist_ok=True)
            if history is not None:
                (out_dir / "milestone_history.json").write_text(json.dumps(history, indent=2), encoding="utf-8")
        except Exception as e:
            print(f"[Trainer] Warning saving milestone history: {e}")
    model.train()
    return metrics, logits


def solve_trace_ridge_wb(
    Z_tr_np: np.ndarray,
    y_tr_np: np.ndarray,
    alphas: list[float],
    device: torch.device,
    label_smoothing: float = 0.05,
) -> dict[float, tuple[np.ndarray, np.ndarray]]:
    """Canonical trace-scaled float64 closed-form Ridge solver (primal when N>=D else dual Gram)."""
    n_tr, dim = Z_tr_np.shape
    y_oh = np.zeros((n_tr, K), dtype=np.float32)
    y_oh[np.arange(n_tr), y_tr_np] = 1.0
    if label_smoothing > 0.0:
        y_oh = (1.0 - label_smoothing) * y_oh + (label_smoothing / float(K))

    X_tr = torch.from_numpy(Z_tr_np).to(device, dtype=torch.float32)
    Y_t = torch.from_numpy(y_oh).to(device, dtype=torch.float32)
    mu_x = X_tr.mean(dim=0, keepdim=True)
    mu_y = Y_t.mean(dim=0, keepdim=True)
    X_c = X_tr - mu_x
    Y_c = Y_t - mu_y
    trace_xx = float((X_c * X_c).sum().item())
    mean_eig = max(trace_xx / float(min(n_tr, dim)), 1e-8)

    out: dict[float, tuple[np.ndarray, np.ndarray]] = {}
    if n_tr >= dim:
        X64, Y64 = X_c.double(), Y_c.double()
        xtx = X64.T @ X64
        xty = X64.T @ Y64
        evals, vecs = torch.linalg.eigh(xtx)
        evals = torch.clamp(evals, min=0.0)
        v_xty = vecs.T @ xty
        mu_x_f = mu_x.double()
        mu_y_f = mu_y.double()
        for a in alphas:
            inv_e = 1.0 / (evals + float(a) * mean_eig)
            W_t = vecs @ (inv_e.unsqueeze(1) * v_xty)
            b_t = (mu_y_f - mu_x_f @ W_t).squeeze(0)
            out[a] = (W_t.T.float().cpu().numpy().astype(np.float32), b_t.float().cpu().numpy().astype(np.float32))
    else:
        X64, Y64 = X_c.double(), Y_c.double()
        K_mat = X64 @ X64.T
        evals, vecs = torch.linalg.eigh(K_mat)
        evals = torch.clamp(evals, min=0.0)
        vt_y = vecs.T @ Y64
        x_t = X64.T
        mu_x_f = mu_x.double()
        mu_y_f = mu_y.double()
        for a in alphas:
            inv_k = 1.0 / (evals + float(a) * mean_eig)
            dual_w = vecs @ (inv_k.unsqueeze(1) * vt_y)
            W_t = x_t @ dual_w
            b_t = (mu_y_f - mu_x_f @ W_t).squeeze(0)
            out[a] = (W_t.T.float().cpu().numpy().astype(np.float32), b_t.float().cpu().numpy().astype(np.float32))
    return out


def gpu_trace_ridge_eval(
    Z_tr_np: np.ndarray,
    y_tr_np: np.ndarray,
    Z_te_np: np.ndarray,
    alphas: list[float],
    device: torch.device,
    label_smoothing: float = 0.05,
) -> dict[float, np.ndarray]:
    """Evaluates closed-form trace-scaled Ridge linear weights on Z_te_np across candidate alphas."""
    wb_map = solve_trace_ridge_wb(Z_tr_np, y_tr_np, alphas=alphas, device=device, label_smoothing=label_smoothing)
    X_te = torch.from_numpy(Z_te_np).to(device, dtype=torch.float32)
    return {a: (X_te @ torch.from_numpy(w).to(device).T + torch.from_numpy(b).to(device)).cpu().numpy() for a, (w, b) in wb_map.items()}


In [ ]:
# === MODULE: src/namethatllm/training/embeddings.py ===
"""
Feature and Layer-by-Layer Embedding Extraction and Kaggle Dataset Bundle Export.
"""

from __future__ import annotations

import functools
import json
import math
import shutil
import time
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import torch
from safetensors.torch import save_file
from torch.utils.data import DataLoader



def save_extracted_features_bundle(
    output_dir: Path | str,
    target: str,
    features_concat: np.ndarray,
    features_by_layer: np.ndarray | None,
    texts: list[str],
    labels: list[int] | np.ndarray,
    model_names: list[str],
    prompt_ids: list[Any] | None = None,
    eval_features_concat: np.ndarray | None = None,
    eval_features_by_layer: np.ndarray | None = None,
    eval_texts: list[str] | None = None,
    eval_labels: list[int] | None = None,
    eval_model_names: list[str] | None = None,
    dataset_title: str | None = None,
    dataset_slug: str | None = None,
    checkpoints_volume: Any | None = None,
    champion_logits_plain: np.ndarray | None = None,
    champion_logits_centered: np.ndarray | None = None,
    head_tail_cos: np.ndarray | None = None,
    token_std_norms: np.ndarray | None = None,
    layer_norms: np.ndarray | None = None,
    eval_champion_logits_plain: np.ndarray | None = None,
    eval_champion_logits_centered: np.ndarray | None = None,
    eval_head_tail_cos: np.ndarray | None = None,
    eval_token_std_norms: np.ndarray | None = None,
    eval_layer_norms: np.ndarray | None = None,
) -> Path:
    """Saves extracted features, structural multiview tensors, and metadata in Kaggle Dataset format."""
    out_p = Path(output_dir)
    out_p.mkdir(parents=True, exist_ok=True)

    print(f"\n[Features Export] Saving features bundle for target '{target}' to: {out_p}...")

    feat_cat_t = torch.from_numpy(features_concat).to(torch.float16)
    save_file({"features_concat": feat_cat_t}, str(out_p / "features_concat.safetensors"))
    sz_cat_mb = (out_p / "features_concat.safetensors").stat().st_size / (1024**2)
    print(f"  Saved features_concat.safetensors: shape={list(feat_cat_t.shape)}, size={sz_cat_mb:.1f} MB")

    sz_layer_mb = 0.0
    if features_by_layer is not None:
        feat_layers_t = torch.from_numpy(features_by_layer).to(torch.float16)
        save_file({"features_by_layer": feat_layers_t}, str(out_p / "features_by_layer.safetensors"))
        sz_layer_mb = (out_p / "features_by_layer.safetensors").stat().st_size / (1024**2)
        print(f"  Saved features_by_layer.safetensors: shape={list(feat_layers_t.shape)}, size={sz_layer_mb:.1f} MB")

    champ_dict = {}
    if champion_logits_plain is not None:
        champ_dict["logits_plain"] = torch.from_numpy(champion_logits_plain).to(torch.float32)
    if champion_logits_centered is not None:
        champ_dict["logits_centered"] = torch.from_numpy(champion_logits_centered).to(torch.float32)
    if head_tail_cos is not None:
        champ_dict["head_tail_cos"] = torch.from_numpy(head_tail_cos).to(torch.float32)
    if token_std_norms is not None:
        champ_dict["token_std_norms"] = torch.from_numpy(token_std_norms).to(torch.float32)
    if champ_dict:
        save_file(champ_dict, str(out_p / "champion_logits.safetensors"))
        print(f"  Saved champion_logits.safetensors: keys={list(champ_dict.keys())}")

    if layer_norms is not None:
        save_file({"layer_norms": torch.from_numpy(layer_norms).to(torch.float32)}, str(out_p / "layer_norms.safetensors"))

    meta_df = pd.DataFrame({
        "id": np.arange(len(texts), dtype=np.int64),
        "model": model_names,
        "label": np.asarray(labels, dtype=np.int64),
        "prompt_id": prompt_ids if prompt_ids is not None else [-1] * len(texts),
        "text_hash": [_text_hash(t) for t in texts],
        "char_len": [len(t) for t in texts],
    })
    meta_df.to_parquet(out_p / "metadata.parquet", index=False)
    print(f"  Saved metadata.parquet: {len(meta_df):,} rows")

    if eval_features_concat is not None and eval_texts is not None:
        ev_cat_t = torch.from_numpy(eval_features_concat).to(torch.float16)
        save_file({"benchmark_features_concat": ev_cat_t}, str(out_p / "benchmark_features_concat.safetensors"))
        if eval_features_by_layer is not None:
            ev_layer_t = torch.from_numpy(eval_features_by_layer).to(torch.float16)
            save_file({"benchmark_features_by_layer": ev_layer_t}, str(out_p / "benchmark_features_by_layer.safetensors"))

        eval_champ_dict = {}
        if eval_champion_logits_plain is not None:
            eval_champ_dict["logits_plain"] = torch.from_numpy(eval_champion_logits_plain).to(torch.float32)
        if eval_champion_logits_centered is not None:
            eval_champ_dict["logits_centered"] = torch.from_numpy(eval_champion_logits_centered).to(torch.float32)
        if eval_head_tail_cos is not None:
            eval_champ_dict["head_tail_cos"] = torch.from_numpy(eval_head_tail_cos).to(torch.float32)
        if eval_token_std_norms is not None:
            eval_champ_dict["token_std_norms"] = torch.from_numpy(eval_token_std_norms).to(torch.float32)
        if eval_champ_dict:
            save_file(eval_champ_dict, str(out_p / "benchmark_champion_logits.safetensors"))
            print(f"  Saved benchmark_champion_logits.safetensors: keys={list(eval_champ_dict.keys())}")

        if eval_layer_norms is not None:
            save_file({"layer_norms": torch.from_numpy(eval_layer_norms).to(torch.float32)}, str(out_p / "benchmark_layer_norms.safetensors"))

        eval_models = eval_model_names if eval_model_names is not None else [
            LABELS[i] if i < len(LABELS) else f"class_{i}" for i in (eval_labels if eval_labels is not None else [0] * len(eval_texts))
        ]
        bench_df = pd.DataFrame({
            "id": np.arange(len(eval_texts), dtype=np.int64),
            "model": eval_models,
            "label": np.asarray(eval_labels, dtype=np.int64) if eval_labels is not None else np.zeros(len(eval_texts), dtype=np.int64),
            "text_hash": [_text_hash(t) for t in eval_texts],
            "char_len": [len(t) for t in eval_texts],
        })
        bench_df.to_parquet(out_p / "benchmark_metadata.parquet", index=False)
        print(f"  Saved benchmark features & metadata: {len(bench_df):,} samples")

    slug = dataset_slug or f"ntl-{target.lower()}-features"
    title = dataset_title or f"NTL {target.upper()} Extracted Features"
    kaggle_meta = {
        "title": title,
        "id": f"anilmittal/{slug}",
        "licenses": [{"name": "CC0-1.0"}],
        "description": (
            f"Pre-extracted high-dimensional feature representations for {target.upper()} models "
            f"using Multi-Layer backbone representations."
        ),
    }
    (out_p / "dataset-metadata.json").write_text(json.dumps(kaggle_meta, indent=2), encoding="utf-8")

    summary = {
        "target": target,
        "num_train_samples": len(texts),
        "num_classes": len(set(model_names)),
        "class_counts": {k: int(v) for k, v in pd.Series(model_names).value_counts().items()},
        "feature_dim_concat": int(features_concat.shape[1]),
        "features_by_layer_shape": list(features_by_layer.shape) if features_by_layer is not None else None,
        "has_benchmark": eval_features_concat is not None,
        "num_benchmark_samples": len(eval_texts) if eval_texts is not None else 0,
        "total_bundle_size_mb": round(sz_cat_mb + sz_layer_mb, 2),
        "created_at": time.strftime("%Y-%m-%d %H:%M:%S UTC", time.gmtime()),
    }
    (out_p / "features_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")

    if checkpoints_volume is not None and hasattr(checkpoints_volume, "commit"):
        try:
            checkpoints_volume.commit()
        except Exception:
            pass

    print(f"[Features Export] Complete! Bundle ready for Kaggle Dataset creation at: {out_p} ({summary['total_bundle_size_mb']:.1f} MB)\n")
    return out_p


def extract_and_export_embeddings(
    model: MultiLayerLoRAClassifier,
    tok: Any,
    df_pool: pd.DataFrame,
    df_train: pd.DataFrame | None = None,
    target: str = "gemini",
    save_all_layers: bool = False,
    save_multiview: bool = False,
    output_dir: Path | str = "export_embeddings",
    samples_per_class: int = 0,
    negative_ratio: float = 0.10,
    eval_texts: list[str] | None = None,
    eval_labels: list[int] | None = None,
    eval_model_names: list[str] | None = None,
    batch_size: int = 16,
    max_len: int | None = None,
    end_id: int | None = None,
    device: torch.device | None = None,
    dataset_title: str | None = None,
    dataset_slug: str | None = None,
    checkpoints_volume: Any | None = None,
    eval_champion_logits_plain: np.ndarray | None = None,
) -> Path:
    """Extracts multi-layer embeddings batch-by-batch across a model and writes a Kaggle dataset bundle."""
    if device is None:
        device = next(model.parameters()).device

    fam_hint = getattr(model, "backbone_family", "unknown")
    adapter_cfg: dict[str, Any] = {}
    if fam_hint and fam_hint != "unknown":
        adapter_cfg["backbone_family"] = fam_hint
    else:
        bb_obj = getattr(model, "backbone", model)
        adapter_cfg["backbone"] = f"{type(bb_obj)} {getattr(bb_obj, 'config', '')}"
    adapter = resolve_backbone_adapter(adapter_cfg)

    tok_spec = getattr(model, "tok_spec", None)
    if tok_spec is None:
        tok_spec = adapter.tok_spec_from_tokenizer(tok, max_len=max_len, end_id=end_id)

    eff_max_len = int(max_len if max_len is not None else tok_spec.max_len)
    batch_size = adapter.clamp_extraction_batch_size(batch_size)

    target_clean = target.lower().strip()
    print("\n==========================================================================================")
    print(f" [Features Extraction] Target: '{target_clean}' | Save All Layers: {save_all_layers} | Multiview: {save_multiview} | Max Len: {eff_max_len}")
    print("==========================================================================================")

    if eval_texts is not None and len(eval_texts) > 0:
        bench_hashes = {_text_hash(t) for t in eval_texts}
        if df_pool is not None and len(df_pool) > 0:
            init_p = len(df_pool)
            pool_hashes = [_text_hash(t) for t in df_pool["text"].fillna("")]
            df_pool = df_pool[[h not in bench_hashes for h in pool_hashes]].copy()
            quar_p = init_p - len(df_pool)
            if quar_p > 0:
                print(f"[Features Extraction] Quarantined {quar_p} pool samples matching benchmark.")
        if df_train is not None and len(df_train) > 0:
            init_t = len(df_train)
            tr_hashes = [_text_hash(t) for t in df_train["text"].fillna("")]
            df_train = df_train[[h not in bench_hashes for h in tr_hashes]].copy()
            quar_t = init_t - len(df_train)
            if quar_t > 0:
                print(f"[Features Extraction] Quarantined {quar_t} gold train samples matching benchmark.")

    if target_clean in ("gold", "train"):
        assert df_train is not None and len(df_train) > 0, "No gold df_train provided for target='gold'"
        df_target_all = df_train.copy()
        df_neg_all = pd.DataFrame()
    elif target_clean == "all":
        all_dfs = []
        for c in LABELS:
            sub_tr = df_train[df_train["model"] == c] if (df_train is not None and len(df_train) > 0) else pd.DataFrame()
            sub_pool = df_pool[df_pool["model"] == c] if (df_pool is not None and len(df_pool) > 0) else pd.DataFrame()
            if samples_per_class > 0:
                n_take = min(samples_per_class - len(sub_tr), len(sub_pool))
                sampled_pool = sub_pool.sample(n=max(n_take, 0), random_state=42) if n_take > 0 else pd.DataFrame()
            else:
                sampled_pool = sub_pool
            c_df = pd.concat([sub_tr, sampled_pool], ignore_index=True)
            all_dfs.append(c_df)
            print(f"  Class {c}: {len(c_df)} samples ({len(sub_tr)} gold, {len(c_df)-len(sub_tr)} pool)")
        df_target_all = pd.concat(all_dfs, ignore_index=True)
        df_neg_all = pd.DataFrame()
    elif target_clean in FAMILIES:
        target_cols = FAMILIES[target_clean]
        target_classes = [LABELS[i] for i in target_cols]
        cap = samples_per_class if samples_per_class > 0 else 3800
        fam_dfs = []
        for c in target_classes:
            sub_tr = df_train[df_train["model"] == c] if (df_train is not None and len(df_train) > 0) else pd.DataFrame()
            sub_pool = df_pool[df_pool["model"] == c] if (df_pool is not None and len(df_pool) > 0) else pd.DataFrame()
            n_take = min(cap - len(sub_tr), len(sub_pool))
            sampled_pool = sub_pool.sample(n=max(n_take, 0), random_state=42) if n_take > 0 else pd.DataFrame()
            c_df = pd.concat([sub_tr, sampled_pool], ignore_index=True)
            fam_dfs.append(c_df)
            print(f"  {target_clean.upper()} class {c}: {len(c_df)} samples ({len(sub_tr)} gold, {len(c_df)-len(sub_tr)} pool)")
        df_target_all = pd.concat(fam_dfs, ignore_index=True)

        if negative_ratio > 0.0 and df_pool is not None and len(df_pool) > 0:
            n_neg = int(len(df_target_all) * negative_ratio)
            df_non_fam = df_pool[~df_pool["model"].isin(target_classes)].copy()
            other_fams = sorted({"".join(ch for ch in m.split("-")[0].split(".")[0] if ch.isalpha()) for m in LABELS if m not in target_classes})
            neg_dfs = []
            n_per_fam = max(n_neg // max(len(other_fams), 1), 1)
            for fam in other_fams:
                fam_sub = df_non_fam[df_non_fam["model"].str.startswith(fam)]
                if len(fam_sub) > 0:
                    neg_dfs.append(fam_sub.sample(n=min(n_per_fam, len(fam_sub)), random_state=42))
            df_neg_all = pd.concat(neg_dfs, ignore_index=True) if neg_dfs else pd.DataFrame()
            print(f"  Sampled {len(df_neg_all)} negative distractors across non-{target_clean.upper()} families.")
        else:
            df_neg_all = pd.DataFrame()
    else:
        raise ValueError(f"Unknown target '{target}'. Must be one of: 'all', 'gold', or a valid family {list(FAMILIES.keys())}.")

    train_texts = df_target_all["text"].tolist() + (df_neg_all["text"].tolist() if len(df_neg_all) > 0 else [])
    model_names = df_target_all["model"].tolist() + (["__OTHER__"] * len(df_neg_all) if len(df_neg_all) > 0 else [])

    if target_clean in FAMILIES:
        ordered_models = [LABELS[i] for i in FAMILIES[target_clean]] + (["__OTHER__"] if len(df_neg_all) > 0 else [])
    else:
        ordered_models = list(LABELS) + (["__OTHER__"] if len(df_neg_all) > 0 else [])
    lab_map = {m: i for i, m in enumerate(ordered_models)}
    train_labels = [lab_map[m] for m in model_names]

    target_pids = (
        df_target_all["prompt_id"].fillna("-1").astype(str).tolist()
        if "prompt_id" in df_target_all.columns
        else ["-1"] * len(df_target_all)
    )
    neg_pids = (
        df_neg_all["prompt_id"].fillna("-1").astype(str).tolist()
        if (len(df_neg_all) > 0 and "prompt_id" in df_neg_all.columns)
        else ["-1"] * len(df_neg_all)
    )
    prompt_ids = target_pids + neg_pids

    amp_dtype = (
        torch.bfloat16
        if (torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8)
        else torch.float16
    )

    has_plain = hasattr(model, "plain_head") and model.plain_head is not None
    has_centered = hasattr(model, "centered_head") and model.centered_head is not None

    print(f"[Features Extraction] Extracting features on {len(train_texts):,} samples (batch_size={batch_size})...")

    out_p = Path(output_dir)
    out_p.mkdir(parents=True, exist_ok=True)
    if save_all_layers:
        try:
            _, _, free_bytes = shutil.disk_usage(out_p)
            n_lay_est = adapter.default_layer_count_for_disk_estimate(model)
            h_dim_est = int(
                getattr(model, "hidden_size", 0)
                or getattr(getattr(getattr(model, "backbone", model), "config", None), "hidden_size", 0)
                or (getattr(model, "total_dim", 0) // max(len(getattr(model, "layers", [1])), 1))
            )
            est_all_layers_bytes = len(train_texts) * n_lay_est * max(h_dim_est, 1) * 2
            if est_all_layers_bytes > 0.80 * free_bytes:
                print(
                    f"\n[WARNING][Disk Safety Guard] Estimated {n_lay_est}-layer tensor size ({est_all_layers_bytes / (1024**3):.2f} GB) "
                    f"exceeds 80% of available free disk space ({free_bytes / (1024**3):.2f} GB) in {out_p}."
                )
                print(
                    f"  -> Automatically falling back to saving {len(getattr(model, 'layers', []))} tapped layers in "
                    f"features_by_layer.safetensors ([N, {len(getattr(model, 'layers', []))}, {h_dim_est}]) to prevent OSError.\n"
                )
                save_all_layers = False
        except Exception as e:
            print(f"[Features Extraction] Note: Could not query disk usage ({e}); proceeding with requested save_all_layers={save_all_layers}.")

    pad_id = tok_spec.pad_id
    cls_id = tok_spec.cls_id
    eff_end_id = tok_spec.end_id if end_id is None else end_id
    drop_first_token = tok_spec.drop_first_token
    collate_fn_ext = functools.partial(collate_text_batch, pad_token_id=pad_id)

    ds_tr = TextSequenceDataset(train_texts, train_labels, None, tok, max_len=eff_max_len, end_id=eff_end_id, cls_id=cls_id)
    loader_tr = DataLoader(ds_tr, batch_size=batch_size, shuffle=False, collate_fn=collate_fn_ext)

    Z_cat_list = []
    Z_layers_list = []
    cos_drift_list = []
    std_norms_list = []
    logits_p_list = []
    logits_c_list = []

    with torch.no_grad():
        with torch.amp.autocast(device.type, enabled=(amp_dtype == torch.bfloat16), dtype=amp_dtype):
            for batch in loader_tr:
                mb_ids = batch["input_ids"].to(device)
                mb_mask = batch["attention_mask"].to(device)
                max_l = max(int(mb_mask.sum(dim=1).max().item()), 2)
                if save_multiview:
                    res = model.forward_features(
                        mb_ids[:, :max_l],
                        mb_mask[:, :max_l],
                        drop_first_token=drop_first_token,
                        return_by_layer=True,
                        extract_all_layers=save_all_layers,
                        return_multiview=True,
                    )
                    if len(res) == 4:
                        z_b, z_l, ht_cos_t, std_norm_t = res
                        cos_drift_list.append(ht_cos_t.cpu().float().numpy())
                        std_norms_list.append(std_norm_t.cpu().float().numpy())
                    else:
                        z_b, z_l, mv_d = res
                        cos_drift_list.append(mv_d["head_tail_cos"].cpu().float().numpy())
                        std_norms_list.append(mv_d["token_std_norms"].cpu().float().numpy())
                else:
                    z_b, z_l = model.forward_features(
                        mb_ids[:, :max_l],
                        mb_mask[:, :max_l],
                        drop_first_token=drop_first_token,
                        return_by_layer=True,
                        extract_all_layers=save_all_layers,
                        return_multiview=False,
                    )

                if len(Z_cat_list) == 0:
                    d_chunk = max(z_b.shape[1] // max(len(model.layers), 1), 1)
                    chunk_stds = [float(z_b[:, i * d_chunk : (i + 1) * d_chunk].std().item()) for i in range(len(model.layers))]
                    layer_stds = (
                        [float(z_l[:, i, :].std().item()) for i in range(z_l.shape[1])]
                        if (save_all_layers and z_l is not None and z_l.ndim == 3)
                        else chunk_stds
                    )
                    dead_layers = [i for i, s in enumerate(layer_stds) if s < 1e-6 or not math.isfinite(s)]
                    print(
                        f"[Sanity Check Batch 0] Active layers: {len(layer_stds) - len(dead_layers)}/{len(layer_stds)} | "
                        f"Layer std range: [{min(layer_stds):.4f}, {max(layer_stds):.4f}] | "
                        f"Tapped {len(model.layers)}-layer std range: [{min(chunk_stds):.4f}, {max(chunk_stds):.4f}]",
                        flush=True,
                    )
                    if dead_layers:
                        raise RuntimeError(f"Batch 0 Sanity Check FAILED: Layers {dead_layers} have zero/NaN activations!")

                Z_cat_list.append(z_b.cpu().float().numpy())
                Z_layers_list.append(z_l.cpu().to(torch.float16).numpy())
                if has_plain:
                    logits_p_list.append(model.predict_plain(z_b).cpu().float().numpy())
                if has_centered:
                    logits_c_list.append(model.predict_centered(z_b).cpu().float().numpy())

    features_concat = np.vstack(Z_cat_list)
    features_by_layer = np.vstack(Z_layers_list)
    champ_p = np.vstack(logits_p_list) if logits_p_list else None
    champ_c = np.vstack(logits_c_list) if logits_c_list else None
    ht_cos = np.vstack(cos_drift_list) if cos_drift_list else None
    t_std = np.vstack(std_norms_list) if std_norms_list else None

    eval_cat = None
    eval_layers = None
    eval_champ_p = None
    eval_champ_c = None
    eval_ht_cos = None
    eval_t_std = None

    if eval_texts is not None and len(eval_texts) > 0:
        print(f"[Features Extraction] Extracting benchmark features on {len(eval_texts)} holdout samples...")
        dummy_y = [0] * len(eval_texts)
        ds_ev = TextSequenceDataset(eval_texts, dummy_y, None, tok, max_len=eff_max_len, end_id=eff_end_id, cls_id=cls_id)
        loader_ev = DataLoader(ds_ev, batch_size=batch_size, shuffle=False, collate_fn=collate_fn_ext)
        ev_cat_list = []
        ev_layers_list = []
        ev_cos_list = []
        ev_std_list = []
        ev_lp_list = []
        ev_lc_list = []

        with torch.no_grad():
            with torch.amp.autocast(device.type, enabled=(amp_dtype == torch.bfloat16), dtype=amp_dtype):
                for batch in loader_ev:
                    mb_ids = batch["input_ids"].to(device)
                    mb_mask = batch["attention_mask"].to(device)
                    max_l = max(int(mb_mask.sum(dim=1).max().item()), 2)
                    if save_multiview:
                        res = model.forward_features(
                            mb_ids[:, :max_l],
                            mb_mask[:, :max_l],
                            drop_first_token=drop_first_token,
                            return_by_layer=True,
                            extract_all_layers=save_all_layers,
                            return_multiview=True,
                        )
                        if len(res) == 4:
                            z_b, z_l, ht_cos_t, std_norm_t = res
                            ev_cos_list.append(ht_cos_t.cpu().float().numpy())
                            ev_std_list.append(std_norm_t.cpu().float().numpy())
                        else:
                            z_b, z_l, mv_d = res
                            ev_cos_list.append(mv_d["head_tail_cos"].cpu().float().numpy())
                            ev_std_list.append(mv_d["token_std_norms"].cpu().float().numpy())
                    else:
                        z_b, z_l = model.forward_features(
                            mb_ids[:, :max_l],
                            mb_mask[:, :max_l],
                            drop_first_token=drop_first_token,
                            return_by_layer=True,
                            extract_all_layers=save_all_layers,
                            return_multiview=False,
                        )
                    ev_cat_list.append(z_b.cpu().float().numpy())
                    ev_layers_list.append(z_l.cpu().to(torch.float16).numpy())
                    if has_plain:
                        ev_lp_list.append(model.predict_plain(z_b).cpu().float().numpy())
                    if has_centered:
                        ev_lc_list.append(model.predict_centered(z_b).cpu().float().numpy())

        eval_cat = np.vstack(ev_cat_list)
        eval_layers = np.vstack(ev_layers_list)
        eval_champ_p = np.vstack(ev_lp_list) if ev_lp_list else None
        eval_champ_c = np.vstack(ev_lc_list) if ev_lc_list else None
        eval_ht_cos = np.vstack(ev_cos_list) if ev_cos_list else None
        eval_t_std = np.vstack(ev_std_list) if ev_std_list else None

    out_dir_path = save_extracted_features_bundle(
        output_dir=output_dir,
        target=target_clean,
        features_concat=features_concat,
        features_by_layer=features_by_layer,
        texts=train_texts,
        labels=train_labels,
        model_names=model_names,
        prompt_ids=prompt_ids,
        eval_features_concat=eval_cat,
        eval_features_by_layer=eval_layers,
        eval_texts=eval_texts,
        eval_labels=eval_labels,
        eval_model_names=eval_model_names,
        dataset_title=dataset_title,
        dataset_slug=dataset_slug,
        checkpoints_volume=checkpoints_volume,
        champion_logits_plain=champ_p if getattr(model, "num_classes", 0) == len(LABELS) else None,
        champion_logits_centered=champ_c if getattr(model, "num_classes", 0) == len(LABELS) else None,
        head_tail_cos=ht_cos,
        token_std_norms=t_std,
        layer_norms=None,
        eval_champion_logits_plain=eval_champion_logits_plain if eval_champion_logits_plain is not None else (eval_champ_p if getattr(model, "num_classes", 0) == len(LABELS) else None),
        eval_champion_logits_centered=eval_champ_c if getattr(model, "num_classes", 0) == len(LABELS) else None,
        eval_head_tail_cos=eval_ht_cos,
        eval_token_std_norms=eval_t_std,
        eval_layer_norms=None,
    )
    if hasattr(model, "norms") and model.norms is not None:
        norms_dict = {}
        for i, norm in enumerate(model.norms):
            norms_dict[f"norms.{i}.weight"] = norm.weight.detach().cpu().to(torch.float32).contiguous()
            if hasattr(norm, "bias") and norm.bias is not None:
                norms_dict[f"norms.{i}.bias"] = norm.bias.detach().cpu().to(torch.float32).contiguous()
        save_file(norms_dict, str(Path(output_dir) / "heads.safetensors"))
    return out_dir_path


In [ ]:
# === MODULE: src/namethatllm/training/gates.py ===
"""
Phase 3 Intra-Family Sister-Decoupling Gate Heads, Supervised Subspace Training, and Offline Sweeps.
"""

from __future__ import annotations

import copy
import functools
import gc
import json
import math
import shutil
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from safetensors import safe_open
from safetensors.torch import load_file, save_file
from scipy.special import softmax as sp_softmax
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import GroupKFold, StratifiedKFold
from torch import nn
from torch.utils.data import DataLoader, TensorDataset



def fit_family_gate_head(
    Z_train: np.ndarray,
    y_train: np.ndarray,
    num_gate_classes: int,
    alphas: list[float] | None = None,
    label_smoothing: float = 0.10,
    Z_eval: np.ndarray | None = None,
    y_eval_true: np.ndarray | None = None,
    eval_mask: np.ndarray | None = None,
    num_family_classes: int | None = None,
    device: torch.device | None = None,
    prompt_ids: list[Any] | np.ndarray | None = None,
) -> tuple[np.ndarray, np.ndarray, float, float, np.ndarray | None, float]:
    """
    Fits an L2-regularized Plain Ridge Head for intra-family gating using 5-fold CV on Z_train.
    Returns: (best_W, best_b, best_alpha, eval_or_oof_acc, eval_logits, best_oof_bacc)
    """
    torch.manual_seed(42)
    np.random.seed(42)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(42)

    if alphas is None:
        alphas = [1.0, 2.0, 5.0, 10.0, 25.0, 50.0, 100.0, 200.0]

    if device is None:
        device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

    N_tr, _ = Z_train.shape
    K_eval_classes = num_family_classes if num_family_classes is not None else num_gate_classes
    fam_mask_all = y_train < K_eval_classes

    def _solve_ridge_split(X_sub: np.ndarray, y_sub: np.ndarray, alpha_val: float) -> tuple[np.ndarray, np.ndarray]:
        N_s, D_s = X_sub.shape
        Y_oh = np.zeros((N_s, num_gate_classes), dtype=np.float32)
        for i_s, lab_s in enumerate(y_sub):
            if 0 <= lab_s < num_gate_classes:
                Y_oh[i_s, lab_s] = 1.0
        Y_sm = (1.0 - label_smoothing) * Y_oh + (label_smoothing / float(num_gate_classes))

        X_t_sub = torch.from_numpy(X_sub).to(device, dtype=torch.float32)
        Y_t_sub = torch.from_numpy(Y_sm).to(device, dtype=torch.float32)

        sub_fam_m = torch.from_numpy(y_sub < K_eval_classes).to(device)
        mu_X_s = X_t_sub[sub_fam_m].mean(dim=0, keepdim=True) if sub_fam_m.sum() > 0 else X_t_sub.mean(dim=0, keepdim=True)
        mu_Y_s = Y_t_sub.mean(dim=0, keepdim=True)

        X_c_s = X_t_sub - mu_X_s
        Y_c_s = Y_t_sub - mu_Y_s

        scale_s = float(torch.sum(X_c_s ** 2).item() / max(N_s, 1))
        eff_a = max(alpha_val * scale_s, 1e-6)

        if N_s >= D_s:
            Cov_s = X_c_s.T @ X_c_s
            reg_s = eff_a * torch.eye(D_s, device=device, dtype=torch.float32)
            W_s = torch.linalg.solve(Cov_s + reg_s, X_c_s.T @ Y_c_s)
        else:
            K_s = X_c_s @ X_c_s.T
            reg_s = eff_a * torch.eye(N_s, device=device, dtype=torch.float32)
            A_s = torch.linalg.solve(K_s + reg_s, Y_c_s)
            W_s = X_c_s.T @ A_s

        b_s = (mu_Y_s - mu_X_s @ W_s).squeeze(0)
        W_np_s = W_s.cpu().numpy()
        b_np_s = b_s.cpu().numpy()

        fam_logits_s = (X_sub[y_sub < K_eval_classes] @ W_np_s + b_np_s)[:, :K_eval_classes]
        row_std_s = float(np.std(fam_logits_s, axis=1).mean()) if len(fam_logits_s) > 0 else 1.0
        temp_scale_s = (3.2 / row_std_s) if row_std_s > 1e-4 else 1.0
        W_np_s = (W_np_s * temp_scale_s).astype(np.float32)
        b_np_s = (b_np_s * temp_scale_s).astype(np.float32)

        del X_t_sub, Y_t_sub, X_c_s, Y_c_s, W_s, b_s
        return W_np_s, b_np_s

    n_splits = 5
    pids_arr = np.asarray(prompt_ids) if prompt_ids is not None else None
    unique_groups = len(set(prompt_ids)) if prompt_ids is not None else 0
    max_group_frac = 1.0
    if pids_arr is not None and len(pids_arr) > 0:
        _, counts = np.unique(pids_arr, return_counts=True)
        max_group_frac = float(counts.max() / len(pids_arr))

    if pids_arr is not None and unique_groups >= 10 and max_group_frac < 0.05:
        splitter = list(GroupKFold(n_splits=n_splits).split(Z_train, y_train, groups=pids_arr))
    else:
        min_cls = min(int(np.bincount(y_train).min()) if len(y_train) > 0 else 0, n_splits)
        if min_cls >= 2:
            splitter = list(StratifiedKFold(n_splits=min_cls, shuffle=True, random_state=42).split(Z_train, y_train))
        else:
            fold_sizes = np.full(min(n_splits, N_tr), N_tr // min(n_splits, N_tr), dtype=int)
            fold_sizes[: N_tr % min(n_splits, N_tr)] += 1
            indices = np.arange(N_tr)
            splitter = []
            cur = 0
            for fs in fold_sizes:
                val_idx = indices[cur : cur + fs]
                tr_idx = np.concatenate([indices[:cur], indices[cur + fs :]])
                splitter.append((tr_idx, val_idx))
                cur += fs

    best_alpha = alphas[0]
    best_oof_bacc = -1.0
    for alpha in alphas:
        oof_logits = np.zeros((N_tr, num_gate_classes), dtype=np.float32)
        for tr_idx, val_idx in splitter:
            W_fold, b_fold = _solve_ridge_split(Z_train[tr_idx], y_train[tr_idx], alpha)
            oof_logits[val_idx] = Z_train[val_idx] @ W_fold + b_fold

        if fam_mask_all.sum() > 0:
            oof_preds = oof_logits[fam_mask_all, :K_eval_classes].argmax(axis=1)
            oof_acc = float(balanced_accuracy_score(y_train[fam_mask_all], oof_preds))
        else:
            oof_acc = float(balanced_accuracy_score(y_train, oof_logits.argmax(axis=1)))

        if oof_acc > best_oof_bacc:
            best_oof_bacc = oof_acc
            best_alpha = alpha

    best_w, best_b = _solve_ridge_split(Z_train, y_train, best_alpha)

    best_eval_logits = None
    if Z_eval is not None and y_eval_true is not None and eval_mask is not None and eval_mask.sum() > 0:
        best_eval_logits = Z_eval @ best_w + best_b
        fam_pred = best_eval_logits[eval_mask, :K_eval_classes].argmax(axis=1)
        best_acc = float(balanced_accuracy_score(y_eval_true, fam_pred))
    else:
        best_acc = best_oof_bacc

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return best_w, best_b, best_alpha, best_acc, best_eval_logits, best_oof_bacc


class FamilyMLPHead(nn.Module):
    """2-Layer Non-Linear MLP Head for intra-family sister decoupling."""

    def __init__(
        self,
        in_features: int = 40960,
        hidden_dim: int = 128,
        num_classes: int = 5,
        dropout: float = 0.25,
        in_dropout: float = 0.50,
    ):
        super().__init__()
        self.in_drop = nn.Dropout(in_dropout)
        self.fc1 = nn.Linear(in_features, hidden_dim)
        self.act = nn.GELU()
        self.ln = nn.LayerNorm(hidden_dim)
        self.drop = nn.Dropout(dropout)
        self.fc2 = nn.Linear(hidden_dim, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = self.fc1(self.in_drop(x))
        h = self.act(h)
        h = self.ln(h)
        h = self.drop(h)
        return self.fc2(h)

    def forward_eval(self, x: torch.Tensor) -> torch.Tensor:
        h = self.fc1(x)
        h = self.act(h)
        h = self.ln(h)
        return self.fc2(h)


def fit_family_mlp_head(
    Z_train: np.ndarray,
    y_train: np.ndarray,
    num_gate_classes: int,
    num_family_classes: int,
    hidden_dim: int = 64,
    epochs: int = 12,
    lr: float = 2e-4,
    dropout: float = 0.35,
    weight_decay: float = 0.10,
    batch_size: int = 64,
    label_smoothing: float = 0.10,
    Z_eval: np.ndarray | None = None,
    y_eval_true: np.ndarray | None = None,
    eval_mask: np.ndarray | None = None,
    device: torch.device | None = None,
    pca_components: np.ndarray | None = None,
    pca_mean: np.ndarray | None = None,
    raw_Z_eval: np.ndarray | None = None,
) -> tuple[dict[str, torch.Tensor], float, np.ndarray | None, float]:
    """Trains a regularized 2-layer MLP head on intra-family features and folds PCA + centering into Layer 1."""
    torch.manual_seed(42)
    np.random.seed(42)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(42)

    if device is None:
        device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

    N_tr, D_feat = Z_train.shape

    if pca_components is not None:
        Z_tr_std = Z_train.astype(np.float32)
        mu_family = None
        sd_family = None
    else:
        fam_mask_tr = y_train < num_family_classes
        if fam_mask_tr.sum() > 1:
            mu_family = Z_train[fam_mask_tr].mean(axis=0, keepdims=True).astype(np.float32)
            sd_family = Z_train[fam_mask_tr].std(axis=0, keepdims=True).clip(min=1e-3).astype(np.float32)
        else:
            mu_family = Z_train.mean(axis=0, keepdims=True).astype(np.float32)
            sd_family = Z_train.std(axis=0, keepdims=True).clip(min=1e-3).astype(np.float32)
        Z_tr_std = ((Z_train - mu_family) / sd_family).astype(np.float32)

    min_c = min(int((y_train == c).sum()) for c in range(num_gate_classes)) if num_gate_classes > 0 else 5
    n_splits_eff = max(min(5, min_c), 2)
    skf = StratifiedKFold(n_splits=n_splits_eff, shuffle=True, random_state=42)
    fit_idx, val_idx = next(skf.split(Z_tr_std, y_train))

    X_fit_t = torch.from_numpy(Z_tr_std[fit_idx]).float()
    y_fit_t = torch.from_numpy(y_train[fit_idx]).long()
    X_val_t = torch.from_numpy(Z_tr_std[val_idx]).float().to(device)
    y_val_np = y_train[val_idx]
    val_fam_mask = y_val_np < num_family_classes

    ds = TensorDataset(X_fit_t, y_fit_t)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=True)

    in_drop_rate = 0.15 if D_feat <= 256 else 0.45
    mlp = FamilyMLPHead(
        in_features=D_feat,
        hidden_dim=hidden_dim,
        num_classes=num_gate_classes,
        dropout=dropout,
        in_dropout=in_drop_rate,
    ).to(device)

    optimizer = torch.optim.AdamW(mlp.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=lr * 0.05)

    sib_mask = torch.zeros((num_gate_classes, num_gate_classes), dtype=torch.float32, device=device)
    for i in range(num_family_classes):
        for j in range(num_family_classes):
            if i != j:
                sib_mask[i, j] = 1.0
    if num_family_classes == 4:
        sib_mask[2, 3] = 1.6
        sib_mask[3, 2] = 1.6
    elif num_family_classes == 5:
        for i in (1, 2, 3):
            for j in (1, 2, 3):
                if i != j:
                    sib_mask[i, j] = 1.4

    best_val_acc = -1.0
    best_state = None
    for _ in range(epochs):
        mlp.train()
        for bx, by in loader:
            bx, by = bx.to(device), by.to(device)
            bx = bx + 0.15 * torch.randn_like(bx)
            optimizer.zero_grad()
            logits = mlp(bx)
            margin_boost = sib_mask[by] * 0.35
            loss = F.cross_entropy(logits + margin_boost, by, label_smoothing=label_smoothing)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(mlp.parameters(), 1.0)
            optimizer.step()
        scheduler.step()

        mlp.eval()
        with torch.no_grad():
            val_logits = mlp.forward_eval(X_val_t).cpu().numpy()
        if val_fam_mask.sum() > 0:
            val_acc = float(
                balanced_accuracy_score(
                    y_val_np[val_fam_mask],
                    val_logits[val_fam_mask, :num_family_classes].argmax(axis=1),
                )
            )
        else:
            val_acc = float(balanced_accuracy_score(y_val_np, val_logits.argmax(axis=1)))

        if val_acc >= best_val_acc or best_state is None:
            best_val_acc = val_acc
            best_state = copy.deepcopy(mlp.state_dict())

    if best_state is not None:
        mlp.load_state_dict(best_state)
    mlp.eval()

    with torch.no_grad():
        fam_mask = y_train < num_family_classes
        if fam_mask.sum() > 0:
            X_fam_t = torch.from_numpy(Z_tr_std[fam_mask]).float().to(device)
            fam_logits_tr = mlp.forward_eval(X_fam_t).cpu().numpy()[:, :num_family_classes]
            row_std = float(np.std(fam_logits_tr, axis=1).mean())
            temp_scale = (3.2 / row_std) if row_std > 1e-4 else 1.0
            mlp.fc2.weight.data.mul_(temp_scale)
            mlp.fc2.bias.data.mul_(temp_scale)
            del X_fam_t

    if pca_components is not None and pca_mean is not None:
        W1_64 = mlp.fc1.weight.detach().cpu().numpy().astype(np.float64)
        b1_64 = mlp.fc1.bias.detach().cpu().numpy().astype(np.float64)
        comp_64 = pca_components.astype(np.float64)
        mu_64 = pca_mean.reshape(1, -1).astype(np.float64)

        W1_folded_64 = W1_64 @ comp_64
        b1_folded_64 = b1_64 - (mu_64 @ W1_folded_64.T).squeeze(0)

        W1_folded = torch.from_numpy(W1_folded_64.astype(np.float32)).contiguous()
        b1_folded = torch.from_numpy(b1_folded_64.astype(np.float32)).contiguous()
    else:
        with torch.no_grad():
            W1 = mlp.fc1.weight.detach().float()
            b1 = mlp.fc1.bias.detach().float()
            mu_t = torch.from_numpy(mu_family).to(device=W1.device, dtype=torch.float32)
            sd_t = torch.from_numpy(sd_family).to(device=W1.device, dtype=torch.float32)
            W1_folded = (W1 / sd_t).contiguous()
            b1_folded = (b1 - ((mu_t / sd_t) @ W1.T).squeeze(0)).contiguous()

    best_eval_acc = best_val_acc
    best_eval_logits = None
    eval_target = raw_Z_eval if (raw_Z_eval is not None and pca_components is not None) else Z_eval
    if eval_target is not None:
        with torch.no_grad():
            Z_ev_raw_t = torch.from_numpy(eval_target).float().to(device)
            h = F.gelu(Z_ev_raw_t @ W1_folded.to(device).T + b1_folded.to(device))
            h = mlp.ln(h)
            best_eval_logits = mlp.fc2(h).cpu().numpy()
            del Z_ev_raw_t, h
        if y_eval_true is not None and eval_mask is not None and eval_mask.sum() > 0:
            fam_pred = best_eval_logits[eval_mask, :num_family_classes].argmax(axis=1)
            best_eval_acc = float(balanced_accuracy_score(y_eval_true, fam_pred))

    W_ridge, b_ridge, *_ = fit_family_gate_head(
        Z_train=Z_train,
        y_train=y_train,
        num_gate_classes=num_gate_classes,
        Z_eval=Z_eval,
        y_eval_true=y_eval_true,
        eval_mask=eval_mask,
        num_family_classes=num_family_classes,
        device=device,
    )

    if pca_components is not None and pca_mean is not None:
        comp_64 = pca_components.astype(np.float64)
        mu_64 = pca_mean.reshape(1, -1).astype(np.float64)
        W_r_64 = W_ridge.astype(np.float64)
        b_r_64 = b_ridge.astype(np.float64)
        W_r_eff_64 = comp_64.T @ W_r_64
        b_r_eff_64 = b_r_64 - (mu_64 @ W_r_eff_64).squeeze(0)
        plain_w = torch.from_numpy(W_r_eff_64.T.astype(np.float32)).contiguous()
        plain_b = torch.from_numpy(b_r_eff_64.astype(np.float32)).contiguous()
    else:
        plain_w = torch.from_numpy(W_ridge.T).to(torch.float32).contiguous()
        plain_b = torch.from_numpy(b_ridge).to(torch.float32).contiguous()

    head_dict = {
        "mlp_w1": W1_folded.cpu().to(torch.float32).contiguous(),
        "mlp_b1": b1_folded.cpu().to(torch.float32).contiguous(),
        "unfolded_w1": mlp.fc1.weight.detach().cpu().to(torch.float32).contiguous(),
        "unfolded_b1": mlp.fc1.bias.detach().cpu().to(torch.float32).contiguous(),
        "ln_weight": mlp.ln.weight.detach().cpu().to(torch.float32).contiguous(),
        "ln_bias": mlp.ln.bias.detach().cpu().to(torch.float32).contiguous(),
        "mlp_w2": mlp.fc2.weight.detach().cpu().to(torch.float32).contiguous(),
        "mlp_b2": mlp.fc2.bias.detach().cpu().to(torch.float32).contiguous(),
        "plain.weight": plain_w,
        "plain.bias": plain_b,
    }

    del X_fit_t, y_fit_t, X_val_t, ds, mlp
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return head_dict, best_eval_acc, best_eval_logits, best_val_acc


def train_family_gate_from_dataset(
    dataset_dir: Path | str,
    target_family: str = "gemini",
    output_dir: Path | str | None = None,
    head_type: str = "mlp",
    use_supervised_subspace: bool = True,
    subspace_dim: int | None = None,
    pca_dim: int | None = None,
    hidden_dim: int = 64,
    mlp_epochs: int = 12,
    mlp_lr: float = 2e-4,
    mlp_dropout: float = 0.35,
    device: torch.device | None = None,
    run_probing_sweep: bool = False,
    checkpoints_volume: Any | None = None,
    gate_weight: float = 0.40,
    select_layers: list[int] | str | None = None,
    include_velocity: bool = True,
    layer_weighting: bool = True,
    use_multiview: bool = False,
    hard_sister_pair: tuple[int, int] | None = None,
) -> dict[str, Any]:
    """Offline entrypoint to train and evaluate any family gate directly from pre-extracted features."""
    if subspace_dim is None and pca_dim is not None:
        subspace_dim = pca_dim

    if device is None:
        device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

    ds_p = Path(dataset_dir)
    assert (ds_p / "features_concat.safetensors").exists(), f"features_concat.safetensors not found in {ds_p}"
    assert (ds_p / "metadata.parquet").exists(), f"metadata.parquet not found in {ds_p}"

    target_family = target_family.lower().strip()
    assert target_family in FAMILIES, f"Unknown family {target_family}. Must be one of {list(FAMILIES.keys())}"
    family_cols = FAMILIES[target_family]
    family_classes = [LABELS[i] for i in family_cols]
    num_family_classes = len(family_classes)

    print("\n" + "=" * 88)
    print(f" [Offline Dataset Gate Training] Loading pre-extracted features from {ds_p} for '{target_family.upper()}'...")
    print("=" * 88)

    feat_sd = load_file(str(ds_p / "features_concat.safetensors"))
    Z_train = feat_sd["features_concat"].cpu().float().numpy()

    Z_train_layers = None
    if (ds_p / "features_by_layer.safetensors").exists():
        feat_l_sd = load_file(str(ds_p / "features_by_layer.safetensors"))
        Z_train_layers = feat_l_sd["features_by_layer"].cpu().numpy()

    df_meta = pd.read_parquet(ds_p / "metadata.parquet")
    model_names = df_meta["model"].tolist()
    prompt_ids = df_meta["prompt_id"].tolist() if "prompt_id" in df_meta.columns else None

    has_negatives = any(m == "__OTHER__" for m in model_names)
    num_gate_classes = num_family_classes + (1 if has_negatives else 0)

    fam_map = {name: i for i, name in enumerate(family_classes)}
    y_train = np.array([fam_map.get(m, num_family_classes if has_negatives else 0) for m in model_names], dtype=np.int64)

    eff_hard_pair = hard_sister_pair
    if eff_hard_pair is None:
        if num_family_classes == 4:
            eff_hard_pair = (2, 3)
        elif num_family_classes == 5:
            eff_hard_pair = (1, 2)
        elif num_family_classes == 2:
            eff_hard_pair = (0, 1)

    if run_probing_sweep and Z_train_layers is not None:
        probe_out_dir = Path(output_dir) if output_dir else ds_p
        run_all_layers_probe_sweep(
            features_by_layer=Z_train_layers,
            y=y_train,
            num_family_classes=num_family_classes,
            hard_sister_pair=eff_hard_pair or (2, 3),
            prompts=np.asarray(prompt_ids) if prompt_ids is not None else None,
            output_dir=probe_out_dir,
        )

    chosen_layers = None
    if select_layers is not None and Z_train_layers is not None:
        if select_layers == "auto_top8":
            probe_out_dir = Path(output_dir) if output_dir else ds_p
            probe_rep = run_all_layers_probe_sweep(
                features_by_layer=Z_train_layers,
                y=y_train,
                num_family_classes=num_family_classes,
                hard_sister_pair=eff_hard_pair or (2, 3),
                prompts=np.asarray(prompt_ids) if prompt_ids is not None else None,
                output_dir=probe_out_dir,
            )
            chosen_layers = probe_rep["top_8_layers"]
        else:
            chosen_layers = list(select_layers)

        if chosen_layers is not None and len(chosen_layers) > 0:
            if max(chosen_layers) >= Z_train_layers.shape[1]:
                print(
                    f"[{target_family.upper()} Gate] Warning: select_layers {chosen_layers} exceeds available layers ({Z_train_layers.shape[1]}). Using all available layers."
                )
                chosen_layers = None
            else:
                print(f"[{target_family.upper()} Gate] Slicing select_layers={chosen_layers} from features_by_layer...")
                Z_train_sel = Z_train_layers[:, chosen_layers, :].astype(np.float32)
                m_l = Z_train_sel.mean(axis=-1, keepdims=True)
                s_l = Z_train_sel.std(axis=-1, keepdims=True) + 1e-5
                Z_train_sel = (Z_train_sel - m_l) / s_l
                Z_train = Z_train_sel.reshape(Z_train_sel.shape[0], -1).astype(np.float32)

    Z_eval, y_ev_arr, main_logits_eval = None, None, None
    Z_ev_layers = None
    if (ds_p / "benchmark_features_concat.safetensors").exists() and (ds_p / "benchmark_metadata.parquet").exists():
        bench_sd = load_file(str(ds_p / "benchmark_features_concat.safetensors"))
        Z_eval = bench_sd["benchmark_features_concat"].cpu().float().numpy()
        df_bench = pd.read_parquet(ds_p / "benchmark_metadata.parquet")
        y_ev_arr = df_bench["label"].to_numpy()

    if (ds_p / "benchmark_features_by_layer.safetensors").exists():
        bench_l_sd = load_file(str(ds_p / "benchmark_features_by_layer.safetensors"))
        Z_ev_layers = bench_l_sd["benchmark_features_by_layer"].cpu().numpy()

    if chosen_layers is not None and Z_ev_layers is not None and max(chosen_layers) < Z_ev_layers.shape[1]:
        Z_ev_sel = Z_ev_layers[:, chosen_layers, :].astype(np.float32)
        m_ev = Z_ev_sel.mean(axis=-1, keepdims=True)
        s_ev = Z_ev_sel.std(axis=-1, keepdims=True) + 1e-5
        Z_ev_sel = (Z_ev_sel - m_ev) / s_ev
        Z_eval = Z_ev_sel.reshape(Z_ev_sel.shape[0], -1).astype(np.float32)

    if (ds_p / "benchmark_champion_logits.safetensors").exists():
        champ_ev_sd = load_file(str(ds_p / "benchmark_champion_logits.safetensors"))
        if "logits_plain" in champ_ev_sd:
            main_logits_eval = champ_ev_sd["logits_plain"].cpu().float().numpy()
        elif "champion_logits" in champ_ev_sd:
            main_logits_eval = champ_ev_sd["champion_logits"].cpu().float().numpy()
        elif len(champ_ev_sd) > 0 and any("logit" in k.lower() for k in champ_ev_sd):
            for k in champ_ev_sd:
                if "logit" in k.lower():
                    main_logits_eval = champ_ev_sd[k].cpu().float().numpy()
                    break
    elif (ds_p / "champion_logits.safetensors").exists() and Z_eval is not None:
        champ_sd = load_file(str(ds_p / "champion_logits.safetensors"))
        if "logits_plain" in champ_sd and len(champ_sd["logits_plain"]) == len(Z_eval):
            main_logits_eval = champ_sd["logits_plain"].cpu().float().numpy()
        elif "champion_logits" in champ_sd and len(champ_sd["champion_logits"]) == len(Z_eval):
            main_logits_eval = champ_sd["champion_logits"].cpu().float().numpy()

    num_tapped = len(chosen_layers) if chosen_layers is not None else 8
    pca_components = None
    pca_mean = None
    if use_supervised_subspace:
        Z_tr_active, Z_ev_active, pca_components, pca_mean, _ = build_oof_supervised_family_subspace(
            Z_train=Z_train,
            y_train=y_train,
            Z_eval=Z_eval,
            num_family_classes=num_family_classes,
            subspace_dim=subspace_dim,
            num_tapped_layers=num_tapped,
            include_velocity=include_velocity,
            layer_weighting=layer_weighting,
            hard_sister_pair=eff_hard_pair,
            prompt_ids=prompt_ids,
            device=device,
        )
    else:
        Z_tr_active = Z_train
        Z_ev_active = Z_eval

    if use_multiview:
        ht_tr, std_tr = None, None
        if (ds_p / "champion_logits.safetensors").exists():
            c_sd = load_file(str(ds_p / "champion_logits.safetensors"))
            if "head_tail_cos" in c_sd:
                ht_tr = c_sd["head_tail_cos"].cpu().float().numpy()
            if "token_std_norms" in c_sd:
                std_tr = c_sd["token_std_norms"].cpu().float().numpy()

        f_by_layer_tr = Z_train_layers if (Z_eval is None or Z_ev_layers is not None) else None
        mv_tr = engineer_multiview_features(
            features_concat=Z_train,
            features_by_layer=f_by_layer_tr,
            head_tail_cos=ht_tr,
            token_std_norms=std_tr,
            num_tapped_layers=num_tapped,
        )
        mv_mean = mv_tr.mean(axis=0, keepdims=True)
        mv_std = mv_tr.std(axis=0, keepdims=True) + 1e-5
        mv_tr_norm = (mv_tr - mv_mean) / mv_std
        Z_tr_active = np.hstack([Z_tr_active, mv_tr_norm]).astype(np.float32)

        if Z_eval is not None:
            ht_ev, std_ev = None, None
            if (ds_p / "benchmark_champion_logits.safetensors").exists():
                b_sd = load_file(str(ds_p / "benchmark_champion_logits.safetensors"))
                if "head_tail_cos" in b_sd:
                    ht_ev = b_sd["head_tail_cos"].cpu().float().numpy()
                if "token_std_norms" in b_sd:
                    std_ev = b_sd["token_std_norms"].cpu().float().numpy()
            f_by_layer_ev = Z_ev_layers if (Z_train_layers is not None) else None
            mv_ev = engineer_multiview_features(
                features_concat=Z_eval,
                features_by_layer=f_by_layer_ev,
                head_tail_cos=ht_ev,
                token_std_norms=std_ev,
                num_tapped_layers=num_tapped,
            )
            mv_ev_norm = (mv_ev - mv_mean) / mv_std
            Z_ev_active = np.hstack([Z_ev_active, mv_ev_norm]).astype(np.float32)

        pca_components = None
        pca_mean = None

    fam_eval_mask = np.isin(y_ev_arr, family_cols) if y_ev_arr is not None else None
    col_to_local = {c: i for i, c in enumerate(family_cols)}
    y_fam_true = np.array([col_to_local[v] for v in y_ev_arr[fam_eval_mask]]) if fam_eval_mask is not None else None

    acc_plain_within = 0.0
    if main_logits_eval is not None and fam_eval_mask is not None and fam_eval_mask.sum() > 0:
        if main_logits_eval.shape[1] == len(LABELS):
            plain_fam_pred = main_logits_eval[fam_eval_mask][:, family_cols].argmax(axis=1)
        else:
            plain_fam_pred = main_logits_eval[fam_eval_mask][:, :num_family_classes].argmax(axis=1)
        acc_plain_within = float(balanced_accuracy_score(y_fam_true, plain_fam_pred))

    best_w_pca, best_b_pca, ridge_alpha, ridge_acc, ridge_logits_ev, ridge_oof_bacc = fit_family_gate_head(
        Z_train=Z_tr_active,
        y_train=y_train,
        num_gate_classes=num_gate_classes,
        alphas=[1.0, 2.0, 5.0, 10.0, 25.0, 50.0, 100.0, 200.0],
        label_smoothing=0.10,
        Z_eval=Z_ev_active,
        y_eval_true=y_fam_true,
        eval_mask=fam_eval_mask,
        num_family_classes=num_family_classes,
        device=device,
        prompt_ids=prompt_ids,
    )

    if pca_components is not None:
        comp_64 = pca_components.astype(np.float64)
        mu_64 = pca_mean.reshape(1, -1).astype(np.float64)
        W_r_64 = best_w_pca.astype(np.float64)
        b_r_64 = best_b_pca.astype(np.float64)
        W_r_eff_64 = comp_64.T @ W_r_64
        b_r_eff_64 = b_r_64 - (mu_64 @ W_r_eff_64).squeeze(0)
        best_w = W_r_eff_64.astype(np.float32)
        best_b = b_r_eff_64.astype(np.float32)
    else:
        best_w = best_w_pca
        best_b = best_b_pca

    plain_w_tensor = torch.from_numpy(best_w.T).to(torch.float32).contiguous()
    plain_b_tensor = torch.from_numpy(best_b).to(torch.float32).contiguous()

    mlp_head_dict, mlp_acc, mlp_logits_ev, mlp_val_acc = fit_family_mlp_head(
        Z_train=Z_tr_active,
        y_train=y_train,
        num_gate_classes=num_gate_classes,
        num_family_classes=num_family_classes,
        hidden_dim=hidden_dim,
        epochs=mlp_epochs,
        lr=mlp_lr,
        dropout=mlp_dropout,
        label_smoothing=0.10,
        Z_eval=Z_ev_active,
        y_eval_true=y_fam_true,
        eval_mask=fam_eval_mask,
        device=device,
        pca_components=pca_components,
        pca_mean=pca_mean,
        raw_Z_eval=Z_eval if pca_components is not None else None,
    )

    hybrid_logits_ev = None
    if mlp_logits_ev is not None and ridge_logits_ev is not None:
        raw_hyb = 0.65 * mlp_logits_ev + 0.35 * ridge_logits_ev
        hyb_fam = raw_hyb[:, :num_family_classes]
        hyb_std = float(np.std(hyb_fam, axis=1).mean()) if len(hyb_fam) > 0 else 1.0
        hyb_scale = (3.2 / hyb_std) if hyb_std > 1e-4 else 1.0
        hybrid_logits_ev = (raw_hyb * hyb_scale).astype(np.float32)

    default_gw = float(gate_weight)

    def _eval_head_metrics(h_logits: np.ndarray | None):
        if h_logits is None:
            return 0.0, 0.0, {}
        ev_acc = 0.0
        if fam_eval_mask is not None and fam_eval_mask.sum() > 0 and y_fam_true is not None:
            fam_pred = h_logits[fam_eval_mask, :num_family_classes].argmax(axis=1)
            ev_acc = float(balanced_accuracy_score(y_fam_true, fam_pred))

        hard_acc = ev_acc
        if y_fam_true is not None and eff_hard_pair is not None and fam_eval_mask is not None and fam_eval_mask.sum() > 0:
            s_a, s_b = eff_hard_pair
            pair_mask_ev = (y_fam_true == s_a) | (y_fam_true == s_b)
            if pair_mask_ev.sum() > 0 and len(np.unique(y_fam_true[pair_mask_ev])) == 2:
                fam_l = h_logits[fam_eval_mask, :num_family_classes]
                spec_pair_preds = np.where(fam_l[pair_mask_ev, s_a] >= fam_l[pair_mask_ev, s_b], s_a, s_b)
                hard_acc = float(balanced_accuracy_score(y_fam_true[pair_mask_ev], spec_pair_preds))

        p_res = {}
        if main_logits_eval is not None and y_ev_arr is not None and main_logits_eval.shape[1] == len(LABELS):
            P_main = sp_softmax(main_logits_eval, axis=1)
            acc_plain_solo = float(balanced_accuracy_score(y_ev_arr, P_main.argmax(axis=1)))
            cand_mask = np.isin(P_main.argmax(axis=1), family_cols) | (P_main[:, family_cols].sum(axis=1) >= 0.10)
            spec_probs = sp_softmax(h_logits[:, :num_family_classes], axis=1)

            best_pipe_acc = 0.0
            best_pipe_w = default_gw
            best_raw_bacc = 0.0
            best_sk_bacc = 0.0
            default_raw_bacc = 0.0
            default_sk_bacc = 0.0
            sweep_weights = sorted(set([0.10, 0.20, 0.30, 0.35, 0.40, 0.45, 0.50, 0.60, default_gw]))

            for gw in sweep_weights:
                P_fused = P_main.copy()
                for idx in np.flatnonzero(cand_mask):
                    orig_mass = float(P_main[idx, family_cols].sum())
                    if orig_mass > 1e-6:
                        gw_eff = gw * min(orig_mass / 0.50, 1.0)
                        main_norm = P_main[idx, family_cols] / orig_mass
                        fused_fam = (1.0 - gw_eff) * main_norm + gw_eff * spec_probs[idx]
                        P_fused[idx, family_cols] = fused_fam * orig_mass

                acc_gated = float(balanced_accuracy_score(y_ev_arr, P_fused.argmax(axis=1)))
                P_sk = sinkhorn_balancing(P_fused, iters=50, sinkhorn_strength=0.65, target_mix=0.35)
                acc_gated_sk = float(balanced_accuracy_score(y_ev_arr, P_sk.argmax(axis=1)))

                if abs(gw - default_gw) < 1e-4:
                    default_raw_bacc = acc_gated
                    default_sk_bacc = acc_gated_sk

                if acc_gated_sk > best_pipe_acc:
                    best_pipe_acc = acc_gated_sk
                    best_pipe_w = gw
                    best_raw_bacc = acc_gated
                    best_sk_bacc = acc_gated_sk

            p_res = {
                "plain_solo_bacc": acc_plain_solo,
                "best_pipe_bacc": best_pipe_acc,
                "best_gate_weight": best_pipe_w,
                "best_raw_bacc": best_raw_bacc,
                "best_sinkhorn_bacc": best_sk_bacc,
                "default_raw_bacc": default_raw_bacc,
                "default_sinkhorn_bacc": default_sk_bacc,
                "lift_over_plain": best_pipe_acc - acc_plain_solo,
            }
        return ev_acc, hard_acc, p_res

    r_ev_acc, r_hard_acc, r_pipe_res = _eval_head_metrics(ridge_logits_ev)
    m_ev_acc, m_hard_acc, m_pipe_res = _eval_head_metrics(mlp_logits_ev)
    h_ev_acc, h_hard_acc, h_pipe_res = _eval_head_metrics(hybrid_logits_ev)

    heads_breakdown = {
        "ridge": {
            "eval_fam_acc": r_ev_acc if ridge_logits_ev is not None else float(ridge_acc),
            "hard_pair_acc": r_hard_acc if ridge_logits_ev is not None else float(ridge_acc),
            "oof_acc": float(ridge_oof_bacc),
            "pipe_results": r_pipe_res,
            "logits": ridge_logits_ev,
        },
        "mlp": {
            "eval_fam_acc": m_ev_acc if mlp_logits_ev is not None else float(mlp_acc),
            "hard_pair_acc": m_hard_acc if mlp_logits_ev is not None else float(mlp_acc),
            "oof_acc": float(mlp_val_acc),
            "pipe_results": m_pipe_res,
            "logits": mlp_logits_ev,
        },
        "hybrid": {
            "eval_fam_acc": h_ev_acc if hybrid_logits_ev is not None else float(mlp_acc),
            "hard_pair_acc": h_hard_acc if hybrid_logits_ev is not None else float(mlp_acc),
            "oof_acc": float(0.65 * mlp_val_acc + 0.35 * ridge_oof_bacc),
            "pipe_results": h_pipe_res,
            "logits": hybrid_logits_ev,
        },
    }

    if head_type.lower() == "mlp":
        chosen_head = heads_breakdown["mlp"]
        best_fam_acc = chosen_head["eval_fam_acc"]
        best_fam_logits = mlp_logits_ev
        best_alpha = ridge_alpha
        final_head_dict = mlp_head_dict
        final_head_dict["plain.weight"] = plain_w_tensor
        final_head_dict["plain.bias"] = plain_b_tensor
    elif head_type.lower() in ("hybrid", "hybrid_65mlp_35ridge"):
        chosen_head = heads_breakdown["hybrid"]
        best_fam_acc = chosen_head["eval_fam_acc"]
        best_fam_logits = hybrid_logits_ev if hybrid_logits_ev is not None else mlp_logits_ev
        best_alpha = ridge_alpha
        final_head_dict = mlp_head_dict
        final_head_dict["plain.weight"] = plain_w_tensor
        final_head_dict["plain.bias"] = plain_b_tensor
    else:
        chosen_head = heads_breakdown["ridge"]
        best_fam_acc = chosen_head["eval_fam_acc"]
        best_fam_logits = ridge_logits_ev
        best_alpha = ridge_alpha
        final_head_dict = {
            "plain.weight": plain_w_tensor,
            "plain.bias": plain_b_tensor,
        }

    hard_pair_eval_acc = chosen_head["hard_pair_acc"]
    pipe_results = chosen_head["pipe_results"]
    best_pipe_w = pipe_results.get("best_gate_weight", default_gw)

    if output_dir is not None:
        out_p = Path(output_dir)
        out_p.mkdir(parents=True, exist_ok=True)
        norms_src = ds_p / "heads.safetensors"
        if norms_src.exists():
            try:
                src_sd = load_file(str(norms_src))
                for k, v in src_sd.items():
                    if k.startswith("norms."):
                        final_head_dict[k] = v
            except Exception:
                pass
        save_file(final_head_dict, str(out_p / "heads.safetensors"))
        meta_fam = {
            "classes": family_classes + (["__OTHER__"] if has_negatives else []),
            "labels": family_classes + (["__OTHER__"] if has_negatives else []),
            "target_family": target_family,
            "head_type": head_type.lower(),
            "default_gate_weight": default_gw,
            "best_sweep_gate_weight": float(best_pipe_w),
            "use_supervised_subspace": use_supervised_subspace,
            "subspace_dim": int(pca_components.shape[0]) if pca_components is not None else None,
            "pca_dim": pca_dim,
            "hidden_dim": hidden_dim if "mlp" in head_type.lower() or "hybrid" in head_type.lower() else None,
            "alpha": float(best_alpha),
            "ridge_alpha": float(ridge_alpha),
            "ridge_within_family_acc": float(ridge_acc),
            "mlp_within_family_acc": float(mlp_acc),
            "within_family_acc": float(best_fam_acc),
            "plain_within_family_acc": float(acc_plain_within),
            "hard_pair_eval_acc": float(hard_pair_eval_acc),
            "lift_over_plain": float(best_fam_acc - acc_plain_within),
        }
        (out_p / "meta.json").write_text(json.dumps(meta_fam, indent=2), encoding="utf-8")
        if checkpoints_volume is not None and hasattr(checkpoints_volume, "commit"):
            checkpoints_volume.commit()
        print(f"[{target_family.upper()} Gate] Checkpoint successfully saved to {out_p}.")

    return {
        "target_family": target_family,
        "within_family_balanced_acc": float(best_fam_acc),
        "plain_within_family_acc": float(acc_plain_within),
        "delta_within_family": float(best_fam_acc - acc_plain_within),
        "hard_pair_eval_acc": float(hard_pair_eval_acc),
        "best_alpha": float(best_alpha),
        "ridge_acc": float(ridge_acc),
        "mlp_acc": float(mlp_acc),
        "subspace_dim": int(pca_components.shape[0]) if pca_components is not None else 0,
        "pipe_results": pipe_results,
        "heads_breakdown": heads_breakdown,
        "spec_logits": best_fam_logits[:, :num_family_classes].tolist() if best_fam_logits is not None else [],
        "spec_classes": family_classes,
    }


def train_family_gate(
    model: nn.Module,
    tok: Any,
    df_pool: pd.DataFrame,
    df_train: pd.DataFrame | None,
    target_family: str,
    samples_per_class: int = 0,
    negative_ratio: float = 0.0,
    eval_texts: list[str] | None = None,
    eval_labels: list[int] | None = None,
    batch_size: int = 32,
    max_len: int | None = None,
    end_id: int | None = None,
    device: torch.device | None = None,
    output_dir: Path | str | None = None,
    checkpoints_volume: Any | None = None,
    head_type: str = "mlp",
    use_supervised_subspace: bool = True,
    subspace_dim: int | None = None,
    pca_dim: int | None = None,
    hidden_dim: int = 64,
    mlp_epochs: int = 12,
    mlp_lr: float = 2e-4,
    mlp_dropout: float = 0.35,
    export_embeddings: bool = False,
    save_all_layers: bool = False,
    embeddings_output_dir: Path | str | None = None,
    gate_weight: float = 0.40,
    train_cfg: dict[str, Any] | None = None,
    include_velocity: bool = True,
    layer_weighting: bool = True,
    hard_sister_pair: tuple[int, int] | None = None,
) -> dict[str, Any]:
    """End-to-end family gate training pipeline extracting representations on model forward_features."""
    if subspace_dim is None and pca_dim is not None:
        subspace_dim = pca_dim

    if device is None:
        device = next(model.parameters()).device

    target_family = target_family.lower()
    assert target_family in FAMILIES, f"Unknown family {target_family}. Must be one of {list(FAMILIES.keys())}"
    family_cols = FAMILIES[target_family]
    family_classes = [LABELS[i] for i in family_cols]
    num_family_classes = len(family_classes)
    has_negatives = bool(negative_ratio > 0.0 and df_pool is not None and len(df_pool) > 0)
    num_gate_classes = num_family_classes + (1 if has_negatives else 0)

    if samples_per_class <= 0:
        default_samples = {"glm": 4000, "gemini": 3800, "gpt": 2500, "claude": 2500}
        samples_per_class = default_samples.get(target_family, 2500)

    fam_hint = getattr(model, "backbone_family", "unknown")
    adapter_cfg: dict[str, Any] = {}
    if fam_hint and fam_hint != "unknown":
        adapter_cfg["backbone_family"] = fam_hint
    else:
        bb_obj = getattr(model, "backbone", model)
        adapter_cfg["backbone"] = f"{type(bb_obj)} {getattr(bb_obj, 'config', '')}"
    adapter = resolve_backbone_adapter(adapter_cfg)

    tok_spec = getattr(model, "tok_spec", None)
    if tok_spec is None:
        tok_spec = adapter.tok_spec_from_tokenizer(tok, max_len=max_len, end_id=end_id)

    batch_size = adapter.clamp_extraction_batch_size(batch_size)

    if eval_texts is not None and len(eval_texts) > 0:
        bench_hashes = {_text_hash(t) for t in eval_texts}
        init_pool_len = len(df_pool)
        pool_hashes = [_text_hash(t) for t in df_pool["text"].fillna("")]
        df_pool = df_pool[[h not in bench_hashes for h in pool_hashes]].copy()
        if df_train is not None and len(df_train) > 0:
            tr_hashes = [_text_hash(t) for t in df_train["text"].fillna("")]
            df_train = df_train[[h not in bench_hashes for h in tr_hashes]].copy()
        quarantined = init_pool_len - len(df_pool)
        print(f"[{target_family.upper()} Gate] Quarantined {quarantined} benchmark overlap rows. Clean pool: {len(df_pool):,}.")

    fam_dfs = []
    for c in family_classes:
        sub_tr = df_train[df_train["model"] == c] if (df_train is not None and len(df_train) > 0) else pd.DataFrame()
        sub_pool = df_pool[df_pool["model"] == c]
        n_take = max(min(samples_per_class - len(sub_tr), len(sub_pool)), 0)
        sampled_pool = sub_pool.sample(n=n_take, random_state=42) if n_take > 0 else pd.DataFrame()
        c_df = pd.concat([sub_tr, sampled_pool], ignore_index=True)
        fam_dfs.append(c_df)
        print(f"  {target_family.upper()} class {c}: {len(c_df)} samples ({len(sub_tr)} train, {len(c_df)-len(sub_tr)} pool)")

    df_fam_all = pd.concat(fam_dfs, ignore_index=True)
    total_fam_samples = len(df_fam_all)

    if has_negatives:
        n_neg = int(total_fam_samples * negative_ratio)
        df_non_fam = df_pool[~df_pool["model"].isin(family_classes)].copy()
        other_fams = sorted({"".join(ch for ch in m.split("-")[0].split(".")[0] if ch.isalpha()) for m in LABELS if m not in family_classes})
        neg_dfs = []
        n_per_fam = max(n_neg // max(len(other_fams), 1), 1)
        for fam in other_fams:
            fam_sub = df_non_fam[df_non_fam["model"].str.startswith(fam)]
            if len(fam_sub) > 0:
                neg_dfs.append(fam_sub.sample(n=min(n_per_fam, len(fam_sub)), random_state=42))
        df_neg_all = pd.concat(neg_dfs, ignore_index=True) if neg_dfs else pd.DataFrame()
        print(f"[{target_family.upper()} Gate] Sampled {len(df_neg_all)} negative distractors across {len(other_fams)} non-{target_family.upper()} families.")
    else:
        df_neg_all = pd.DataFrame()

    train_texts = df_fam_all["text"].tolist() + (df_neg_all["text"].tolist() if len(df_neg_all) > 0 else [])
    fam_map = {name: i for i, name in enumerate(family_classes)}
    train_labels = [fam_map[m] for m in df_fam_all["model"].tolist()] + ([num_family_classes] * len(df_neg_all) if len(df_neg_all) > 0 else [])
    fam_pids = (
        df_fam_all["prompt_id"].fillna("-1").astype(str).tolist()
        if "prompt_id" in df_fam_all.columns
        else [f"pool_{i}" for i in range(len(df_fam_all))]
    )
    neg_pids = (
        df_neg_all["prompt_id"].fillna("-1").astype(str).tolist()
        if (len(df_neg_all) > 0 and "prompt_id" in df_neg_all.columns)
        else [f"neg_{i}" for i in range(len(df_neg_all))]
    )
    prompt_ids = fam_pids + neg_pids
    other_str = " + 1 Other" if len(df_neg_all) > 0 else ""
    print(f"[{target_family.upper()} Gate] Total training dataset: {len(train_texts):,} samples ({num_gate_classes} classes: {num_family_classes} {target_family.upper()}{other_str}).")

    amp_dtype = (
        torch.bfloat16
        if (torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8)
        else torch.float16
    )

    print(f"[{target_family.upper()} Gate] Extracting Z_train on {len(train_texts)} samples (batch_size={batch_size})...")
    pad_id = tok_spec.pad_id
    cls_id = tok_spec.cls_id
    eff_end_id = tok_spec.end_id if end_id is None else end_id
    eff_max_len = int(max_len if max_len is not None else tok_spec.max_len)
    drop_first_token = tok_spec.drop_first_token
    collate_fn_gate = functools.partial(collate_text_batch, pad_token_id=pad_id)

    ds_tr = TextSequenceDataset(train_texts, train_labels, None, tok, max_len=eff_max_len, end_id=eff_end_id, cls_id=cls_id)
    loader_tr = DataLoader(ds_tr, batch_size=batch_size, shuffle=False, collate_fn=collate_fn_gate)

    Z_tr_list = []
    Z_tr_layers_list = []
    tr_cos_list = []
    tr_std_list = []
    need_layers = bool(export_embeddings or save_all_layers)
    if save_all_layers and export_embeddings:
        try:
            emb_target_dir = Path(embeddings_output_dir) if embeddings_output_dir else (Path(output_dir) / "embeddings" if output_dir else Path("export_embeddings"))
            emb_target_dir.mkdir(parents=True, exist_ok=True)
            _, _, free_bytes = shutil.disk_usage(emb_target_dir)
            n_lay_est = adapter.default_layer_count_for_disk_estimate(model)
            h_dim_est = int(
                getattr(model, "hidden_size", 0)
                or getattr(getattr(getattr(model, "backbone", model), "config", None), "hidden_size", 0)
                or (getattr(model, "total_dim", 0) // max(len(getattr(model, "layers", [1])), 1))
            )
            est_all_layers_bytes = len(train_texts) * n_lay_est * max(h_dim_est, 1) * 2
            if est_all_layers_bytes > 0.80 * free_bytes:
                print(
                    f"\n[WARNING][Disk Safety Guard] Estimated {n_lay_est}-layer tensor size ({est_all_layers_bytes / (1024**3):.2f} GB) "
                    f"exceeds 80% of available free disk space ({free_bytes / (1024**3):.2f} GB) in {emb_target_dir}."
                )
                print(
                    f"  -> Automatically falling back to saving {len(getattr(model, 'layers', []))} tapped layers in "
                    f"features_by_layer.safetensors ([N, {len(getattr(model, 'layers', []))}, {h_dim_est}]).\n"
                )
                save_all_layers = False
        except Exception:
            pass
    with torch.no_grad():
        with torch.amp.autocast(device.type, enabled=(amp_dtype == torch.bfloat16), dtype=amp_dtype):
            for batch in loader_tr:
                mb_ids = batch["input_ids"].to(device)
                mb_mask = batch["attention_mask"].to(device)
                max_l = max(int(mb_mask.sum(dim=1).max().item()), 2)
                if need_layers:
                    res = model.forward_features(
                        mb_ids[:, :max_l],
                        mb_mask[:, :max_l],
                        drop_first_token=drop_first_token,
                        return_by_layer=True,
                        extract_all_layers=save_all_layers,
                        return_multiview=True,
                    )
                    if isinstance(res, tuple) and len(res) == 4:
                        z_b, z_l, ht_cos_t, std_norm_t = res
                        tr_cos_list.append(ht_cos_t.cpu().to(torch.float16).numpy())
                        tr_std_list.append(std_norm_t.cpu().to(torch.float16).numpy())
                    elif isinstance(res, tuple) and len(res) == 2:
                        z_b, z_l = res
                    Z_tr_layers_list.append(z_l.cpu().to(torch.float16).numpy())
                else:
                    z_b = model.forward_features(mb_ids[:, :max_l], mb_mask[:, :max_l], drop_first_token=drop_first_token)

                if len(Z_tr_list) == 0:
                    d_chunk = max(z_b.shape[1] // max(len(model.layers), 1), 1)
                    chunk_stds = [float(z_b[:, i * d_chunk : (i + 1) * d_chunk].std().item()) for i in range(len(model.layers))]
                    layer_stds = (
                        [float(z_l[:, i, :].std().item()) for i in range(z_l.shape[1])]
                        if (need_layers and z_l is not None and z_l.ndim == 3)
                        else chunk_stds
                    )
                    dead_layers = [i for i, s in enumerate(layer_stds) if s < 1e-6 or not math.isfinite(s)]
                    print(
                        f"[Sanity Check Batch 0] Active layers: {len(layer_stds) - len(dead_layers)}/{len(layer_stds)} | "
                        f"Layer std range: [{min(layer_stds):.4f}, {max(layer_stds):.4f}] | "
                        f"Tapped {len(model.layers)}-layer std range: [{min(chunk_stds):.4f}, {max(chunk_stds):.4f}]",
                        flush=True,
                    )
                    if dead_layers:
                        raise RuntimeError(f"Batch 0 Sanity Check FAILED: Layers {dead_layers} have zero/NaN activations!")

                Z_tr_list.append(z_b.cpu().float().numpy())
    Z_train = np.vstack(Z_tr_list)
    Z_train_layers = np.vstack(Z_tr_layers_list) if Z_tr_layers_list else None
    head_tail_cos_tr = np.concatenate(tr_cos_list, axis=0) if tr_cos_list else None
    token_std_tr = np.concatenate(tr_std_list, axis=0) if tr_std_list else None
    y_train = np.array(train_labels, dtype=np.int64)
    print(
        f"[{target_family.upper()} Gate] Extracted Z_train: shape={Z_train.shape}, dtype={Z_train.dtype}"
        + (f", layers_shape={Z_train_layers.shape}" if Z_train_layers is not None else "")
    )

    Z_eval = None
    Z_eval_layers = None
    head_tail_cos_ev = None
    token_std_ev = None
    main_logits_eval = None
    main_logits_train = None
    centered_logits_train = None
    centered_logits_eval = None
    if eval_texts is not None and len(eval_texts) > 0:
        print(f"[{target_family.upper()} Gate] Extracting Z_eval on {len(eval_texts)} benchmark samples...")
        dummy_eval_y = [0] * len(eval_texts)
        ds_ev = TextSequenceDataset(eval_texts, dummy_eval_y, None, tok, max_len=eff_max_len, end_id=eff_end_id, cls_id=cls_id)
        loader_ev = DataLoader(ds_ev, batch_size=batch_size, shuffle=False, collate_fn=collate_fn_gate)
        Z_ev_list = []
        Z_ev_layers_list = []
        ev_cos_list = []
        ev_std_list = []
        with torch.no_grad():
            with torch.amp.autocast(device.type, enabled=(amp_dtype == torch.bfloat16), dtype=amp_dtype):
                for batch in loader_ev:
                    mb_ids = batch["input_ids"].to(device)
                    mb_mask = batch["attention_mask"].to(device)
                    max_l = max(int(mb_mask.sum(dim=1).max().item()), 2)
                    if need_layers:
                        res = model.forward_features(
                            mb_ids[:, :max_l],
                            mb_mask[:, :max_l],
                            drop_first_token=drop_first_token,
                            return_by_layer=True,
                            extract_all_layers=save_all_layers,
                            return_multiview=True,
                        )
                        if isinstance(res, tuple) and len(res) == 4:
                            z_b, z_l, ht_cos_t, std_norm_t = res
                            ev_cos_list.append(ht_cos_t.cpu().to(torch.float16).numpy())
                            ev_std_list.append(std_norm_t.cpu().to(torch.float16).numpy())
                        elif isinstance(res, tuple) and len(res) == 2:
                            z_b, z_l = res
                        Z_ev_layers_list.append(z_l.cpu().to(torch.float16).numpy())
                    else:
                        z_b = model.forward_features(mb_ids[:, :max_l], mb_mask[:, :max_l], drop_first_token=drop_first_token)
                    Z_ev_list.append(z_b.cpu().float().numpy())
        Z_eval = np.vstack(Z_ev_list)
        Z_eval_layers = np.vstack(Z_ev_layers_list) if Z_ev_layers_list else None
        head_tail_cos_ev = np.concatenate(ev_cos_list, axis=0) if ev_cos_list else None
        token_std_ev = np.concatenate(ev_std_list, axis=0) if ev_std_list else None

    if hasattr(model, "plain_head") and model.plain_head is not None:
        W_p = model.plain_head.weight.detach().cpu().float().numpy()
        b_p = model.plain_head.bias.detach().cpu().float().numpy()
        main_logits_train = Z_train @ W_p.T + b_p
        if Z_eval is not None:
            main_logits_eval = Z_eval @ W_p.T + b_p

    if hasattr(model, "centered_head") and model.centered_head is not None:
        W_c = model.centered_head.weight.detach().cpu().float().numpy()
        b_c = model.centered_head.bias.detach().cpu().float().numpy()
        centered_logits_train = Z_train @ W_c.T + b_c
        if Z_eval is not None:
            centered_logits_eval = Z_eval @ W_c.T + b_c

    if export_embeddings:
        emb_dir = Path(embeddings_output_dir) if embeddings_output_dir else (Path(output_dir) / "embeddings" if output_dir else Path("export_embeddings"))
        save_extracted_features_bundle(
            output_dir=emb_dir,
            target=target_family,
            features_concat=Z_train,
            features_by_layer=Z_train_layers,
            texts=train_texts,
            labels=train_labels,
            model_names=df_fam_all["model"].tolist() + ["__OTHER__"] * len(df_neg_all),
            prompt_ids=prompt_ids,
            champion_logits_plain=main_logits_train,
            champion_logits_centered=centered_logits_train,
            head_tail_cos=head_tail_cos_tr,
            token_std_norms=token_std_tr,
            eval_features_concat=Z_eval,
            eval_features_by_layer=Z_eval_layers,
            eval_texts=eval_texts,
            eval_labels=eval_labels,
            eval_champion_logits_plain=main_logits_eval,
            eval_champion_logits_centered=centered_logits_eval,
            eval_head_tail_cos=head_tail_cos_ev,
            eval_token_std_norms=token_std_ev,
            dataset_title=f"NTL {target_family.upper()} Extracted Features",
            dataset_slug=f"ntl-{target_family.lower()}-features",
            checkpoints_volume=checkpoints_volume,
        )
        if hasattr(model, "norms") and model.norms is not None:
            norms_dict = {}
            for i, norm in enumerate(model.norms):
                norms_dict[f"norms.{i}.weight"] = norm.weight.detach().cpu().to(torch.float32).contiguous()
                if hasattr(norm, "bias") and norm.bias is not None:
                    norms_dict[f"norms.{i}.bias"] = norm.bias.detach().cpu().to(torch.float32).contiguous()
            save_file(norms_dict, str(emb_dir / "heads.safetensors"))

    norms = np.linalg.norm(Z_train, axis=1)
    q25, q50, q75 = np.percentile(norms, [25, 50, 75])
    iqr = q75 - q25
    norm_outliers_3iqr = int(((norms < q25 - 3.0 * iqr) | (norms > q75 + 3.0 * iqr)).sum())
    norm_outliers_15iqr = int(((norms < q25 - 1.5 * iqr) | (norms > q75 + 1.5 * iqr)).sum())

    num_tapped = len(getattr(model, "layers", [2, 4, 6, 9, 13, 17, 22, 27]))
    chunk_dim = Z_train.shape[1] // max(num_tapped, 1)

    fam_mask_stats = y_train < num_family_classes
    if fam_mask_stats.sum() > 0:
        mu_fam = Z_train[fam_mask_stats].mean(axis=0, keepdims=True)
        dists = np.linalg.norm(Z_train[fam_mask_stats] - mu_fam, axis=1)
        dq25, dq50, dq75 = np.percentile(dists, [25, 50, 75])
        diqr = dq75 - dq25
        dist_outliers_3iqr = int((dists > dq75 + 3.0 * diqr).sum())
        dist_outliers_15iqr = int((dists > dq75 + 1.5 * diqr).sum())
    else:
        dists = np.zeros(len(Z_train))
        dq50, diqr = 0.0, 0.0
        dist_outliers_3iqr, dist_outliers_15iqr = 0, 0

    print("\n" + "=" * 80)
    print(f" [{target_family.upper()} Gate] Step 2: Quality Inspection, Normalization Check & Outlier Statistics:")
    print(f"  - Total training samples: {len(Z_train):,} ({len(df_fam_all)} intra-family across {num_family_classes} classes)")
    for i, c in enumerate(family_classes):
        print(f"    * Class {i} ({c}): {int((y_train == i).sum()):,} samples")
    print(f"  - Feature dimension: {Z_train.shape[1]:,} ({num_tapped} tapped layers x {chunk_dim}-dim LayerNormed)")
    print(f"  - Sample L2 norm stats: mean={norms.mean():.2f}, std={norms.std():.2f}, median={q50:.2f}, IQR={iqr:.2f}")
    print(f"  - Centroid distance stats: mean={dists.mean():.2f}, std={dists.std():.2f}, median={dq50:.2f}, IQR={diqr:.2f}")
    print(f"  - Outliers by norm (>1.5/3.0 IQR): {norm_outliers_15iqr}/{norm_outliers_3iqr} | dist: {dist_outliers_15iqr}/{dist_outliers_3iqr}")
    print("=" * 80 + "\n")

    if Z_train_layers is not None:
        hard_pair = (2, 3) if target_family == "gemini" else ((1, 2) if target_family == "gpt" else (0, 1))
        run_all_layers_probe_sweep(
            features_by_layer=Z_train_layers,
            y=y_train,
            num_family_classes=num_family_classes,
            hard_sister_pair=hard_pair,
            output_dir=output_dir,
            prompts=np.asarray(prompt_ids) if prompt_ids is not None else None,
        )

    subspace_components = None
    subspace_mean = None
    Z_tr_active = Z_train
    Z_ev_active = Z_eval

    if use_supervised_subspace:
        print(f"[{target_family.upper()} Gate] Step 3: Building Pure Supervised Sister-Decoupling Subspace...")
        Z_tr_active, Z_ev_active, subspace_components, subspace_mean, sub_diag = build_oof_supervised_family_subspace(
            Z_train=Z_train,
            y_train=y_train,
            Z_eval=Z_eval,
            num_family_classes=num_family_classes,
            num_tapped_layers=num_tapped,
            subspace_dim=subspace_dim,
            device=device,
            include_velocity=include_velocity,
            layer_weighting=layer_weighting,
            hard_sister_pair=hard_sister_pair,
            prompt_ids=prompt_ids,
        )
        print(f"[{target_family.upper()} Gate] Step 3 Complete: {sub_diag['total_subspace_dim']} whitened dims.")

    pca_components = subspace_components
    pca_mean = subspace_mean

    y_ev_arr = np.array(eval_labels) if eval_labels is not None else None
    fam_eval_mask = np.isin(y_ev_arr, family_cols) if y_ev_arr is not None else None
    col_to_local = {c: i for i, c in enumerate(family_cols)}
    y_fam_true = np.array([col_to_local[v] for v in y_ev_arr[fam_eval_mask]]) if fam_eval_mask is not None else None

    acc_plain_within = 0.0
    if main_logits_eval is not None and fam_eval_mask is not None and fam_eval_mask.sum() > 0:
        if main_logits_eval.shape[1] == len(LABELS):
            plain_fam_pred = main_logits_eval[fam_eval_mask][:, family_cols].argmax(axis=1)
        else:
            plain_fam_pred = main_logits_eval[fam_eval_mask][:, :num_family_classes].argmax(axis=1)
        acc_plain_within = float(balanced_accuracy_score(y_fam_true, plain_fam_pred))

    best_w_pca, best_b_pca, ridge_alpha, ridge_acc, ridge_logits_ev, _ = fit_family_gate_head(
        Z_train=Z_tr_active,
        y_train=y_train,
        num_gate_classes=num_gate_classes,
        alphas=[1.0, 2.0, 5.0, 10.0, 25.0, 50.0, 100.0, 200.0],
        label_smoothing=0.10,
        Z_eval=Z_ev_active,
        y_eval_true=y_fam_true,
        eval_mask=fam_eval_mask,
        num_family_classes=num_family_classes,
        device=device,
        prompt_ids=prompt_ids,
    )

    if pca_components is not None:
        comp_64 = pca_components.astype(np.float64)
        mu_64 = pca_mean.reshape(1, -1).astype(np.float64)
        W_r_eff_64 = comp_64.T @ best_w_pca.astype(np.float64)
        b_r_eff_64 = best_b_pca.astype(np.float64) - (mu_64 @ W_r_eff_64).squeeze(0)
        best_w = W_r_eff_64.astype(np.float32)
        best_b = b_r_eff_64.astype(np.float32)
    else:
        best_w = best_w_pca
        best_b = best_b_pca

    plain_w_tensor = torch.from_numpy(best_w.T).to(torch.float32).contiguous()
    plain_b_tensor = torch.from_numpy(best_b).to(torch.float32).contiguous()

    mlp_head_dict, mlp_acc, mlp_logits_ev, _ = fit_family_mlp_head(
        Z_train=Z_tr_active,
        y_train=y_train,
        num_gate_classes=num_gate_classes,
        num_family_classes=num_family_classes,
        hidden_dim=hidden_dim,
        epochs=mlp_epochs,
        lr=mlp_lr,
        dropout=mlp_dropout,
        label_smoothing=0.10,
        Z_eval=Z_ev_active,
        y_eval_true=y_fam_true,
        eval_mask=fam_eval_mask,
        device=device,
        pca_components=pca_components,
        pca_mean=pca_mean,
        raw_Z_eval=Z_eval if pca_components is not None else None,
    )

    hybrid_logits_ev = None
    if mlp_logits_ev is not None and ridge_logits_ev is not None:
        raw_hyb = 0.65 * mlp_logits_ev + 0.35 * ridge_logits_ev
        hyb_fam = raw_hyb[:, :num_family_classes]
        hyb_std = float(np.std(hyb_fam, axis=1).mean()) if len(hyb_fam) > 0 else 1.0
        hyb_scale = (3.2 / hyb_std) if hyb_std > 1e-4 else 1.0
        hybrid_logits_ev = (raw_hyb * hyb_scale).astype(np.float32)

    if head_type.lower() == "mlp":
        best_fam_acc = mlp_acc
        best_fam_logits = mlp_logits_ev
        best_alpha = ridge_alpha
        final_head_dict = mlp_head_dict
        final_head_dict["plain.weight"] = plain_w_tensor
        final_head_dict["plain.bias"] = plain_b_tensor
    elif head_type.lower() in ("hybrid", "hybrid_65mlp_35ridge"):
        if hybrid_logits_ev is not None and fam_eval_mask is not None and fam_eval_mask.sum() > 0:
            fam_hyb_pred = hybrid_logits_ev[fam_eval_mask, :num_family_classes].argmax(axis=1)
            hybrid_acc = float(balanced_accuracy_score(y_fam_true, fam_hyb_pred))
        else:
            hybrid_acc = mlp_acc
        best_fam_acc = hybrid_acc
        best_fam_logits = hybrid_logits_ev if hybrid_logits_ev is not None else mlp_logits_ev
        best_alpha = ridge_alpha
        final_head_dict = mlp_head_dict
        final_head_dict["plain.weight"] = plain_w_tensor
        final_head_dict["plain.bias"] = plain_b_tensor
    else:
        best_fam_acc = ridge_acc
        best_fam_logits = ridge_logits_ev
        best_alpha = ridge_alpha
        final_head_dict = {"plain.weight": plain_w_tensor, "plain.bias": plain_b_tensor}

    default_gw = float(train_cfg.get("gate_weight", gate_weight)) if isinstance(train_cfg, dict) else float(gate_weight)
    best_pipe_w = default_gw
    pipe_results = {}
    if main_logits_eval is not None and best_fam_logits is not None and y_ev_arr is not None and main_logits_eval.shape[1] == len(LABELS):
        P_main = sp_softmax(main_logits_eval, axis=1)
        acc_plain_solo = float(balanced_accuracy_score(y_ev_arr, P_main.argmax(axis=1)))
        best_pipe_acc = 0.0
        cand_mask = np.isin(P_main.argmax(axis=1), family_cols) | (P_main[:, family_cols].sum(axis=1) >= 0.10)
        spec_probs = sp_softmax(best_fam_logits[:, :num_family_classes], axis=1)
        sweep_weights = sorted(set([0.10, 0.20, 0.30, 0.35, 0.40, 0.45, 0.50, 0.60, default_gw]))

        for gw in sweep_weights:
            P_fused = P_main.copy()
            for idx in np.flatnonzero(cand_mask):
                orig_mass = float(P_main[idx, family_cols].sum())
                if orig_mass > 1e-6:
                    gw_eff = gw * min(orig_mass / 0.50, 1.0)
                    main_norm = P_main[idx, family_cols] / orig_mass
                    fused_fam = (1.0 - gw_eff) * main_norm + gw_eff * spec_probs[idx]
                    P_fused[idx, family_cols] = fused_fam * orig_mass

            P_sk = sinkhorn_balancing(P_fused, iters=50, sinkhorn_strength=0.65, target_mix=0.35)
            acc_gated_sk = float(balanced_accuracy_score(y_ev_arr, P_sk.argmax(axis=1)))
            if acc_gated_sk > best_pipe_acc:
                best_pipe_acc = acc_gated_sk
                best_pipe_w = gw

        pipe_results = {
            "plain_solo_bacc": acc_plain_solo,
            "best_pipe_bacc": best_pipe_acc,
            "best_gate_weight": best_pipe_w,
            "lift_over_plain": best_pipe_acc - acc_plain_solo,
        }

    hard_pair_eval_acc = float(best_fam_acc)
    pair_check = hard_sister_pair
    if pair_check is None:
        if num_family_classes == 4:
            pair_check = (2, 3)
        elif num_family_classes == 5:
            pair_check = (1, 2)
        elif num_family_classes == 2:
            pair_check = (0, 1)

    if y_fam_true is not None and best_fam_logits is not None and pair_check is not None and fam_eval_mask is not None and fam_eval_mask.sum() > 0:
        s_a, s_b = pair_check
        pair_mask_ev = (y_fam_true == s_a) | (y_fam_true == s_b)
        if pair_mask_ev.sum() > 0 and len(np.unique(y_fam_true[pair_mask_ev])) == 2:
            fam_logits = best_fam_logits[fam_eval_mask, :num_family_classes]
            spec_pair_preds = np.where(fam_logits[pair_mask_ev, s_a] >= fam_logits[pair_mask_ev, s_b], s_a, s_b)
            hard_pair_eval_acc = float(balanced_accuracy_score(y_fam_true[pair_mask_ev], spec_pair_preds))

    if output_dir is not None:
        out_p = Path(output_dir)
        out_p.mkdir(parents=True, exist_ok=True)
        if hasattr(model, "norms") and model.norms is not None:
            for i, norm in enumerate(model.norms):
                final_head_dict[f"norms.{i}.weight"] = norm.weight.detach().cpu().to(torch.float32).contiguous()
                if hasattr(norm, "bias") and norm.bias is not None:
                    final_head_dict[f"norms.{i}.bias"] = norm.bias.detach().cpu().to(torch.float32).contiguous()
        save_file(final_head_dict, str(out_p / "heads.safetensors"))
        meta_fam = {
            "classes": family_classes + (["__OTHER__"] if has_negatives else []),
            "labels": family_classes + (["__OTHER__"] if has_negatives else []),
            "target_family": target_family,
            "head_type": head_type.lower(),
            "default_gate_weight": default_gw,
            "best_sweep_gate_weight": float(best_pipe_w),
            "use_supervised_subspace": use_supervised_subspace,
            "subspace_dim": int(pca_components.shape[0]) if pca_components is not None else None,
            "pca_dim": pca_dim,
            "hidden_dim": hidden_dim if ("mlp" in head_type.lower() or "hybrid" in head_type.lower()) else None,
            "alpha": float(best_alpha),
            "ridge_alpha": float(ridge_alpha),
            "ridge_within_family_acc": float(ridge_acc),
            "mlp_within_family_acc": float(mlp_acc),
            "within_family_acc": float(best_fam_acc),
            "plain_within_family_acc": float(acc_plain_within),
            "hard_pair_eval_acc": float(hard_pair_eval_acc),
            "lift_over_plain": float(best_fam_acc - acc_plain_within),
        }
        (out_p / "meta.json").write_text(json.dumps(meta_fam, indent=2), encoding="utf-8")
        if checkpoints_volume is not None and hasattr(checkpoints_volume, "commit"):
            checkpoints_volume.commit()
        print(f"[{target_family.upper()} Gate] Checkpoint saved to {out_p}.")

    try:
        del Z_train, Z_train_layers, Z_eval, Z_eval_layers, Z_tr_active, Z_ev_active
    except Exception:
        pass
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    should_sweep = export_embeddings and (train_cfg.get("run_sweep", True) if isinstance(train_cfg, dict) else True)
    if should_sweep:
        print(f"\n[{target_family.upper()} Gate] Automatically launching offline feature & head sweep...")
        run_sweep(
            embeddings_dir=emb_dir,
            target_family=target_family,
            output_dir=output_dir,
            ledger_dir=Path(output_dir) / "experiments" if output_dir else "experiments",
            device=device,
        )

    return {
        "target_family": target_family,
        "within_family_balanced_acc": float(best_fam_acc),
        "plain_within_family_acc": float(acc_plain_within),
        "delta_within_family": float(best_fam_acc - acc_plain_within),
        "hard_pair_eval_acc": float(hard_pair_eval_acc),
        "best_alpha": float(best_alpha),
        "ridge_acc": float(ridge_acc),
        "mlp_acc": float(mlp_acc),
        "subspace_dim": int(pca_components.shape[0]) if pca_components is not None else 0,
        "pipe_results": pipe_results,
        "spec_logits": best_fam_logits[:, :num_family_classes].tolist() if best_fam_logits is not None else [],
        "spec_classes": family_classes,
    }


def run_sweep(
    embeddings_dir: Path | str,
    target_family: str,
    output_dir: Path | str | None = None,
    ledger_dir: Path | str = "experiments",
    device: torch.device | None = None,
    exp_prefix: str = "EXP",
) -> dict[str, Any]:
    """Executes offline ablation sweep across feature engineering variants and heads in <30s total."""
    ds_p = Path(embeddings_dir)
    target_family = target_family.lower().strip()
    if output_dir is None:
        output_dir = Path(f"export_{target_family}_gate_mlp")
    out_p = Path(output_dir)
    out_p.mkdir(parents=True, exist_ok=True)

    ledger = ExperimentLedger(ledger_dir=ledger_dir)

    print("\n" + "=" * 92)
    print(f" [RUNNER] STARTING OFFLINE GATE EXPERIMENT SWEEP: '{target_family.upper()}' ({ds_p})")
    print("=" * 92)

    family_cols = FAMILIES[target_family]
    eff_hard_pair = (2, 3) if target_family == "gemini" else ((1, 2) if target_family in ("gpt", "claude") else (0, 1))

    y_ev_arr = None
    main_logits_eval = None
    baseline_raw_bacc = 0.0
    baseline_sk_bacc = 0.0
    fam_solo_bacc = 0.0
    hard_pair_solo_acc = 0.0

    if (ds_p / "benchmark_metadata.parquet").exists():
        df_bench = pd.read_parquet(ds_p / "benchmark_metadata.parquet")
        y_ev_arr = df_bench["label"].to_numpy()

    if (ds_p / "benchmark_champion_logits.safetensors").exists():
        champ_ev_sd = load_file(str(ds_p / "benchmark_champion_logits.safetensors"))
        if "logits_plain" in champ_ev_sd:
            main_logits_eval = champ_ev_sd["logits_plain"].cpu().float().numpy()
        elif "champion_logits" in champ_ev_sd:
            main_logits_eval = champ_ev_sd["champion_logits"].cpu().float().numpy()
        elif len(champ_ev_sd) > 0 and any("logit" in k.lower() for k in champ_ev_sd):
            for k in champ_ev_sd:
                if "logit" in k.lower():
                    main_logits_eval = champ_ev_sd[k].cpu().float().numpy()
                    break

    if y_ev_arr is not None and main_logits_eval is not None:
        if main_logits_eval.shape[1] == len(LABELS):
            P_main = sp_softmax(main_logits_eval, axis=1)
            baseline_raw_bacc = float(balanced_accuracy_score(y_ev_arr, P_main.argmax(axis=1)))
            P_sk = sinkhorn_balancing(P_main, iters=50, sinkhorn_strength=0.65, target_mix=0.35)
            baseline_sk_bacc = float(balanced_accuracy_score(y_ev_arr, P_sk.argmax(axis=1)))

            fam_eval_mask = np.isin(y_ev_arr, family_cols)
            if fam_eval_mask.sum() > 0:
                col_to_local = {c: i for i, c in enumerate(family_cols)}
                y_fam_true = np.array([col_to_local[v] for v in y_ev_arr[fam_eval_mask]])
                plain_fam_pred = main_logits_eval[fam_eval_mask][:, family_cols].argmax(axis=1)
                fam_solo_bacc = float(balanced_accuracy_score(y_fam_true, plain_fam_pred))

                s_a, s_b = eff_hard_pair
                pair_mask = (y_fam_true == s_a) | (y_fam_true == s_b)
                if pair_mask.sum() > 0 and len(np.unique(y_fam_true[pair_mask])) == 2:
                    pair_preds = np.where(
                        main_logits_eval[fam_eval_mask][pair_mask, family_cols[s_a]]
                        >= main_logits_eval[fam_eval_mask][pair_mask, family_cols[s_b]],
                        s_a,
                        s_b,
                    )
                    hard_pair_solo_acc = float(balanced_accuracy_score(y_fam_true[pair_mask], pair_preds))
                else:
                    hard_pair_solo_acc = fam_solo_bacc
        else:
            fam_eval_mask = np.isin(y_ev_arr, family_cols)
            if fam_eval_mask.sum() > 0:
                col_to_local = {c: i for i, c in enumerate(family_cols)}
                y_fam_true = np.array([col_to_local[v] for v in y_ev_arr[fam_eval_mask]])
                plain_fam_pred = main_logits_eval[fam_eval_mask][:, : len(family_cols)].argmax(axis=1)
                fam_solo_bacc = float(balanced_accuracy_score(y_fam_true, plain_fam_pred))

                s_a, s_b = eff_hard_pair
                pair_mask = (y_fam_true == s_a) | (y_fam_true == s_b)
                if pair_mask.sum() > 0 and len(np.unique(y_fam_true[pair_mask])) == 2:
                    pair_preds = np.where(
                        main_logits_eval[fam_eval_mask][pair_mask, s_a] >= main_logits_eval[fam_eval_mask][pair_mask, s_b],
                        s_a,
                        s_b,
                    )
                    hard_pair_solo_acc = float(balanced_accuracy_score(y_fam_true[pair_mask], pair_preds))
                else:
                    hard_pair_solo_acc = fam_solo_bacc

    ledger.baseline_bacc = baseline_sk_bacc
    exp00_entry = {
        "exp_id": "EXP-00",
        "phase": "P1_baseline",
        "target_family": target_family,
        "feature_set": "F0_plain_solo",
        "subspace_dim": 0,
        "head_type": "plain",
        "gate_weight": 0.0,
        "oof_family_bacc": None,
        "holdout_family_bacc": fam_solo_bacc,
        "hard_sister_acc": hard_pair_solo_acc,
        "holdout_18cls_raw_bacc": baseline_raw_bacc,
        "holdout_18cls_sinkhorn_bacc": baseline_sk_bacc,
        "delta_vs_baseline_bacc": 0.0,
        "notes": "18-class Champion Plain Head solo baseline on benchmark holdout",
    }
    existing_df = ledger.get_dataframe()
    if existing_df is None or existing_df.empty or "exp_id" not in existing_df.columns or "EXP-00" not in existing_df["exp_id"].values:
        ledger.add_entry(exp00_entry)
    print(f"EXP-00 Complete: Solo BAcc={baseline_raw_bacc*100:.2f}% | Sinkhorn={baseline_sk_bacc*100:.2f}% | Family BAcc={fam_solo_bacc*100:.2f}%")

    pfx = exp_prefix if exp_prefix else "EXP"
    sep = "_" if ("_" in pfx or "-" in pfx) else "-"
    feature_specs = [
        (f"{pfx}{sep}01", "F1_fisher_per_layer", False, False, None, False, "Pure Per-Layer Fisher Subspace"),
        (f"{pfx}{sep}02", "F2_fisher_velocity", True, False, None, False, "+ Inter-Layer Velocity Fisher Subspace"),
        (f"{pfx}{sep}03", "F3_scalarmix_fisher", True, True, None, False, "+ ScalarMix Layer Importance Weighting"),
    ]
    has_by_layer = False
    if (ds_p / "features_by_layer.safetensors").exists():
        try:
            with safe_open(str(ds_p / "features_by_layer.safetensors"), framework="pt") as f:
                shp = f.get_slice("features_by_layer").get_shape()
                has_by_layer = len(shp) == 3 and shp[1] > 8
        except Exception:
            has_by_layer = False
    if has_by_layer:
        feature_specs.append((f"{pfx}{sep}04", "F4_auto_top8_fisher", True, True, "auto_top8", False, "+ Auto-Top-8 Layer Slicing"))
    feature_specs.append((f"{pfx}{sep}05", "F5_fisher_multiview", True, True, None, True, "+ Offline Multi-View Features (Upper Bound)"))

    sweep_results = []
    best_foldable_score = -1.0
    best_foldable_run = None
    suffix_map = {"ridge": "a", "mlp": "b", "hybrid": "c"}

    for base_id, feat_set, inc_vel, l_wt, sel_l, use_mv, desc in feature_specs:
        print(f"\n--- [{base_id}] Running {feat_set} ({desc}) ---")
        tmp_run_out = out_p / f"_tmp_{base_id.lower()}"
        try:
            res = train_family_gate_from_dataset(
                dataset_dir=ds_p,
                target_family=target_family,
                output_dir=tmp_run_out,
                head_type="hybrid",
                select_layers=sel_l,
                include_velocity=inc_vel,
                layer_weighting=l_wt,
                use_multiview=use_mv,
                device=device,
                gate_weight=0.40,
            )

            heads_bd = res.get("heads_breakdown", {})
            for h_type in ["ridge", "mlp", "hybrid"]:
                exp_id = f"{base_id}{suffix_map[h_type]}"
                h_info = heads_bd.get(h_type, {})
                p_res = h_info.get("pipe_results", {})
                hld_18_raw = p_res.get("best_raw_bacc", 0.0)
                hld_18_sk = p_res.get("best_pipe_bacc", 0.0)
                gw_best = p_res.get("best_gate_weight", 0.40)
                delta_lift = hld_18_sk - baseline_sk_bacc

                record = {
                    "exp_id": exp_id,
                    "phase": "P2_features" if h_type == "ridge" else "P3_heads",
                    "target_family": target_family,
                    "feature_set": feat_set,
                    "subspace_dim": res.get("subspace_dim", 0),
                    "head_type": h_type,
                    "gate_weight": gw_best,
                    "oof_family_bacc": h_info.get("oof_acc"),
                    "holdout_family_bacc": h_info.get("eval_fam_acc", 0.0),
                    "hard_sister_acc": h_info.get("hard_pair_acc", 0.0),
                    "holdout_18cls_raw_bacc": hld_18_raw,
                    "holdout_18cls_sinkhorn_bacc": hld_18_sk,
                    "delta_vs_baseline_bacc": delta_lift,
                    "notes": f"{desc} ({h_type.upper()}, gw={gw_best:.2f})",
                }
                arts = []
                if (tmp_run_out / "meta.json").exists():
                    arts.append(tmp_run_out / "meta.json")
                if (tmp_run_out / "layer_probe_report.json").exists():
                    arts.append(tmp_run_out / "layer_probe_report.json")
                ledger.add_entry(record, artifacts=arts)
                sweep_results.append((record, tmp_run_out))

                if sel_l is None and not use_mv and hld_18_sk > best_foldable_score:
                    best_foldable_score = hld_18_sk
                    best_foldable_run = (exp_id, tmp_run_out, record)

                print(
                    f"[{exp_id}] Result: Head={h_type.upper()} | Fam BAcc={record['holdout_family_bacc']*100:.2f}% | "
                    f"Hard Sis={record['hard_sister_acc']*100:.2f}% | 18c SK={hld_18_sk*100:.2f}% (d {delta_lift*100:+.2f}%)"
                )

        except Exception as e:
            print(f"[{base_id}] Encountered error: {e}")

    if best_foldable_run is not None:
        best_id, best_tmp_dir, best_rec = best_foldable_run
        print("\n" + "=" * 92)
        print(f" [WINNER] Selecting Best Foldable Checkpoint: {best_id} ({best_rec['feature_set']} + {best_rec['head_type']})")
        print("=" * 92)

        for src_f in best_tmp_dir.glob("*"):
            if src_f.is_file():
                shutil.copy2(str(src_f), str(out_p / src_f.name))
        meta_p = out_p / "meta.json"
        if meta_p.exists():
            try:
                m_data = json.loads(meta_p.read_text(encoding="utf-8"))
                m_data["winning_exp_id"] = best_id
                m_data["head_type"] = best_rec["head_type"]
                m_data["best_sweep_gate_weight"] = float(best_rec["gate_weight"])
                meta_p.write_text(json.dumps(m_data, indent=2), encoding="utf-8")
            except Exception:
                pass

    for _, tmp_dir in sweep_results:
        if tmp_dir.exists():
            try:
                shutil.rmtree(str(tmp_dir))
            except Exception:
                pass

    ledger.export_copy(out_p)
    print(f"\n[RUNNER] Sweep Complete. Exported champion gate and ledgers to: {out_p}")
    return {
        "target_family": target_family,
        "baseline_sk_bacc": baseline_sk_bacc,
        "best_foldable_score": best_foldable_score,
        "best_run": best_foldable_run[0] if best_foldable_run else None,
        "total_experiments": len(sweep_results) + 1,
    }


In [ ]:
# === MODULE: src/namethatllm/training/backbones/qwen.py ===
"""
Qwen Backbone Adapter (Qwen3-14B, Qwen2.5-14B, Qwen3-8B) and FP16 / Norm Stabilization Helpers.
"""

from __future__ import annotations

import copy
import types
from pathlib import Path
from typing import Any

import torch
from torch import nn



def _disable_final_norm(mod: nn.Module) -> None:
    """Replaces an autoregressive LM's final LayerNorm / RMSNorm with nn.Identity()."""
    for path in ["norm", "model.norm", "base_model.norm", "base_model.model.norm", "base_model.model.model.norm"]:
        curr = mod
        parts = path.split(".")
        for p in parts[:-1]:
            curr = getattr(curr, p, None)
            if curr is None:
                break
        if curr is not None and hasattr(curr, parts[-1]) and getattr(curr, parts[-1]) is not None:
            setattr(curr, parts[-1], nn.Identity())


def _stabilize_fp16_backbone(mod: nn.Module) -> None:
    """Prevents float16 overflow (>65,504 -> Inf -> NaN) in Qwen2.5/Qwen3 on Turing T4 GPUs."""
    for _, sub in mod.named_modules():
        cls_name = sub.__class__.__name__.lower()
        if "rmsnorm" in cls_name and not getattr(sub, "_fp32_norm_patched", False):
            orig_norm_fwd = sub.forward

            def _safe_rms_fwd(self_norm, x, _orig=orig_norm_fwd):
                x_clean = torch.nan_to_num(x.float(), nan=0.0, posinf=65000.0, neginf=-65000.0)
                variance = x_clean.pow(2).mean(-1, keepdim=True)
                x_normed = x_clean * torch.rsqrt(variance + getattr(self_norm, "variance_epsilon", 1e-6))
                w = self_norm.weight
                return (w.float() * x_normed).to(x.dtype)

            sub.forward = types.MethodType(_safe_rms_fwd, sub)
            sub._fp32_norm_patched = True

        if (
            hasattr(sub, "gate_proj")
            and hasattr(sub, "up_proj")
            and hasattr(sub, "down_proj")
            and hasattr(sub, "act_fn")
            and not getattr(sub, "_fp32_mlp_patched", False)
        ):
            def _safe_mlp_fwd(self_mlp, x):
                in_dtype = x.dtype
                x_safe = torch.nan_to_num(x, nan=0.0, posinf=65000.0, neginf=-65000.0)
                gate_f = self_mlp.act_fn(self_mlp.gate_proj(x_safe).float())
                up_f = self_mlp.up_proj(x_safe).float()
                inter_f = gate_f * up_f
                max_val = inter_f.abs().amax(dim=-1, keepdim=True).clamp(min=1.0)
                scale = (max_val / 16384.0).clamp(min=1.0).detach()
                inter_scaled = (inter_f / scale).to(in_dtype)
                down_out = self_mlp.down_proj(inter_scaled).float() * scale
                return torch.clamp(down_out, min=-65000.0, max=65000.0).to(in_dtype)

            sub.forward = types.MethodType(_safe_mlp_fwd, sub)
            sub._fp32_mlp_patched = True

        if "decoderlayer" in cls_name and not getattr(sub, "_fp16_clamp_hooked", False):
            def _clamp_layer_out(module, inputs, output):
                if isinstance(output, torch.Tensor):
                    return torch.nan_to_num(output, nan=0.0, posinf=65000.0, neginf=-65000.0).clamp(-65000.0, 65000.0)
                if isinstance(output, tuple) and len(output) > 0 and isinstance(output[0], torch.Tensor):
                    h0 = torch.nan_to_num(output[0], nan=0.0, posinf=65000.0, neginf=-65000.0).clamp(-65000.0, 65000.0)
                    return (h0,) + output[1:]
                return output

            sub.register_forward_hook(_clamp_layer_out)
            sub._fp16_clamp_hooked = True


@register_backbone("qwen")
class QwenAdapter(BaseBackboneAdapter):
    """Strategy adapter for Qwen causal decoder backbones (Qwen3-14B, Qwen2.5-14B, Qwen3-8B)."""

    family_name = "qwen"
    default_backbone = "Qwen/Qwen3-14B-Base"
    default_output_dir = "export_champion_qwen3_14b"
    default_keep_layers = 27
    default_layers = (2, 4, 6, 9, 13, 17, 22, 27)
    default_max_len = 384
    default_end_id = 151643
    preserves_working_checkpoint = True

    def default_lora_targets(self) -> list[str]:
        return ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

    def resolve_model_source(self, cfg: dict[str, Any], input_dir: Path) -> str:
        backbone_name = str(cfg.get("backbone", self.default_backbone))
        local_base = None
        if input_dir.exists():
            b_lower = backbone_name.lower()
            if "2.5" in b_lower or "qwen25" in b_lower:
                local_base = find_base_model(input_dir, "qwen2", 5120, 48) or find_base_model(input_dir, "qwen2", 3584, 28)
            elif "14b" in b_lower:
                local_base = find_base_model(input_dir, "qwen3", 5120, 40) or find_base_model(input_dir, "qwen2", 5120, 48)
            elif "8b" in b_lower:
                local_base = find_base_model(input_dir, "qwen3", 4096, 36)
        return str(local_base) if local_base else backbone_name

    def resolve_precision(self, cfg: dict[str, Any], device: torch.device) -> PrecisionSpec:
        base = super().resolve_precision(cfg, device)
        n_gpus = torch.cuda.device_count()
        has_high_vram_single_gpu = bool(
            torch.cuda.is_available() and n_gpus == 1 and torch.cuda.get_device_properties(0).total_memory >= 40 * 1024**3
        )
        use_fa2 = False
        if has_high_vram_single_gpu and base.use_bf16:
            try:
                import flash_attn  # noqa: F401

                use_fa2 = True
            except ImportError:
                use_fa2 = False
        return PrecisionSpec(
            compute_dtype=base.compute_dtype,
            use_bf16=base.use_bf16,
            use_fa2=use_fa2,
            total_vram_gb=base.total_vram_gb,
        )

    def tok_spec_from_tokenizer(
        self, tok: Any, max_len: int | None = None, end_id: int | None = None
    ) -> TokenizationSpec:
        tok_pad = getattr(tok, "pad_token_id", None)
        pad_id = int(tok_pad) if tok_pad is not None else self.default_end_id
        eff_end_id = int(end_id) if end_id is not None else self.default_end_id
        try:
            vocab_sz = len(tok)
            if eff_end_id >= vocab_sz:
                tok_eos = getattr(tok, "eos_token_id", None)
                tok_sep = getattr(tok, "sep_token_id", None)
                if tok_eos is not None:
                    eff_end_id = int(tok_eos)
                elif tok_sep is not None:
                    eff_end_id = int(tok_sep)
                else:
                    eff_end_id = 2
        except Exception:
            pass
        return TokenizationSpec(
            max_len=int(max_len if max_len is not None else self.default_max_len),
            pad_id=pad_id,
            cls_id=None,
            end_id=eff_end_id,
            drop_first_token=True,
        )

    def load_tokenizer(self, model_source: str, cfg: dict[str, Any], is_smoke: bool = False) -> tuple[Any, TokenizationSpec]:
        from transformers import AutoTokenizer

        mock_tok = cfg.get("mock_tokenizer")
        if mock_tok is not None:
            tok = mock_tok
        else:
            try:
                tok = AutoTokenizer.from_pretrained(model_source, trust_remote_code=True, use_fast=True)
                if tok.pad_token is None:
                    tok.pad_token = tok.eos_token
            except Exception:
                if is_smoke:
                    class _FallbackTok:
                        pad_token_id = 0
                        cls_token_id = None
                        sep_token_id = None
                        eos_token_id = 2
                        pad_token = "<pad>"
                        eos_token = "</s>"

                        def __call__(self, text, **kwargs):
                            return {"input_ids": [ord(c) % 500 + 10 for c in str(text)[:30]]}

                        def __len__(self):
                            return 128100

                    tok = _FallbackTok()
                else:
                    raise
        max_len = int(cfg.get("max_len", self.default_max_len))
        cfg_end = cfg.get("end_id")
        end_id = int(cfg_end) if cfg_end is not None else self.default_end_id
        return tok, self.tok_spec_from_tokenizer(tok, max_len=max_len, end_id=end_id)

    def load_backbone(
        self, model_source: str, cfg: dict[str, Any], precision: PrecisionSpec, device: torch.device
    ) -> tuple[nn.Module, int, int, list[int]]:
        from transformers import AutoConfig, AutoModel

        keep_layers = int(cfg.get("keep_layers", self.default_keep_layers))
        layers = list(cfg.get("layers", self.default_layers))
        mock_bb = cfg.get("mock_backbone")
        if mock_bb is not None:
            base_model = copy.deepcopy(mock_bb).to(device)
            base_cfg = getattr(base_model, "config", type("Config", (), {"hidden_size": 32, "num_hidden_layers": 24})())
            return base_model, int(base_cfg.hidden_size), keep_layers, layers

        base_cfg = AutoConfig.from_pretrained(model_source, trust_remote_code=True)
        max_backbone_layers = int(getattr(base_cfg, "num_hidden_layers", self.default_keep_layers))
        keep_layers = min(keep_layers, max_backbone_layers)
        if any(l > keep_layers for l in layers):
            layers = [l for l in layers if l <= keep_layers]
        base_cfg.num_hidden_layers = keep_layers
        base_cfg.use_cache = False
        if getattr(base_cfg, "layer_types", None) is not None:
            base_cfg.layer_types = base_cfg.layer_types[:keep_layers]

        extra_kwargs: dict[str, Any] = {}
        if precision.use_fa2:
            extra_kwargs["attn_implementation"] = "flash_attention_2"

        n_gpus = torch.cuda.device_count()
        has_high_vram_single_gpu = bool(
            torch.cuda.is_available() and n_gpus == 1 and torch.cuda.get_device_properties(0).total_memory >= 40 * 1024**3
        )
        if has_high_vram_single_gpu:
            print(f"[Trainer] High-VRAM single GPU detected ({precision.total_vram_gb:.1f} GiB). Loading directly on {device} in {precision.compute_dtype}...")
            base_model = AutoModel.from_pretrained(
                model_source, config=base_cfg, torch_dtype=precision.compute_dtype, low_cpu_mem_usage=True, trust_remote_code=True, **extra_kwargs
            ).to(device)
        elif n_gpus >= 2:
            max_memory = {0: "10.0GiB", 1: "13.5GiB"}
            base_model = AutoModel.from_pretrained(
                model_source, config=base_cfg, torch_dtype=precision.compute_dtype, device_map="auto", max_memory=max_memory, low_cpu_mem_usage=True, trust_remote_code=True
            )
        elif n_gpus == 1:
            base_model = AutoModel.from_pretrained(
                model_source, config=base_cfg, torch_dtype=precision.compute_dtype, device_map="auto", low_cpu_mem_usage=True, trust_remote_code=True
            )
        else:
            base_model = AutoModel.from_pretrained(
                model_source, config=base_cfg, torch_dtype=torch.float32, low_cpu_mem_usage=True, trust_remote_code=True
            )
        return base_model, int(base_cfg.hidden_size), keep_layers, layers

    def prepare_backbone(
        self, base_model: nn.Module, precision: PrecisionSpec, cfg: dict[str, Any], is_eval_only: bool
    ) -> nn.Module:
        _disable_final_norm(base_model)
        if (
            precision.compute_dtype == torch.float16
            and cfg.get("mock_backbone") is None
            and bool(cfg.get("stabilize_fp16", True))
        ):
            _stabilize_fp16_backbone(base_model)
            print("[Trainer] Stabilized FP16 backbone with detached SwiGLU dynamic scaling and residual clamping.")
        return base_model

    def build_scalers(self, precision: PrecisionSpec, device: torch.device) -> tuple[Any | None, Any | None]:
        enabled = bool(precision.compute_dtype == torch.float16 and device.type == "cuda" and torch.cuda.is_available())
        return None, _ManualFP16LossScaler(enabled=enabled)

    def adjust_stage1_batch_config(
        self, stage1_cfg: dict[str, Any], precision: PrecisionSpec
    ) -> tuple[int, int, int]:
        bs1 = int(stage1_cfg.get("batch_size", 2))
        accum1 = int(stage1_cfg.get("gradient_accumulation_steps", 8))
        is_low_vram = torch.cuda.is_available() and precision.total_vram_gb < 24.0
        if is_low_vram and bs1 > 4:
            old_bs1 = bs1
            bs1 = 2
            accum1 = max(accum1 * (old_bs1 // 2), 8)
            print(f"[Trainer] Low VRAM GPU ({precision.total_vram_gb:.1f} GiB) detected: adjusted Stage 1 batch_size from {old_bs1} to {bs1}, accum_steps to {accum1}")
        default_eval_bs = 4 if is_low_vram else 8
        eval_bs = int(stage1_cfg.get("eval_batch_size", default_eval_bs))
        return bs1, accum1, eval_bs

    def clamp_extraction_batch_size(self, batch_size: int) -> int:
        if torch.cuda.is_available():
            vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
            if vram_gb < 18.0 and batch_size > 8:
                print(f"[Features Extraction] Low VRAM GPU ({vram_gb:.1f} GiB) detected: clamping batch_size from {batch_size} to 8.")
                return 8
        return batch_size


In [ ]:
# === MODULE: src/namethatllm/training/backbones/deberta.py ===
"""
DeBERTa Backbone Adapter (DeBERTa-v3-large) and Specialist Experiment Suite.
"""

from __future__ import annotations

import copy
import gc
import importlib
import json
import shutil
from pathlib import Path
from typing import Any

import pandas as pd
import torch
from safetensors.torch import load_file
from scipy.special import softmax as sp_softmax
from sklearn.metrics import balanced_accuracy_score
from torch import nn
from transformers import AutoConfig, AutoModel, AutoTokenizer, PretrainedConfig



@register_backbone("deberta")
class DebertaAdapter(BaseBackboneAdapter):
    """Strategy adapter for DeBERTa encoder backbones (microsoft/deberta-v3-large)."""

    family_name = "deberta"
    default_backbone = "microsoft/deberta-v3-large"
    default_output_dir = "export_champion_deberta"
    default_keep_layers = 24
    default_layers = (3, 6, 9, 12, 15, 18, 21, 24)
    default_max_len = 512
    default_end_id = 0

    def default_lora_targets(self) -> list[str]:
        return ["query_proj", "key_proj", "value_proj", "pos_key_proj", "pos_query_proj", "dense"]

    def resolve_model_source(self, cfg: dict[str, Any], input_dir: Path) -> str:
        backbone_name = str(cfg.get("backbone", self.default_backbone))
        if input_dir.exists():
            for cfg_p in input_dir.rglob("config.json"):
                try:
                    c_data = json.loads(cfg_p.read_text(encoding="utf-8"))
                    if "deberta" in str(c_data.get("model_type", "")).lower():
                        return str(cfg_p.parent)
                except Exception:
                    continue
        return backbone_name

    def tok_spec_from_tokenizer(
        self, tok: Any, max_len: int | None = None, end_id: int | None = None
    ) -> TokenizationSpec:
        tok_pad = getattr(tok, "pad_token_id", None)
        pad_id = int(tok_pad) if tok_pad is not None else 0
        raw_cls = getattr(tok, "cls_token_id", None)
        cls_id = int(raw_cls) if raw_cls is not None else None
        raw_sep = getattr(tok, "sep_token_id", None)
        if raw_sep is not None:
            eff_end_id = int(raw_sep)
        elif end_id is not None:
            eff_end_id = int(end_id)
        else:
            eff_end_id = self.default_end_id
        try:
            vocab_sz = len(tok)
            if eff_end_id >= vocab_sz:
                tok_eos = getattr(tok, "eos_token_id", None)
                tok_sep = getattr(tok, "sep_token_id", None)
                if tok_eos is not None:
                    eff_end_id = int(tok_eos)
                elif tok_sep is not None:
                    eff_end_id = int(tok_sep)
                else:
                    eff_end_id = 2
        except Exception:
            pass
        return TokenizationSpec(
            max_len=int(max_len if max_len is not None else self.default_max_len),
            pad_id=pad_id,
            cls_id=cls_id,
            end_id=eff_end_id,
            drop_first_token=False,
        )

    def load_tokenizer(self, model_source: str, cfg: dict[str, Any], is_smoke: bool = False) -> tuple[Any, TokenizationSpec]:
        mock_tok = cfg.get("mock_tokenizer")
        if mock_tok is not None:
            tok = mock_tok
        else:
            try:
                tok = AutoTokenizer.from_pretrained(model_source, trust_remote_code=True, use_fast=True)
                if tok.pad_token is None:
                    tok.pad_token = tok.eos_token
            except Exception:
                if is_smoke:
                    class _FallbackTok:
                        pad_token_id = 0
                        cls_token_id = 1
                        sep_token_id = 2
                        eos_token_id = 2
                        pad_token = "<pad>"
                        eos_token = "</s>"

                        def __call__(self, text, **kwargs):
                            return {"input_ids": [ord(c) % 500 + 10 for c in str(text)[:30]]}

                        def __len__(self):
                            return 128100

                    tok = _FallbackTok()
                else:
                    raise
        max_len = int(cfg.get("max_len", self.default_max_len))
        cfg_end = cfg.get("end_id")
        end_id = int(cfg_end) if cfg_end is not None else self.default_end_id
        return tok, self.tok_spec_from_tokenizer(tok, max_len=max_len, end_id=end_id)

    def load_backbone(
        self, model_source: str, cfg: dict[str, Any], precision: PrecisionSpec, device: torch.device
    ) -> tuple[nn.Module, int, int, list[int]]:
        keep_layers = int(cfg.get("keep_layers", self.default_keep_layers))
        layers = list(cfg.get("layers", self.default_layers))
        mock_bb = cfg.get("mock_backbone")
        if mock_bb is not None:
            base_model = copy.deepcopy(mock_bb).to(device)
            base_cfg = getattr(base_model, "config", type("Config", (), {"hidden_size": 32, "num_hidden_layers": 24})())
            return base_model, int(base_cfg.hidden_size), keep_layers, layers

        base_cfg = AutoConfig.from_pretrained(model_source, trust_remote_code=True)
        max_backbone_layers = int(getattr(base_cfg, "num_hidden_layers", 24))
        keep_layers = min(keep_layers, max_backbone_layers, 24)
        if any(l > keep_layers for l in layers) or len(layers) == 0:
            layers = list(self.default_layers)
        base_cfg.num_hidden_layers = keep_layers
        base_cfg.use_cache = False

        print(f"[Trainer] DeBERTa model detected. Loading in torch.float32 directly on {device} (bypassing device_map='auto')...")
        base_model = AutoModel.from_pretrained(
            model_source, config=base_cfg, torch_dtype=torch.float32, low_cpu_mem_usage=True, trust_remote_code=True
        ).to(device)
        return base_model, int(base_cfg.hidden_size), keep_layers, layers

    def build_scalers(self, precision: PrecisionSpec, device: torch.device) -> tuple[Any | None, Any | None]:
        enabled = bool(precision.compute_dtype == torch.float16 and device.type == "cuda" and torch.cuda.is_available())
        return torch.amp.GradScaler("cuda", enabled=enabled), None

    def adjust_stage1_batch_config(
        self, stage1_cfg: dict[str, Any], precision: PrecisionSpec
    ) -> tuple[int, int, int]:
        bs1 = int(stage1_cfg.get("batch_size", 2))
        accum1 = int(stage1_cfg.get("gradient_accumulation_steps", 8))
        eval_bs = int(stage1_cfg.get("eval_batch_size", 32))
        return bs1, accum1, eval_bs

    def stage2_eval_batch_size(self) -> int:
        return 16

    def specialist_phase_tag(self, target_family: str) -> str:
        return "P3_deberta_finetuned"

    def specialist_feature_set_tag(self, target_family: str) -> str:
        return "F6b_finetuned_deberta_direct"

    def specialist_notes_label(self) -> str:
        return "DeBERTa"

    def maybe_run_pipeline(
        self, train_cfg: dict[str, Any], step_callback: Any | None = None, **kwargs: Any
    ) -> Any | None:
        if train_cfg.get("pipeline_mode") == "deberta_specialist_suite":
            return run_deberta_specialist_suite(train_cfg, step_callback=step_callback, run_pipeline_fn=kwargs.get("run_pipeline_fn"))
        return None


def run_deberta_specialist_suite(
    train_cfg: dict[str, Any],
    step_callback: Any = None,
    run_pipeline_fn: Any = None,
) -> dict[str, Any]:
    """Executes the High-Sample DeBERTa-v3-Large Specialist Experiment Suite (EXP-06a, EXP-06b, EXP-06c)."""
    if run_pipeline_fn is None:
        run_pipeline_fn = globals().get("run_training_pipeline") or importlib.import_module("namethatllm.training.engine").run_training_pipeline

    adapter = DebertaAdapter()
    backbone_name = str(train_cfg.get("backbone", adapter.default_backbone))
    target_family = str(train_cfg.get("target_family", "gemini")).lower().strip()
    keep_layers = min(int(train_cfg.get("keep_layers", 24)), 24)
    layers = list(train_cfg.get("layers", adapter.default_layers))
    if any(l > keep_layers for l in layers):
        layers = list(adapter.default_layers)
    max_len = int(train_cfg.get("max_len", 446))
    samples_per_class = int(train_cfg.get("samples_per_class", 8500))
    negative_ratio = float(train_cfg.get("negative_ratio", 0.10))
    batch_size = int(train_cfg.get("batch_size", 32))
    out_dir = Path(train_cfg.get("output_dir", "export_deberta_gemini_specialist"))
    gate_out_dir = Path(train_cfg.get("gate_output_dir", "export_deberta_gemini_gate_mlp"))
    emb_out_dir = Path(train_cfg.get("embeddings_output_dir", "export_embeddings_deberta_finetuned"))
    input_dir = Path(train_cfg.get("input_dir", "/kaggle/input"))
    is_smoke = bool(train_cfg.get("smoke", False))
    device = torch.device(train_cfg.get("device", "cuda:0" if torch.cuda.is_available() else "cpu"))
    checkpoints_volume = train_cfg.get("checkpoints_volume", None)
    ledger_dir = Path(train_cfg.get("ledger_dir", "experiments"))
    ledger_dir.mkdir(parents=True, exist_ok=True)
    ledger = ExperimentLedger(ledger_dir=ledger_dir)

    fam_cols = FAMILIES[target_family]
    fam_classes = [LABELS[i] for i in fam_cols]
    num_fam_classes = len(fam_classes)
    eff_hard_pair = (2, 3) if target_family == "gemini" else ((1, 2) if target_family in ("gpt", "claude") else (0, 1))

    print("\n" + "=" * 92)
    print(f" [DEBERTA SPECIALIST SUITE] High-Sample ({samples_per_class}/class) Suite for '{target_family.upper()}'")
    print(f" Output Specialist: {out_dir} | Output Gate: {gate_out_dir} | Embeddings: {emb_out_dir}")
    print("=" * 92)

    bench_paths = list(input_dir.rglob("clean_benchmark_1800_rows.parquet"))
    if not bench_paths and Path("clean_benchmark_1800_rows.parquet").exists():
        bench_paths = [Path("clean_benchmark_1800_rows.parquet")]

    df_bench = None
    if bench_paths and bench_paths[0].exists():
        try:
            df_bench = pd.read_parquet(bench_paths[0])
        except Exception as e:
            print(f"[DeBERTa Suite] Note reading benchmark parquet: {e}")

    if df_bench is None or len(df_bench) == 0:
        if is_smoke:
            syn_records = [{"text": f"Benchmark sample passage text for {m} sample {rep} test", "model": m} for m in LABELS for rep in range(2)]
            df_bench = pd.DataFrame(syn_records)
        else:
            raise FileNotFoundError("clean_benchmark_1800_rows.parquet not found for DeBERTa specialist suite.")

    eval_texts = df_bench["text"].tolist()
    eval_labels = [LAB2I[m] for m in df_bench["model"].tolist()]
    eval_model_names = df_bench["model"].tolist()
    bench_hashes = {_text_hash(t) for t in eval_texts}
    print(f"[DeBERTa Suite] Loaded clean benchmark with {len(eval_texts)} passages ({len(bench_hashes)} unique hashes).")

    champ_paths = (
        list(input_dir.rglob("benchmark_champion_logits.safetensors"))
        + list(input_dir.rglob("champion_logits.safetensors"))
        + list(Path(".").rglob("*champion_logits.safetensors"))
    )
    champ_main_logits = None
    baseline_raw_bacc = 0.0
    baseline_sk_bacc = 0.0
    fam_solo_bacc = 0.0
    hard_pair_solo_acc = 0.0

    if champ_paths:
        try:
            champ_sd = load_file(str(champ_paths[0]))
            c_key = "logits_plain" if "logits_plain" in champ_sd else list(champ_sd.keys())[0]
            cand_l = champ_sd[c_key].cpu().float().numpy()
            if len(cand_l) == len(eval_texts) and cand_l.shape[1] == len(LABELS):
                champ_main_logits = cand_l
                P_main = sp_softmax(champ_main_logits, axis=1)
                baseline_raw_bacc = float(balanced_accuracy_score(eval_labels, P_main.argmax(axis=1)))
                P_sk = sinkhorn_balancing(P_main, iters=50, sinkhorn_strength=0.65, target_mix=0.35)
                baseline_sk_bacc = float(balanced_accuracy_score(eval_labels, P_sk.argmax(axis=1)))

                fam_eval_mask = np.isin(eval_labels, fam_cols)
                if fam_eval_mask.sum() > 0:
                    y_fam_true = np.array([fam_cols.index(c) for c in np.array(eval_labels)[fam_eval_mask]])
                    solo_fam_pred = champ_main_logits[fam_eval_mask][:, fam_cols].argmax(axis=1)
                    fam_solo_bacc = float(balanced_accuracy_score(y_fam_true, solo_fam_pred))
                    pair_mask = np.isin(y_fam_true, [eff_hard_pair[0], eff_hard_pair[1]])
                    if pair_mask.sum() > 0 and len(np.unique(y_fam_true[pair_mask])) == 2:
                        s_a, s_b = eff_hard_pair
                        fam_logits_pair = champ_main_logits[fam_eval_mask][pair_mask][:, fam_cols]
                        pair_preds = np.where(fam_logits_pair[:, s_a] >= fam_logits_pair[:, s_b], s_a, s_b)
                        hard_pair_solo_acc = float(balanced_accuracy_score(y_fam_true[pair_mask], pair_preds))
                print(f"[DeBERTa Suite] Baseline Champion: Raw={baseline_raw_bacc*100:.2f}% | Sinkhorn={baseline_sk_bacc*100:.2f}% | Gemini Solo={fam_solo_bacc*100:.2f}%")
        except Exception as e:
            print(f"[DeBERTa Suite] Note on reading champion logits: {e}")

    ledger.add_entry({
        "exp_id": "EXP-00",
        "phase": "P0_baseline",
        "target_family": target_family,
        "feature_set": "F0_plain_solo",
        "subspace_dim": 0,
        "head_type": "plain",
        "gate_weight": 0.0,
        "oof_family_bacc": 0.0,
        "holdout_family_bacc": fam_solo_bacc,
        "hard_sister_acc": hard_pair_solo_acc,
        "holdout_18cls_raw_bacc": baseline_raw_bacc,
        "holdout_18cls_sinkhorn_bacc": baseline_sk_bacc,
        "delta_vs_baseline_bacc": 0.0,
        "notes": "18-Class In-House Champion Plain Solo Baseline",
    })

    train_path, _, pool_file = resolve_dataset_paths(input_dir if input_dir.exists() else Path("."))
    all_pools = resolve_all_pool_files(input_dir if input_dir.exists() else Path("."))

    df_train = None
    if train_path and train_path.exists():
        try:
            df_train = pd.read_csv(train_path)
            tr_hashes = [_text_hash(t) for t in df_train["text"].fillna("")]
            df_train = df_train[[h not in bench_hashes for h in tr_hashes]].reset_index(drop=True)
            print(f"[DeBERTa Suite] Retained {len(df_train)} clean non-quarantined train.csv rows.")
        except Exception as e:
            print(f"[DeBERTa Suite] Note reading train_path: {e}")

    if df_train is None or len(df_train) == 0:
        if is_smoke:
            syn_tr = [{"text": f"Train passage for {m} prompt {rep} text content", "model": m, "prompt_id": f"p_{rep}"} for m in LABELS for rep in range(3)]
            df_train = pd.DataFrame(syn_tr)
        else:
            raise FileNotFoundError("train.csv not found for specialist suite.")

    dfs_pool = []
    pool_list = all_pools if all_pools else ([pool_file] if pool_file else [])
    for p in pool_list:
        try:
            df_one = pd.read_parquet(p, columns=["text", "model"]) if str(p).endswith(".parquet") else pd.read_csv(p, usecols=["text", "model"])
            dfs_pool.append(df_one)
        except Exception:
            pass

    if dfs_pool:
        df_pool = pd.concat(dfs_pool, ignore_index=True)
    elif is_smoke:
        df_pool = pd.DataFrame([{"text": f"Pool sample passage text for {m} item {rep} with enough characters to be valid", "model": m} for m in LABELS for rep in range(8)])
    else:
        df_pool = pd.DataFrame(columns=["text", "model"])

    if len(df_pool) > 0:
        df_pool = df_pool.dropna(subset=["text", "model"])
        df_pool["text"] = df_pool["text"].astype(str)
        p_hashes = [_text_hash(t) for t in df_pool["text"]]
        df_pool = df_pool[[h not in bench_hashes for h in p_hashes]].reset_index(drop=True)
        print(f"[DeBERTa Suite] Retained {len(df_pool)} clean non-quarantined pool rows (0.0% overlap with benchmark).")

    model_source = adapter.resolve_model_source(train_cfg, input_dir)
    mock_backbone = train_cfg.get("mock_backbone")
    tok, tok_spec = adapter.load_tokenizer(model_source, {**train_cfg, "max_len": max_len}, is_smoke=is_smoke)

    def _make_fallback_deberta(h_dim: int = 32, n_l: int = 24) -> nn.Module:
        class _FallbackConfig(PretrainedConfig):
            model_type = "deberta-v2"

            def __init__(self, hidden_size=32, num_hidden_layers=24, **kwargs):
                super().__init__(**kwargs)
                self.hidden_size = hidden_size
                self.num_hidden_layers = num_hidden_layers

        class _FallbackDeberta(nn.Module):
            def __init__(self, hd=32, nl=24):
                super().__init__()
                self.config = _FallbackConfig(hidden_size=hd, num_hidden_layers=nl)
                self.embed = nn.Embedding(128100, hd)
                self.layers = nn.ModuleList([nn.Linear(hd, hd) for _ in range(nl)])

            def forward(self, input_ids, **kwargs):
                h = self.embed(input_ids)
                states = [h]
                for l in self.layers:
                    h = l(h)
                    states.append(h)
                return type("Output", (), {"hidden_states": states, "last_hidden_state": h})()

        return _FallbackDeberta(h_dim, n_l).to(device)

    # EXP-06a: Frozen Public DeBERTa
    print("\n" + "=" * 88)
    print(" [EXP-06a] Extracting Frozen Public DeBERTa Embeddings & Training Heads...")
    print("=" * 88)
    exp_6a_emb_dir = Path("export_embeddings_deberta_public_frozen")
    found_pre_6a = None
    if (exp_6a_emb_dir / "features_concat.safetensors").exists() and (exp_6a_emb_dir / "metadata.parquet").exists():
        found_pre_6a = exp_6a_emb_dir
    elif input_dir.exists():
        for cand in input_dir.rglob("features_concat.safetensors"):
            cand_dir = cand.parent
            if (cand_dir / "metadata.parquet").exists() and "deberta" in str(cand_dir).lower():
                found_pre_6a = cand_dir
                break

    if found_pre_6a is not None:
        exp_6a_emb_dir = found_pre_6a
    else:
        exp_6a_emb_dir.mkdir(parents=True, exist_ok=True)
        if mock_backbone is not None:
            base_model_frozen = mock_backbone
            hidden_sz = getattr(base_model_frozen.config, "hidden_size", 64)
        else:
            try:
                base_cfg_f = AutoConfig.from_pretrained(model_source)
                base_cfg_f.num_hidden_layers = keep_layers
                base_model_frozen = AutoModel.from_pretrained(model_source, config=base_cfg_f, torch_dtype=torch.float32).to(device)
                hidden_sz = base_cfg_f.hidden_size
            except Exception:
                if is_smoke:
                    base_model_frozen = _make_fallback_deberta(32, keep_layers)
                    hidden_sz = 32
                else:
                    raise

        model_frozen = MultiLayerLoRAClassifier(
            base_model_frozen,
            hidden_size=hidden_sz,
            layers=layers,
            num_classes=num_fam_classes,
            backbone_family=adapter.family_name,
            tok_spec=tok_spec,
        ).to(device)
        model_frozen.eval()

        extract_and_export_embeddings(
            model=model_frozen,
            tok=tok,
            df_pool=df_pool,
            df_train=df_train,
            target=target_family,
            save_all_layers=False,
            save_multiview=False,
            output_dir=exp_6a_emb_dir,
            samples_per_class=min(samples_per_class, 8 if is_smoke else samples_per_class),
            negative_ratio=negative_ratio,
            eval_texts=eval_texts,
            eval_labels=eval_labels,
            eval_model_names=eval_model_names,
            batch_size=min(batch_size, 4 if is_smoke else batch_size),
            max_len=min(max_len, 64 if is_smoke else max_len),
            end_id=tok_spec.end_id,
            device=device,
            eval_champion_logits_plain=champ_main_logits,
        )

    res_6a = train_family_gate_from_dataset(
        dataset_dir=exp_6a_emb_dir,
        target_family=target_family,
        output_dir=None,
        mlp_epochs=2 if is_smoke else int(train_cfg.get("mlp_epochs", 12)),
        gate_weight=float(train_cfg.get("gate_weight", 0.40)),
        device=device,
    )

    heads_bk = res_6a.get("heads_breakdown", {})
    for h_name in ["ridge", "mlp", "hybrid"]:
        if h_name in heads_bk:
            h_info = heads_bk[h_name]
            p_res = h_info.get("pipe_results", {})
            h_sk = p_res.get("best_sinkhorn_bacc", 0.0)
            h_raw = p_res.get("best_raw_bacc", 0.0)
            gw_val = p_res.get("best_gate_weight", 0.40)
            ledger.add_entry({
                "exp_id": f"EXP-06a_{h_name}",
                "phase": "P2_deberta_frozen",
                "target_family": target_family,
                "feature_set": "F6a_frozen_public_deberta",
                "subspace_dim": res_6a.get("subspace_dim", 0),
                "head_type": h_name,
                "gate_weight": gw_val,
                "oof_family_bacc": h_info.get("oof_acc", 0.0),
                "holdout_family_bacc": h_info.get("eval_fam_acc", 0.0),
                "hard_sister_acc": h_info.get("hard_pair_acc", 0.0),
                "holdout_18cls_raw_bacc": h_raw,
                "holdout_18cls_sinkhorn_bacc": h_sk,
                "delta_vs_baseline_bacc": (h_sk - baseline_sk_bacc) if baseline_sk_bacc > 0 else 0.0,
                "notes": f"Frozen Public DeBERTa + OOF Fisher ({h_name.upper()}, gw={gw_val:.2f})",
            })

    if "model_frozen" in locals():
        del model_frozen
    if "base_model_frozen" in locals():
        del base_model_frozen
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    if found_pre_6a is None:
        shutil.rmtree(str(exp_6a_emb_dir), ignore_errors=True)

    # EXP-06b: Fine-Tuned DeBERTa Specialist Model
    print("\n" + "=" * 88)
    print(" [EXP-06b] Fine-Tuning DeBERTa-v3-Large Specialist Model...")
    print("=" * 88)
    cfg_b = copy.deepcopy(train_cfg)
    cfg_b["pipeline_mode"] = "specialist"
    cfg_b["backbone_family"] = "deberta"
    cfg_b["output_dir"] = str(out_dir)
    cfg_b["exp_prefix"] = "EXP-06b"
    if is_smoke:
        cfg_b["smoke"] = True
        s1 = cfg_b.setdefault("stage1", {})
        s1["epochs"] = 1
        s1["max_samples"] = 16
        s2 = cfg_b.setdefault("stage2", {})
        s2["epochs"] = 1
        s2["n_splits"] = 2
    run_pipeline_fn(cfg_b, step_callback=step_callback)

    # EXP-06c: Fine-Tuned Embeddings + Probing Sweep
    print("\n" + "=" * 88)
    print(" [EXP-06c] Extracting Multi-Layer Embeddings from Fine-Tuned Specialist...")
    print("=" * 88)
    emb_out_dir.mkdir(parents=True, exist_ok=True)
    if mock_backbone is not None:
        merged_backbone = mock_backbone
        h_sz = getattr(merged_backbone.config, "hidden_size", 64)
    else:
        try:
            base_cfg_c = AutoConfig.from_pretrained(model_source)
            base_cfg_c.num_hidden_layers = keep_layers
            base_c = AutoModel.from_pretrained(model_source, config=base_cfg_c, torch_dtype=torch.float32).to(device)
            from peft import PeftModel

            adapter_load_dir = out_dir / "adapter" if (out_dir / "adapter" / "adapter_config.json").exists() else out_dir
            peft_m = PeftModel.from_pretrained(base_c, str(adapter_load_dir))
            merged_backbone = peft_m.merge_and_unload()
            h_sz = base_cfg_c.hidden_size
        except Exception:
            if is_smoke:
                merged_backbone = _make_fallback_deberta(32, keep_layers)
                h_sz = 32
            else:
                raise

    model_finetuned = MultiLayerLoRAClassifier(
        merged_backbone,
        hidden_size=h_sz,
        layers=layers,
        num_classes=num_fam_classes,
        backbone_family=adapter.family_name,
        tok_spec=tok_spec,
    ).to(device)
    heads_ckpt = out_dir / "heads.safetensors"
    if heads_ckpt.exists():
        load_and_slice_champion_heads(heads_ckpt, model_finetuned, fam_cols=None)
    model_finetuned.eval()

    extract_and_export_embeddings(
        model=model_finetuned,
        tok=tok,
        df_pool=df_pool,
        df_train=df_train,
        target=target_family,
        save_all_layers=True,
        save_multiview=True,
        output_dir=emb_out_dir,
        samples_per_class=min(samples_per_class, 8 if is_smoke else samples_per_class),
        negative_ratio=negative_ratio,
        eval_texts=eval_texts,
        eval_labels=eval_labels,
        eval_model_names=eval_model_names,
        batch_size=min(batch_size, 4 if is_smoke else batch_size),
        max_len=min(max_len, 64 if is_smoke else max_len),
        end_id=tok_spec.end_id,
        device=device,
        eval_champion_logits_plain=champ_main_logits,
    )

    del model_finetuned, merged_backbone
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    sweep_summary = run_sweep(
        embeddings_dir=emb_out_dir,
        target_family=target_family,
        output_dir=gate_out_dir,
        ledger_dir=ledger_dir,
        device=device,
        exp_prefix="EXP-06c",
    )

    ledger.export_copy(out_dir)
    ledger.export_copy(gate_out_dir)
    return {
        "suite": "deberta_specialist_suite",
        "specialist_dir": str(out_dir),
        "gate_dir": str(gate_out_dir),
        "embeddings_dir": str(emb_out_dir),
        "sweep_summary": sweep_summary,
    }


In [ ]:
# === MODULE: src/namethatllm/training/backbones/bge.py ===
"""
BGE Backbone Adapter (BAAI/bge-m3), Export Context Layer Restorer, and Encoder Ridge Packager.
"""

from __future__ import annotations

import copy
import gc
import importlib
import json
import shutil
from collections.abc import Iterator
from contextlib import contextmanager
from pathlib import Path
from typing import Any

import numpy as np
import torch
from peft import LoraConfig, PeftModel, get_peft_model
from safetensors.torch import load_file, save_file
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import GroupKFold
from torch import nn
from transformers import AutoConfig, AutoModel, AutoTokenizer



@register_backbone("bge")
class BgeAdapter(BaseBackboneAdapter):
    """Strategy adapter for bidirectional BGE encoder backbones (BAAI/bge-m3)."""

    family_name = "bge"
    default_backbone = "BAAI/bge-m3"
    default_output_dir = "checkpoints/export_bge_m3_v2"
    default_keep_layers = 24
    default_layers = (4, 7, 10, 13, 16, 19, 22, 24)
    default_max_len = 512
    default_end_id = 2

    def default_lora_targets(self) -> list[str]:
        return ["query", "key", "value", "dense"]

    def resolve_model_source(self, cfg: dict[str, Any], input_dir: Path) -> str:
        backbone_name = str(cfg.get("backbone", self.default_backbone))
        if input_dir.exists():
            for cfg_p in input_dir.rglob("config.json"):
                try:
                    c_data = json.loads(cfg_p.read_text(encoding="utf-8"))
                    m_type = str(c_data.get("model_type", "")).lower()
                    if "xlm-roberta" in m_type or "bge" in str(cfg_p.parent).lower():
                        return str(cfg_p.parent)
                except Exception:
                    continue
        return backbone_name

    def tok_spec_from_tokenizer(
        self, tok: Any, max_len: int | None = None, end_id: int | None = None
    ) -> TokenizationSpec:
        tok_pad = getattr(tok, "pad_token_id", None)
        pad_id = int(tok_pad) if tok_pad is not None else 0
        raw_cls = getattr(tok, "cls_token_id", None)
        cls_id = int(raw_cls) if raw_cls is not None else None
        if end_id is not None:
            raw_end = end_id
        else:
            tok_eos = getattr(tok, "eos_token_id", None)
            tok_sep = getattr(tok, "sep_token_id", None)
            if tok_eos is not None:
                raw_end = tok_eos
            elif tok_sep is not None:
                raw_end = tok_sep
            else:
                raw_end = self.default_end_id
        eff_end_id = int(raw_end)
        return TokenizationSpec(
            max_len=int(max_len if max_len is not None else self.default_max_len),
            pad_id=pad_id,
            cls_id=cls_id,
            end_id=eff_end_id,
            drop_first_token=True,
        )

    def load_tokenizer(self, model_source: str, cfg: dict[str, Any], is_smoke: bool = False) -> tuple[Any, TokenizationSpec]:
        mock_tok = cfg.get("mock_tokenizer")
        if mock_tok is not None:
            tok = mock_tok
        else:
            try:
                tok = AutoTokenizer.from_pretrained(model_source, trust_remote_code=True, use_fast=True)
                if tok.pad_token is None and tok.eos_token is not None:
                    tok.pad_token = tok.eos_token
            except Exception:
                if is_smoke:
                    class _FallbackTok:
                        pad_token_id = 0
                        cls_token_id = 0
                        sep_token_id = 2
                        eos_token_id = 2
                        pad_token = "<pad>"
                        eos_token = "</s>"

                        def __call__(self, text, **kwargs):
                            return {"input_ids": [ord(c) % 500 + 10 for c in str(text)[:30]]}

                        def __len__(self):
                            return 250002

                    tok = _FallbackTok()
                else:
                    raise
        arch_cfg = cfg.get("architecture", {})
        max_len = int(cfg.get("max_len", self.default_max_len))
        end_id = cfg.get("end_id", None)
        spec = self.tok_spec_from_tokenizer(tok, max_len=max_len, end_id=end_id)
        drop_first = bool(arch_cfg.get("drop_first_token", cfg.get("drop_first_token", True)))
        if drop_first != spec.drop_first_token:
            spec = TokenizationSpec(
                max_len=spec.max_len,
                pad_id=spec.pad_id,
                cls_id=spec.cls_id,
                end_id=spec.end_id,
                drop_first_token=drop_first,
            )
        return tok, spec

    def load_backbone(
        self, model_source: str, cfg: dict[str, Any], precision: PrecisionSpec, device: torch.device
    ) -> tuple[nn.Module, int, int, list[int]]:
        arch_cfg = cfg.get("architecture", {})
        layers = [int(x) for x in arch_cfg.get("tap_layers", cfg.get("layers", self.default_layers))]
        keep_layers = int(cfg.get("keep_layers", max(layers) if layers else self.default_keep_layers))
        mock_bb = cfg.get("mock_backbone")
        if mock_bb is not None:
            base_model = copy.deepcopy(mock_bb).to(device)
            base_cfg = getattr(base_model, "config", type("Config", (), {"hidden_size": int(cfg.get("hidden_size", 32)), "num_hidden_layers": keep_layers})())
            return base_model, int(getattr(base_cfg, "hidden_size", cfg.get("hidden_size", 32))), keep_layers, layers

        base_cfg = AutoConfig.from_pretrained(model_source, trust_remote_code=True)
        dtype = torch.bfloat16 if precision.use_bf16 else torch.float32
        base_model = AutoModel.from_pretrained(model_source, config=base_cfg, torch_dtype=dtype, trust_remote_code=True).to(device)
        return base_model, int(base_cfg.hidden_size), keep_layers, layers

    def prepare_backbone(
        self, base_model: nn.Module, precision: PrecisionSpec, cfg: dict[str, Any], is_eval_only: bool
    ) -> nn.Module:
        arch_cfg = cfg.get("architecture", {})
        tap_layers = [int(x) for x in arch_cfg.get("tap_layers", cfg.get("layers", self.default_layers))]
        max_active_block = max(tap_layers) if tap_layers else int(cfg.get("keep_layers", self.default_keep_layers))
        encoder = getattr(base_model, "encoder", None)
        if encoder is not None and hasattr(encoder, "layer") and len(encoder.layer) > max_active_block:
            base_model._orig_encoder_layers = encoder.layer
            base_model._max_active_block = max_active_block
            encoder.layer = nn.ModuleList(list(encoder.layer)[:max_active_block])
        return base_model

    @staticmethod
    def _find_encoder_holder(backbone: nn.Module) -> nn.Module | None:
        cands = [
            backbone,
            getattr(backbone, "base_model", None),
            getattr(getattr(backbone, "base_model", None), "model", None),
        ]
        if hasattr(backbone, "get_base_model"):
            try:
                cands.append(backbone.get_base_model())
            except Exception:
                pass
        for cand in cands:
            if cand is not None and hasattr(cand, "_orig_encoder_layers") and hasattr(cand, "encoder"):
                return cand
        return None

    @contextmanager
    def export_context(self, backbone: nn.Module) -> Iterator[None]:
        holder = self._find_encoder_holder(backbone)
        if holder is None:
            yield
            return
        orig_layers = holder._orig_encoder_layers
        max_active = int(getattr(holder, "_max_active_block", len(orig_layers)))
        holder.encoder.layer = orig_layers
        try:
            yield
        finally:
            holder.encoder.layer = nn.ModuleList(list(orig_layers)[:max_active])

    def apply_lora(self, base_model: nn.Module, cfg: dict[str, Any], is_eval_only: bool) -> nn.Module:
        lora_block = cfg.get("lora")
        if not isinstance(lora_block, dict):
            return super().apply_lora(base_model, cfg, is_eval_only=is_eval_only)

        lora_cfg = LoraConfig(
            r=int(lora_block.get("r", 32)),
            lora_alpha=int(lora_block.get("alpha", lora_block.get("lora_alpha", 64))),
            lora_dropout=float(lora_block.get("dropout", 0.05)),
            bias="none",
            target_modules=list(lora_block.get("target_modules", self.default_lora_targets())),
            layers_to_transform=[int(i) for i in lora_block["layers_to_transform"]] if "layers_to_transform" in lora_block else None,
            layers_pattern=str(lora_block.get("layers_pattern", "layer")) if "layers_to_transform" in lora_block else None,
        )
        peft_model = get_peft_model(base_model, lora_cfg)
        for p in peft_model.parameters():
            if p.requires_grad and p.dtype != torch.float32:
                p.data = p.data.float()
        if is_eval_only:
            peft_model.eval()
        return peft_model

    def build_scalers(self, precision: PrecisionSpec, device: torch.device) -> tuple[Any | None, Any | None]:
        enabled = bool(precision.compute_dtype == torch.float16 and device.type == "cuda" and torch.cuda.is_available())
        return torch.amp.GradScaler("cuda", enabled=enabled), None

    def maybe_run_pipeline(
        self, train_cfg: dict[str, Any], step_callback: Any | None = None, **kwargs: Any
    ) -> Any | None:
        pipeline_mode = train_cfg.get("pipeline_mode")
        if pipeline_mode in ("encoder_stage1", "bge_v2_stage1") or (
            pipeline_mode is None and "stage1" in train_cfg and "stage2" not in train_cfg
        ):
            run_encoder_stage1_fn = kwargs.get("run_encoder_stage1_fn")
            if run_encoder_stage1_fn is None:
                run_encoder_stage1_fn = globals().get("run_encoder_stage1") or importlib.import_module("namethatllm.training.engine").run_encoder_stage1

            input_dir = Path(train_cfg.get("input_dir", "/kaggle/input"))
            out_dir = Path(train_cfg.get("output_dir", self.default_output_dir))
            checkpoints_volume = train_cfg.get("checkpoints_volume", None)
            res_enc = run_encoder_stage1_fn(
                config=train_cfg,
                datasets_dir=Path(train_cfg.get("datasets_dir", input_dir)),
                checkpoint_base=out_dir if "output_dir" in train_cfg else None,
                on_checkpoint_saved=getattr(checkpoints_volume, "commit", None),
            )
            return Path(res_enc.get("checkpoint_dir", out_dir))
        return None


def package_encoder_ridge_checkpoint(
    cfg: dict[str, Any],
    datasets_dir: Path,
    checkpoint_root: Path | None = None,
    on_checkpoint_saved: Any = None,
) -> dict[str, Any]:
    """Merges an encoder Stage-1 LoRA checkpoint to FP16 and calibrates Plain + Centered trace-scaled Ridge heads."""
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    tap_layers = [int(x) for x in cfg["architecture"]["tap_layers"]]
    max_len = int(cfg["max_len"])
    hidden_size = int(cfg["hidden_size"])
    backbone_name = str(cfg["backbone"])

    ckpt_root = Path(checkpoint_root) if checkpoint_root is not None else Path(cfg.get("output", {}).get("checkpoint_dir", "checkpoints/export_bge_m3_v2"))
    stage1_dir = ckpt_root / "stage1_best"
    final_dir = ckpt_root / "final"
    if final_dir.exists():
        shutil.rmtree(final_dir)
    final_dir.mkdir(parents=True, exist_ok=True)

    tok = AutoTokenizer.from_pretrained(backbone_name)
    base_f32 = AutoModel.from_pretrained(backbone_name, torch_dtype=torch.float32)
    peft_m = PeftModel.from_pretrained(base_f32, str(stage1_dir / "adapter"))
    merged = peft_m.merge_and_unload().half()
    merged.save_pretrained(str(final_dir), safe_serialization=True)
    tok.save_pretrained(str(final_dir))
    del peft_m, base_f32, merged
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    reloaded = AutoModel.from_pretrained(str(final_dir), torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32).to(device).eval()
    stage1_heads = load_file(str(stage1_dir / "heads.safetensors"), device="cpu")
    model = MultiLayerLoRAClassifier(backbone=reloaded, hidden_size=hidden_size, layers=tap_layers, num_classes=K, backbone_family="bge").to(device).eval()
    model.norms.float()
    with torch.no_grad():
        for i in range(len(tap_layers)):
            model.norms[i].weight.copy_(stage1_heads[f"norms.{i}.weight"].to(device, dtype=torch.float32))
            model.norms[i].bias.copy_(stage1_heads[f"norms.{i}.bias"].to(device, dtype=torch.float32))
    W_neural = stage1_heads["plain.weight"].float().numpy().astype(np.float32)
    b_neural = stage1_heads["plain.bias"].float().numpy().astype(np.float32)

    df_pool, df_train, df_bench = load_zero_leak_stage1_pool(datasets_dir)
    y_bench = np.array([LAB2I[m] for m in df_bench["model"]], dtype=np.int64)
    y_gold = np.array([LAB2I[m] for m in df_train["model"]], dtype=np.int64)
    prompts_gold = df_train["prompt_id"].values
    pool_labels = df_pool["label_idx"].to_numpy()

    rng = np.random.RandomState(42)
    n_per = max(1, int(cfg.get("output", {}).get("ridge_probe_samples", 36000)) // K)
    probe_idxs = np.concatenate([rng.choice(np.where(pool_labels == c_i)[0], size=min(n_per, int((pool_labels == c_i).sum())), replace=False) for c_i in range(K)])
    Z_bench, _ = extract_features_and_logits(model, df_bench["text"].astype(str).tolist(), tok, max_len=max_len, device=device)
    Z_gold, _ = extract_features_and_logits(model, df_train["text"].astype(str).tolist(), tok, max_len=max_len, device=device)
    Z_probe, _ = extract_features_and_logits(model, [str(df_pool["text"].iloc[int(i)]) for i in probe_idxs], tok, max_len=max_len, device=device)

    alpha_p = 0.25
    W_rp, b_rp = solve_trace_ridge_wb(Z_probe, pool_labels[probe_idxs], alphas=[alpha_p], device=device)[alpha_p]
    scale_rp = float(np.linalg.norm(W_neural)) / max(float(np.linalg.norm(W_rp)), 1e-8)
    W_rp_s, b_rp_s = W_rp * scale_rp, (b_rp - b_rp.mean()) * scale_rp + b_neural.mean()
    best_wn, W_plain, b_plain = max(
        ((wn, wn * W_neural + (1.0 - wn) * W_rp_s, wn * b_neural + (1.0 - wn) * b_rp_s) for wn in [0.0, 0.35, 0.50, 0.65, 0.80, 1.0]),
        key=lambda item: balanced_accuracy_score(y_bench, (Z_bench @ item[1].T + item[2]).argmax(axis=1)),
    )
    blend_detail = compute_detailed_metrics(Z_bench @ W_plain.T + b_plain, y_bench)

    Z_gold_cent = Z_gold.copy()
    for pid in np.unique(prompts_gold):
        m = prompts_gold == pid
        if m.sum() > 1:
            Z_gold_cent[m] -= Z_gold_cent[m].mean(axis=0, keepdims=True)
    cand_alphas = [0.05, 0.1, 0.25, 0.5, 1.0, 2.0]
    splits = list(GroupKFold(n_splits=5).split(Z_gold, y_gold, groups=prompts_gold))
    oof_p_map = {a: np.zeros((len(y_gold), K), dtype=np.float32) for a in cand_alphas}
    oof_c_map = {a: np.zeros((len(y_gold), K), dtype=np.float32) for a in cand_alphas}
    for tr_i, val_i in splits:
        wp = solve_trace_ridge_wb(Z_gold[tr_i], y_gold[tr_i], cand_alphas, device=device)
        wc = solve_trace_ridge_wb(Z_gold_cent[tr_i], y_gold[tr_i], cand_alphas, device=device)
        for a in cand_alphas:
            oof_p_map[a][val_i] = Z_gold[val_i] @ wp[a][0].T + wp[a][1]
            oof_c_map[a][val_i] = Z_gold_cent[val_i] @ wc[a][0].T + wc[a][1]
    oof_p_best = max(float(balanced_accuracy_score(y_gold, oof_p_map[a].argmax(axis=1))) for a in cand_alphas)
    best_alpha_c, oof_c_best = max(
        ((a, float(balanced_accuracy_score(y_gold, oof_c_map[a].argmax(axis=1)))) for a in cand_alphas),
        key=lambda item: item[1],
    )
    W_cent_raw, b_cent_raw = solve_trace_ridge_wb(Z_gold_cent, y_gold, [best_alpha_c], device=device)[best_alpha_c]
    scale_c = float(np.linalg.norm(W_plain)) / max(float(np.linalg.norm(W_cent_raw)), 1e-8)

    heads_dict = {
        **{f"norms.{i}.weight": ln.weight.detach().cpu().contiguous() for i, ln in enumerate(model.norms)},
        **{f"norms.{i}.bias": ln.bias.detach().cpu().contiguous() for i, ln in enumerate(model.norms)},
        "plain.weight": torch.from_numpy(np.ascontiguousarray(W_plain)).float(),
        "plain.bias": torch.from_numpy(np.ascontiguousarray(b_plain)).float(),
        "centered.weight": torch.from_numpy(np.ascontiguousarray((W_cent_raw * scale_c).astype(np.float32))).float(),
        "centered.bias": torch.from_numpy(np.ascontiguousarray((b_cent_raw * scale_c).astype(np.float32))).float(),
    }
    save_file(heads_dict, str(final_dir / "heads.safetensors"))

    wpca = fit_shared_gaussian_lowram(Z_gold, y_gold, prompts_gold, d_target=min(256, Z_gold.shape[1]))
    np.savez_compressed(
        str(final_dir / "wPCA_bank.npz"),
        mx=wpca["mx"].astype(np.float32),
        sx=wpca["sx"].astype(np.float32),
        mu=wpca["mx"].astype(np.float32),
        sd=wpca["sx"].astype(np.float32),
        W=wpca["W"].astype(np.float32),
        m=wpca["m"].astype(np.float32),
        lam=wpca["lam"].astype(np.float32),
        V_top64=wpca["V_top64"].astype(np.float32),
        layers=np.array(tap_layers, dtype=np.int32),
    )

    metrics = {
        "bench_neural_bacc": float(balanced_accuracy_score(y_bench, (Z_bench @ W_neural.T + b_neural).argmax(axis=1))),
        "bench_plain_ridge_bacc": float(balanced_accuracy_score(y_bench, (Z_bench @ W_rp.T + b_rp).argmax(axis=1))),
        "bench_plain_blend_bacc": float(blend_detail["bacc"]),
        "bench_plain_blend_nll": float(blend_detail["nll"]),
        "bench_plain_blend_gem4": float(blend_detail["gem_within_bacc"]),
        "bench_plain_blend_gem_ret": float(blend_detail["gem_retention"]),
        "bench_plain_blend_sister_37_38": float(blend_detail["sister_37_38_acc"]),
        "plain_blend_w_neural": float(best_wn),
        "plain_ridge_alpha": float(alpha_p),
        "gold_oof5_plain_bacc": oof_p_best,
        "gold_oof5_centered_bacc": oof_c_best,
        "centered_ridge_alpha": float(best_alpha_c),
    }
    meta_json = {
        "labels": list(LABELS),
        "classes": list(LABELS),
        "target_family": "",
        "backbone": backbone_name,
        "cfg": {
            "keep": len(tap_layers),
            "layers": tap_layers,
            "max_len": max_len,
            "end_token_id": int(tok.eos_token_id) if getattr(tok, "eos_token_id", None) is not None else 2,
            "backbone": backbone_name,
            "pooling": "mean_drop_cls",
            "drop_first_token": True,
            "per_layer_layernorm": True,
            "feature_dim": len(tap_layers) * hidden_size,
            "metrics": metrics,
        },
    }
    (final_dir / "meta.json").write_text(json.dumps(meta_json, indent=2), encoding="utf-8")
    (final_dir / "training_report.json").write_text(json.dumps({"metrics": metrics, "bench_blend_detail": blend_detail}, indent=2), encoding="utf-8")
    if on_checkpoint_saved is not None:
        on_checkpoint_saved()
    return {"final_dir": str(final_dir), "metrics": metrics, "meta_json": meta_json, "training_report": {"metrics": metrics, "bench_blend_detail": blend_detail}}


In [ ]:
# === MODULE: src/namethatllm/training/backbones/__init__.py ===
"""
Backbone Adapter Registry Exports. Importing this module registers `qwen`, `deberta`, and `bge`.
"""

from __future__ import annotations


__all__ = [
    "BaseBackboneAdapter",
    "BgeAdapter",
    "DebertaAdapter",
    "PrecisionSpec",
    "QwenAdapter",
    "TokenizationSpec",
    "_BACKBONE_REGISTRY",
    "_disable_final_norm",
    "_stabilize_fp16_backbone",
    "package_encoder_ridge_checkpoint",
    "register_backbone",
    "resolve_backbone_adapter",
    "run_deberta_specialist_suite",
]


In [ ]:
# === MODULE: src/namethatllm/training/engine.py ===
"""
Model-Agnostic Training Engine: Stage 1 & Stage 2 Loops, Resume Support, and Encoder Stage 1 Entrypoint.
"""

from __future__ import annotations

import copy
import functools
import gc
import importlib.util
import json
import math
import os
import shutil
import time
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import torch
from peft import PeftModel, set_peft_model_state_dict
from safetensors.torch import load_file
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import GroupKFold
from torch import nn
from torch.utils.data import DataLoader
from transformers import AutoModel, get_cosine_schedule_with_warmup



def _setup_training_env() -> None:
    """Configures PyTorch memory allocator and neutralizes incompatible optional packages."""
    os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
    orig_find_spec = importlib.util.find_spec

    def _safe_find_spec(name: str, *args: Any, **kwargs: Any) -> Any:
        if name == "torchao":
            return None
        return orig_find_spec(name, *args, **kwargs)

    importlib.util.find_spec = _safe_find_spec
    try:
        import peft.import_utils

        peft.import_utils.is_torchao_available = lambda: False
    except Exception:
        pass


_configure_training_environment = _setup_training_env


def train_prompt_group_step(
    model: MultiLayerLoRAClassifier,
    opt: torch.optim.Optimizer,
    input_ids: torch.Tensor,
    attention_mask: torch.Tensor,
    labels: torch.Tensor,
    micro_batch_size: int = 2,
    centered_weight: float = 0.5,
    family_margin: float = 0.35,
    supcon_weight: float = 0.20,
    label_smoothing: float = 0.10,
    accum_steps: int = 1,
    compute_dtype: torch.dtype = torch.float16,
    weight: torch.Tensor | None = None,
    drop_first_token: bool = True,
    scaler: Any | None = None,
    boost_hard_sisters: bool = True,
    manual_scaler: Any | None = None,
) -> tuple[float, float, float]:
    """Two-pass micro-batched training step for a single prompt group with singleton guard and FP16 loss scaling."""
    N = input_ids.shape[0]
    if N == 0:
        return 0.0, 0.0, 0.0
    dev = input_ids.device
    labels_dev = labels.to(dev)
    use_manual = manual_scaler is not None and getattr(manual_scaler, "is_enabled", lambda: False)()
    use_grad_scaler = (not use_manual) and scaler is not None and getattr(scaler, "is_enabled", lambda: False)()

    if N <= 1:
        model.train()
        mb_mask = attention_mask[:1]
        max_l = max(int(mb_mask.sum(dim=1).max().item()), 2)
        with torch.amp.autocast("cuda", enabled=torch.cuda.is_available(), dtype=compute_dtype):
            Z_mb = model.forward_features(input_ids[:1, :max_l], mb_mask[:, :max_l], drop_first_token=drop_first_token)
            mb_dev = Z_mb.device
            logits_p = model.predict_plain(Z_mb)
            loss_p = family_margin_cross_entropy(
                logits_p,
                labels_dev[:1].to(mb_dev),
                family_margin=family_margin,
                label_smoothing=label_smoothing,
                weight=weight,
                boost_hard_sisters=boost_hard_sisters,
            )
            scaled_loss = loss_p / float(max(accum_steps, 1))
            loss_to_bw = manual_scaler.scale_loss(scaled_loss) if use_manual else scaled_loss
        if use_grad_scaler:
            scaler.scale(loss_to_bw).backward()
        else:
            loss_to_bw.backward()
        loss_p_val = float(loss_p.item())
        return loss_p_val, loss_p_val, 0.0

    was_training = model.training
    model.eval()
    z_chunks = []
    with torch.no_grad():
        with torch.amp.autocast("cuda", enabled=torch.cuda.is_available(), dtype=compute_dtype):
            for s in range(0, N, 4):
                e = min(s + 4, N)
                mb_mask = attention_mask[s:e]
                max_l = max(int(mb_mask.sum(dim=1).max().item()), 2)
                z_c = model.forward_features(input_ids[s:e, :max_l], mb_mask[:, :max_l], drop_first_token=drop_first_token)
                z_chunks.append(z_c)
    Z_all_no_grad = torch.cat(z_chunks, dim=0)
    if N >= 8 and torch.rand(1).item() < 0.25:
        min_k = min(N, max(8, N // 2))
        k = int(torch.randint(min_k, N + 1, (1,)).item())
        sub_indices = torch.randperm(N, device=Z_all_no_grad.device)[:k]
        mu_prompt = Z_all_no_grad[sub_indices].mean(dim=0, keepdim=True)
    else:
        mu_prompt = Z_all_no_grad.mean(dim=0, keepdim=True)
    Z_group_centered = (Z_all_no_grad - mu_prompt).detach()
    del z_chunks

    total_loss_val = 0.0
    plain_loss_val = 0.0
    centered_loss_val = 0.0

    if was_training or not model.training:
        model.train()
    for s in range(0, N, micro_batch_size):
        e = min(s + micro_batch_size, N)
        mb_mask = attention_mask[s:e]
        max_l = max(int(mb_mask.sum(dim=1).max().item()), 2)
        mb_ids = input_ids[s:e, :max_l]
        mb_mask_slice = mb_mask[:, :max_l]

        with torch.amp.autocast("cuda", enabled=torch.cuda.is_available(), dtype=compute_dtype):
            Z_mb = model.forward_features(mb_ids, mb_mask_slice, drop_first_token=drop_first_token)
            mb_dev = Z_mb.device

            logits_p = model.predict_plain(Z_mb)
            loss_p = family_margin_cross_entropy(
                logits_p,
                labels_dev[s:e].to(mb_dev),
                family_margin=family_margin,
                label_smoothing=label_smoothing,
                weight=weight,
                boost_hard_sisters=boost_hard_sisters,
            )

            Z_c_mb = Z_mb - mu_prompt.to(mb_dev)
            logits_c = model.predict_centered(Z_c_mb)
            loss_c = family_margin_cross_entropy(
                logits_c,
                labels_dev[s:e].to(mb_dev),
                family_margin=family_margin,
                label_smoothing=label_smoothing,
                weight=weight,
                boost_hard_sisters=boost_hard_sisters,
            )

            loss_mb = loss_p + float(centered_weight) * loss_c
            if supcon_weight > 0.0:
                loss_sup = compute_supcon_loss(
                    Z_c_mb,
                    labels_dev[s:e],
                    temperature=0.10,
                    prototypes=model.centered_head.weight.detach(),
                    Z_group_centered=Z_group_centered,
                    group_slice=(s, e),
                )
                loss_mb = loss_mb + float(supcon_weight) * loss_sup

            weight_frac = (e - s) / float(N)
            scaled_loss = (loss_mb * weight_frac) / float(max(accum_steps, 1))
            loss_to_bw = manual_scaler.scale_loss(scaled_loss) if use_manual else scaled_loss

        if use_grad_scaler:
            scaler.scale(loss_to_bw).backward()
        else:
            loss_to_bw.backward()

        total_loss_val += loss_mb.item() * weight_frac
        plain_loss_val += loss_p.item() * weight_frac
        centered_loss_val += loss_c.item() * weight_frac

    return total_loss_val, plain_loss_val, centered_loss_val


def _load_pool_and_bench_for_gate(
    input_dir: Path,
    all_pools: list[Path],
    pool_file: Path | None,
    train_path: Path | None,
    meta_cols: list[str],
) -> tuple[pd.DataFrame, pd.DataFrame | None, list[str] | None, list[int] | None]:
    """Loads pool, gold train, and clean benchmark dataframe for export or gate modes."""
    pool_list = all_pools if all_pools else ([pool_file] if pool_file else [])
    dfs = []
    for p in pool_list:
        try:
            df_one = pd.read_parquet(p) if str(p).endswith(".parquet") else pd.read_csv(p)
            keep_cols = [c for c in meta_cols if c in df_one.columns]
            dfs.append(df_one[keep_cols])
        except Exception:
            pass
    df_p = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame(columns=["text", "model"])

    df_tr = None
    if train_path and Path(train_path).exists():
        try:
            df_tr = pd.read_csv(train_path)
        except Exception:
            pass

    eval_texts, eval_labels = None, None
    bench_paths = list(input_dir.rglob("clean_benchmark_1800_rows.parquet")) if input_dir.exists() else []
    if bench_paths:
        try:
            df_bench = pd.read_parquet(bench_paths[0])
            eval_texts = df_bench["text"].tolist()
            eval_labels = [LAB2I[m] for m in df_bench["model"].tolist()]
            print(f"[Trainer] Loaded {len(eval_texts)} benchmark samples from {bench_paths[0]}")
        except Exception as e:
            print(f"[Trainer] Note on reading clean benchmark: {e}")
    return df_p, df_tr, eval_texts, eval_labels


def _load_champion_adapter_and_heads(
    model: MultiLayerLoRAClassifier,
    input_dir: Path,
    backbone_name: str,
    lora_r_val: int,
    target_family: str | None,
    merge_adapter: bool,
    fail_on_error: bool,
    adapter_obj: BaseBackboneAdapter,
    precision: PrecisionSpec,
    train_cfg: dict[str, Any],
) -> None:
    """Finds and loads compatible champion LoRA adapter and head weights for extraction/gate pipelines."""
    adapter_found = find_compatible_champion_lora(
        input_dir,
        backbone_name=backbone_name,
        lora_r=lora_r_val,
        target_family=target_family or "",
    )
    if adapter_found is not None:
        try:
            cand_files = [
                adapter_found / "adapter_model.safetensors",
                adapter_found / "adapter" / "adapter_model.safetensors",
                adapter_found / "adapter_model.bin",
            ]
            adapter_file = next((p for p in cand_files if p.exists()), cand_files[0])
            adapter_sd = load_file(str(adapter_file)) if str(adapter_file).endswith(".safetensors") else torch.load(str(adapter_file), map_location="cpu", weights_only=True)

            model_sd = model.backbone.state_dict()
            mismatches = [
                f"{k_sd}: checkpoint {v_sd.shape} vs model {model_sd[k_sd].shape}"
                for k_sd, v_sd in adapter_sd.items()
                if k_sd in model_sd and model_sd[k_sd].shape != v_sd.shape
            ]
            if mismatches:
                raise ValueError(f"Shape mismatches in adapter {adapter_found}: {mismatches[:3]}...")

            set_peft_model_state_dict(model.backbone, adapter_sd)
            print(f"[Trainer] Loaded champion LoRA weights from {adapter_found}.")
            if merge_adapter and hasattr(model.backbone, "merge_and_unload"):
                try:
                    model.backbone = model.backbone.merge_and_unload()
                    print("[Trainer] Successfully merged and unloaded champion LoRA weights into base model.")
                except Exception as e:
                    print(f"[Trainer] Note on merge_and_unload: {e}")
        except Exception as e:
            print(f"[Trainer] Error loading champion adapter weights: {e}")
            if fail_on_error:
                raise RuntimeError(f"Failed to load champion LoRA adapter from {adapter_found}: {e}") from e
    elif fail_on_error and input_dir.exists():
        found_all = list(input_dir.rglob("adapter_model.safetensors"))
        if found_all:
            raise RuntimeError(f"Discovered adapters on disk but none were compatible with {backbone_name} (r={lora_r_val}): {found_all}")

    head_candidates: list[Path] = []
    if adapter_found is not None:
        head_candidates.extend([
            adapter_found / "heads.safetensors",
            adapter_found.parent / "heads.safetensors",
            adapter_found / "heads.pt",
            adapter_found.parent / "heads.pt",
        ])
    all_heads = (list(input_dir.rglob("heads.safetensors")) + list(input_dir.rglob("heads.pt"))) if input_dir.exists() else []
    priority_terms = ("champion", "inhouse", "stage1", "backbone", "18class", "full")
    head_candidates.extend([c for c in all_heads if any(term in str(c).lower() for term in priority_terms)])
    head_candidates.extend(all_heads)

    for cand in head_candidates:
        if cand.exists() and load_and_slice_champion_heads(cand, model, fam_cols=None):
            print(f"[Trainer] Loaded champion norms & heads from {cand}")
            break


def _resume_checkpoint(
    model: MultiLayerLoRAClassifier,
    resume_ckpt: str | Path,
    target_family: str | None,
    is_specialist: bool,
) -> None:
    """Restores adapter weights and verified head tensors from a compatible checkpoint directory."""
    resume_dir = Path(str(resume_ckpt))
    adapter_candidates = [
        resume_dir / "adapter" / "adapter_model.safetensors",
        resume_dir / "adapter_model.safetensors",
        resume_dir / "adapter" / "adapter_model.bin",
        resume_dir / "adapter_model.bin",
    ]
    heads_candidates = [
        resume_dir / "heads.safetensors",
        resume_dir / "heads.pt",
    ]
    adapter_file = next((p for p in adapter_candidates if p.exists()), None)
    heads_file = next((p for p in heads_candidates if p.exists()), None)
    if not resume_dir.exists() or adapter_file is None or heads_file is None:
        raise FileNotFoundError(
            f"resume_from_checkpoint={resume_ckpt!r} must contain adapter/ (or adapter_model.safetensors) and heads.safetensors"
        )
    if str(adapter_file).endswith(".safetensors"):
        adapter_sd = load_file(str(adapter_file))
    else:
        adapter_sd = torch.load(str(adapter_file), map_location="cpu", weights_only=True)
    set_peft_model_state_dict(model.backbone, adapter_sd)
    heads_ok = load_and_slice_champion_heads(
        heads_file,
        model,
        fam_cols=FAMILIES[target_family] if is_specialist and target_family in FAMILIES else None,
    )
    if not heads_ok:
        raise RuntimeError(f"Failed to load compatible heads from {heads_file}")
    print(f"[Trainer] Resumed checkpoint from {resume_dir} (adapter={adapter_file.name}, heads={heads_file.name}).")


def _run_stage1(
    model: MultiLayerLoRAClassifier,
    tok: Any,
    tok_spec: TokenizationSpec,
    adapter: BaseBackboneAdapter,
    precision: PrecisionSpec,
    device: torch.device,
    train_cfg: dict[str, Any],
    stage1_cfg: dict[str, Any],
    input_dir: Path,
    out_dir: Path,
    train_path: Path | None,
    pool_file: Path | None,
    all_pools: list[Path],
    bench_bundle: tuple[list[str], np.ndarray, list[str], np.ndarray, np.ndarray] | None,
    active_labels: list[str],
    sub_lab2i: dict[str, int],
    layers: list[int],
    keep_layers: int,
    backbone_name: str,
    target_family: str | None,
    is_specialist: bool,
    family_margin: float,
    milestone_history: list[dict[str, Any]],
    step_callback: Any | None = None,
) -> None:
    """Executes Stage 1 pool pretraining with per-class weighting, opt-step milestones, and clean budget exit."""
    max_stage1_hours = float(stage1_cfg.get("max_hours", 12.0))
    s1_start = time.time()
    max_s1_sec = max_stage1_hours * 3600.0
    raw_pool_list = all_pools if all_pools else ([pool_file] if pool_file else [])
    use_v2_only = bool(stage1_cfg.get("use_v2_only", True))
    pool_list = _select_stage1_pool_files(raw_pool_list, use_v2_only=use_v2_only)
    print(f"[Trainer] Stage 1 Pretraining on {len(pool_list)} pool file(s) (use_v2_only={use_v2_only}): {[Path(p).name for p in pool_list]}")

    use_mw = bool(stage1_cfg.get("use_match_weight", True))
    mw_clip = tuple(stage1_cfg.get("match_weight_clip", (0.5, 2.5)))
    df_p, pool_stats = load_clean_stage1_pool(
        pool_list=pool_list,
        input_dir=input_dir,
        train_path=train_path,
        bench_bundle=bench_bundle,
        active_labels=active_labels,
        exclude_train_overlap=bool(stage1_cfg.get("exclude_train_overlap", True)),
        weight_clip=mw_clip,
        use_match_weight=use_mw,
        min_char_len=40,
        include_unseen_holdout=True,
    )

    word_lens = [len(t.split()) for t in df_p["text"]]
    tok_lens = [min(len(t) // 4 + 1, tok_spec.max_len) for t in df_p["text"]]
    class_dist = df_p["model"].value_counts().to_dict()

    def _calc_percentiles(vals: list[int]) -> dict[str, float]:
        if not vals:
            return {}
        v = np.array(vals)
        return {k: float(np.percentile(v, p)) for k, p in (("p10", 10), ("p25", 25), ("p50", 50), ("p75", 75), ("p90", 90), ("p99", 99))}

    audit_report = {
        "source_files": pool_stats["source_counts"],
        "total_raw_rows": pool_stats["total_raw_rows"],
        "deduplicated_rows": pool_stats["dedup_rows"],
        "quarantined_holdout_rows": pool_stats["quarantine_dropped"],
        "retained_pool_rows": int(len(df_p)),
        "class_distribution": {str(k): int(v) for k, v in class_dist.items()},
        "word_length_percentiles": _calc_percentiles(word_lens),
        "token_length_percentiles": _calc_percentiles(tok_lens),
    }
    out_dir.mkdir(parents=True, exist_ok=True)
    audit_file = out_dir / "data_audit_report.json"
    audit_file.write_text(json.dumps(audit_report, indent=2), encoding="utf-8")

    print("\n==========================================================================================")
    print("                         SACRED DATA INTEGRITY AUDIT REPORT                               ")
    print("==========================================================================================")
    for fname, cnt in pool_stats["source_counts"].items():
        print(f"   - {fname:<32}: {cnt:>8,} rows")
    print(f" Clean Retained Pool Rows: {len(df_p):,} | Quarantined: {pool_stats['quarantine_dropped']:,}")
    for lbl in active_labels:
        c_cnt = class_dist.get(lbl, 0)
        print(f"   - {lbl:<22}: {c_cnt:>6,} ({(c_cnt / max(len(df_p), 1)) * 100:5.2f}%)")
    print("==========================================================================================\n")

    epochs1 = int(stage1_cfg.get("epochs", 1))
    num_classes = len(active_labels)
    max_s = int(stage1_cfg.get("max_samples", 2000 * num_classes if is_specialist else 150000))
    explicit_quotas = stage1_cfg.get("class_quotas")
    oversample_to_target = bool(stage1_cfg.get("oversample_to_target", explicit_quotas is not None))
    present_labels = [lbl for lbl in active_labels if class_dist.get(lbl, 0) > 0]
    present_lab2i = {lbl: idx for idx, lbl in enumerate(present_labels)}
    present_y_arr = np.array([present_lab2i[m] for m in df_p["model"]], dtype=np.int64)

    hard_set = set(HARD_CLASS_INDICES)
    if explicit_quotas:
        quotas_map = {lbl: int(explicit_quotas.get(lbl, max_s // max(len(present_labels), 1))) for lbl in present_labels}
        if "max_samples" in stage1_cfg:
            quota_sum = sum(int(v) for v in explicit_quotas.values())
            if quota_sum != int(stage1_cfg["max_samples"]):
                print(
                    f"[Trainer Warning] stage1.class_quotas sum ({quota_sum}) != stage1.max_samples ({int(stage1_cfg['max_samples'])}); using class_quotas."
                )
        epoch_plans = [
            build_targeted_epoch_indices(present_y_arr, quotas_map, epoch=ep, seed=42, class_labels=present_labels)
            for ep in range(epochs1)
        ]
    elif len(df_p) > max_s or oversample_to_target:
        cls_multipliers = {lbl: (1.30 if LAB2I.get(lbl) in hard_set else 1.0) for lbl in present_labels}
        total_multiplier = max(sum(cls_multipliers.values()), 1e-6)
        quotas_map = {lbl: max(1, int(round(max_s * (cls_multipliers[lbl] / total_multiplier)))) for lbl in present_labels}
        epoch_plans = [
            build_targeted_epoch_indices(present_y_arr, quotas_map, epoch=ep, seed=42, class_labels=present_labels)
            for ep in range(epochs1)
        ]
    else:
        epoch_plans = [np.random.RandomState(42 + ep).permutation(len(df_p)) for ep in range(epochs1)]

    print("[Trainer Stage 1] Epoch-1 Scheduled Sequence Counts:", {lbl: int((df_p.iloc[epoch_plans[0]]["model"] == lbl).sum()) for lbl in present_labels})

    bs1, accum1, bench_eval_bs = adapter.adjust_stage1_batch_config(stage1_cfg, precision)
    collate_fn_s1 = functools.partial(collate_text_batch, pad_token_id=tok_spec.pad_id)

    lora_ratio1 = float(stage1_cfg.get("lora_plus_ratio", 2.5))
    param_groups1 = _get_loraplus_param_groups(
        model,
        lr_lora=float(stage1_cfg.get("lr_lora", 1.5e-4)),
        lr_head=float(stage1_cfg.get("lr_head", 8e-4)),
        lora_plus_ratio=lora_ratio1,
        include_centered_head=False,
    )
    opt1 = torch.optim.AdamW(param_groups1, weight_decay=float(train_cfg.get("weight_decay", 0.01)))
    total_steps1 = sum(math.ceil(len(ep_idx) / bs1) for ep_idx in epoch_plans)
    total_opt_steps1 = max(1, sum(math.ceil(math.ceil(len(ep_idx) / bs1) / accum1) for ep_idx in epoch_plans))
    warmup1 = min(int(stage1_cfg.get("warmup_steps", 65)), max(1, total_opt_steps1 // 5))
    sched1 = get_cosine_schedule_with_warmup(opt1, num_warmup_steps=warmup1, num_training_steps=total_opt_steps1)

    stage1_class_weights = _build_class_weight_tensor(
        active_labels,
        sub_lab2i,
        stage1_cfg,
        default_hard_weight=float(train_cfg.get("hard_class_weight", 1.0)),
        device=device,
    )
    boost_hard = bool(train_cfg.get("boost_hard_sisters", True))
    s1_supcon_w = float(stage1_cfg.get("supcon_weight", 0.0))
    s1_label_smooth = float(stage1_cfg.get("label_smoothing", 0.0))

    scaler1, manual_scaler1 = adapter.build_scalers(precision, device)
    if manual_scaler1 is not None and manual_scaler1.is_enabled():
        print(f"[Trainer Stage 1] Enabled _ManualFP16LossScaler (initial scale={manual_scaler1.scale:.1f})")
    eval_steps1 = int(stage1_cfg.get("eval_steps", 0))
    save_steps = int(stage1_cfg.get("save_steps", 500))
    best_s1_bacc = -1.0
    best_s1_state: dict[str, torch.Tensor] | None = None

    def _snapshot_trainable() -> dict[str, torch.Tensor]:
        return {n: p.detach().cpu().clone() for n, p in model.named_parameters() if p.requires_grad}

    model.train()
    opt1.zero_grad(set_to_none=True)
    global_step1 = 0
    opt_step1 = 0
    skipped_steps1 = 0
    stop_stage1 = False
    for ep in range(epochs1):
        if stop_stage1:
            break
        df_ep = df_p.iloc[epoch_plans[ep]].reset_index(drop=True)
        ep_lengths = [min(len(t) // 4 + 1, tok_spec.max_len) for t in df_ep["text"]]
        ep_weights = df_ep["_sample_weight"].astype(float).tolist() if "_sample_weight" in df_ep.columns else None
        ds_ep = TextSequenceDataset(
            df_ep["text"].tolist(),
            [sub_lab2i[m] for m in df_ep["model"]],
            None,
            tok,
            tok_spec=tok_spec,
            sample_weights=ep_weights,
        )
        bucket_sampler = BucketedLengthBatchSampler(ep_lengths, batch_size=bs1, shuffle=True, seed=42 + ep)
        bucket_sampler.set_epoch(ep)
        loader1 = DataLoader(ds_ep, batch_sampler=bucket_sampler, collate_fn=collate_fn_s1)
        n_batches1 = len(loader1)
        opt1.zero_grad(set_to_none=True)
        for step_idx, batch in enumerate(loader1):
            global_step1 += 1
            mb_ids = batch["input_ids"].to(device)
            mb_mask = batch["attention_mask"].to(device)
            max_l = max(int(mb_mask.sum(dim=1).max().item()), 2)
            mb_ids = mb_ids[:, :max_l]
            mb_mask = mb_mask[:, :max_l]
            labels_dev = batch["labels"].to(device)

            with torch.amp.autocast("cuda", enabled=torch.cuda.is_available(), dtype=precision.compute_dtype):
                Z_s1, logits_p, _ = model(mb_ids, mb_mask, drop_first_token=tok_spec.drop_first_token)
                sw_s1 = batch.get("sample_weights")
                loss_ce = family_margin_cross_entropy(
                    logits_p,
                    labels_dev,
                    family_margin=family_margin,
                    label_smoothing=s1_label_smooth,
                    weight=stage1_class_weights,
                    boost_hard_sisters=boost_hard,
                    sample_weight=sw_s1,
                )
                loss_sup = (
                    compute_supcon_loss(Z_s1, labels_dev, temperature=0.10, prototypes=model.plain_head.weight.detach())
                    if s1_supcon_w > 0.0
                    else (Z_s1 * 0.0).sum()
                )
                eff_accum = (
                    min(accum1, n_batches1 - (step_idx - (step_idx % accum1)))
                    if (step_idx // accum1) == ((n_batches1 - 1) // accum1) and (n_batches1 % accum1 != 0)
                    else accum1
                )
                loss = (loss_ce + s1_supcon_w * loss_sup) / max(eff_accum, 1)

            if scaler1 is not None and getattr(scaler1, "is_enabled", lambda: False)():
                scaler1.scale(loss).backward()
            elif manual_scaler1 is not None and manual_scaler1.is_enabled():
                manual_scaler1.scale_loss(loss).backward()
            else:
                loss.backward()

            did_step_now = (step_idx + 1) % accum1 == 0 or (step_idx + 1) == n_batches1
            if did_step_now:
                did_step = _step_with_grad_guard(
                    model,
                    opt1,
                    sched1,
                    scaler=scaler1,
                    manual_scaler=manual_scaler1,
                    max_grad_norm=1.0,
                )
                skipped_steps1 += int(not did_step)
                opt_step1 += 1

                if bench_bundle is not None and eval_steps1 > 0 and opt_step1 % eval_steps1 == 0 and (total_opt_steps1 - opt_step1) > 25:
                    _, _, sub_texts, sub_y, _ = bench_bundle
                    m_sub, _ = _evaluate_bench_checkpoint(
                        model=model,
                        tok=tok,
                        texts=sub_texts,
                        y_true=sub_y,
                        max_len=tok_spec.max_len,
                        device=device,
                        compute_dtype=precision.compute_dtype,
                        drop_first_token=tok_spec.drop_first_token,
                        tag=f"S1_OptStep{opt_step1}",
                        end_id=tok_spec.end_id,
                        cls_id=tok_spec.cls_id,
                        pad_id=tok_spec.pad_id,
                        batch_size=bench_eval_bs,
                        out_dir=out_dir,
                        history=milestone_history,
                    )
                    last_sub_bacc = float(m_sub["bacc"])
                    if last_sub_bacc > best_s1_bacc:
                        best_s1_bacc = last_sub_bacc
                        best_s1_state = _snapshot_trainable()
                        with adapter.export_context(model.backbone):
                            export_compatible_checkpoint(
                                model=model,
                                output_dir=out_dir / "stage1_best",
                                layers=layers,
                                keep=keep_layers,
                                max_len=tok_spec.max_len,
                                end_id=tok_spec.end_id if tok_spec.end_id is not None else 0,
                                backbone_name=backbone_name,
                                class_labels=active_labels,
                                target_family=target_family if is_specialist else None,
                            )
                        print(f"[Trainer Stage 1] Saved new best milestone checkpoint (Sub-Bench BAcc={best_s1_bacc*100:.2f}%)")

                if save_steps > 0 and opt_step1 % save_steps == 0:
                    try:
                        with adapter.export_context(model.backbone):
                            export_compatible_checkpoint(
                                model=model,
                                output_dir=out_dir,
                                layers=layers,
                                keep=keep_layers,
                                max_len=tok_spec.max_len,
                                end_id=tok_spec.end_id if tok_spec.end_id is not None else 0,
                                backbone_name=backbone_name,
                                class_labels=active_labels,
                                target_family=target_family if is_specialist else None,
                            )
                        print(f"[Trainer Stage 1] Intermediate checkpoint auto-saved at opt step {opt_step1} (global {global_step1}) to {out_dir}")
                        if step_callback is not None:
                            step_callback("checkpoint_saved", opt_step1)
                    except Exception as e:
                        print(f"[Trainer Stage 1 Warning] Auto-save failed at opt step {opt_step1}: {e}")

            if global_step1 % 200 == 0:
                elapsed_m = (time.time() - s1_start) / 60.0
                lr_now = sched1.get_last_lr()[0] if hasattr(sched1, "get_last_lr") else 0.0
                scale_str = f" | Scale: {manual_scaler1.scale:.0f}" if (manual_scaler1 is not None and manual_scaler1.is_enabled()) else ""
                print(
                    f"[Trainer Stage 1] Epoch {ep + 1}/{epochs1} Step {step_idx + 1}/{n_batches1} (Opt {opt_step1}/{total_opt_steps1}, Global {global_step1}/{total_steps1}) "
                    f"| Loss: {loss.item() * max(eff_accum, 1):.4f} | LR: {lr_now:.2e}{scale_str} | Skipped: {skipped_steps1} | Elapsed: {elapsed_m:.1f}m"
                )

            if (time.time() - s1_start) >= max_s1_sec:
                print(
                    f"[Trainer Stage 1] Reached {stage1_cfg.get('max_hours', 12.0)}h time budget at opt step {opt_step1} (global step {global_step1}); discarding partial accum window and advancing to Stage 2."
                )
                opt1.zero_grad(set_to_none=True)
                stop_stage1 = True
                break

        del ds_ep, loader1, bucket_sampler

    opt1.zero_grad(set_to_none=True)
    if bench_bundle is not None:
        full_texts, full_y, _, sub_y, sub_idx_arr = bench_bundle
        _, full_lg = _evaluate_bench_checkpoint(
            model=model,
            tok=tok,
            texts=full_texts,
            y_true=full_y,
            max_len=tok_spec.max_len,
            device=device,
            compute_dtype=precision.compute_dtype,
            drop_first_token=tok_spec.drop_first_token,
            tag="S1_End_Full",
            end_id=tok_spec.end_id,
            cls_id=tok_spec.cls_id,
            pad_id=tok_spec.pad_id,
            batch_size=bench_eval_bs,
            out_dir=out_dir,
            history=milestone_history,
        )
        sub_idx_arr = np.asarray(sub_idx_arr, dtype=np.int64)
        end_sub_bacc = float(balanced_accuracy_score(sub_y, full_lg[sub_idx_arr].argmax(axis=1)))
        if best_s1_state is not None and bool(stage1_cfg.get("restore_best_before_stage2", True)) and best_s1_bacc > end_sub_bacc + 0.005:
            print(
                f"[Trainer Stage 1] Restoring best milestone weights (Sub BAcc={best_s1_bacc*100:.2f}% > End Sub BAcc={end_sub_bacc*100:.2f}%) before Stage 2."
            )
            with torch.no_grad():
                for n, p in model.named_parameters():
                    if n in best_s1_state:
                        p.data.copy_(best_s1_state[n].to(p.device, dtype=p.dtype))
    del opt1, sched1, df_p, best_s1_state
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    try:
        with adapter.export_context(model.backbone):
            export_compatible_checkpoint(
                model=model,
                output_dir=out_dir,
                layers=layers,
                keep=keep_layers,
                max_len=tok_spec.max_len,
                end_id=tok_spec.end_id if tok_spec.end_id is not None else 0,
                backbone_name=backbone_name,
                class_labels=active_labels,
                target_family=target_family if is_specialist else None,
            )
        print(f"[Trainer] Safe intermediate Stage 1 checkpoint exported to {out_dir}")
        if step_callback is not None:
            step_callback("checkpoint_saved", "stage1_finished")
    except Exception as e:
        print(f"[Trainer] Warning: could not export Stage 1 intermediate checkpoint: {e}")


def _run_stage2(
    model: MultiLayerLoRAClassifier,
    tok: Any,
    tok_spec: TokenizationSpec,
    adapter: BaseBackboneAdapter,
    precision: PrecisionSpec,
    device: torch.device,
    train_cfg: dict[str, Any],
    stage2_cfg: dict[str, Any],
    train_path: Path | None,
    out_dir: Path,
    bench_bundle: tuple[list[str], np.ndarray, list[str], np.ndarray, np.ndarray] | None,
    active_labels: list[str],
    sub_lab2i: dict[str, int],
    layers: list[int],
    keep_layers: int,
    backbone_name: str,
    target_family: str | None,
    is_specialist: bool,
    family_margin: float,
    supcon_weight: float,
    step_callback: Any | None = None,
) -> tuple[np.ndarray, dict[str, Any]]:
    """Executes Stage 2 prompt-grouped contrastive fine-tuning, SWA averaging, and OOF/Ridge calibration."""
    if bool(stage2_cfg.get("freeze_backbone", False)):
        print("[Trainer] Freezing backbone parameters for Stage 2 (exclusively training norms, plain_head, and centered_head)...")
        for p in model.backbone.parameters():
            p.requires_grad = False

    df_tr = pd.read_csv(train_path)
    if bench_bundle is not None:
        bench_norm_set = {norm_text_hash(t) for t in bench_bundle[0]}
        before_s2 = len(df_tr)
        df_tr = df_tr[~df_tr["text"].astype(str).map(norm_text_hash).isin(bench_norm_set)].reset_index(drop=True)
        if len(df_tr) < before_s2:
            print(f"[Trainer Stage 2] Quarantined {before_s2 - len(df_tr)} overlapping benchmark rows from train.csv.")
    if is_specialist:
        df_tr = df_tr[df_tr["model"].isin(active_labels)].reset_index(drop=True)
    num_classes = len(active_labels)
    print(f"[Trainer] Stage 2 Fine-Tuning on {train_path} ({len(df_tr)} samples, {num_classes} classes)...")

    bs2 = int(stage2_cfg.get("batch_size", 2))
    accum2 = int(stage2_cfg.get("gradient_accumulation_steps", 1))
    collate_fn = functools.partial(collate_text_batch, pad_token_id=tok_spec.pad_id)
    ds2 = TextSequenceDataset(
        df_tr["text"].tolist(),
        [sub_lab2i[m] for m in df_tr["model"]],
        df_tr["prompt_id"].tolist(),
        tok,
        tok_spec=tok_spec,
    )

    if bool(stage2_cfg.get("enabled", True)):
        if bool(stage2_cfg.get("warmstart_centered_from_plain", bool(train_cfg.get("fresh_init", True)))):
            with torch.no_grad():
                w_p_init = model.plain_head.weight.detach().float()
                row_norms = w_p_init.norm(dim=1, keepdim=True).clamp(min=1e-6)
                target_norm = float(stage2_cfg.get("centered_init_norm", 0.65))
                w_c_init = (w_p_init / row_norms) * target_norm
                model.centered_head.weight.copy_(w_c_init.to(model.centered_head.weight.device, dtype=model.centered_head.weight.dtype))
                model.centered_head.bias.zero_()
            print(f"[Trainer Stage 2] Warm-started centered_head from unit-normalized plain_head directions (row_norm={target_norm:.2f}).")

        sampler2 = PromptGroupBatchSampler(df_tr["prompt_id"].tolist(), batch_size=bs2, shuffle=True)
        loader2 = DataLoader(ds2, batch_sampler=sampler2, collate_fn=collate_fn)
        n_batches2 = len(loader2)

        lora_ratio2 = float(stage2_cfg.get("lora_plus_ratio", 2.5))
        param_groups2 = _get_loraplus_param_groups(
            model,
            lr_lora=float(stage2_cfg.get("lr_lora", 3e-5)),
            lr_head=float(stage2_cfg.get("lr_head", 2e-4)),
            lora_plus_ratio=lora_ratio2,
            include_centered_head=True,
        )
        opt2 = torch.optim.AdamW(param_groups2, weight_decay=float(train_cfg.get("weight_decay", 0.01)))
        epochs2 = int(stage2_cfg.get("epochs", 2))
        total_steps2 = max(1, n_batches2 * epochs2)
        total_opt_steps2 = max(1, epochs2 * max(1, math.ceil(n_batches2 / max(accum2, 1))))
        warmup2 = min(int(stage2_cfg.get("warmup_steps", 20)), max(1, total_opt_steps2 // 4))
        sched2 = get_cosine_schedule_with_warmup(opt2, num_warmup_steps=warmup2, num_training_steps=total_opt_steps2)

        swa_enabled = bool(stage2_cfg.get("swa", True))
        swa_start_frac = float(stage2_cfg.get("swa_start_frac", 0.5))
        swa_min_step = 0 if n_batches2 < 4 else int(n_batches2 * swa_start_frac)
        trainable_params = [p for p in model.parameters() if p.requires_grad]
        swa_shadow = {p: torch.zeros_like(p.data, dtype=torch.float32, device="cpu") for p in trainable_params} if swa_enabled else {}
        swa_steps = 0

        stage2_class_weights = _build_class_weight_tensor(
            active_labels,
            sub_lab2i,
            stage2_cfg,
            default_hard_weight=float(stage2_cfg.get("hard_class_weight", 1.0)),
            device=device,
        )
        supcon_s2_target = float(stage2_cfg.get("supcon_weight", supcon_weight))
        supcon_warmup_batches = max(1, int(total_steps2 * float(stage2_cfg.get("supcon_warmup_frac", 0.25))))
        s2_label_smooth = float(stage2_cfg.get("label_smoothing", 0.05 if is_specialist else 0.10))
        s2_centered_w = float(stage2_cfg.get("centered_weight", 0.50))

        scaler2, manual_scaler2 = adapter.build_scalers(precision, device)
        if manual_scaler2 is not None and manual_scaler2.is_enabled():
            print(f"[Trainer Stage 2] Enabled _ManualFP16LossScaler (initial scale={manual_scaler2.scale:.1f})")
        s2_boost_hard = bool(train_cfg.get("boost_hard_sisters", True))
        model.train()
        global_step2 = 0
        skipped_steps2 = 0
        s2_start = time.time()
        for ep in range(epochs2):
            sampler2.set_epoch(ep)
            opt2.zero_grad(set_to_none=True)
            for step_idx, batch in enumerate(loader2):
                global_step2 += 1
                win_start = step_idx - (step_idx % max(accum2, 1))
                eff_accum2 = min(accum2, n_batches2 - win_start) if (accum2 > 1 and win_start + accum2 > n_batches2) else max(accum2, 1)
                sup_ramp = min(1.0, float(global_step2) / float(supcon_warmup_batches))
                _, loss2_p, loss2_c = train_prompt_group_step(
                    model=model,
                    opt=opt2,
                    input_ids=batch["input_ids"].to(device),
                    attention_mask=batch["attention_mask"].to(device),
                    labels=batch["labels"].to(device),
                    micro_batch_size=bs2,
                    centered_weight=s2_centered_w,
                    family_margin=family_margin,
                    supcon_weight=supcon_s2_target * sup_ramp,
                    label_smoothing=s2_label_smooth,
                    accum_steps=eff_accum2,
                    compute_dtype=precision.compute_dtype,
                    weight=stage2_class_weights,
                    drop_first_token=tok_spec.drop_first_token,
                    scaler=scaler2,
                    boost_hard_sisters=s2_boost_hard,
                    manual_scaler=manual_scaler2,
                )
                if (step_idx + 1) % max(accum2, 1) == 0 or (step_idx + 1) == n_batches2:
                    did_step = _step_with_grad_guard(
                        model, opt2, sched2, scaler=scaler2, manual_scaler=manual_scaler2, max_grad_norm=1.0
                    )
                    skipped_steps2 += int(not did_step)
                    if did_step and swa_enabled and ep == (epochs2 - 1) and step_idx >= swa_min_step:
                        swa_steps += 1
                        for p in trainable_params:
                            swa_shadow[p].add_(p.data.detach().float().cpu())

                if global_step2 % 25 == 0:
                    elapsed_s2_m = (time.time() - s2_start) / 60.0
                    lr2_now = sched2.get_last_lr()[0] if hasattr(sched2, "get_last_lr") else 0.0
                    scale2_str = f" | Scale: {manual_scaler2.scale:.0f}" if (manual_scaler2 is not None and manual_scaler2.is_enabled()) else ""
                    print(
                        f"[Trainer Stage 2] Epoch {ep + 1}/{epochs2} Group {step_idx + 1}/{n_batches2} (Global {global_step2}/{total_steps2}) "
                        f"| Plain: {loss2_p:.4f} | Centered: {loss2_c:.4f} | LR: {lr2_now:.2e}{scale2_str} "
                        f"| Skipped: {skipped_steps2} | Elapsed: {elapsed_s2_m:.1f}m"
                    )

        if swa_enabled and swa_steps > 0:
            print(f"[Trainer] Applying SWA weight averaging across {swa_steps} steps in final Stage 2 epoch...")
            for p in trainable_params:
                p.data.copy_((swa_shadow[p] / float(swa_steps)).to(p.device, p.dtype))
            del swa_shadow

        del loader2, opt2, sched2, sampler2
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        try:
            with adapter.export_context(model.backbone):
                export_compatible_checkpoint(
                    model=model,
                    output_dir=out_dir,
                    layers=layers,
                    keep=keep_layers,
                    max_len=tok_spec.max_len,
                    end_id=tok_spec.end_id if tok_spec.end_id is not None else 0,
                    backbone_name=backbone_name,
                    class_labels=active_labels,
                    target_family=target_family if is_specialist else None,
                )
            print(f"[Trainer] Saved intermediate Stage 2 SWA checkpoint to {out_dir}")
            if step_callback is not None:
                step_callback("checkpoint_saved", "stage2_swa")
        except Exception as e:
            print(f"[Trainer] Warning saving Stage 2 intermediate checkpoint: {e}")

    print("[Trainer] Extracting Stage 2 embeddings for calibration & 5-Stage OOF Cross-Validation...")
    model.eval()
    loader_eval = DataLoader(ds2, batch_size=adapter.stage2_eval_batch_size(), shuffle=False, collate_fn=collate_fn)
    Z_list = []
    with torch.no_grad():
        with torch.amp.autocast("cuda", enabled=torch.cuda.is_available(), dtype=precision.compute_dtype):
            for batch in loader_eval:
                b_ids = batch["input_ids"].to(device)
                b_mask = batch["attention_mask"].to(device)
                max_l = max(int(b_mask.sum(dim=1).max().item()), 2)
                z_b = model.forward_features(b_ids[:, :max_l], b_mask[:, :max_l], drop_first_token=tok_spec.drop_first_token)
                Z_list.append(z_b.float().cpu().numpy())
    Z_train = np.vstack(Z_list)
    y_train = np.array([sub_lab2i[m] for m in df_tr["model"]], dtype=np.int64)
    prompts_train = df_tr["prompt_id"].to_numpy()

    del loader_eval, ds2
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    default_ridge_w = 0.35 if is_specialist else 0.0
    default_alpha = 8.0 if is_specialist else 10.0
    default_ls = 0.05 if is_specialist else 0.10

    w_ridge_p = float(stage2_cfg.get("plain_ridge_weight", default_ridge_w))
    w_ridge_c = float(stage2_cfg.get("centered_ridge_weight", default_ridge_w))
    if w_ridge_p > 0.0 or w_ridge_c > 0.0:
        ridge_alpha = float(stage2_cfg.get("ridge_alpha", default_alpha))
        ridge_ls = float(stage2_cfg.get("ridge_label_smoothing", stage2_cfg.get("label_smoothing", default_ls)))
        W_p, b_p, W_c, b_c = fit_ridge_heads(Z_train, y_train, prompts_train, alpha=ridge_alpha, label_smoothing=ridge_ls, num_classes=num_classes)
        if w_ridge_p > 0.0:
            _blend_head(model.plain_head, W_p, b_p, w_neural=1.0 - w_ridge_p, w_ridge=w_ridge_p)
        if w_ridge_c > 0.0:
            _blend_head(model.centered_head, W_c, b_c, w_neural=1.0 - w_ridge_c, w_ridge=w_ridge_c)

    oof_report: dict[str, Any] = {}
    if bool(stage2_cfg.get("check_overfitting", True)) and len(np.unique(prompts_train)) >= 2:
        try:
            oof_report = evaluate_stage2_oof_cv(
                Z_train,
                y_train,
                prompts_train,
                n_splits=int(stage2_cfg.get("n_splits", 5)),
                alpha=float(stage2_cfg.get("cv_alpha", default_alpha)),
                label_smoothing=float(stage2_cfg.get("cv_label_smoothing", default_ls)),
                num_classes=num_classes,
                class_labels=active_labels,
                output_dir=out_dir,
            )
        except Exception as e:
            print(f"[Stage 2 CV Warning] OOF cross-validation failed ({e}); continuing to checkpoint export.")

    with adapter.export_context(model.backbone):
        export_compatible_checkpoint(
            model=model,
            output_dir=out_dir,
            layers=layers,
            keep=keep_layers,
            max_len=tok_spec.max_len,
            end_id=tok_spec.end_id if tok_spec.end_id is not None else 0,
            backbone_name=backbone_name,
            Z_train=Z_train,
            y_train=y_train,
            prompts_train=prompts_train,
            class_labels=active_labels,
            target_family=target_family if is_specialist else None,
        )
    return Z_train, oof_report


def run_training_pipeline(
    train_cfg: dict[str, Any] | None = None,
    step_callback: Any | None = None,
) -> Any:
    """Executes the unified multi-layer LoRA training pipeline delegated by backbone strategy adapter."""
    _setup_training_env()
    train_cfg = train_cfg or {}

    adapter = resolve_backbone_adapter(train_cfg)
    backbone_name = str(train_cfg.get("backbone", adapter.default_backbone))
    out_dir = Path(train_cfg.get("output_dir", adapter.default_output_dir))
    family_margin = float(train_cfg.get("family_margin", 0.35))
    supcon_weight = float(train_cfg.get("supcon_weight", 0.20))
    is_smoke = bool(train_cfg.get("smoke", False))
    checkpoints_volume = train_cfg.get("checkpoints_volume", None)
    input_dir = Path(train_cfg.get("input_dir", "/kaggle/input"))
    target_family = train_cfg.get("target_family")

    special_res = adapter.maybe_run_pipeline(
        train_cfg,
        step_callback=step_callback,
        run_pipeline_fn=run_training_pipeline,
        run_encoder_stage1_fn=run_encoder_stage1,
    )
    if special_res is not None:
        return special_res

    is_gate_or_export = bool(
        train_cfg.get("pipeline_mode") in ("family_gate", "export_embeddings", "extract_features")
        or train_cfg.get("mode") == "family_gate"
        or train_cfg.get("gate_training")
    )

    fresh_init = bool(train_cfg.get("fresh_init", False))
    working_champ = Path("/kaggle/working") / adapter.default_output_dir
    if adapter.preserves_working_checkpoint and not fresh_init and not is_gate_or_export and input_dir.exists() and not working_champ.exists():
        for cand in input_dir.rglob(adapter.default_output_dir):
            if cand.is_dir() and (cand / "heads.safetensors").exists():
                try:
                    shutil.copytree(str(cand), str(working_champ))
                    print(f"[Trainer] Preserved existing checkpoint from {cand} to {working_champ}")
                    break
                except Exception as e:
                    print(f"[Trainer] Note on preserving checkpoint: {e}")

    if train_cfg.get("specialist_jobs") and isinstance(train_cfg["specialist_jobs"], list):
        final_out = out_dir
        for job in train_cfg["specialist_jobs"]:
            job_cfg = copy.deepcopy(train_cfg)
            job_cfg.pop("specialist_jobs", None)
            for k, v in job.items():
                if isinstance(v, dict) and isinstance(job_cfg.get(k), dict):
                    job_cfg[k] = {**job_cfg[k], **v}
                else:
                    job_cfg[k] = v
            final_out = run_training_pipeline(job_cfg, step_callback=step_callback)
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        return final_out

    from_emb_dir = train_cfg.get("from_embeddings_dir")
    if from_emb_dir:
        print(f"[Trainer] Fast offline dataset gate training mode triggered from: {from_emb_dir}")
        train_family_gate_from_dataset(
            dataset_dir=Path(from_emb_dir),
            target_family=train_cfg.get("target_family", "gemini"),
            output_dir=out_dir,
            head_type=train_cfg.get("head_type", "mlp"),
            subspace_dim=train_cfg.get("subspace_dim", train_cfg.get("pca_dim", None)),
            hidden_dim=int(train_cfg.get("hidden_dim", 64)),
            mlp_epochs=int(train_cfg.get("mlp_epochs", 12)),
            mlp_lr=float(train_cfg.get("mlp_lr", 2e-4)),
            mlp_dropout=float(train_cfg.get("mlp_dropout", 0.35)),
            device=torch.device("cuda:0" if torch.cuda.is_available() else "cpu"),
            checkpoints_volume=checkpoints_volume,
        )
        return out_dir

    is_specialist = bool(target_family and target_family in FAMILIES and not is_gate_or_export)
    if is_specialist:
        fam_cols = FAMILIES[target_family]
        active_labels = [LABELS[i] for i in fam_cols]
        sub_lab2i = {m: i for i, m in enumerate(active_labels)}
        num_classes = len(active_labels)
        print(f"[Trainer] Training {target_family} family specialist with {num_classes} classes: {active_labels}")
    else:
        active_labels = list(LABELS)
        sub_lab2i = dict(LAB2I)
        num_classes = K

    model_source = adapter.resolve_model_source(train_cfg, input_dir)
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    precision = adapter.resolve_precision(train_cfg, device)
    train_path, _, pool_file = resolve_dataset_paths(input_dir if input_dir.exists() else Path("."))
    all_pools = resolve_all_pool_files(input_dir if input_dir.exists() else Path("."))

    tok, tok_spec = adapter.load_tokenizer(model_source, train_cfg, is_smoke=is_smoke)
    base_model, hidden_size, keep_layers, layers = adapter.load_backbone(model_source, train_cfg, precision, device)
    print(
        f"[Trainer] Family: {adapter.family_name}, Device: {device}, GPUs: {torch.cuda.device_count()} "
        f"(VRAM: {precision.total_vram_gb:.1f} GiB), Dtype: {precision.compute_dtype}, Source: {model_source}, Keep: {keep_layers}"
    )

    base_model = adapter.prepare_backbone(base_model, precision, train_cfg, is_eval_only=is_gate_or_export)
    base_model = adapter.apply_lora(base_model, train_cfg, is_eval_only=is_gate_or_export)

    model = MultiLayerLoRAClassifier(
        base_model,
        hidden_size=hidden_size,
        layers=layers,
        num_classes=num_classes,
        backbone_family=adapter.family_name,
        tok_spec=tok_spec,
    )
    if is_gate_or_export:
        model.eval()
    if hasattr(base_model, "hf_device_map") and len(getattr(base_model, "hf_device_map", {})) > 1:
        model.plain_head = model.plain_head.to(device)
        model.centered_head = model.centered_head.to(device)
        model.dropouts = model.dropouts.to(device)
        for idx in range(len(model.norms)):
            model.norms[idx] = model.norms[idx].to(device)
    else:
        model = model.to(device)

    if train_cfg.get("pipeline_mode") in ("export_embeddings", "extract_features"):
        exp_cfg = train_cfg.get("export_embeddings", {})
        if not isinstance(exp_cfg, dict):
            exp_cfg = {}
        target = exp_cfg.get("target") or train_cfg.get("embeddings_target") or target_family or "gemini"
        save_all = bool(exp_cfg.get("save_all_layers", train_cfg.get("save_all_layers", False)))
        save_mv = bool(exp_cfg.get("save_multiview", train_cfg.get("save_multiview", False)))
        emb_out = exp_cfg.get("output_dir") or train_cfg.get("embeddings_output_dir") or out_dir
        samples_per_cls = int(exp_cfg.get("samples_per_class", train_cfg.get("samples_per_class", 0)))
        neg_ratio = float(exp_cfg.get("negative_ratio", train_cfg.get("negative_ratio", 0.10)))
        ext_bs = int(exp_cfg.get("batch_size", train_cfg.get("batch_size", 16)))
        ext_max_len = int(exp_cfg.get("max_len", train_cfg.get("max_len", tok_spec.max_len)))
        ds_title = exp_cfg.get("dataset_title", train_cfg.get("dataset_title"))
        ds_slug = exp_cfg.get("dataset_slug", train_cfg.get("dataset_slug"))

        df_p, df_tr, eval_texts, eval_labels = _load_pool_and_bench_for_gate(
            input_dir, all_pools, pool_file, train_path, ["text", "model", "prompt_id"]
        )
        _load_champion_adapter_and_heads(
            model, input_dir, backbone_name, int(train_cfg.get("lora_r", 32)), target_family, True, False, adapter, precision, train_cfg
        )
        extract_and_export_embeddings(
            model=model,
            tok=tok,
            df_pool=df_p,
            df_train=df_tr,
            target=target,
            save_all_layers=save_all,
            save_multiview=save_mv,
            output_dir=emb_out,
            samples_per_class=samples_per_cls,
            negative_ratio=neg_ratio,
            eval_texts=eval_texts,
            eval_labels=eval_labels,
            batch_size=ext_bs,
            max_len=ext_max_len,
            end_id=tok_spec.end_id,
            device=device,
            dataset_title=ds_title,
            dataset_slug=ds_slug,
            checkpoints_volume=checkpoints_volume,
        )
        return Path(emb_out)

    if is_gate_or_export:
        print(f"[Trainer] Running Phase 3 Family Gate Pipeline for target family: {target_family} (head_type={train_cfg.get('head_type', 'mlp')})...")
        _load_champion_adapter_and_heads(
            model, input_dir, backbone_name, int(train_cfg.get("lora_r", 32)), target_family, True, not is_smoke, adapter, precision, train_cfg
        )
        df_p, df_tr, eval_texts, eval_labels = _load_pool_and_bench_for_gate(
            input_dir, all_pools, pool_file, train_path, ["text", "model", "prompt_id"]
        )
        exp_cfg = train_cfg.get("export_embeddings", {})
        if isinstance(exp_cfg, bool):
            exp_enabled, exp_save_all, exp_out = exp_cfg, bool(train_cfg.get("save_all_layers", False)), train_cfg.get("embeddings_output_dir")
        elif isinstance(exp_cfg, dict):
            exp_enabled = bool(exp_cfg.get("enabled", True))
            exp_save_all = bool(exp_cfg.get("save_all_layers", train_cfg.get("save_all_layers", False)))
            exp_out = exp_cfg.get("output_dir") or train_cfg.get("embeddings_output_dir")
        else:
            exp_enabled = bool(train_cfg.get("export_embeddings", False))
            exp_save_all = bool(train_cfg.get("save_all_layers", False))
            exp_out = train_cfg.get("embeddings_output_dir")

        gate_res = train_family_gate(
            model=model,
            tok=tok,
            df_pool=df_p,
            df_train=df_tr,
            target_family=target_family or "gemini",
            samples_per_class=int(train_cfg.get("samples_per_class", 3800)),
            negative_ratio=float(train_cfg.get("negative_ratio", 0.10)),
            eval_texts=eval_texts,
            eval_labels=eval_labels,
            batch_size=int(train_cfg.get("batch_size", 32)),
            max_len=int(train_cfg.get("max_len", tok_spec.max_len)),
            end_id=tok_spec.end_id,
            device=device,
            output_dir=out_dir,
            checkpoints_volume=checkpoints_volume,
            head_type=train_cfg.get("head_type", "mlp"),
            use_supervised_subspace=bool(train_cfg.get("use_supervised_subspace", True)),
            subspace_dim=train_cfg.get("subspace_dim", train_cfg.get("pca_dim", None)),
            hidden_dim=int(train_cfg.get("hidden_dim", 64)),
            mlp_epochs=int(train_cfg.get("mlp_epochs", 12)),
            mlp_lr=float(train_cfg.get("mlp_lr", 2e-4)),
            mlp_dropout=float(train_cfg.get("mlp_dropout", 0.35)),
            export_embeddings=exp_enabled,
            save_all_layers=exp_save_all,
            embeddings_output_dir=exp_out,
            gate_weight=float(train_cfg.get("gate_weight", 0.40)),
            train_cfg=train_cfg,
        )
        print(f"[Trainer] Phase 3 Family Gate Training Complete: {gate_res}")
        return out_dir

    milestone_history: list[dict[str, Any]] = []
    stage1_cfg = train_cfg.get("stage1", {})
    _, _, bench_eval_bs = adapter.adjust_stage1_batch_config(stage1_cfg, precision)
    bench_sub_size = int(stage1_cfg.get("eval_subset_size", 540))
    bench_bundle = (
        _load_benchmark_holdout(input_dir, subset_size=bench_sub_size, seed=42)
        if not is_specialist and bool(train_cfg.get("eval_benchmark", True))
        else None
    )

    resume_ckpt = train_cfg.get("resume_from_checkpoint")
    if resume_ckpt:
        _resume_checkpoint(model, resume_ckpt, target_family, is_specialist)
    else:
        print(f"[Trainer] Starting fresh training initialization (fresh_init={bool(train_cfg.get('fresh_init', True))}).")

    if stage1_cfg.get("enabled", True) and (all_pools or pool_file):
        _run_stage1(
            model=model,
            tok=tok,
            tok_spec=tok_spec,
            adapter=adapter,
            precision=precision,
            device=device,
            train_cfg=train_cfg,
            stage1_cfg=stage1_cfg,
            input_dir=input_dir,
            out_dir=out_dir,
            train_path=train_path,
            pool_file=pool_file,
            all_pools=all_pools,
            bench_bundle=bench_bundle,
            active_labels=active_labels,
            sub_lab2i=sub_lab2i,
            layers=layers,
            keep_layers=keep_layers,
            backbone_name=backbone_name,
            target_family=target_family,
            is_specialist=is_specialist,
            family_margin=family_margin,
            milestone_history=milestone_history,
            step_callback=step_callback,
        )

    stage2_cfg = train_cfg.get("stage2", {})
    Z_train, oof_report = _run_stage2(
        model=model,
        tok=tok,
        tok_spec=tok_spec,
        adapter=adapter,
        precision=precision,
        device=device,
        train_cfg=train_cfg,
        stage2_cfg=stage2_cfg,
        train_path=train_path,
        out_dir=out_dir,
        bench_bundle=bench_bundle,
        active_labels=active_labels,
        sub_lab2i=sub_lab2i,
        layers=layers,
        keep_layers=keep_layers,
        backbone_name=backbone_name,
        target_family=target_family,
        is_specialist=is_specialist,
        family_margin=family_margin,
        supcon_weight=supcon_weight,
        step_callback=step_callback,
    )

    if is_specialist:
        try:
            bench_paths = list(input_dir.rglob("clean_benchmark_1800_rows.parquet"))
            if not bench_paths and Path("clean_benchmark_1800_rows.parquet").exists():
                bench_paths = [Path("clean_benchmark_1800_rows.parquet")]
            if bench_paths:
                df_bench = pd.read_parquet(bench_paths[0])
                bench_texts = df_bench["text"].tolist()
                bench_labels = [LAB2I[m] for m in df_bench["model"].tolist()]
                bench_ds = TextSequenceDataset(bench_texts, bench_labels, None, tok, tok_spec=tok_spec)
                collate_fn = functools.partial(collate_text_batch, pad_token_id=tok_spec.pad_id)
                bench_loader = DataLoader(bench_ds, batch_size=4, shuffle=False, collate_fn=collate_fn)
                model.eval()
                Z_b_list = []
                with torch.no_grad():
                    with torch.amp.autocast("cuda", enabled=torch.cuda.is_available(), dtype=precision.compute_dtype):
                        for batch in bench_loader:
                            b_ids = batch["input_ids"].to(device)
                            b_mask = batch["attention_mask"].to(device)
                            max_l = max(int(b_mask.sum(dim=1).max().item()), 2)
                            z_b = model.forward_features(b_ids[:, :max_l], b_mask[:, :max_l], drop_first_token=tok_spec.drop_first_token)
                            Z_b_list.append(z_b.float().cpu().numpy())
                Z_bench = np.vstack(Z_b_list)
                W_p_curr = model.plain_head.weight.detach().float().cpu().numpy()
                b_p_curr = model.plain_head.bias.detach().float().cpu().numpy()
                logits_spec = Z_bench @ W_p_curr.T + b_p_curr
                W_c_curr = model.centered_head.weight.detach().float().cpu().numpy()
                b_c_curr = model.centered_head.bias.detach().float().cpu().numpy()
                logits_blend = compute_grouped_inference_logits(
                    Z_bench, W_p_curr, b_p_curr, W_c_curr, b_c_curr, groups=None, blend_centered=0.20
                )["logits_blend"]

                fam_cols = FAMILIES[target_family]
                fam_bench_mask = np.isin(bench_labels, fam_cols)
                holdout_fam_bacc_plain = 0.0
                hard_sister_acc_plain = 0.0
                holdout_fam_bacc_blend = 0.0
                hard_sister_acc_blend = 0.0

                if fam_bench_mask.sum() > 0:
                    y_fam_true = np.array([fam_cols.index(l) for l in np.array(bench_labels)[fam_bench_mask]])
                    holdout_fam_bacc_plain = float(balanced_accuracy_score(y_fam_true, logits_spec[fam_bench_mask, : len(fam_cols)].argmax(axis=1)))
                    holdout_fam_bacc_blend = float(balanced_accuracy_score(y_fam_true, logits_blend[fam_bench_mask, : len(fam_cols)].argmax(axis=1)))
                    eff_pair = (2, 3) if (target_family == "gemini" and len(fam_cols) == 4) else (0, 1)
                    s_a, s_b = eff_pair
                    pair_mask = (y_fam_true == s_a) | (y_fam_true == s_b)
                    if pair_mask.sum() > 0 and len(np.unique(y_fam_true[pair_mask])) == 2:
                        p_preds_plain = np.where(logits_spec[fam_bench_mask][pair_mask, s_a] >= logits_spec[fam_bench_mask][pair_mask, s_b], s_a, s_b)
                        hard_sister_acc_plain = float(balanced_accuracy_score(y_fam_true[pair_mask], p_preds_plain))
                        p_preds_blend = np.where(logits_blend[fam_bench_mask][pair_mask, s_a] >= logits_blend[fam_bench_mask][pair_mask, s_b], s_a, s_b)
                        hard_sister_acc_blend = float(balanced_accuracy_score(y_fam_true[pair_mask], p_preds_blend))

                champ_paths = (
                    list(input_dir.rglob("benchmark_champion_logits.safetensors"))
                    + list(input_dir.rglob("champion_logits.safetensors"))
                    + list(Path(".").rglob("*champion_logits.safetensors"))
                )

                def _eval_gate_fused(spec_lg: np.ndarray) -> tuple[float, float, float, float]:
                    b_gw, b_raw, b_sk, base_sk = 0.35, 0.0, 0.0, 0.5294
                    if champ_paths:
                        try:
                            champ_sd = load_file(str(champ_paths[0]))
                            c_key = "logits_plain" if "logits_plain" in champ_sd else list(champ_sd.keys())[0]
                            logits_champ = champ_sd[c_key].cpu().float().numpy()
                            if len(logits_champ) == len(bench_labels):
                                P_main = softmax(logits_champ, axis=1)
                                P_main_sk = sinkhorn_balancing(P_main, iters=50, sinkhorn_strength=0.65, target_mix=0.35)
                                base_sk = float(balanced_accuracy_score(bench_labels, P_main_sk.argmax(axis=1)))
                                cand_mask = np.isin(P_main.argmax(axis=1), fam_cols) | (P_main[:, fam_cols].sum(axis=1) >= 0.10)
                                s_slice = spec_lg[:, : len(fam_cols)]
                                s_std = float(np.std(s_slice, axis=1).mean()) if len(s_slice) > 0 else 1.0
                                spec_probs = softmax(s_slice * ((3.2 / s_std) if s_std > 1e-4 else 1.0), axis=1)
                                for gw in [0.10, 0.20, 0.30, 0.35, 0.40, 0.45, 0.50, 0.60]:
                                    P_fused = P_main.copy()
                                    for idx in np.flatnonzero(cand_mask):
                                        orig_mass = float(P_main[idx, fam_cols].sum())
                                        if orig_mass > 1e-6:
                                            gw_eff = gw * min(orig_mass / 0.50, 1.0)
                                            main_norm = P_main[idx, fam_cols] / orig_mass
                                            P_fused[idx, fam_cols] = ((1.0 - gw_eff) * main_norm + gw_eff * spec_probs[idx]) * orig_mass
                                    r_bacc = float(balanced_accuracy_score(bench_labels, P_fused.argmax(axis=1)))
                                    p_sk = sinkhorn_balancing(P_fused, iters=50, sinkhorn_strength=0.65, target_mix=0.35)
                                    s_bacc = float(balanced_accuracy_score(bench_labels, p_sk.argmax(axis=1)))
                                    if s_bacc > b_sk:
                                        b_sk, b_raw, b_gw = s_bacc, r_bacc, gw
                        except Exception as e:
                            print(f"[Trainer] Gate fusion eval note: {e}")
                    return b_gw, b_raw, b_sk, base_sk

                best_gw_p, best_raw_p, best_sink_p, base_sk_p = _eval_gate_fused(logits_spec)
                best_gw_b, best_raw_b, best_sink_b, base_sk_b = _eval_gate_fused(logits_blend)
                oof_plain = float(oof_report.get("oof_plain_bacc", oof_report.get("oof_blend_bacc", 0.0)))
                oof_blend = float(oof_report.get("oof_blend_bacc", oof_report.get("oof_sink_bacc", 0.0)))
                ledger = ExperimentLedger(ledger_dir=train_cfg.get("ledger_dir", "experiments"))
                exp_pfx = str(train_cfg.get("exp_prefix") or "EXP-06")
                phase_tag = adapter.specialist_phase_tag(target_family)
                feat_tag = adapter.specialist_feature_set_tag(target_family)
                notes_lbl = adapter.specialist_notes_label()

                ledger.add_entry({
                    "exp_id": f"{exp_pfx}_plain",
                    "phase": phase_tag,
                    "target_family": target_family,
                    "feature_set": feat_tag,
                    "subspace_dim": int(Z_train.shape[1]),
                    "head_type": "plain",
                    "gate_weight": best_gw_p,
                    "oof_family_bacc": oof_plain,
                    "holdout_family_bacc": holdout_fam_bacc_plain,
                    "hard_sister_acc": hard_sister_acc_plain,
                    "holdout_18cls_raw_bacc": best_raw_p,
                    "holdout_18cls_sinkhorn_bacc": best_sink_p,
                    "delta_vs_baseline_bacc": (best_sink_p - base_sk_p) if best_sink_p > 0 else 0.0,
                    "notes": f"{notes_lbl} {target_family} specialist direct plain head",
                })
                ledger.add_entry({
                    "exp_id": f"{exp_pfx}_blend",
                    "phase": phase_tag,
                    "target_family": target_family,
                    "feature_set": feat_tag,
                    "subspace_dim": int(Z_train.shape[1]),
                    "head_type": "blend",
                    "gate_weight": best_gw_b,
                    "oof_family_bacc": oof_blend,
                    "holdout_family_bacc": holdout_fam_bacc_blend,
                    "hard_sister_acc": hard_sister_acc_blend,
                    "holdout_18cls_raw_bacc": best_raw_b,
                    "holdout_18cls_sinkhorn_bacc": best_sink_b,
                    "delta_vs_baseline_bacc": (best_sink_b - base_sk_b) if best_sink_b > 0 else 0.0,
                    "notes": f"{notes_lbl} {target_family} specialist grouped inference blend head",
                })
                if ledger.md_path.exists():
                    shutil.copy(str(ledger.md_path), str(out_dir / "experiment_ledger.md"))

                meta_p = out_dir / "meta.json"
                if meta_p.exists():
                    try:
                        m_d = json.loads(meta_p.read_text(encoding="utf-8"))
                        m_d.update({
                            "oof_plain_bacc": oof_plain,
                            "oof_sink_bacc": oof_blend,
                            "within_family_acc": max(holdout_fam_bacc_plain, holdout_fam_bacc_blend),
                            "hard_pair_eval_acc": max(hard_sister_acc_plain, hard_sister_acc_blend),
                            "holdout_family_bacc": max(holdout_fam_bacc_plain, holdout_fam_bacc_blend),
                            "holdout_18cls_sinkhorn_bacc": max(best_sink_p, best_sink_b),
                            "default_gate_weight": best_gw_b if best_sink_b >= best_sink_p else best_gw_p,
                            "best_gate_weight": best_gw_b if best_sink_b >= best_sink_p else best_gw_p,
                        })
                        meta_p.write_text(json.dumps(m_d, indent=2), encoding="utf-8")
                    except Exception as e:
                        print(f"[Trainer] Note updating meta.json: {e}")
        except Exception as e:
            print(f"[Trainer] Benchmark holdout evaluation note: {e}")
    elif not is_specialist and bench_bundle is not None:
        try:
            full_texts, full_y, _, _, _ = bench_bundle
            bench_extras: dict[str, Any] = {}
            m_plain, _ = _evaluate_bench_checkpoint(
                model=model,
                tok=tok,
                texts=full_texts,
                y_true=full_y,
                max_len=tok_spec.max_len,
                device=device,
                compute_dtype=precision.compute_dtype,
                drop_first_token=tok_spec.drop_first_token,
                tag="S2_Final_Plain",
                end_id=tok_spec.end_id,
                cls_id=tok_spec.cls_id,
                pad_id=tok_spec.pad_id,
                batch_size=bench_eval_bs,
                blend_centered=0.0,
                out_dir=out_dir,
                history=milestone_history,
                eval_extras_out=bench_extras,
            )
            print_prediction_table(m_plain, header="FINAL STAGE 2 BENCH-1800 (PLAIN HEAD)")
            m_sk = bench_extras.get("plain_sinkhorn", {})
            m_bl = bench_extras.get("blend_global_approx", {})
            m_bsk = bench_extras.get("blend_global_approx_sinkhorn", {})
            if m_sk:
                print(
                    f"[Bench-1800 Calibration] Plain={m_plain['bacc']*100:.2f}% | "
                    f"Plain+Sink={m_sk['bacc']*100:.2f}% (Gem4={m_sk['gem_within_bacc']*100:.2f}%) | "
                    f"Blend(GlobalApprox)={m_bl['bacc']*100:.2f}% | Blend(GlobalApprox)+Sink={m_bsk['bacc']*100:.2f}% (3.7v3.8={m_bsk['sister_37_38_acc']*100:.2f}%)"
                )
            cm_arr = np.asarray(m_plain["confusion_matrix"], dtype=np.int64)
            print("\n=== PROBLEM-CLASS TOP CONFUSION SINKS (BENCH-1800) ===")
            for cname in PROBLEM_WATCH_CLASSES:
                c_i = LAB2I.get(cname)
                if c_i is None:
                    continue
                row = cm_arr[c_i]
                top_sinks = sorted(
                    ((LABELS[j], int(row[j])) for j in range(K) if j != c_i and row[j] > 0),
                    key=lambda x: -x[1],
                )[:3]
                print(f"  {cname:<18} | true={int(row.sum()):3d} pred={int(cm_arr[:, c_i].sum()):3d} correct={int(row[c_i]):3d} | top sinks: {top_sinks}")

            eval_out = {"plain": m_plain, **bench_extras, "milestone_history": milestone_history, "oof_report": oof_report}
            (out_dir / "benchmark_1800_eval.json").write_text(json.dumps(eval_out, indent=2), encoding="utf-8")
            meta_p = out_dir / "meta.json"
            if meta_p.exists():
                m_d = json.loads(meta_p.read_text(encoding="utf-8"))
                m_d["bench_1800_plain_bacc"] = float(m_plain["bacc"])
                m_d["bench_1800_gem4_bacc"] = float(m_plain["gem_within_bacc"])
                m_d["bench_1800_37_38_acc"] = float(m_plain["sister_37_38_acc"])
                if m_sk:
                    m_d["bench_1800_plain_sink_bacc"] = float(m_sk["bacc"])
                if m_bsk:
                    m_d["bench_1800_blend_sink_bacc"] = float(m_bsk["bacc"])
                meta_p.write_text(json.dumps(m_d, indent=2), encoding="utf-8")
        except Exception as e:
            print(f"[Trainer] 18-class benchmark evaluation note: {e}")

    if step_callback is not None:
        try:
            step_callback("checkpoint_saved", "training_complete")
        except Exception as e:
            print(f"[Trainer Callback Warning] {e}")
    print(f"[Trainer] Checkpoint exported to {out_dir}")
    return out_dir


def run_encoder_stage1(
    config: dict[str, Any],
    datasets_dir: Path,
    checkpoint_base: Path | None = None,
    on_checkpoint_saved: Any = None,
) -> dict[str, Any]:
    """Unified Stage-1 Multi-Layer LoRA trainer for bidirectional encoder backbones."""
    t0 = time.time()
    s1_cfg = config["stage1"]
    seed = int(s1_cfg["seed"])
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True

    adapter = resolve_backbone_adapter(config)
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    precision = adapter.resolve_precision(config, device)
    backbone_name = str(config["backbone"])
    loss_cfg = config["loss"]
    out_cfg = config["output"]

    save_root = Path(checkpoint_base) if checkpoint_base is not None else Path(out_cfg["checkpoint_dir"])
    save_root.mkdir(parents=True, exist_ok=True)
    df_pool, df_train, df_bench = load_zero_leak_stage1_pool(
        datasets_dir=datasets_dir,
        weight_clip=tuple(loss_cfg["match_weight_clip"]),
    )
    y_bench = np.array([LAB2I[m] for m in df_bench["model"]], dtype=np.int64)
    y_train_gold = np.array([LAB2I[m] for m in df_train["model"]], dtype=np.int64)
    prompts_gold = df_train["prompt_id"].values
    pool_labels = df_pool["label_idx"].to_numpy()
    pool_weights = df_pool["sample_weight"].to_numpy()
    pool_texts = df_pool["text"].astype(str).tolist()

    model_source = adapter.resolve_model_source(config, datasets_dir)
    tok, tok_spec = adapter.load_tokenizer(model_source, config, is_smoke=bool(config.get("smoke", False)))
    pool_tok_lens: list[int] = []
    for i in range(0, len(pool_texts), 4096):
        pool_tok_lens.extend(tok(pool_texts[i : i + 4096], truncation=True, max_length=tok_spec.max_len, return_length=True)["length"])

    base_model, hidden_size, _, tap_layers = adapter.load_backbone(model_source, config, precision, device)
    base_model = adapter.prepare_backbone(base_model, precision, config, is_eval_only=False)
    peft_backbone = adapter.apply_lora(base_model, config, is_eval_only=False)

    model = MultiLayerLoRAClassifier(
        backbone=peft_backbone,
        hidden_size=hidden_size,
        layers=tap_layers,
        num_classes=K,
        backbone_family=adapter.family_name,
        tok_spec=tok_spec,
    ).to(device)
    model.norms.float()
    model.plain_head.float()
    model.centered_head.float()
    for p in model.centered_head.parameters():
        p.requires_grad = False

    def _safe_export(out_p: Path, Z_tr: np.ndarray | None = None, y_tr: np.ndarray | None = None, p_tr: np.ndarray | None = None) -> None:
        with adapter.export_context(model.backbone):
            export_compatible_checkpoint(
                model=model,
                output_dir=out_p,
                layers=tap_layers,
                keep=len(tap_layers),
                max_len=tok_spec.max_len,
                end_id=tok_spec.end_id if tok_spec.end_id is not None else 2,
                backbone_name=backbone_name,
                Z_train=Z_tr,
                y_train=y_tr,
                prompts_train=p_tr,
            )

    lora_cfg = config["lora"]
    epochs = int(s1_cfg["epochs"])
    batch_size = int(s1_cfg["batch_size"])
    max_toks_fwd = int(s1_cfg.get("max_tokens_per_forward", 4096))
    class_quotas = dict(s1_cfg["class_quotas"])
    epoch_plans = [build_targeted_epoch_indices(pool_labels, class_quotas, epoch=e, seed=seed) for e in range(epochs)]
    total_steps = sum(math.ceil(len(ep_idx) / batch_size) for ep_idx in epoch_plans)
    warmup_steps = max(1, int(round(total_steps * float(s1_cfg["warmup_ratio"]))))

    param_groups = _get_loraplus_param_groups(
        model,
        lr_lora=float(s1_cfg["lr_lora"]),
        lr_head=float(s1_cfg["lr_head"]),
        lora_plus_ratio=float(lora_cfg["lora_plus_ratio"]),
        include_centered_head=False,
    )
    optimizer = torch.optim.AdamW(param_groups, weight_decay=float(loss_cfg["weight_decay"]))
    scheduler = get_cosine_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps)
    scaler_enc, manual_scaler_enc = adapter.build_scalers(precision, device)
    max_grad_norm = float(loss_cfg["max_grad_norm"])

    eval_every_samples = int(s1_cfg.get("eval_every_samples", 0))
    bench_texts_list = df_bench["text"].astype(str).tolist()
    watch_models = tuple(PROBLEM_WATCH_CLASSES[:8])

    epoch_logs, scaling_curve = [], []
    best_bacc, best_epoch, best_samples_seen = -1.0, -1, -1
    total_samples_seen, global_step = 0, 0
    next_eval_target = eval_every_samples if eval_every_samples > 0 else 10**12
    best_dir, last_dir = save_root / "stage1_best", save_root / "stage1_last"

    def _record_scaling_point(tag: str, win_loss: float, ep_num: int) -> dict[str, Any]:
        nonlocal best_bacc, best_epoch, best_samples_seen
        _, b_logits = extract_features_and_logits(
            model,
            bench_texts_list,
            tok,
            max_len=tok_spec.max_len,
            device=device,
            batch_size=64,
            drop_first_token=tok_spec.drop_first_token,
            compute_dtype=precision.compute_dtype,
            end_id=tok_spec.end_id,
            cls_id=tok_spec.cls_id,
        )
        m = compute_detailed_metrics(b_logits, y_bench)
        pc_map = {r["model"]: r for r in m["per_class"]}
        pt = {
            "tag": tag,
            "epoch": ep_num,
            "step": global_step,
            "samples_seen": total_samples_seen,
            "elapsed_sec": float(time.time() - t0),
            "train_loss": float(win_loss),
            "bacc": m["bacc"],
            "nll": m["nll"],
            "gem_within_bacc": m["gem_within_bacc"],
            "gem_retention": m["gem_retention"],
            "sister_37_38_acc": m["sister_37_38_acc"],
            "zero_recall_classes": m["zero_recall_classes"],
            "watch_recall": {k: pc_map[k]["recall"] for k in watch_models if k in pc_map},
            "watch_bias": {k: pc_map[k]["bias"] for k in watch_models if k in pc_map},
        }
        scaling_curve.append(pt)
        if m["bacc"] > best_bacc:
            best_bacc, best_epoch, best_samples_seen = m["bacc"], ep_num, total_samples_seen
            _safe_export(best_dir)
            print(f"  -> [New Best Saved] {tag} ({total_samples_seen:,} samples): BAcc={best_bacc*100:.2f}% -> {best_dir}", flush=True)
        model.train()
        return m

    for ep in range(epochs):
        ep_t0 = time.time()
        ep_indices = epoch_plans[ep]
        sampler = BucketedLengthBatchSampler(
            lengths=[pool_tok_lens[int(i)] for i in ep_indices],
            batch_size=batch_size,
            bucket_size_multiplier=60,
            shuffle=True,
            seed=seed + ep * 31,
        )
        model.train()
        running_loss = 0.0
        n_batches = len(sampler)
        for step_i, rel_batch in enumerate(sampler):
            abs_batch = ep_indices[np.asarray(rel_batch, dtype=np.int64)]
            b_texts = [pool_texts[int(i)] for i in abs_batch]
            b_labels = torch.from_numpy(pool_labels[abs_batch]).to(device, dtype=torch.long)
            b_weights = torch.from_numpy(pool_weights[abs_batch]).to(device, dtype=torch.float32)
            ds_batch = TextSequenceDataset(b_texts, None, None, tok, tok_spec=tok_spec)
            enc = collate_text_batch([ds_batch[k] for k in range(len(b_texts))], pad_token_id=tok_spec.pad_id)
            seq_len, bsz = int(enc["input_ids"].shape[1]), len(abs_batch)
            optimizer.zero_grad(set_to_none=True)

            chunk_sz = max(2, min(bsz, max_toks_fwd // max(seq_len, 1)))
            total_w = b_weights.sum().clamp(min=1e-6)
            step_loss_val = 0.0
            for start in range(0, bsz, chunk_sz):
                end = min(start + chunk_sz, bsz)
                sub_ids = enc["input_ids"][start:end].to(device)
                sub_mask = enc["attention_mask"][start:end].to(device)
                sub_y, sub_w = b_labels[start:end], b_weights[start:end]
                w_frac = sub_w.sum() / total_w
                with torch.amp.autocast("cuda", enabled=torch.cuda.is_available(), dtype=precision.compute_dtype):
                    feats = model.forward_features(sub_ids, sub_mask, drop_first_token=tok_spec.drop_first_token)
                    loss = family_margin_cross_entropy(
                        model.predict_plain(feats),
                        sub_y,
                        family_margin=float(loss_cfg["family_margin"]),
                        label_smoothing=float(loss_cfg["label_smoothing"]),
                        boost_hard_sisters=bool(loss_cfg["boost_hard_sisters"]),
                        sample_weight=sub_w,
                    )
                if scaler_enc is not None and getattr(scaler_enc, "is_enabled", lambda: False)():
                    scaler_enc.scale(loss * w_frac).backward()
                elif manual_scaler_enc is not None and manual_scaler_enc.is_enabled():
                    manual_scaler_enc.scale_loss(loss * w_frac).backward()
                else:
                    (loss * w_frac).backward()
                step_loss_val += float(loss.item()) * float(w_frac.item())

            _step_with_grad_guard(
                model,
                optimizer,
                scheduler,
                scaler=scaler_enc,
                manual_scaler=manual_scaler_enc,
                max_grad_norm=max_grad_norm,
            )
            running_loss += step_loss_val
            global_step += 1
            total_samples_seen += bsz

            if total_samples_seen >= next_eval_target and (step_i + 1) < (n_batches - 15):
                _record_scaling_point(f"{total_samples_seen//1000}k", running_loss / float(step_i + 1), ep + 1)
                next_eval_target += eval_every_samples

        ep_metrics = _record_scaling_point(f"Ep{ep+1}_End", running_loss / max(n_batches, 1), ep + 1)
        ep_metrics["epoch"] = ep + 1
        ep_metrics["train_avg_loss"] = float(running_loss / max(n_batches, 1))
        ep_metrics["epoch_sec"] = float(time.time() - ep_t0)
        print_prediction_table(ep_metrics, header=f"Epoch {ep+1} Bench-1800")
        epoch_logs.append(ep_metrics)
        _safe_export(last_dir)
        if on_checkpoint_saved is not None:
            on_checkpoint_saved()

    if best_samples_seen != total_samples_seen and (best_dir / "heads.safetensors").exists():
        del model, peft_backbone, base_model, optimizer, scheduler
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        dtype_re = torch.bfloat16 if precision.use_bf16 else torch.float32
        base_re = AutoModel.from_pretrained(model_source, torch_dtype=dtype_re)
        peft_re = PeftModel.from_pretrained(base_re, str(best_dir / "adapter"))
        adapter.prepare_backbone(base_re, precision, config, is_eval_only=True)
        model = MultiLayerLoRAClassifier(
            backbone=peft_re,
            hidden_size=hidden_size,
            layers=tap_layers,
            num_classes=K,
            backbone_family=adapter.family_name,
            tok_spec=tok_spec,
        ).to(device)
        model.norms.float()
        model.plain_head.float()
        model.centered_head.float()
        hd = load_file(str(best_dir / "heads.safetensors"), device=str(device))
        for i in range(len(tap_layers)):
            model.norms[i].weight.data.copy_(hd[f"norms.{i}.weight"].float())
            if f"norms.{i}.bias" in hd:
                model.norms[i].bias.data.copy_(hd[f"norms.{i}.bias"].float())
        model.plain_head.weight.data.copy_(hd["plain.weight"].float())
        model.plain_head.bias.data.copy_(hd["plain.bias"].float())

    n_probe_per_cls = max(1, int(out_cfg.get("ridge_probe_samples", 36000)) // K)
    rng_pr = np.random.RandomState(42)
    probe_idxs = np.concatenate([
        rng_pr.choice(np.where(pool_labels == c_i)[0], size=min(n_probe_per_cls, int((pool_labels == c_i).sum())), replace=False)
        for c_i in range(K)
    ])
    Z_bench, lg_bench_neural = extract_features_and_logits(
        model, bench_texts_list, tok, max_len=tok_spec.max_len, device=device, batch_size=64, drop_first_token=tok_spec.drop_first_token, compute_dtype=precision.compute_dtype, end_id=tok_spec.end_id, cls_id=tok_spec.cls_id
    )
    Z_gold, lg_gold_neural = extract_features_and_logits(
        model, df_train["text"].astype(str).tolist(), tok, max_len=tok_spec.max_len, device=device, batch_size=64, drop_first_token=tok_spec.drop_first_token, compute_dtype=precision.compute_dtype, end_id=tok_spec.end_id, cls_id=tok_spec.cls_id
    )
    Z_probe, _ = extract_features_and_logits(
        model, [pool_texts[int(i)] for i in probe_idxs], tok, max_len=tok_spec.max_len, device=device, batch_size=64, drop_first_token=tok_spec.drop_first_token, compute_dtype=precision.compute_dtype, end_id=tok_spec.end_id, cls_id=tok_spec.cls_id
    )

    ridge_map = gpu_trace_ridge_eval(Z_probe, pool_labels[probe_idxs], Z_bench, alphas=[0.05, 0.25, 1.0, 5.0], device=device)
    best_r_alpha, best_r_logits = max(ridge_map.items(), key=lambda kv: balanced_accuracy_score(y_bench, kv[1].argmax(axis=1)))

    def _row_z(x: np.ndarray) -> np.ndarray:
        return (x - x.mean(axis=1, keepdims=True)) / np.clip(x.std(axis=1, keepdims=True), 1e-5, None)

    best_w_neural, best_blend_logits = max(
        ((w_n, w_n * _row_z(lg_bench_neural) + (1.0 - w_n) * _row_z(best_r_logits)) for w_n in [0.0, 0.25, 0.5, 0.7, 0.85, 1.0]),
        key=lambda item: balanced_accuracy_score(y_bench, item[1].argmax(axis=1)),
    )

    Z_gold_cent = Z_gold.copy()
    for pid in np.unique(prompts_gold):
        m = prompts_gold == pid
        if m.sum() > 1:
            Z_gold_cent[m] -= Z_gold_cent[m].mean(axis=0, keepdims=True)

    oof_alphas = [0.05, 0.25, 1.0, 5.0]
    oof_p_map = {a: np.zeros((len(y_train_gold), K), dtype=np.float32) for a in oof_alphas}
    oof_c_map = {a: np.zeros((len(y_train_gold), K), dtype=np.float32) for a in oof_alphas}
    for tr_i, val_i in GroupKFold(n_splits=5).split(Z_gold, y_train_gold, groups=prompts_gold):
        rp = gpu_trace_ridge_eval(Z_gold[tr_i], y_train_gold[tr_i], Z_gold[val_i], oof_alphas, device=device)
        rc = gpu_trace_ridge_eval(Z_gold_cent[tr_i], y_train_gold[tr_i], Z_gold_cent[val_i], oof_alphas, device=device)
        for a in oof_alphas:
            oof_p_map[a][val_i] = rp[a]
            oof_c_map[a][val_i] = rc[a]

    _safe_export(best_dir, Z_tr=Z_gold, y_tr=y_train_gold, p_tr=prompts_gold)
    best_oof_c = max(oof_c_map.values(), key=lambda lg: balanced_accuracy_score(y_train_gold, lg.argmax(axis=1)))
    return {
        "checkpoint_dir": str(best_dir),
        "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
        "elapsed_sec": float(time.time() - t0),
        "config": config,
        "best_epoch": best_epoch,
        "best_samples_seen": best_samples_seen,
        "best_neural_bench_bacc": float(best_bacc),
        "scaling_curve": scaling_curve,
        "epoch_logs": epoch_logs,
        "bench_ridge": {"best_alpha": float(best_r_alpha), **compute_detailed_metrics(best_r_logits, y_bench)},
        "bench_neural_ridge_blend": {"w_neural": float(best_w_neural), **compute_detailed_metrics(best_blend_logits, y_bench)},
        "gold_train_eval": {
            "neural_plain_bacc": float(balanced_accuracy_score(y_train_gold, lg_gold_neural.argmax(axis=1))),
            "oof5_ridge_plain_bacc": max(float(balanced_accuracy_score(y_train_gold, oof_p_map[a].argmax(axis=1))) for a in oof_alphas),
            "oof5_ridge_centered_bacc": max(float(balanced_accuracy_score(y_train_gold, oof_c_map[a].argmax(axis=1))) for a in oof_alphas),
        },
        "arrays": {
            "y_bench": y_bench.tolist(),
            "bench_neural_logits": lg_bench_neural.astype(np.float16).tolist(),
            "bench_ridge_logits": best_r_logits.astype(np.float16).tolist(),
            "y_gold": y_train_gold.tolist(),
            "gold_oof_centered_logits": best_oof_c.astype(np.float16).tolist(),
        },
    }


In [ ]:
# === MODULE: src/namethatllm/trainer.py ===
"""
Name That LLM - Thin Backward-Compatible Training Facade & CLI Entrypoint.
Re-exports all modular training primitives from `namethatllm.training.*`.
"""

from __future__ import annotations

import argparse
import json
from pathlib import Path


run_bge_v2_stage1 = run_encoder_stage1


def main() -> None:
    parser = argparse.ArgumentParser(description="NameThatLLM Training & Gate Pipeline")
    parser.add_argument("--config", type=str, default=None, help="Path to config JSON")
    parser.add_argument("--from-embeddings-dir", type=str, default=None, help="Offline training from exported embeddings")
    parser.add_argument("--target-family", type=str, default="gemini", help="Target family for gate training")
    parser.add_argument("--output-dir", type=str, default=None, help="Output directory")
    parser.add_argument("--head-type", type=str, default="mlp", choices=["mlp", "ridge", "hybrid"])
    parser.add_argument("--select-layers", type=str, default=None, help="'auto_top8' or comma-separated layer indices")
    parser.add_argument("--use-multiview", action="store_true", help="Add engineered multi-view features")
    parser.add_argument("--no-velocity", action="store_true", help="Disable inter-layer velocity directions")
    parser.add_argument("--no-layer-weighting", action="store_true", help="Disable layer-probing importance weighting")
    args, _ = parser.parse_known_args()
    if args.from_embeddings_dir:
        out_p = Path(args.output_dir) if args.output_dir else Path(f"export_{args.target_family}_gate_{args.head_type}")
        sel_l = args.select_layers
        if sel_l is not None and sel_l != "auto_top8":
            sel_l = [int(x.strip()) for x in sel_l.split(",") if x.strip()]
        train_family_gate_from_dataset(
            dataset_dir=Path(args.from_embeddings_dir),
            target_family=args.target_family,
            output_dir=out_p,
            head_type=args.head_type,
            select_layers=sel_l,
            use_multiview=args.use_multiview,
            include_velocity=not args.no_velocity,
            layer_weighting=not args.no_layer_weighting,
        )
    elif args.config:
        run_training_pipeline(json.loads(Path(args.config).read_text(encoding="utf-8")))


In [ ]:
if __name__ == '__main__':
    run_training_pipeline(TRAINING_CONFIG)
